In [ ]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()

print("GPU cache cleared")

GPU cache cleared


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from google.colab import drive

# اتصال به گوگل درایو
drive.mount('/content/drive')

print("کتابخانه‌ها بارگذاری و درایو متصل شد.")

ValueError: mount failed

In [ ]:
import pandas as pd
import torch
import os
from collections import Counter

# 1. Load the confirmed Ground Truth file
gt_path = '/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv'
df_gt = pd.read_csv(gt_path)

print(f'--- Final Ground Truth Analysis ---')
print(f'Total records found: {len(df_gt):,}')
print(f'Columns: {df_gt.columns.tolist()}')

# 2. Build Character Vocabulary (itos/stoi)
# Combine all Persian text to find unique characters
all_text = " ".join(df_gt['text'].astype(str).tolist())
char_counts = Counter(all_text)
special_tokens = ['<PAD>', '<BOS>', '<EOS>', '<UNK>']
# Sort unique characters to maintain index consistency
chars = sorted([c for c in char_counts.keys()])
vocab = special_tokens + chars

stoi = {char: i for i, char in enumerate(vocab)}
itos = {i: char for i, char in enumerate(vocab)}

print(f'Vocabulary Size: {len(vocab)} unique characters')
print(f'Sample characters: {vocab[10:35]}')

# 3. Tokenizer Function
def tokenize(text, max_len=128):
    tokens = [stoi['<BOS>']]
    for char in str(text):
        tokens.append(stoi.get(char, stoi['<UNK>']))
    tokens.append(stoi['<EOS>'])

    # Apply Padding
    if len(tokens) < max_len:
        tokens.extend([stoi['<PAD>']] * (max_len - len(tokens)))
    return tokens[:max_len]

# 4. Generate REAL Target IDs for validation
# We take the first 8 rows to match our pipeline batch size
sample_sentences = df_gt['text'].head(8).tolist()
target_ids_batch = torch.tensor([tokenize(s) for s in sample_sentences])

print(f'\nTarget IDs Tensor Shape: {target_ids_batch.shape}')
print(f'First target sequence sample: {target_ids_batch[0][:15]}...')

# Save vocabulary artifacts
torch.save({
    'stoi': stoi,
    'itos': itos,
    'vocab': vocab,
    'vocab_size': len(vocab)
}, '/content/idpl_vocab.pt')

print('\n✓ Ground Truth verified and Vocabulary saved as idpl_vocab.pt')
print('✓ System is now ready for Stage 7: Real OCR Training.')

--- Final Ground Truth Analysis ---
Total records found: 27,120
Columns: ['split', 'hf_index', 'image_path', 'image_name', 'text']
Vocabulary Size: 169 unique characters
Sample characters: ['*', ',', '-', '.', '/', ':', '?', '[', ']', '`', '{', '}', '«', '\xad', '²', '»', 'ï', '،', '؛', '؟', 'آ', 'أ', 'ؤ', 'إ', 'ئ']

Target IDs Tensor Shape: torch.Size([8, 128])
First target sequence sample: tensor([ 1, 90, 35, 36, 43,  5, 63,  5, 58, 35, 45,  5, 40, 60, 62])...

✓ Ground Truth verified and Vocabulary saved as idpl_vocab.pt
✓ System is now ready for Stage 7: Real OCR Training.


In [ ]:
import os
import json

# مسیرهای شناسایی شده
file_list_path = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_ocr_final_backup/dataset_file_list.txt'
json_path = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json'

print('--- بررسی دقیق محتوای فایل‌های کاندید ---')

# ۱. بررسی نمونه محتوای متنی
if os.path.exists(file_list_path):
    print(f'\n[!] نمونه محتوای dataset_file_list.txt:')
    with open(file_list_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = [f.readline().strip() for _ in range(5)]
        for i, line in enumerate(lines):
            print(f'   {i+1}: {line}')

# ۲. بررسی ساختار JSON
if os.path.exists(json_path):
    print(f'\n[!] ساختار ورودی‌های image_paths.json:')
    with open(json_path, 'r') as f:
        data = json.load(f)
        print(f'   نوع داده: {type(data)}')
        print(f'   نمونه مورد اول: {data[0] if len(data)>0 else "خالی"}')

# ۳. لیست نهایی فایل‌های غیر مدل/تصویر (Extra Files)
print('\n--- لیست کامل فایل‌های جانبی (Metadata/Extra) ---')
feature_path = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features'
extra_inventory = []
for root, dirs, files in os.walk(feature_path):
    for f in files:
        if not f.endswith(('.pth', '.pt', '.png', '.tif', '.jpg')):
            full_path = os.path.join(root, f)
            size = os.path.getsize(full_path) / 1024
            extra_inventory.append(f'{f} ({size:.2f} KB) -> {root}')

for item in sorted(extra_inventory):
    print(f'- {item}')

--- بررسی دقیق محتوای فایل‌های کاندید ---

[!] نمونه محتوای dataset_file_list.txt:
   1: /content/drive/MyDrive/colab_ocr_project/test/ا/ا.png
   2: /content/drive/MyDrive/colab_ocr_project/test/ا/ا_9.png
   3: /content/drive/MyDrive/colab_ocr_project/test/آ/آ_4.png
   4: /content/drive/MyDrive/colab_ocr_project/test/ب/ب.png
   5: /content/drive/MyDrive/colab_ocr_project/test/ب/ب_9.png

[!] ساختار ورودی‌های image_paths.json:
   نوع داده: <class 'list'>
   نمونه مورد اول: /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001 (1).tif

--- لیست کامل فایل‌های جانبی (Metadata/Extra) ---
- class_names.pkl (0.18 KB) -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/colab_ocr_project
- dataset_file_list.txt (1895.75 KB) -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_ocr_final_backup
- experiment_config.json (0.30 KB) -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS-Net_Research2026_Results
- image_pa

In [ ]:
import os
import subprocess
import re

print('--- جستجوی نهایی و عمیق برای یافتن فایل Ground Truth ---')

# شناسه‌های نمونه برای تطبیق
sample_ids = ['27868', '27935', '00001', '00002']

try:
    # جستجوی فایل‌های متنی و داده‌ای بالای ۵۰۰ کیلوبایت که تصویر یا مدل نیستند
    cmd = "find /content/drive -type f -not -path '*/.*' -size +500k -not -name '*.pth' -not -name '*.pt' -not -name '*.tif' -not -name '*.jpg' -not -name '*.png' -not -name '*.ipynb'"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    candidates = result.stdout.splitlines()

    print(f'تعداد {len(candidates)} فایل کاندید پیدا شد. در حال بررسی محتوا...')

    found_any = False
    for path in candidates:
        try:
            # بررسی نمونه محتوا برای وجود متون فارسی و شناسه‌های عددی
            with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                sample = f.read(20000)

                # شرط ۱: وجود حروف فارسی
                has_persian = any('\u0600' <= char <= '\u06FF' for char in sample)
                # شرط ۲: وجود حداقل یکی از شناسه‌های نمونه
                has_ids = any(sid in sample for sid in sample_ids)

                if has_persian or has_ids:
                    found_any = True
                    file_size = os.path.getsize(path) / 1024
                    print(f'\n[!!!] مورد مشکوک یافت شد: {path}')
                    print(f'    حجم فایل: {file_size:.2f} KB')
                    print(f'    دارای متن فارسی: {has_persian} | دارای شناسه: {has_ids}')
                    print(f'    پیش‌نمایش: {sample[:200].strip()}...')
        except:
            continue

    if not found_any:
        print('\nمتاسفانه فایلی با مشخصات مورد نظر در این جستجو پیدا نشد.')

except Exception as e:
    print(f'Search error: {e}')

--- جستجوی نهایی و عمیق برای یافتن فایل Ground Truth ---
تعداد 4 فایل کاندید پیدا شد. در حال بررسی محتوا...

[!!!] مورد مشکوک یافت شد: /content/drive/MyDrive/FILE_02a27e28-5854-478a-bd83-2c03ca115b0f.zip
    حجم فایل: 56788.50 KB
    دارای متن فارسی: True | دارای شناسه: False
    پیش‌نمایش: PK  X\        ܑy-   FILE_02a27e28-5854-478a-bd83-2c03ca115b0f.pdfw4\m`5HQD3eF5D0uFFM0z%ABHhI!2D}?<[?~{\>yǱ
IY)3=,LB2B^n,LHyG"oaܜ~H4IAHNHFTUUkg-m|CG)_i놺*m䈳
d\H %/'#...

[!!!] مورد مشکوک یافت شد: /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv
    حجم فایل: 6087.69 KB
    دارای متن فارسی: True | دارای شناسه: False
    پیش‌نمایش: ﻿split,hf_index,image_path,image_name,text
train,0,/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00490.tif,00490.tif,"یابد و كار جمهوری اسلامی را یكسره كند. كافی است به تبلیغات «اكس»گونه یك...

[!!!] مورد مشکوک یافت شد: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_ocr_final_backup/dataset_file_

In [ ]:
import os
import pickle
import torch
import re

feature_path = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features'
image_dir = '/content/drive/Othercomputers/My Laptop/Desktop/idplimgl'

# شناسه‌های نمونه برای جستجو
sample_image_ids = ['27868', '27935', '00001', '00002']

def contains_persian(text):
    return bool(re.search(r'[\u0600-\u06FF]', str(text)))

print('--- بررسی فایل‌های باینری برای یافتن نگاشت متن فارسی ---')

for root, dirs, files in os.walk(feature_path):
    for f in files:
        if f.endswith(('.pkl', '.pt', '.pth')):
            full_path = os.path.join(root, f)
            # نادیده گرفتن فایل‌های بسیار حجیم مدل برای جلوگیری از کرش
            if os.path.getsize(full_path) > 500 * 1024 * 1024:
                continue

            try:
                if f.endswith('.pkl'):
                    with open(full_path, 'rb') as pf:
                        data = pickle.load(pf)
                else:
                    # بررسی فایل‌های Torch (ممکن است دیکشنری حاوی متادیتا باشند)
                    data = torch.load(full_path, map_location='cpu', weights_only=False)

                print(f'\nبررسی فایل: {full_path} | نوع داده: {type(data)}')

                # اگر دیکشنری باشد، کلیدها و مقادیر را برای متن فارسی چک می‌کنیم
                if isinstance(data, dict):
                    has_gt = False
                    for k, v in data.items():
                        if contains_persian(v) or any(sid in str(k) for sid in sample_image_ids):
                            print(f' [!] مورد مشکوک یافت شد -> کلید: {k} | نمونه مقدار: {str(v)[:100]}')
                            has_gt = True
                    if not has_gt:
                        print(f' کلیدهای موجود: {list(data.keys())[:10]}')

                # اگر لیست بزرگی باشد، احتمال دارد لیست جملات باشد
                elif isinstance(data, list) and len(data) > 1000:
                    print(f' [!] لیست با طول {len(data)} یافت شد. نمونه مورد اول: {str(data[0])[:100]}')
                    if contains_persian(data[0]):
                        print('  -> این لیست حاوی متن فارسی است.')

            except Exception as e:
                pass

--- بررسی فایل‌های باینری برای یافتن نگاشت متن فارسی ---

بررسی فایل: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/colab_ocr_project/class_names.pkl | نوع داده: <class 'list'>

بررسی فایل: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/colab_ocr_project/resnet_farsi.pth | نوع داده: <class 'collections.OrderedDict'>
 کلیدهای موجود: ['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean']

بررسی فایل: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/colab_ocr_project/efficientnet_farsi.pth | نوع داده: <class 'collections.OrderedDict'>
 کلیدهای موجود: ['conv_stem.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'blocks.0.0.conv_dw.weight', 'blocks.0.0.bn1.weight', 'blocks.0.0.bn1.bias', 'blocks.0.0.bn1.running_mean']

بررس

In [ ]:
import os

base = "/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features"

target_ids = ["27938", "27907", "28007", "27931", "28012"]

print("Searching for target IDs in all project files...\n")

for root, dirs, files in os.walk(base):

    for fname in files:

        # فایل‌هایی که اسمشان مستقیماً شامل شناسه است
        if any(tid in fname for tid in target_ids):
            print("FILE NAME MATCH:")
            print(os.path.join(root, fname))

        # فایل‌های متنی را هم جستجو می‌کنیم
        path = os.path.join(root, fname)

        try:
            if os.path.getsize(path) > 50 * 1024 * 1024:
                continue

            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()

            matches = [tid for tid in target_ids if tid in content]

            if matches:
                print("\nCONTENT MATCH:")
                print("File:", path)
                print("IDs:", matches)

        except:
            pass

print("\nSearch finished.")

Searching for target IDs in all project files...


CONTENT MATCH:
File: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_ocr_final_backup/dataset_file_list.txt
IDs: ['27938', '27907', '28007', '27931', '28012']

CONTENT MATCH:
File: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_split/train.txt
IDs: ['27938', '27907', '28007']

CONTENT MATCH:
File: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_split/validation.txt
IDs: ['27931', '28012']

CONTENT MATCH:
File: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json
IDs: ['27938', '27907', '28007', '27931', '28012']

Search finished.


In [ ]:
import os

base = "/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_split"

for fname in ["train.txt", "validation.txt"]:
    path = os.path.join(base, fname)

    print("\n" + "="*80)
    print(fname)
    print("="*80)

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    print("Number of lines:", len(lines))

    print("\nFirst 20 lines:")
    for i, line in enumerate(lines[:20]):
        print(i, repr(line.strip()))

In [ ]:
import os

paths = [
    "/content/drive/MyDrive/idplimgl_full",
    "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
]

for p in paths:
    print("\n" + "="*80)
    print("PATH:", p)
    print("="*80)

    if not os.path.exists(p):
        print("❌ مسیر وجود ندارد")
        continue

    files = os.listdir(p)

    images = [
        f for f in files
        if f.lower().endswith((".tif", ".tiff", ".png", ".jpg", ".jpeg"))
    ]

    print("Total files:", len(files))
    print("Image files:", len(images))

    print("\nFirst 10:")
    for f in images[:10]:
        print(f)

    print("\nLast 10:")
    for f in images[-10:]:
        print(f)

In [ ]:
import os
import pandas as pd

ROOT = "/content/drive"

# پسوندهای احتمالی فایل Ground Truth / metadata
extensions = {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".json"}

candidates = []

print("🔎 Searching Drive for possible IDPL-PFOD metadata files...\n")

for base, dirs, files in os.walk(ROOT):
    # حذف پوشه‌های غیرضروری
    dirs[:] = [
        d for d in dirs
        if d not in {
            ".cache", ".config", ".local", "node_modules",
            "__pycache__", ".ipynb_checkpoints"
        }
    ]

    for f in files:
        ext = os.path.splitext(f)[1].lower()

        if ext in extensions:
            path = os.path.join(base, f)

            try:
                size_mb = os.path.getsize(path) / (1024**2)
            except:
                continue

            name_lower = f.lower()

            # فایل‌هایی که از نظر نام یا اندازه ارزش بررسی دارند
            if any(k in name_lower for k in [
                "idpl", "pfod", "miras", "text",
                "annotation", "annot", "ground",
                "label", "metadata", "meta", "info"
            ]):
                candidates.append((path, size_mb))

print("=" * 100)
print(f"FOUND CANDIDATES: {len(candidates)}")
print("=" * 100)

for path, size_mb in sorted(candidates):
    print(f"{size_mb:10.2f} MB | {path}")

In [ ]:
# ============================================================
# STEP 1 — بررسی دسترسی به IDPL-PFOD واقعی
# ============================================================

!pip -q install datasets

from datasets import load_dataset

print("Loading IDPL-PFOD metadata...")
ds = load_dataset("myrkur/IDPL-PFOD", split="train")

print("\n" + "="*80)
print("DATASET INFORMATION")
print("="*80)

print("Number of rows:", len(ds))
print("Columns:", ds.column_names)

print("\nFirst sample:")
sample = ds[0]

for k, v in sample.items():
    if k == "image":
        print("image:", type(v), getattr(v, "size", None))
    else:
        print(k, ":", v)

print("\n" + "="*80)
print("TEXT SAMPLES")
print("="*80)

for i in range(5):
    print(f"\n[{i}]")
    print(ds[i]["text"])

Loading IDPL-PFOD metadata...

DATASET INFORMATION
Number of rows: 25617
Columns: ['image', 'text']

First sample:
image: <class 'PIL.PngImagePlugin.PngImageFile'> (700, 50)
text : یابد و كار جمهوری اسلامی را یكسره كند. كافی است به تبلیغات «اكس»گونه یك ماه


TEXT SAMPLES

[0]
یابد و كار جمهوری اسلامی را یكسره كند. كافی است به تبلیغات «اكس»گونه یك ماه


[1]
تا این عدد ۱۱۰ گرم به ۲۰۰ تا ۲۵۰ گرم برسد. وی با اشاره به


[2]
احساس كرد كه در برابر شاه تحقیر شده است. چند روز بعد ابتهاج توسط مافوق


[3]
بخشی از آرای ما، مجازات اداری است كه هیچ مقامی در مقابل آن قرار نمی‌گیرد.


[4]
در سه ماهه نخست سال جاری رشد بیمه های زندگی ١٨ درصد بود. همتی نفوذ



In [ ]:
# ============================================================
# PFMS-SERIES-GT38-2026
# STEP 2 — Exact Image Content Matching
# HuggingFace IDPL-PFOD ↔ Local TIFF Images
# ============================================================

import os
import hashlib
from PIL import Image
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
LOCAL_DIR = "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"

print("=" * 90)
print("PFMS-SERIES-GT38-2026 | IMAGE ↔ GT MATCHING")
print("=" * 90)

# ------------------------------------------------------------
# Function: pixel hash
# ------------------------------------------------------------
def image_pixel_hash(img):
    """
    Creates a hash from actual RGB pixel content.
    This ignores TIFF/PNG container differences.
    """
    img = img.convert("RGB")
    return hashlib.sha256(img.tobytes()).hexdigest()


# ------------------------------------------------------------
# STEP A — Hash all local TIFF images
# ------------------------------------------------------------
local_hash_to_path = {}
local_errors = []

local_files = [
    f for f in os.listdir(LOCAL_DIR)
    if f.lower().endswith((".tif", ".tiff"))
]

print(f"\nLocal TIFF files: {len(local_files)}")
print("Creating pixel hashes...\n")

for fname in tqdm(local_files, desc="Hashing local TIFFs"):
    path = os.path.join(LOCAL_DIR, fname)

    try:
        with Image.open(path) as img:
            h = image_pixel_hash(img)

        local_hash_to_path[h] = path

    except Exception as e:
        local_errors.append((fname, str(e)))

print("\nLocal hashing finished.")
print("Unique local image hashes:", len(local_hash_to_path))
print("Errors:", len(local_errors))


# ------------------------------------------------------------
# STEP B — Match HuggingFace train + test
# ------------------------------------------------------------

all_matches = []
unmatched_hf = []

def process_split(dataset, split_name):

    print("\n" + "=" * 90)
    print(f"PROCESSING SPLIT: {split_name}")
    print("=" * 90)

    matches = 0

    for i in tqdm(range(len(dataset)), desc=f"Matching {split_name}"):

        try:
            sample = dataset[i]

            hf_img = sample["image"]
            text = sample["text"]

            h = image_pixel_hash(hf_img)

            local_path = local_hash_to_path.get(h)

            if local_path is not None:

                matches += 1

                all_matches.append({
                    "split": split_name,
                    "hf_index": i,
                    "image_path": local_path,
                    "image_name": os.path.basename(local_path),
                    "text": text
                })

            else:
                unmatched_hf.append({
                    "split": split_name,
                    "hf_index": i,
                    "text": text
                })

        except Exception as e:
            unmatched_hf.append({
                "split": split_name,
                "hf_index": i,
                "text": None,
                "error": str(e)
            })

    print(f"\nMatched {split_name}: {matches:,} / {len(dataset):,}")

    return matches


# Train
train_matches = process_split(ds, "train")

# Test
test_ds = load_dataset(
    "myrkur/IDPL-PFOD",
    split="test"
)

test_matches = process_split(test_ds, "test")


# ------------------------------------------------------------
# STEP C — Final statistics
# ------------------------------------------------------------

total_hf = len(ds) + len(test_ds)
total_matched = len(all_matches)

print("\n")
print("=" * 90)
print("FINAL MATCHING RESULT")
print("=" * 90)

print(f"HuggingFace total : {total_hf:,}")
print(f"Local TIFF total  : {len(local_files):,}")
print(f"Matched images    : {total_matched:,}")
print(f"Unmatched HF      : {len(unmatched_hf):,}")

coverage = 100 * total_matched / total_hf

print(f"GT coverage       : {coverage:.2f}%")

print("=" * 90)


# ------------------------------------------------------------
# STEP D — Show first matches
# ------------------------------------------------------------

print("\nFIRST 10 VERIFIED MATCHES")
print("-" * 90)

for item in all_matches[:10]:

    print("\nImage :", item["image_name"])
    print("Split :", item["split"])
    print("HF idx:", item["hf_index"])
    print("GT   :", item["text"])


# ------------------------------------------------------------
# STEP E — Save mapping
# ------------------------------------------------------------

import pandas as pd

mapping_df = pd.DataFrame(all_matches)

OUT_CSV = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"

mapping_df.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("\n")
print("=" * 90)
print("MAPPING SAVED")
print("=" * 90)
print(OUT_CSV)
print("Rows:", len(mapping_df))

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from google.colab import drive

# اتصال به گوگل درایو
drive.mount('/content/drive')

print("کتابخانه‌ها بارگذاری و درایو متصل شد.")

### رسم نمودارهای تحلیل داده (Exploratory Data Analysis)

In [ ]:
# فرض می‌کنیم دیتافریم df حاوی اطلاعات تصاویر است
# در صورت لزوم این بخش را با داده‌های واقعی خود جایگزین کنید

def plot_dataset_stats(df):
    if df is None or df.empty:
        print("داده‌ای برای رسم نمودار یافت نشد.")
        return

    plt.figure(figsize=(12, 5))

    # نمودار توزیع طول متن‌ها
    plt.subplot(1, 2, 1)
    sns.histplot(df['text'].str.len(), kde=True, color='skyblue')
    plt.title('Distribution of Text Lengths')
    plt.xlabel('Character Count')

    # نمودار نسبت داده‌های آموزش و تست
    plt.subplot(1, 2, 2)
    if 'split' in df.columns:
        df['split'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['lightgreen', 'salmon'])
        plt.title('Train/Test Split Ratio')

    plt.tight_layout()
    plt.show()

# فراخوانی تابع (پس از بارگذاری دیتاست)
# plot_dataset_stats(matches_df)

In [ ]:
# بررسی وجود فایل و بارگذاری دیتاست برای رسم نمودار
import pandas as pd
import os

# مسیر فایل خروجی حاصل از سلول انطباق (Cell 21)
output_csv = "/content/pfms_series_dataset/idpl_official_local_matching.csv"

if 'matches_df' not in globals():
    if os.path.exists(output_csv):
        print("در حال بارگذاری دیتاست از فایل ذخیره شده در دیسک...")
        matches_df = pd.read_csv(output_csv)
    else:
        print("خطا: داده‌های انطباق یافته یافت نشد.")
        print("لطفاً ابتدا سلول مربوط به انطباق تصاویر و متون (سلول شماره 21) را اجرا کنید تا فایل مورد نظر ساخته شود.")
        matches_df = None

if matches_df is not None:
    print(f"دیتاست با موفقیت بارگذاری شد. تعداد کل رکوردها: {len(matches_df)}")
    # فراخوانی تابع رسم نمودار که در سلول‌های قبلی تعریف شده است
    plot_dataset_stats(matches_df)


In [ ]:
# ================================================================
# PFMS-Net — Sequential / Series Experiment
# ================================================================
# Title:
# طراحی یک سامانه شناسایی نویسه‌های نوری متون فارسی
# مبتنی بر یادگیری عمیق و سازوکار توجه
#
# Experiment:
# Swin Transformer → MambaVision → DTrOCR
#
# Dataset:
# IDPL-PFOD
#
# Version:
# PFMS-Net-Series-Final
#
# Project Code:
# PFMS-SERIES-T-SNE-2026
# ================================================================

print("=" * 70)
print("PFMS-Net — Sequential / Series Experiment")
print("Architecture: Swin → MambaVision → DTrOCR")
print("Dataset: IDPL-PFOD")
print("Version: PFMS-Net-Series-Final")
print("=" * 70)

PFMS-Net — Sequential / Series Experiment
Architecture: Swin → MambaVision → DTrOCR
Dataset: IDPL-PFOD
Version: PFMS-Net-Series-Final


In [ ]:
# ================================================================
# CELL 2 — Install Required Libraries
# ================================================================

!pip -q install timm transformers sentencepiece accelerate

In [ ]:
# ================================================================
# CELL 3 — Imports
# ================================================================

import os
import sys
import json
import math
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms

import timm

from transformers import (
    VisionEncoderDecoderModel,
    TrOCRProcessor
)

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# ================================================================
# CELL 4 — Mount Google Drive
# ================================================================

from google.colab import drive

drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Google Drive mounted successfully.


In [ ]:
# ================================================================
# CELL 5 — Verify IDPL-PFOD Dataset
# ================================================================

DATASET_PATH = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
)

print("=" * 70)
print("DATASET VERIFICATION")
print("=" * 70)

print("Dataset path:")
print(DATASET_PATH)

print("\nPath exists:", DATASET_PATH.exists())
print("Is directory:", DATASET_PATH.is_dir())

if DATASET_PATH.exists():

    # Supported image formats
    image_extensions = {
        ".jpg", ".jpeg", ".png",
        ".bmp", ".tif", ".tiff",
        ".webp"
    }

    image_files = [
        p for p in DATASET_PATH.rglob("*")
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    print("\nTotal image files:", len(image_files))

    # Show first 10 files
    print("\nFirst 10 image files:")
    for i, img_path in enumerate(image_files[:10], 1):
        print(f"{i:02d}. {img_path}")

    # File extensions
    extension_counts = {}

    for p in image_files:
        ext = p.suffix.lower()
        extension_counts[ext] = extension_counts.get(ext, 0) + 1

    print("\nImage extensions:")
    for ext, count in sorted(extension_counts.items()):
        print(f"{ext}: {count}")

else:
    print("\nERROR: Dataset path does not exist.")

DATASET VERIFICATION
Dataset path:
/content/drive/Othercomputers/My Laptop/Desktop/idplimgl

Path exists: True
Is directory: True

Total image files: 27178

First 10 image files:
01. /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27868.tif
02. /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27935.tif
03. /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27901.tif
04. /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27896.tif
05. /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27899.tif
06. /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27931.tif
07. /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27928.tif
08. /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27895.tif
09. /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27917.tif
10. /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27908.tif

Image extensions:
.tif: 27178


In [ ]:
# ================================================================
# CELL 6 — Inspect IDPL-PFOD Image Properties
# ================================================================

print("=" * 70)
print("IMAGE PROPERTY INSPECTION")
print("=" * 70)

sample_files = image_files[:20]

image_info = []

for img_path in sample_files:
    try:
        with Image.open(img_path) as img:
            image_info.append({
                "file": img_path.name,
                "width": img.width,
                "height": img.height,
                "mode": img.mode,
                "format": img.format
            })
    except Exception as e:
        print(f"Error reading {img_path.name}: {e}")

df_image_info = pd.DataFrame(image_info)

print("\nSample image information:")
display(df_image_info)

print("\nUnique dimensions in sample:")
print(
    df_image_info[["width", "height"]]
    .drop_duplicates()
    .to_string(index=False)
)

print("\nImage modes:")
print(df_image_info["mode"].value_counts())

print("\nImage formats:")
print(df_image_info["format"].value_counts())

IMAGE PROPERTY INSPECTION

Sample image information:


,file,width,height,mode,format
0,27868.tif,700,50,RGB,TIFF
1,27935.tif,700,50,RGB,TIFF
2,27901.tif,700,50,L,TIFF
3,27896.tif,700,50,RGB,TIFF
4,27899.tif,700,50,RGB,TIFF
5,27931.tif,700,50,RGB,TIFF
6,27928.tif,700,50,RGB,TIFF
7,27895.tif,700,50,L,TIFF
8,27917.tif,700,50,RGB,TIFF
9,27908.tif,700,50,L,TIFF



Unique dimensions in sample:
 width  height
   700      50

Image modes:
mode
RGB    12
L       8
Name: count, dtype: int64

Image formats:
format
TIFF    20
Name: count, dtype: int64


In [ ]:
# ================================================================
# CELL 7 — Image Dimension Statistics
# ================================================================

print("=" * 70)
print("IMAGE DIMENSION STATISTICS")
print("=" * 70)

MAX_CHECK = min(1000, len(image_files))

widths = []
heights = []
modes = []

for img_path in image_files[:MAX_CHECK]:

    try:
        with Image.open(img_path) as img:
            widths.append(img.width)
            heights.append(img.height)
            modes.append(img.mode)

    except Exception:
        continue

print(f"\nChecked images: {len(widths)}")

print("\nWidth statistics:")
print(f"Min    : {min(widths)}")
print(f"Max    : {max(widths)}")
print(f"Mean   : {np.mean(widths):.2f}")
print(f"Median : {np.median(widths):.2f}")

print("\nHeight statistics:")
print(f"Min    : {min(heights)}")
print(f"Max    : {max(heights)}")
print(f"Mean   : {np.mean(heights):.2f}")
print(f"Median : {np.median(heights):.2f}")

print("\nMost common dimensions:")

dimension_counts = (
    pd.DataFrame({
        "width": widths,
        "height": heights
    })
    .value_counts()
    .head(20)
)

print(dimension_counts)

print("\nImage modes:")
print(pd.Series(modes).value_counts())

IMAGE DIMENSION STATISTICS

Checked images: 1000

Width statistics:
Min    : 700
Max    : 700
Mean   : 700.00
Median : 700.00

Height statistics:
Min    : 50
Max    : 50
Mean   : 50.00
Median : 50.00

Most common dimensions:
width  height
700    50        1000
Name: count, dtype: int64

Image modes:
RGB    548
L      452
Name: count, dtype: int64


In [ ]:
import os
import pandas as pd
from pathlib import Path

print('--- جستجوی گسترده برای یافتن فایل Ground Truth (بیش از ۲۵۰۰۰ ردیف) ---')
search_paths = ['/content/drive/MyDrive', '/content/drive/Othercomputers/My Laptop/Desktop']
candidates = []

for base in search_paths:
    if os.path.exists(base):
        print(f'در حال جستجو در: {base}')
        for path in Path(base).rglob('*'):
            if path.is_file() and path.suffix.lower() in ['.csv', '.txt', '.json', '.tsv']:
                try:
                    # بررسی سریع تعداد خطوط بدون بارگذاری کامل
                    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                        line_count = sum(1 for line in f)

                    if line_count > 25000:
                        candidates.append({
                            'path': str(path),
                            'lines': line_count,
                            'size_mb': os.path.getsize(path) / (1024*1024)
                        })
                except:
                    continue

if candidates:
    print(f'\nتعداد {len(candidates)} فایل با حجم بالا پیدا شد:')
    for c in candidates:
        print(f"- مسیر: {c['path']}\n  تعداد ردیف: {c['lines']:,} | حجم: {c['size_mb']:.2f} MB")
        # نمایش نمونه برای تشخیص محتوا
        try:
            if c['path'].endswith('.csv'):
                display(pd.read_csv(c['path'], nrows=3))
            else:
                with open(c['path'], 'r', encoding='utf-8', errors='ignore') as f:
                    print(f"  نمونه محتوا: {f.readline().strip()[:200]}")
        except: print('  امکان نمایش پیش‌نمایش وجود ندارد.')
else:
    print('\n⚠ متاسفانه فایلی با بیش از ۲۵۰۰۰ ردیف پیدا نشد. این یعنی متادیتا ممکن است در قالبی غیرمتنی (مثل .mat یا .pkl حجیم) یا در مسیر دیگری باشد.')

--- جستجوی گسترده برای یافتن فایل Ground Truth (بیش از ۲۵۰۰۰ ردیف) ---
در حال جستجو در: /content/drive/MyDrive
در حال جستجو در: /content/drive/Othercomputers/My Laptop/Desktop

تعداد 3 فایل با حجم بالا پیدا شد:
- مسیر: /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv
  تعداد ردیف: 54,241 | حجم: 5.95 MB


,split,hf_index,image_path,image_name,text
0,train,0,/content/drive/Othercomputers/My Laptop/Deskto...,00490.tif,یابد و كار جمهوری اسلامی را یكسره كند. كافی اس...
1,train,1,/content/drive/Othercomputers/My Laptop/Deskto...,19617.tif,تا این عدد ۱۱۰ گرم به ۲۰۰ تا ۲۵۰ گرم برسد. وی ...
2,train,2,/content/drive/Othercomputers/My Laptop/Deskto...,23763.tif,احساس كرد كه در برابر شاه تحقیر شده است. چند ر...


- مسیر: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_ocr_final_backup/dataset_file_list.txt
  تعداد ردیف: 29,109 | حجم: 1.85 MB
  نمونه محتوا: /content/drive/MyDrive/colab_ocr_project/test/ا/ا.png
- مسیر: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json
  تعداد ردیف: 27,180 | حجم: 1.87 MB
  نمونه محتوا: [


In [ ]:
import json

path = "/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json"

with open(path, "r", encoding="utf-8") as f:
    paths = json.load(f)

print("Number of paths:", len(paths))

print("\nFirst 20:")
for i, p in enumerate(paths[:20]):
    print(i, "=>", p)

print("\nLast 10:")
for i, p in enumerate(paths[-10:], start=len(paths)-10):
    print(i, "=>", p)

Number of paths: 27178

First 20:
0 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001 (1).tif
1 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001.tif
2 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00002 (1).tif
3 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00002.tif
4 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00003.tif
5 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00004 (1).tif
6 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00004.tif
7 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00005 (1).tif
8 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00005.tif
9 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00006 (1).tif
10 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00006.tif
11 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00007 (1).tif
12 => /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00007.tif
13 

In [ ]:
import os

print("\nChecking corresponding image files:\n")

for p in paths[:20]:
    print(os.path.basename(p), "=>", os.path.exists(p))


Checking corresponding image files:

00001 (1).tif => True
00001.tif => True
00002 (1).tif => True
00002.tif => True
00003.tif => True
00004 (1).tif => True
00004.tif => True
00005 (1).tif => True
00005.tif => True
00006 (1).tif => True
00006.tif => True
00007 (1).tif => True
00007.tif => True
00008 (1).tif => True
00008.tif => True
00009 (1).tif => True
00009.tif => True
00010 (1).tif => True
00010.tif => True
00011 (1).tif => True


In [ ]:
# ================================================================
# CELL 7 — IDPL-PFOD Preprocessing Configuration
# ================================================================

print("=" * 70)
print("PFMS-Net SERIES — DATA PREPROCESSING")
print("=" * 70)

# ------------------------------------------------
# Image configuration
# ------------------------------------------------

IMAGE_HEIGHT = 50
IMAGE_WIDTH  = 700

# Swin will receive RGB images
IMAGE_MODE = "RGB"

# IMPORTANT:
# We preserve the original OCR aspect ratio.
# No 224x224 resizing is applied.
print(f"Input image size : {IMAGE_HEIGHT} × {IMAGE_WIDTH}")
print(f"Image mode       : {IMAGE_MODE}")
print("Resize           : DISABLED")
print("Aspect ratio     : PRESERVED")

# ------------------------------------------------
# Normalization
# ------------------------------------------------

# Standard ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

print("\nPreprocessing pipeline:")
print("TIFF → RGB → Tensor → ImageNet Normalization")

print("\n✓ Preprocessing configuration is ready.")

PFMS-Net SERIES — DATA PREPROCESSING
Input image size : 50 × 700
Image mode       : RGB
Resize           : DISABLED
Aspect ratio     : PRESERVED

Preprocessing pipeline:
TIFF → RGB → Tensor → ImageNet Normalization

✓ Preprocessing configuration is ready.


In [ ]:
# ================================================================
# CELL 11 — Inspect IDPL Dataset Folder Structure
# ================================================================

from pathlib import Path
from collections import Counter

BASE_PATH = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop"
)

folders_to_check = [
    BASE_PATH / "idplimg",
    BASE_PATH / "idplimgl",
    BASE_PATH / "idplimgl_clean"
]

print("=" * 70)
print("IDPL DATASET FOLDER STRUCTURE")
print("=" * 70)

for folder in folders_to_check:

    print("\n" + "-" * 70)
    print("FOLDER:", folder)
    print("Exists:", folder.exists())
    print("Is directory:", folder.is_dir())

    if not folder.exists() or not folder.is_dir():
        continue

    # Direct children
    children = list(folder.iterdir())

    files = [p for p in children if p.is_file()]
    dirs  = [p for p in children if p.is_dir()]

    print("Direct files:", len(files))
    print("Direct subfolders:", len(dirs))

    if dirs:
        print("\nSubfolders:")
        for d in dirs[:50]:
            print("  [DIR] ", d.name)

    if files:
        print("\nFirst files:")
        for f in files[:20]:
            print("  [FILE]", f.name)

    # File extension statistics
    extension_counter = Counter(
        p.suffix.lower()
        for p in files
    )

    print("\nDirect file extensions:")
    if extension_counter:
        for ext, count in extension_counter.most_common():
            print(f"  {ext or '[NO EXTENSION]'} : {count}")
    else:
        print("  No direct files.")

IDPL DATASET FOLDER STRUCTURE

----------------------------------------------------------------------
FOLDER: /content/drive/Othercomputers/My Laptop/Desktop/idplimg
Exists: True
Is directory: True
Direct files: 0
Direct subfolders: 0

Direct file extensions:
  No direct files.

----------------------------------------------------------------------
FOLDER: /content/drive/Othercomputers/My Laptop/Desktop/idplimgl
Exists: True
Is directory: True
Direct files: 27178
Direct subfolders: 0

First files:
  [FILE] 27868.tif
  [FILE] 27935.tif
  [FILE] 27901.tif
  [FILE] 27896.tif
  [FILE] 27899.tif
  [FILE] 27931.tif
  [FILE] 27928.tif
  [FILE] 27895.tif
  [FILE] 27917.tif
  [FILE] 27908.tif
  [FILE] 27866.tif
  [FILE] 27909.tif
  [FILE] 27850.tif
  [FILE] 27879.tif
  [FILE] 27877.tif
  [FILE] 27894.tif
  [FILE] 27855.tif
  [FILE] 27920.tif
  [FILE] 27864.tif
  [FILE] 27907.tif

Direct file extensions:
  .tif : 27178

----------------------------------------------------------------------
FOLDE

In [ ]:
# ================================================================
# CELL 12 — Inspect Previous OCR Code Directory
# ================================================================

OCR_CODE_PATH = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop/code ocr persion"
)

print("=" * 70)
print("INSPECTING PREVIOUS OCR CODE DIRECTORY")
print("=" * 70)

print("Path:", OCR_CODE_PATH)
print("Exists:", OCR_CODE_PATH.exists())

if OCR_CODE_PATH.exists():

    items = list(OCR_CODE_PATH.iterdir())

    print(f"\nTotal direct items: {len(items)}")

    print("\nFiles and folders:")

    for i, p in enumerate(items, 1):

        if p.is_file():
            try:
                size_kb = p.stat().st_size / 1024
            except:
                size_kb = 0

            print(
                f"{i:03d}. [FILE]   "
                f"{p.name}   "
                f"{size_kb:.2f} KB"
            )

        elif p.is_dir():

            print(f"{i:03d}. [FOLDER] {p.name}")

else:

    print("\n❌ OCR code directory not found.")

INSPECTING PREVIOUS OCR CODE DIRECTORY
Path: /content/drive/Othercomputers/My Laptop/Desktop/code ocr persion
Exists: True

Total direct items: 10

Files and folders:
001. [FOLDER] flagged
002. [FILE]   پروپوزال 4.docx   61.52 KB
003. [FOLDER] persian_alphabet_images
004. [FOLDER] .ipynb_checkpoints
005. [FILE]   label_encoder.pkl   2.23 KB
006. [FILE]   Untitled36.ipynb   361.48 KB
007. [FOLDER] My EndNote Library1.Data
008. [FOLDER] PDF
009. [FOLDER] My EndNote Library1.enl
010. [FOLDER] مقاله خرداد1404


In [ ]:
# ================================================================
# CELL 13 — Inspect Previous OCR Notebook for Dataset / Labels
# ================================================================

import json
import re

OLD_NOTEBOOK = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop/code ocr persion/Untitled36.ipynb"
)

print("=" * 70)
print("SEARCHING PREVIOUS NOTEBOOK FOR DATASET / LABEL INFORMATION")
print("=" * 70)

print("Notebook:", OLD_NOTEBOOK)
print("Exists:", OLD_NOTEBOOK.exists())

keywords = [
    "idpl",
    "pfod",
    "label",
    "labels",
    "ground truth",
    "ground_truth",
    "text",
    "transcription",
    "annotation",
    "csv",
    "tif",
    "27868",
    "27178"
]

if OLD_NOTEBOOK.exists():

    with open(
        OLD_NOTEBOOK,
        "r",
        encoding="utf-8"
    ) as f:

        notebook_data = json.load(f)

    cells = notebook_data.get("cells", [])

    print(f"\nTotal notebook cells: {len(cells)}")

    matched_cells = []

    for idx, cell in enumerate(cells):

        source = "".join(cell.get("source", []))

        source_lower = source.lower()

        matched_keywords = [
            k for k in keywords
            if k.lower() in source_lower
        ]

        if matched_keywords:

            matched_cells.append(
                (idx, cell.get("cell_type"), matched_keywords, source)
            )

    print(
        f"\nCells containing relevant keywords: "
        f"{len(matched_cells)}"
    )

    print("\n" + "=" * 70)

    for idx, cell_type, matches, source in matched_cells:

        print(f"\nCELL INDEX: {idx}")
        print(f"CELL TYPE : {cell_type}")
        print(f"KEYWORDS  : {matches}")
        print("-" * 70)

        # Limit output per cell
        print(source[:4000])

        print("\n" + "=" * 70)

else:

    print("\n❌ Notebook not found.")

SEARCHING PREVIOUS NOTEBOOK FOR DATASET / LABEL INFORMATION
Notebook: /content/drive/Othercomputers/My Laptop/Desktop/code ocr persion/Untitled36.ipynb
Exists: True

Total notebook cells: 23

Cells containing relevant keywords: 14


CELL INDEX: 0
CELL TYPE : code
KEYWORDS  : ['text']
----------------------------------------------------------------------
from PIL import Image, ImageDraw, ImageFont
import os

# تعریف حروف الفبای فارسی
persian_alphabet = [
    'ا', 'ب', 'پ', 'ت', 'ث', 'ج', 'چ', 'ح', 'خ', 
    'د', 'ذ', 'ر', 'ز', 'ژ', 'س', 'ش', 'ص', 'ض', 
    'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ک', 'گ', 'ل', 
    'م', 'ن', 'و', 'ه', 'ی'
]

# مسیر ذخیره تصاویر
output_dir = 'persian_alphabet_images'
os.makedirs(output_dir, exist_ok=True)

# فونت فارسی (به فونت Tahoma یا هر فونت دیگر که شامل حروف فارسی باشد تغییر دهید)
font_path = 'C:/Windows/Fonts/bzar.ttf'  # مسیر فونت را مشخص کنید
font_size = 100

for letter in persian_alphabet:
    # ایجاد یک تصویر جدید
    img = Image.new('RGB', (100, 100), c

In [ ]:
# ================================================================
# CELL 14 — Locate IDPL-PFOD CSV / Ground Truth Files
# ================================================================

from pathlib import Path

print("=" * 70)
print("IDPL-PFOD CSV / GROUND TRUTH SEARCH")
print("=" * 70)

# ------------------------------------------------
# Search locations
# ------------------------------------------------

SEARCH_ROOTS = [
    Path("/content/drive/Othercomputers/My Laptop/Desktop"),
    Path("/content/drive/MyDrive")
]

# ------------------------------------------------
# Search for CSV / annotation / metadata files
# ------------------------------------------------

candidate_files = []

for root in SEARCH_ROOTS:

    if not root.exists():
        print(f"\n⚠ Search root not accessible: {root}")
        continue

    print(f"\nSearching in:")
    print(root)

    try:

        for p in root.rglob("*"):

            if not p.is_file():
                continue

            # Only relevant file types
            if p.suffix.lower() not in {
                ".csv",
                ".tsv",
                ".json",
                ".jsonl",
                ".txt"
            }:
                continue

            name = p.name.lower()
            full_path = str(p).lower()

            # Strong IDPL/PFOD indicators
            indicators = [
                "idpl",
                "pfod",
                "ground",
                "truth",
                "annotation",
                "transcription",
                "metadata",
                "label"
            ]

            if any(
                word in name or word in full_path
                for word in indicators
            ):
                candidate_files.append(p)

    except Exception as e:
        print("Search error:", e)


# Remove duplicates
candidate_files = list(dict.fromkeys(candidate_files))

# ------------------------------------------------
# Results
# ------------------------------------------------

print("\n" + "=" * 70)
print("SEARCH RESULTS")
print("=" * 70)

print(f"\nPotential files found: {len(candidate_files)}")

if candidate_files:

    for i, p in enumerate(candidate_files, 1):

        try:
            size_kb = p.stat().st_size / 1024
        except:
            size_kb = 0

        print(
            f"{i:03d}. {p}"
            f"   | Size: {size_kb:.2f} KB"
        )

else:

    print("\n❌ No candidate metadata files found.")

IDPL-PFOD CSV / GROUND TRUTH SEARCH

Searching in:
/content/drive/Othercomputers/My Laptop/Desktop

Searching in:
/content/drive/MyDrive

SEARCH RESULTS

Potential files found: 3
001. /content/drive/MyDrive/labels.csv   | Size: 17.51 KB
002. /content/drive/MyDrive/labels_extended.csv   | Size: 60.00 KB
003. /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv   | Size: 6087.69 KB


In [ ]:
# ================================================================
# CELL 15 — Inspect labels.csv and labels_extended.csv
# ================================================================

import pandas as pd
from pathlib import Path

LABEL_FILES = [
    Path("/content/drive/MyDrive/labels.csv"),
    Path("/content/drive/MyDrive/labels_extended.csv")
]

print("=" * 70)
print("INSPECTING LABEL FILES")
print("=" * 70)

for label_file in LABEL_FILES:

    print("\n" + "=" * 70)
    print(f"FILE: {label_file}")
    print("=" * 70)

    if not label_file.exists():
        print("❌ File not found.")
        continue

    try:

        df = pd.read_csv(label_file)

        print(f"\nRows    : {len(df):,}")
        print(f"Columns : {len(df.columns)}")

        print("\nColumn names:")
        for i, col in enumerate(df.columns, 1):
            print(f"{i:02d}. {col}")

        print("\nData types:")
        print(df.dtypes)

        print("\nFirst 10 rows:")
        display(df.head(10))

        print("\nMissing values:")
        print(df.isnull().sum())

        print("\n" + "-" * 70)
        print("Possible image/file columns:")

        for col in df.columns:

            col_lower = str(col).lower()

            if any(
                key in col_lower
                for key in [
                    "image",
                    "img",
                    "file",
                    "filename",
                    "name",
                    "id"
                ]
            ):
                print(f"  → {col}")

        print("\nPossible text/ground-truth columns:")

        for col in df.columns:

            col_lower = str(col).lower()

            if any(
                key in col_lower
                for key in [
                    "text",
                    "label",
                    "transcription",
                    "truth",
                    "gt",
                    "sentence",
                    "content"
                ]
            ):
                print(f"  → {col}")

    except Exception as e:

        print(f"\n❌ Error reading file:")
        print(e)

INSPECTING LABEL FILES

FILE: /content/drive/MyDrive/labels.csv

Rows    : 1,061
Columns : 2

Column names:
01. image
02. text

Data types:
image    object
text     object
dtype: object

First 10 rows:


,image,text
0,00046.tif,00046
1,00039.tif,00039
2,00024.tif,00024
3,00050.tif,00050
4,00040 (1).tif,00040 (1)
5,00003 (1).tif,00003 (1)
6,00021.tif,00021
7,00031.tif,00031
8,00030.tif,00030
9,00048.tif,00048



Missing values:
image    0
text     0
dtype: int64

----------------------------------------------------------------------
Possible image/file columns:
  → image

Possible text/ground-truth columns:
  → text

FILE: /content/drive/MyDrive/labels_extended.csv

Rows    : 1,061
Columns : 2

Column names:
01. image_path
02. text

Data types:
image_path    object
text          object
dtype: object

First 10 rows:


,image_path,text
0,/content/drive/MyDrive/idplimgl/idplimgl/00046...,00046
1,/content/drive/MyDrive/idplimgl/idplimgl/00039...,00039
2,/content/drive/MyDrive/idplimgl/idplimgl/00024...,00024
3,/content/drive/MyDrive/idplimgl/idplimgl/00050...,00050
4,/content/drive/MyDrive/idplimgl/idplimgl/00040...,00040 (1)
5,/content/drive/MyDrive/idplimgl/idplimgl/00003...,00003 (1)
6,/content/drive/MyDrive/idplimgl/idplimgl/00021...,00021
7,/content/drive/MyDrive/idplimgl/idplimgl/00031...,00031
8,/content/drive/MyDrive/idplimgl/idplimgl/00030...,00030
9,/content/drive/MyDrive/idplimgl/idplimgl/00048...,00048



Missing values:
image_path    0
text          0
dtype: int64

----------------------------------------------------------------------
Possible image/file columns:
  → image_path

Possible text/ground-truth columns:
  → text


In [ ]:
# ================================================================
# CELL 16 — Find Possible Official IDPL-PFOD Metadata CSV
# ================================================================

import os
import pandas as pd
from pathlib import Path

print("=" * 70)
print("SEARCHING FOR LARGE IDPL-PFOD METADATA CSV")
print("=" * 70)

SEARCH_ROOTS = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Othercomputers/My Laptop/Desktop")
]

# IDPL-PFOD official dataset size
EXPECTED_ROWS = 30138

# We accept a small tolerance because some files may have
# a header or slightly different formatting.
MIN_EXPECTED_ROWS = 25000

candidate_csvs = []

for root in SEARCH_ROOTS:

    if not root.exists():
        print(f"\n⚠ Cannot access: {root}")
        continue

    print(f"\nScanning:")
    print(root)

    try:

        for csv_path in root.rglob("*.csv"):

            try:

                # Fast row counting without loading entire CSV
                with open(
                    csv_path,
                    "r",
                    encoding="utf-8",
                    errors="ignore"
                ) as f:

                    row_count = sum(1 for _ in f) - 1

                if row_count >= MIN_EXPECTED_ROWS:

                    candidate_csvs.append(
                        (csv_path, row_count)
                    )

            except Exception as e:

                print(
                    f"Could not inspect {csv_path.name}: {e}"
                )

    except Exception as e:

        print(f"Search error: {e}")


# Remove duplicates
unique_candidates = {}

for path, rows in candidate_csvs:
    unique_candidates[str(path)] = rows

candidate_csvs = [
    (Path(path), rows)
    for path, rows in unique_candidates.items()
]

# Sort by closeness to expected official size
candidate_csvs.sort(
    key=lambda x: abs(x[1] - EXPECTED_ROWS)
)

print("\n" + "=" * 70)
print("CANDIDATE CSV FILES")
print("=" * 70)

print(
    f"\nExpected official IDPL-PFOD rows: "
    f"{EXPECTED_ROWS:,}"
)

print(
    f"Minimum rows considered: "
    f"{MIN_EXPECTED_ROWS:,}"
)

print(
    f"\nCandidates found: "
    f"{len(candidate_csvs)}"
)

if candidate_csvs:

    for i, (path, rows) in enumerate(
        candidate_csvs,
        1
    ):

        try:
            size_mb = path.stat().st_size / (1024 ** 2)
        except:
            size_mb = 0

        difference = rows - EXPECTED_ROWS

        print(
            f"\n{i:03d}. {path}"
        )

        print(
            f"     Rows       : {rows:,}"
        )

        print(
            f"     Difference : {difference:+,}"
        )

        print(
            f"     Size       : {size_mb:.2f} MB"
        )

else:

    print(
        "\n❌ No large CSV file was found."
    )

    print(
        "\nThis means the official metadata CSV "
        "is probably not currently stored in the mounted Drive."
    )

SEARCHING FOR LARGE IDPL-PFOD METADATA CSV

Scanning:
/content/drive/MyDrive

Scanning:
/content/drive/Othercomputers/My Laptop/Desktop

CANDIDATE CSV FILES

Expected official IDPL-PFOD rows: 30,138
Minimum rows considered: 25,000

Candidates found: 1

001. /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv
     Rows       : 54,240
     Difference : +24,102
     Size       : 5.95 MB


In [ ]:
# ================================================================
# CELL 17 — Inspect Official IDPL-PFOD from Hugging Face
# ================================================================

!pip -q install datasets huggingface_hub

from datasets import load_dataset

print("=" * 70)
print("LOADING OFFICIAL IDPL-PFOD DATASET")
print("=" * 70)

HF_DATASET_NAME = "myrkur/IDPL-PFOD"

print("\nDataset:", HF_DATASET_NAME)
print("Loading metadata / dataset structure...")

idpl_hf = load_dataset(
    HF_DATASET_NAME
)

print("\n" + "=" * 70)
print("DATASET LOADED")
print("=" * 70)

print("\nDataset object:")
print(idpl_hf)

print("\nSplits:")
print(idpl_hf)

for split_name in idpl_hf.keys():

    split = idpl_hf[split_name]

    print("\n" + "-" * 70)
    print(f"SPLIT: {split_name}")
    print("-" * 70)

    print("Number of rows:", len(split))
    print("Features:", split.features)

LOADING OFFICIAL IDPL-PFOD DATASET

Dataset: myrkur/IDPL-PFOD
Loading metadata / dataset structure...


README.md:   0%|          | 0.00/434 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  432MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 76.3MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25617 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4521 [00:00<?, ? examples/s]


DATASET LOADED

Dataset object:
DatasetDict({
    train: Dataset({
        features: ['image', 'text'],
        num_rows: 25617
    })
    test: Dataset({
        features: ['image', 'text'],
        num_rows: 4521
    })
})

Splits:
DatasetDict({
    train: Dataset({
        features: ['image', 'text'],
        num_rows: 25617
    })
    test: Dataset({
        features: ['image', 'text'],
        num_rows: 4521
    })
})

----------------------------------------------------------------------
SPLIT: train
----------------------------------------------------------------------
Number of rows: 25617
Features: {'image': Image(mode=None, decode=True), 'text': Value('string')}

----------------------------------------------------------------------
SPLIT: test
----------------------------------------------------------------------
Number of rows: 4521
Features: {'image': Image(mode=None, decode=True), 'text': Value('string')}


In [ ]:
# ================================================================
# CELL 18 — Inspect Official IDPL-PFOD Image + Ground Truth
# ================================================================

print("=" * 70)
print("INSPECTING OFFICIAL IDPL-PFOD IMAGE / TEXT PAIRS")
print("=" * 70)

for split_name in ["train", "test"]:

    print("\n" + "=" * 70)
    print(f"SPLIT: {split_name}")
    print("=" * 70)

    ds = idpl_hf[split_name]

    print("Number of samples:", len(ds))

    # Inspect first 5 samples
    for i in range(min(5, len(ds))):

        sample = ds[i]

        image = sample["image"]
        text = sample["text"]

        print("\n" + "-" * 70)
        print(f"Sample {i}")

        print("Image object type :", type(image))
        print("Image size        :", image.size)
        print("Image mode        :", image.mode)
        print("Ground Truth text :", repr(text))

        # Check whether image object contains a filename/path
        print("Image info        :", image.info)

        if hasattr(image, "filename"):
            print("Image filename    :", image.filename)

        if hasattr(image, "path"):
            print("Image path        :", image.path)

INSPECTING OFFICIAL IDPL-PFOD IMAGE / TEXT PAIRS

SPLIT: train
Number of samples: 25617

----------------------------------------------------------------------
Sample 0
Image object type : <class 'PIL.PngImagePlugin.PngImageFile'>
Image size        : (700, 50)
Image mode        : RGB
Ground Truth text : 'یابد و كار جمهوری اسلامی را یكسره كند. كافی است به تبلیغات «اكس»گونه یك ماه\n'
Image info        : {}
Image filename    : 

----------------------------------------------------------------------
Sample 1
Image object type : <class 'PIL.PngImagePlugin.PngImageFile'>
Image size        : (700, 50)
Image mode        : RGB
Ground Truth text : 'تا این عدد ۱۱۰ گرم به ۲۰۰ تا ۲۵۰ گرم برسد. وی با اشاره به\n'
Image info        : {}
Image filename    : 

----------------------------------------------------------------------
Sample 2
Image object type : <class 'PIL.PngImagePlugin.PngImageFile'>
Image size        : (700, 50)
Image mode        : RGB
Ground Truth text : 'احساس كرد كه در برابر شاه تحقی

In [ ]:
# ================================================================
# CELL 19 — Test Exact Image Matching
# Official IDPL-PFOD ↔ Local TIFF Dataset
# ================================================================

from pathlib import Path
from PIL import Image
import numpy as np
import hashlib

LOCAL_IMAGE_DIR = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
)

print("=" * 70)
print("TESTING OFFICIAL ↔ LOCAL IMAGE MATCHING")
print("=" * 70)

print("\nLocal dataset:", LOCAL_IMAGE_DIR)

local_files = sorted(
    LOCAL_IMAGE_DIR.glob("*.tif")
)

print("Local TIFF files:", len(local_files))


# ------------------------------------------------
# Function: exact RGB pixel hash
# ------------------------------------------------

def image_hash_from_pil(img):
    """
    Convert image to RGB and create SHA-256 hash
    from its raw pixel data.
    """

    img = img.convert("RGB")
    arr = np.asarray(img, dtype=np.uint8)

    return hashlib.sha256(
        arr.tobytes()
    ).hexdigest()


# ------------------------------------------------
# Build hash index for local TIFF images
# ------------------------------------------------

print("\nBuilding local image hash index...")
print("This may take a little time.")

local_hash_to_file = {}

for i, path in enumerate(local_files):

    try:

        with Image.open(path) as img:

            if img.size != (700, 50):
                continue

            h = image_hash_from_pil(img)

        local_hash_to_file[h] = path.name

    except Exception as e:

        print(
            f"Warning: could not read {path.name}: {e}"
        )

    if (i + 1) % 5000 == 0:

        print(
            f"Processed: {i + 1:,} / "
            f"{len(local_files):,}"
        )


print(
    "\nLocal hash index size:",
    len(local_hash_to_file)
)


# ------------------------------------------------
# Test official samples
# ------------------------------------------------

print("\n" + "=" * 70)
print("TESTING FIRST 20 OFFICIAL SAMPLES")
print("=" * 70)

matches = 0
not_found = 0

for split_name in ["train", "test"]:

    ds = idpl_hf[split_name]

    # First 10 from each split
    n = min(10, len(ds))

    print(
        f"\n--- {split_name.upper()} ---"
    )

    for i in range(n):

        sample = ds[i]

        official_image = sample["image"]
        ground_truth = sample["text"]

        h = image_hash_from_pil(
            official_image
        )

        matched_file = local_hash_to_file.get(h)

        if matched_file is not None:

            matches += 1

            print(
                f"\n✓ MATCH"
                f"\n  Official index : {i}"
                f"\n  Local file     : {matched_file}"
                f"\n  GT preview     : {repr(ground_truth[:100])}"
            )

        else:

            not_found += 1

            print(
                f"\n✗ NOT FOUND"
                f"\n  Official index : {i}"
                f"\n  GT preview     : {repr(ground_truth[:100])}"
            )


# ------------------------------------------------
# Summary
# ------------------------------------------------

print("\n" + "=" * 70)
print("MATCHING TEST SUMMARY")
print("=" * 70)

print("Matched    :", matches)
print("Not found  :", not_found)

total_tested = matches + not_found

if total_tested > 0:

    match_rate = (
        matches / total_tested
    ) * 100

    print(
        f"Match rate : {match_rate:.2f}%"
    )

TESTING OFFICIAL ↔ LOCAL IMAGE MATCHING

Local dataset: /content/drive/Othercomputers/My Laptop/Desktop/idplimgl
Local TIFF files: 27178

Building local image hash index...
This may take a little time.
Processed: 5,000 / 27,178
Processed: 10,000 / 27,178
Processed: 15,000 / 27,178
Processed: 20,000 / 27,178
Processed: 25,000 / 27,178

Local hash index size: 27117

TESTING FIRST 20 OFFICIAL SAMPLES

--- TRAIN ---

✓ MATCH
  Official index : 0
  Local file     : 00490.tif
  GT preview     : 'یابد و كار جمهوری اسلامی را یكسره كند. كافی است به تبلیغات «اكس»گونه یك ماه\n'

✓ MATCH
  Official index : 1
  Local file     : 19617.tif
  GT preview     : 'تا این عدد ۱۱۰ گرم به ۲۰۰ تا ۲۵۰ گرم برسد. وی با اشاره به\n'

✓ MATCH
  Official index : 2
  Local file     : 23763.tif
  GT preview     : 'احساس كرد كه در برابر شاه تحقیر شده است. چند روز بعد ابتهاج توسط مافوق\n'

✓ MATCH
  Official index : 3
  Local file     : 01486.tif
  GT preview     : 'بخشی از آرای ما، مجازات اداری است كه هیچ مقامی در مقاب

In [ ]:
# ================================================================
# CELL 20 — Diagnose Unmatched Official Images
# ================================================================

import numpy as np
from PIL import Image, ImageChops
from pathlib import Path

print("=" * 70)
print("DIAGNOSING UNMATCHED OFFICIAL IMAGES")
print("=" * 70)

# ------------------------------------------------
# Unmatched samples discovered in Cell 19
# ------------------------------------------------

unmatched_samples = [
    ("train", 7),
    ("test", 4),
    ("test", 9),
]

# ------------------------------------------------
# Prepare local image list
# ------------------------------------------------

local_files = sorted(
    LOCAL_IMAGE_DIR.glob("*.tif")
)

print(
    f"\nLocal TIFF files: {len(local_files):,}"
)

# ------------------------------------------------
# Compare each unmatched image
# ------------------------------------------------

for split_name, index in unmatched_samples:

    print("\n" + "=" * 70)
    print(
        f"OFFICIAL SAMPLE: {split_name} / index {index}"
    )
    print("=" * 70)

    official_sample = idpl_hf[split_name][index]

    official_img = (
        official_sample["image"]
        .convert("RGB")
    )

    official_text = official_sample["text"]

    official_arr = np.asarray(
        official_img,
        dtype=np.uint8
    )

    print(
        "\nGround Truth:"
    )

    print(
        repr(official_text)
    )

    print(
        "\nOfficial image:"
    )

    print(
        "Size :", official_img.size
    )

    print(
        "Mode :", official_img.mode
    )

    # ------------------------------------------------
    # Find closest local image by mean absolute error
    # ------------------------------------------------

    best_file = None
    best_mae = float("inf")

    checked = 0

    for path in local_files:

        try:

            with Image.open(path) as img:

                if img.size != (700, 50):
                    continue

                local_img = img.convert("RGB")

                local_arr = np.asarray(
                    local_img,
                    dtype=np.uint8
                )

            mae = np.mean(
                np.abs(
                    official_arr.astype(np.int16)
                    -
                    local_arr.astype(np.int16)
                )
            )

            if mae < best_mae:

                best_mae = mae
                best_file = path.name

            checked += 1

        except:
            continue

    print(
        "\nClosest local image:"
    )

    print(
        "File :", best_file
    )

    print(
        f"MAE  : {best_mae:.6f}"
    )

    if best_mae == 0:

        print(
            "✓ Exact pixel match exists."
        )

    elif best_mae < 1:

        print(
            "✓ Very small pixel difference."
        )

    elif best_mae < 5:

        print(
            "⚠ Small image difference."
        )

    else:

        print(
            "⚠ Significant image difference."
        )

DIAGNOSING UNMATCHED OFFICIAL IMAGES

Local TIFF files: 27,178

OFFICIAL SAMPLE: train / index 7

Ground Truth:
'بوده و در پس پرده مشخص نیست که چه اتفاقی در حال رخ دادن است.\n'

Official image:
Size : (700, 50)
Mode : RGB

Closest local image:
File : 10524.tif
MAE  : 3.109800
⚠ Small image difference.

OFFICIAL SAMPLE: test / index 4

Ground Truth:
'و سیمای وقت برای گرفتن رقمی حدود دو میلیارد تومان برای یک سال ، آن\n'

Official image:
Size : (700, 50)
Mode : RGB

Closest local image:
File : 03025.tif
MAE  : 5.239943
⚠ Significant image difference.

OFFICIAL SAMPLE: test / index 9

Ground Truth:
'قیمت ۱۱۷ هزار و ۷۲ تومان به فروش می... است در بازار ارز اما قیمت\n'

Official image:
Size : (700, 50)
Mode : RGB

Closest local image:
File : 09143.tif
MAE  : 4.207343
⚠ Small image difference.


In [ ]:
# ================================================================
# CELL 21 — FULL OFFICIAL ↔ LOCAL DATASET MATCHING
# ================================================================

from pathlib import Path
from PIL import Image
import numpy as np
import hashlib
import pandas as pd
import time

print("=" * 70)
print("FULL IDPL-PFOD IMAGE ↔ GROUND TRUTH MATCHING")
print("=" * 70)

# ------------------------------------------------
# Paths
# ------------------------------------------------

LOCAL_IMAGE_DIR = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
)

OUTPUT_DIR = Path(
    "/content/pfms_series_dataset"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nLocal image directory:")
print(LOCAL_IMAGE_DIR)

print("\nOutput directory:")
print(OUTPUT_DIR)


# ------------------------------------------------
# Collect local TIFF files
# ------------------------------------------------

local_files = sorted(
    LOCAL_IMAGE_DIR.glob("*.tif")
)

print(
    f"\nLocal TIFF files: {len(local_files):,}"
)


# ------------------------------------------------
# Image hash function
# ------------------------------------------------

def image_hash(img):

    img = img.convert("RGB")

    arr = np.asarray(
        img,
        dtype=np.uint8
    )

    return hashlib.sha256(
        arr.tobytes()
    ).hexdigest()


# ------------------------------------------------
# Build local hash index
# ------------------------------------------------

print("\n" + "=" * 70)
print("BUILDING LOCAL IMAGE HASH INDEX")
print("=" * 70)

start_time = time.time()

local_hash_to_files = {}

valid_local = 0
invalid_local = 0

for i, path in enumerate(local_files):

    try:

        with Image.open(path) as img:

            if img.size != (700, 50):
                continue

            h = image_hash(img)

        if h not in local_hash_to_files:
            local_hash_to_files[h] = []

        local_hash_to_files[h].append(
            path.name
        )

        valid_local += 1

    except Exception as e:

        invalid_local += 1

    if (i + 1) % 5000 == 0:

        elapsed = time.time() - start_time

        print(
            f"Processed {i + 1:,} / "
            f"{len(local_files):,} | "
            f"Elapsed: {elapsed:.1f}s"
        )


print("\nLocal images with valid 700×50 size:")
print(f"{valid_local:,}")

print("\nInvalid / skipped:")
print(f"{invalid_local:,}")

print("\nUnique image hashes:")
print(
    f"{len(local_hash_to_files):,}"
)


# ------------------------------------------------
# Official Dataset → local matching
# ------------------------------------------------

print("\n" + "=" * 70)
print("MATCHING OFFICIAL DATASET")
print("=" * 70)

records = []

exact_matches = 0
unmatched = 0
duplicate_hash_matches = 0

overall_start = time.time()


for split_name in ["train", "test"]:

    ds = idpl_hf[split_name]

    print(
        f"\nProcessing split: {split_name}"
    )

    for i in range(len(ds)):

        sample = ds[i]

        official_img = sample["image"]
        ground_truth = sample["text"]

        h = image_hash(
            official_img
        )

        matching_files = (
            local_hash_to_files.get(h, [])
        )

        if len(matching_files) == 1:

            match_type = "exact"
            local_file = matching_files[0]

            exact_matches += 1

        elif len(matching_files) > 1:

            match_type = "exact_duplicate"
            local_file = matching_files[0]

            exact_matches += 1
            duplicate_hash_matches += 1

        else:

            match_type = "unmatched"
            local_file = None

            unmatched += 1

        records.append({

            "official_index": i,
            "split": split_name,
            "local_filename": local_file,
            "ground_truth": ground_truth,
            "match_type": match_type,
            "image_hash": h

        })

        if (i + 1) % 5000 == 0:

            print(
                f"  {i + 1:,} / "
                f"{len(ds):,}"
            )


# ------------------------------------------------
# Create DataFrame
# ------------------------------------------------

matches_df = pd.DataFrame(
    records
)


# ------------------------------------------------
# Save complete matching table
# ------------------------------------------------

matching_csv = (
    OUTPUT_DIR /
    "idpl_official_local_matching.csv"
)

matches_df.to_csv(
    matching_csv,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------
# Summary
# ------------------------------------------------

total_official = len(matches_df)

exact_rate = (
    exact_matches /
    total_official *
    100
)

unmatched_rate = (
    unmatched /
    total_official *
    100
)


print("\n" + "=" * 70)
print("FULL MATCHING RESULTS")
print("=" * 70)

print(
    f"\nOfficial samples      : "
    f"{total_official:,}"
)

print(
    f"Exact matches         : "
    f"{exact_matches:,}"
)

print(
    f"Unmatched             : "
    f"{unmatched:,}"
)

print(
    f"Duplicate exact match : "
    f"{duplicate_hash_matches:,}"
)

print(
    f"\nExact match rate      : "
    f"{exact_rate:.2f}%"
)

print(
    f"Unmatched rate        : "
    f"{unmatched_rate:.2f}%"
)

print(
    f"\nMatching table saved:"
)

print(
    matching_csv
)


# ------------------------------------------------
# Split statistics
# ------------------------------------------------

print("\n" + "=" * 70)
print("MATCHING BY SPLIT")
print("=" * 70)

split_summary = (
    matches_df
    .groupby(
        ["split", "match_type"]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

display(
    split_summary
)


# ------------------------------------------------
# Preview successful matches
# ------------------------------------------------

print("\n" + "=" * 70)
print("FIRST 10 SUCCESSFUL MATCHES")
print("=" * 70)

display(
    matches_df[
        matches_df["match_type"]
        .str.startswith("exact")
    ][
        [
            "official_index",
            "split",
            "local_filename",
            "ground_truth",
            "match_type"
        ]
    ].head(10)
)


# ------------------------------------------------
# Preview unmatched samples
# ------------------------------------------------

print("\n" + "=" * 70)
print("FIRST 20 UNMATCHED SAMPLES")
print("=" * 70)

display(
    matches_df[
        matches_df["match_type"]
        == "unmatched"
    ][
        [
            "official_index",
            "split",
            "ground_truth"
        ]
    ].head(20)
)


print("\n" + "=" * 70)
print("CELL 21 COMPLETED")
print("=" * 70)

print(
    f"Total processing time: "
    f"{time.time() - overall_start:.1f} seconds"
)

FULL IDPL-PFOD IMAGE ↔ GROUND TRUTH MATCHING

Local image directory:
/content/drive/Othercomputers/My Laptop/Desktop/idplimgl

Output directory:
/content/pfms_series_dataset

Local TIFF files: 27,178

BUILDING LOCAL IMAGE HASH INDEX
Processed 5,000 / 27,178 | Elapsed: 39.2s
Processed 10,000 / 27,178 | Elapsed: 55.9s
Processed 15,000 / 27,178 | Elapsed: 72.9s
Processed 20,000 / 27,178 | Elapsed: 89.4s
Processed 25,000 / 27,178 | Elapsed: 105.9s

Local images with valid 700×50 size:
27,178

Invalid / skipped:
0

Unique image hashes:
27,117

MATCHING OFFICIAL DATASET

Processing split: train
  5,000 / 25,617
  10,000 / 25,617
  15,000 / 25,617
  20,000 / 25,617
  25,000 / 25,617

Processing split: test

FULL MATCHING RESULTS

Official samples      : 30,138
Exact matches         : 27,120
Unmatched             : 3,018
Duplicate exact match : 61

Exact match rate      : 89.99%
Unmatched rate        : 10.01%

Matching table saved:
/content/pfms_series_dataset/idpl_official_local_matching.csv


match_type,exact,exact_duplicate,unmatched
split,,,
test,4055,11,455
train,23004,50,2563



FIRST 10 SUCCESSFUL MATCHES


,official_index,split,local_filename,ground_truth,match_type
0,0,train,00490.tif,یابد و كار جمهوری اسلامی را یكسره كند. كافی اس...,exact
1,1,train,19617.tif,تا این عدد ۱۱۰ گرم به ۲۰۰ تا ۲۵۰ گرم برسد. وی ...,exact
2,2,train,23763.tif,احساس كرد كه در برابر شاه تحقیر شده است. چند ر...,exact
3,3,train,01486.tif,بخشی از آرای ما، مجازات اداری است كه هیچ مقامی...,exact
4,4,train,06302.tif,در سه ماهه نخست سال جاری رشد بیمه های زندگی ١٨...,exact
5,5,train,19740.tif,فروشنده مسکن برای تشکیل کابینه جدید در دولت و ...,exact
6,6,train,26460.tif,از آنها كه سالها برای زیارت خانه خدا و تنفس در...,exact
8,8,train,07999.tif,از این اطلاعات برای دسترسی پیدا کردن به اکانت ...,exact
9,9,train,15314.tif,می‌گیریم که افراد باید ۱۰ سال در آن منطقه که م...,exact
11,11,train,22949.tif,بیان می‌کند. به علاوه منجر به کشف آثار باستانی...,exact



FIRST 20 UNMATCHED SAMPLES


,official_index,split,ground_truth
7,7,train,بوده و در پس پرده مشخص نیست که چه اتفاقی در حا...
10,10,train,نور چیده ام از باغ مرقدت شاه چراغ، ای حرم سبز ...
29,29,train,اسلامی طراحی و خدمت مقام معظم رهبری پیشنهاد گر...
39,39,train,آب، تولیدات پیشرفته، معدن کاری، فرودگاهی و هوا...
48,48,train,آن زیر ۱۴۰ کیلومتر بر ساعت است یعنی حدود ۱۳۵ ک...
60,60,train,بزرگ و حدود ۵۰ درصد در سایر شهرها پرداخت شده ک...
72,72,train,معدن نیز شامل موارد مربوط به صادرات و واردات، ...
73,73,train,زیر بارش نمی رود و این زیر بار نرفتن دارد به ب...
76,76,train,است و بهترین آثار برای جشنواره كاوشگران جوان و...
77,77,train,انجام شده برای قطع وابستگی به واردات بنزین و ت...



CELL 21 COMPLETED
Total processing time: 36.2 seconds


In [ ]:
# ================================================================
# CELL 22 — MATCHED DATASET STATISTICAL ANALYSIS
# ================================================================

import pandas as pd
import numpy as np
from collections import Counter

print("=" * 70)
print("PFMS-SERIES — MATCHED DATASET STATISTICS")
print("=" * 70)

# ------------------------------------------------
# Load matching table
# ------------------------------------------------

matching_csv = (
    "/content/pfms_series_dataset/"
    "idpl_official_local_matching.csv"
)

df = pd.read_csv(
    matching_csv,
    encoding="utf-8-sig"
)

print("\nTotal official records:")
print(f"{len(df):,}")


# ------------------------------------------------
# Exact matched records
# ------------------------------------------------

matched_df = df[
    df["match_type"].str.startswith("exact")
].copy()

print("\nExact matched records:")
print(f"{len(matched_df):,}")


# ------------------------------------------------
# Unique local images
# ------------------------------------------------

unique_local_images = (
    matched_df["local_filename"]
    .dropna()
    .nunique()
)

print("\nUnique local images:")
print(f"{unique_local_images:,}")


# ------------------------------------------------
# Duplicate local filenames
# ------------------------------------------------

filename_counts = (
    matched_df["local_filename"]
    .value_counts()
)

duplicated_filenames = (
    filename_counts[
        filename_counts > 1
    ]
)

print("\nLocal images appearing more than once:")
print(
    f"{len(duplicated_filenames):,}"
)

if len(duplicated_filenames) > 0:

    print("\nTop duplicated local images:")

    display(
        duplicated_filenames
        .head(20)
        .to_frame("count")
    )


# ------------------------------------------------
# Split distribution
# ------------------------------------------------

print("\n" + "=" * 70)
print("SPLIT DISTRIBUTION")
print("=" * 70)

split_counts = (
    matched_df["split"]
    .value_counts()
)

display(
    split_counts.to_frame("count")
)


# ------------------------------------------------
# Text cleaning diagnostics
# ------------------------------------------------

matched_df["ground_truth"] = (
    matched_df["ground_truth"]
    .fillna("")
)

matched_df["text_length"] = (
    matched_df["ground_truth"]
    .astype(str)
    .str.replace(
        "\n",
        "",
        regex=False
    )
    .str.len()
)

matched_df["word_count"] = (
    matched_df["ground_truth"]
    .astype(str)
    .str.strip()
    .str.split()
    .str.len()
)


# ------------------------------------------------
# Empty text
# ------------------------------------------------

empty_text = (
    matched_df["ground_truth"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("\nEmpty Ground Truth texts:")
print(empty_text)


# ------------------------------------------------
# Text statistics
# ------------------------------------------------

print("\n" + "=" * 70)
print("TEXT STATISTICS")
print("=" * 70)

print(
    f"\nMinimum text length : "
    f"{matched_df['text_length'].min()}"
)

print(
    f"Maximum text length : "
    f"{matched_df['text_length'].max()}"
)

print(
    f"Mean text length    : "
    f"{matched_df['text_length'].mean():.2f}"
)

print(
    f"Median text length  : "
    f"{matched_df['text_length'].median():.2f}"
)

print(
    f"\nMinimum word count  : "
    f"{matched_df['word_count'].min()}"
)

print(
    f"Maximum word count  : "
    f"{matched_df['word_count'].max()}"
)

print(
    f"Mean word count     : "
    f"{matched_df['word_count'].mean():.2f}"
)

print(
    f"Median word count   : "
    f"{matched_df['word_count'].median():.2f}"
)


# ------------------------------------------------
# Train/Test statistics
# ------------------------------------------------

print("\n" + "=" * 70)
print("TEXT LENGTH BY SPLIT")
print("=" * 70)

split_stats = (
    matched_df
    .groupby("split")
    .agg(
        samples=("local_filename", "count"),
        mean_chars=("text_length", "mean"),
        median_chars=("text_length", "median"),
        max_chars=("text_length", "max"),
        mean_words=("word_count", "mean"),
        max_words=("word_count", "max")
    )
)

display(
    split_stats
)


# ------------------------------------------------
# Character vocabulary
# ------------------------------------------------

print("\n" + "=" * 70)
print("CHARACTER VOCABULARY")
print("=" * 70)

all_text = "".join(
    matched_df["ground_truth"]
    .astype(str)
    .tolist()
)

characters = Counter(
    all_text
)

print(
    "\nUnique characters including spaces/newline:"
)

print(
    len(characters)
)

print(
    "\nTop 30 characters:"
)

display(
    pd.DataFrame(
        characters.most_common(30),
        columns=[
            "character",
            "frequency"
        ]
    )
)


# ------------------------------------------------
# Final summary
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 22 SUMMARY")
print("=" * 70)

print(
    f"Official records       : {len(df):,}"
)

print(
    f"Exact matched records  : {len(matched_df):,}"
)

print(
    f"Unique local images    : {unique_local_images:,}"
)

print(
    f"Unmatched records      : "
    f"{len(df) - len(matched_df):,}"
)

print(
    f"Empty text records     : {empty_text:,}"
)

print("\nNo dataset files were modified.")
print("This cell performs analysis only.")

PFMS-SERIES — MATCHED DATASET STATISTICS

Total official records:
30,138

Exact matched records:
27,120

Unique local images:
27,117

Local images appearing more than once:
3

Top duplicated local images:


,count
local_filename,
15222.tif,2
03358.tif,2
03363.tif,2



SPLIT DISTRIBUTION


,count
split,
train,23054
test,4066



Empty Ground Truth texts:
0

TEXT STATISTICS

Minimum text length : 39
Maximum text length : 114
Mean text length    : 76.21
Median text length  : 76.00

Minimum word count  : 8
Maximum word count  : 20
Mean word count     : 15.00
Median word count   : 15.00

TEXT LENGTH BY SPLIT


,samples,mean_chars,median_chars,max_chars,mean_words,max_words
split,,,,,,
test,4066,76.105263,76.0,104,14.999262,19
train,23054,76.229678,76.0,114,15.000000,20



CHARACTER VOCABULARY

Unique characters including spaces/newline:
165

Top 30 characters:


,character,frequency
0,,379677
1,ا,239507
2,ی,154385
3,ر,138475
4,د,111329
5,ن,107891
6,ه,97216
7,و,88471
8,م,88314
9,ت,80592



CELL 22 SUMMARY
Official records       : 30,138
Exact matched records  : 27,120
Unique local images    : 27,117
Unmatched records      : 3,018
Empty text records     : 0

No dataset files were modified.
This cell performs analysis only.


In [ ]:
# ================================================================
# CELL 23 — DATA LEAKAGE & DUPLICATE CHECK
# ================================================================

import pandas as pd

print("=" * 70)
print("PFMS-SERIES — DATA LEAKAGE CHECK")
print("=" * 70)

matching_csv = (
    "/content/pfms_series_dataset/"
    "idpl_official_local_matching.csv"
)

df = pd.read_csv(
    matching_csv,
    encoding="utf-8-sig"
)

# فقط تطبیق‌های دقیق
matched = df[
    df["match_type"].str.startswith("exact")
].copy()

# ------------------------------------------------
# Train / Test
# ------------------------------------------------

train_df = matched[
    matched["split"] == "train"
].copy()

test_df = matched[
    matched["split"] == "test"
].copy()

print("\nTrain samples:")
print(f"{len(train_df):,}")

print("\nTest samples:")
print(f"{len(test_df):,}")


# ------------------------------------------------
# Shared filenames
# ------------------------------------------------

train_files = set(
    train_df["local_filename"].dropna()
)

test_files = set(
    test_df["local_filename"].dropna()
)

shared_files = (
    train_files.intersection(
        test_files
    )
)

print("\n" + "=" * 70)
print("TRAIN / TEST OVERLAP")
print("=" * 70)

print(
    "\nUnique Train images:",
    len(train_files)
)

print(
    "Unique Test images :",
    len(test_files)
)

print(
    "Shared images      :",
    len(shared_files)
)

if len(shared_files) == 0:

    print(
        "\n✓ NO IMAGE LEAKAGE DETECTED"
    )

else:

    print(
        "\n⚠ IMAGE LEAKAGE DETECTED"
    )

    print(
        "\nShared files:"
    )

    for f in sorted(shared_files):

        print(
            " ",
            f
        )


# ------------------------------------------------
# Duplicate local filenames
# ------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE LOCAL FILENAMES")
print("=" * 70)

filename_counts = (
    matched["local_filename"]
    .value_counts()
)

duplicates = filename_counts[
    filename_counts > 1
]

print(
    "\nNumber of duplicated local filenames:",
    len(duplicates)
)

if len(duplicates) > 0:

    print(
        "\nDuplicated files:"
    )

    display(
        duplicates.to_frame("count")
    )


# ------------------------------------------------
# Same Ground Truth duplicates
# ------------------------------------------------

print("\n" + "=" * 70)
print("GROUND TRUTH DUPLICATION CHECK")
print("=" * 70)

gt_counts = (
    matched["ground_truth"]
    .value_counts()
)

print(
    "\nUnique Ground Truth strings:",
    len(gt_counts)
)

print(
    "Total matched records:",
    len(matched)
)

print(
    "Repeated Ground Truth strings:",
    (gt_counts > 1).sum()
)


# ------------------------------------------------
# Final verdict
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 23 FINAL VERDICT")
print("=" * 70)

if len(shared_files) == 0:

    print(
        "\n✓ TRAIN / TEST IMAGE LEAKAGE = NONE"
    )

else:

    print(
        "\n⚠ TRAIN / TEST IMAGE LEAKAGE EXISTS"
    )

print(
    "\nDataset remains unchanged."
)

PFMS-SERIES — DATA LEAKAGE CHECK

Train samples:
23,054

Test samples:
4,066

TRAIN / TEST OVERLAP

Unique Train images: 23052
Unique Test images : 4066
Shared images      : 1

⚠ IMAGE LEAKAGE DETECTED

Shared files:
  03358.tif

DUPLICATE LOCAL FILENAMES

Number of duplicated local filenames: 3

Duplicated files:


,count
local_filename,
15222.tif,2
03358.tif,2
03363.tif,2



GROUND TRUTH DUPLICATION CHECK

Unique Ground Truth strings: 27065
Total matched records: 27120
Repeated Ground Truth strings: 55

CELL 23 FINAL VERDICT

⚠ TRAIN / TEST IMAGE LEAKAGE EXISTS

Dataset remains unchanged.


In [ ]:
# ================================================================
# CELL 24 — CREATE CLEAN MATCHED DATASET
# ================================================================

import pandas as pd
from pathlib import Path

print("=" * 70)
print("PFMS-SERIES — CREATING CLEAN DATASET")
print("=" * 70)

# ------------------------------------------------
# Load original matching table
# ------------------------------------------------

input_csv = (
    "/content/pfms_series_dataset/"
    "idpl_official_local_matching.csv"
)

df = pd.read_csv(
    input_csv,
    encoding="utf-8-sig"
)

# ------------------------------------------------
# Keep only exact matches
# ------------------------------------------------

clean_df = df[
    df["match_type"].str.startswith("exact")
].copy()

print(
    "\nOriginal exact matched records:",
    len(clean_df)
)


# ------------------------------------------------
# Remove only Train/Test leakage
# ------------------------------------------------

leakage_file = "03358.tif"

train_mask = (
    (clean_df["split"] == "train") &
    (clean_df["local_filename"] == leakage_file)
)

removed_rows = clean_df[
    train_mask
].copy()

print("\nLeakage record found:")
print(len(removed_rows))

if len(removed_rows) > 0:

    print("\nRemoved from TRAIN:")

    display(
        removed_rows[
            [
                "official_index",
                "split",
                "local_filename",
                "ground_truth"
            ]
        ]
    )


clean_df = clean_df[
    ~train_mask
].copy()


# ------------------------------------------------
# Remove duplicated local images inside same split
# ------------------------------------------------

# We keep the first occurrence of each image.
# This does NOT modify the original matching table.

before_dedup = len(clean_df)

clean_df = (
    clean_df
    .drop_duplicates(
        subset=[
            "local_filename"
        ],
        keep="first"
    )
    .copy()
)

after_dedup = len(clean_df)

print(
    "\nRecords removed because of duplicate local images:",
    before_dedup - after_dedup
)


# ------------------------------------------------
# Reset index
# ------------------------------------------------

clean_df = clean_df.reset_index(
    drop=True
)


# ------------------------------------------------
# Verify Train/Test leakage
# ------------------------------------------------

train_files = set(
    clean_df[
        clean_df["split"] == "train"
    ]["local_filename"]
    .dropna()
)

test_files = set(
    clean_df[
        clean_df["split"] == "test"
    ]["local_filename"]
    .dropna()
)

shared_files = (
    train_files.intersection(
        test_files
    )
)


# ------------------------------------------------
# Save clean dataset
# ------------------------------------------------

output_csv = (
    "/content/pfms_series_dataset/"
    "idpl_pfms_series_clean.csv"
)

clean_df.to_csv(
    output_csv,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------
# Final statistics
# ------------------------------------------------

train_clean = clean_df[
    clean_df["split"] == "train"
]

test_clean = clean_df[
    clean_df["split"] == "test"
]

print("\n" + "=" * 70)
print("CLEAN DATASET RESULTS")
print("=" * 70)

print(
    "\nTotal clean samples:",
    f"{len(clean_df):,}"
)

print(
    "\nClean Train:",
    f"{len(train_clean):,}"
)

print(
    "Clean Test :",
    f"{len(test_clean):,}"
)

print(
    "\nUnique Train images:",
    f"{train_clean['local_filename'].nunique():,}"
)

print(
    "Unique Test images :",
    f"{test_clean['local_filename'].nunique():,}"
)

print(
    "\nShared Train/Test images:",
    len(shared_files)
)

if len(shared_files) == 0:

    print(
        "\n✓ DATA LEAKAGE SUCCESSFULLY REMOVED"
    )

else:

    print(
        "\n⚠ DATA LEAKAGE STILL EXISTS"
    )

    print(
        "\nShared files:"
    )

    for f in sorted(shared_files):

        print(" ", f)


# ------------------------------------------------
# Check Ground Truth
# ------------------------------------------------

empty_gt = (
    clean_df["ground_truth"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print(
    "\nEmpty Ground Truth:",
    empty_gt
)


# ------------------------------------------------
# Save split files separately
# ------------------------------------------------

train_csv = (
    "/content/pfms_series_dataset/"
    "train_clean.csv"
)

test_csv = (
    "/content/pfms_series_dataset/"
    "test_clean.csv"
)

train_clean.to_csv(
    train_csv,
    index=False,
    encoding="utf-8-sig"
)

test_clean.to_csv(
    test_csv,
    index=False,
    encoding="utf-8-sig"
)


print("\nSaved files:")

print(
    "1.",
    output_csv
)

print(
    "2.",
    train_csv
)

print(
    "3.",
    test_csv
)

print("\n" + "=" * 70)
print("CELL 24 COMPLETED")
print("=" * 70)

PFMS-SERIES — CREATING CLEAN DATASET

Original exact matched records: 27120

Leakage record found:
1

Removed from TRAIN:


,official_index,split,local_filename,ground_truth
4158,4158,train,03358.tif,برای ضبط مکالمات با ساده ترین حالت ممکن از است...



Records removed because of duplicate local images: 2

CLEAN DATASET RESULTS

Total clean samples: 27,117

Clean Train: 23,051
Clean Test : 4,066

Unique Train images: 23,051
Unique Test images : 4,066

Shared Train/Test images: 0

✓ DATA LEAKAGE SUCCESSFULLY REMOVED

Empty Ground Truth: 0

Saved files:
1. /content/pfms_series_dataset/idpl_pfms_series_clean.csv
2. /content/pfms_series_dataset/train_clean.csv
3. /content/pfms_series_dataset/test_clean.csv

CELL 24 COMPLETED


In [ ]:
# ======================================================================
# CELL 25 — PFMS-SERIES FINAL DATASET INTEGRITY CHECK
# ======================================================================

import os
import random
import hashlib
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter

print("=" * 70)
print("PFMS-SERIES — FINAL DATASET INTEGRITY CHECK")
print("=" * 70)

# ----------------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------------

DATASET_DIR = "/content/pfms_series_dataset"
LOCAL_IMAGE_DIR = "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"

CLEAN_CSV = os.path.join(
    DATASET_DIR,
    "idpl_pfms_series_clean.csv"
)

TRAIN_CSV = os.path.join(
    DATASET_DIR,
    "train_clean.csv"
)

TEST_CSV = os.path.join(
    DATASET_DIR,
    "test_clean.csv"
)

# ----------------------------------------------------------------------
# REPRODUCIBILITY
# ----------------------------------------------------------------------

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)

print("\nReproducibility seed:", SEED)

# ----------------------------------------------------------------------
# LOAD DATA
# ----------------------------------------------------------------------

df = pd.read_csv(
    CLEAN_CSV,
    encoding="utf-8-sig"
)

train_df = pd.read_csv(
    TRAIN_CSV,
    encoding="utf-8-sig"
)

test_df = pd.read_csv(
    TEST_CSV,
    encoding="utf-8-sig"
)

print("\nFiles loaded successfully.")

# ----------------------------------------------------------------------
# BASIC STRUCTURE
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("1. DATASET STRUCTURE")
print("-" * 70)

print("Total records :", f"{len(df):,}")
print("Train records :", f"{len(train_df):,}")
print("Test records  :", f"{len(test_df):,}")

print("\nColumns:")
for col in df.columns:
    print(" -", col)

# ----------------------------------------------------------------------
# SPLIT VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("2. SPLIT VALIDATION")
print("-" * 70)

split_values = sorted(
    df["split"].dropna().astype(str).unique()
)

print("Available splits:", split_values)

print("\nSplit counts:")
print(
    df["split"]
    .value_counts()
    .sort_index()
)

# ----------------------------------------------------------------------
# TRAIN / TEST OVERLAP
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("3. TRAIN / TEST LEAKAGE CHECK")
print("-" * 70)

train_files = set(
    train_df["local_filename"]
    .dropna()
    .astype(str)
)

test_files = set(
    test_df["local_filename"]
    .dropna()
    .astype(str)
)

shared_files = train_files.intersection(
    test_files
)

print("Unique Train images:", f"{len(train_files):,}")
print("Unique Test images :", f"{len(test_files):,}")
print("Shared images      :", len(shared_files))

if len(shared_files) == 0:
    print("\n✓ PASS — No Train/Test image leakage")

else:
    print("\n✗ FAIL — Train/Test leakage detected")

    print("\nShared files:")
    for filename in sorted(shared_files):
        print(" ", filename)

# ----------------------------------------------------------------------
# DUPLICATE CHECK WITHIN SPLITS
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("4. DUPLICATE IMAGE CHECK")
print("-" * 70)

train_duplicate_count = (
    train_df["local_filename"]
    .duplicated()
    .sum()
)

test_duplicate_count = (
    test_df["local_filename"]
    .duplicated()
    .sum()
)

print(
    "Duplicate Train filenames:",
    train_duplicate_count
)

print(
    "Duplicate Test filenames :",
    test_duplicate_count
)

if train_duplicate_count == 0 and test_duplicate_count == 0:
    print("\n✓ PASS — No duplicate image within either split")

else:
    print("\n⚠ Duplicate images detected")

# ----------------------------------------------------------------------
# GROUND TRUTH VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("5. GROUND TRUTH VALIDATION")
print("-" * 70)

df["ground_truth"] = (
    df["ground_truth"]
    .fillna("")
    .astype(str)
)

empty_gt = (
    df["ground_truth"]
    .str.strip()
    .eq("")
    .sum()
)

print("Empty Ground Truth:", empty_gt)

if empty_gt == 0:
    print("✓ PASS — No empty Ground Truth")

else:
    print("✗ FAIL — Empty Ground Truth exists")

# ----------------------------------------------------------------------
# TEXT LENGTH
# ----------------------------------------------------------------------

df["text_length"] = (
    df["ground_truth"]
    .str.replace("\n", "", regex=False)
    .str.len()
)

df["word_count"] = (
    df["ground_truth"]
    .str.replace("\n", " ", regex=False)
    .str.split()
    .str.len()
)

print("\n" + "-" * 70)
print("6. TEXT STATISTICS")
print("-" * 70)

print(
    "Minimum characters :",
    int(df["text_length"].min())
)

print(
    "Maximum characters :",
    int(df["text_length"].max())
)

print(
    "Mean characters    :",
    round(df["text_length"].mean(), 2)
)

print(
    "Median characters  :",
    round(df["text_length"].median(), 2)
)

print(
    "\nMinimum words      :",
    int(df["word_count"].min())
)

print(
    "Maximum words      :",
    int(df["word_count"].max())
)

print(
    "Mean words         :",
    round(df["word_count"].mean(), 2)
)

# ----------------------------------------------------------------------
# LOCAL IMAGE EXISTENCE CHECK
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("7. LOCAL IMAGE EXISTENCE CHECK")
print("-" * 70)

all_filenames = (
    df["local_filename"]
    .dropna()
    .astype(str)
    .unique()
)

missing_images = []

for filename in all_filenames:

    image_path = os.path.join(
        LOCAL_IMAGE_DIR,
        filename
    )

    if not os.path.isfile(image_path):
        missing_images.append(filename)

print(
    "Referenced unique images:",
    f"{len(all_filenames):,}"
)

print(
    "Missing local images:",
    len(missing_images)
)

if len(missing_images) == 0:

    print(
        "\n✓ PASS — All referenced local images exist"
    )

else:

    print(
        "\n✗ FAIL — Missing images detected"
    )

    print("\nFirst missing files:")
    for filename in missing_images[:20]:
        print(" ", filename)

# ----------------------------------------------------------------------
# IMAGE PROPERTY CHECK
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("8. IMAGE PROPERTY CHECK")
print("-" * 70)

sample_size = min(
    100,
    len(all_filenames)
)

sample_files = random.sample(
    list(all_filenames),
    sample_size
)

bad_dimensions = []
bad_modes = []

dimension_counter = Counter()
mode_counter = Counter()

for filename in sample_files:

    image_path = os.path.join(
        LOCAL_IMAGE_DIR,
        filename
    )

    try:

        with Image.open(image_path) as img:

            dimension_counter[
                img.size
            ] += 1

            mode_counter[
                img.mode
            ] += 1

            if img.size != (700, 50):

                bad_dimensions.append(
                    (
                        filename,
                        img.size
                    )
                )

            if img.mode not in [
                "RGB",
                "L"
            ]:

                bad_modes.append(
                    (
                        filename,
                        img.mode
                    )
                )

    except Exception as e:

        bad_dimensions.append(
            (
                filename,
                f"READ ERROR: {e}"
            )
        )

print(
    "Checked sample images:",
    sample_size
)

print("\nDimensions:")
for dimension, count in dimension_counter.items():
    print(
        f"  {dimension}: {count}"
    )

print("\nModes:")
for mode, count in mode_counter.items():
    print(
        f"  {mode}: {count}"
    )

print(
    "\nBad dimensions:",
    len(bad_dimensions)
)

print(
    "Bad modes:",
    len(bad_modes)
)

if len(bad_dimensions) == 0:
    print(
        "✓ PASS — All sampled images are 700×50"
    )
else:
    print(
        "⚠ Dimension problem detected"
    )

# ----------------------------------------------------------------------
# CHARACTER VOCABULARY
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("9. CHARACTER VOCABULARY")
print("-" * 70)

character_counter = Counter()

for text in df["ground_truth"]:

    character_counter.update(text)

print(
    "Unique characters including whitespace:",
    len(character_counter)
)

print("\nMost frequent characters:")

for char, count in character_counter.most_common(20):

    display_char = char

    if char == " ":
        display_char = "[SPACE]"

    elif char == "\n":
        display_char = "[NEWLINE]"

    elif char == "\u200c":
        display_char = "[ZWNJ]"

    print(
        f"{display_char!r:15s} : {count:,}"
    )

# ----------------------------------------------------------------------
# RANDOM SAMPLE INSPECTION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("10. RANDOM IMAGE / TEXT PAIR INSPECTION")
print("-" * 70)

random_indices = random.sample(
    range(len(df)),
    min(5, len(df))
)

for i, idx in enumerate(random_indices, 1):

    row = df.iloc[idx]

    filename = str(
        row["local_filename"]
    )

    gt = str(
        row["ground_truth"]
    )

    image_path = os.path.join(
        LOCAL_IMAGE_DIR,
        filename
    )

    try:

        with Image.open(image_path) as img:

            print(f"\nSample {i}")
            print("Split    :", row["split"])
            print("File     :", filename)
            print("Size     :", img.size)
            print("Mode     :", img.mode)
            print("GT       :", repr(gt[:150]))

    except Exception as e:

        print(
            f"\nSample {i} — ERROR: {e}"
        )

# ----------------------------------------------------------------------
# DATASET HASH
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("11. DATASET REPRODUCIBILITY HASH")
print("-" * 70)

hash_columns = [
    "split",
    "official_index",
    "local_filename",
    "ground_truth"
]

hash_df = df[
    hash_columns
].copy()

hash_df = hash_df.sort_values(
    by=[
        "split",
        "official_index"
    ]
)

csv_bytes = hash_df.to_csv(
    index=False,
    encoding="utf-8"
).encode("utf-8")

dataset_hash = hashlib.sha256(
    csv_bytes
).hexdigest()

print("SHA-256:")
print(dataset_hash)

# ----------------------------------------------------------------------
# FINAL VERDICT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 25 — FINAL VERDICT")
print("=" * 70)

checks = {
    "Train/Test leakage": len(shared_files) == 0,
    "Train duplicates": train_duplicate_count == 0,
    "Test duplicates": test_duplicate_count == 0,
    "Empty Ground Truth": empty_gt == 0,
    "Missing images": len(missing_images) == 0,
    "Bad dimensions": len(bad_dimensions) == 0,
}

print()

for name, passed in checks.items():

    if passed:
        print(f"✓ PASS — {name}")
    else:
        print(f"✗ FAIL — {name}")

all_passed = all(checks.values())

print("\n" + "-" * 70)

if all_passed:

    print(
        "✓ DATASET INTEGRITY CHECK PASSED"
    )

    print(
        "✓ DATASET IS READY FOR PFMS-SERIES EXPERIMENTS"
    )

else:

    print(
        "⚠ DATASET INTEGRITY CHECK REQUIRES REVIEW"
    )

print("-" * 70)

print("\nFinal dataset:")
print(
    "  Train:",
    f"{len(train_df):,}"
)
print(
    "  Test :",
    f"{len(test_df):,}"
)
print(
    "  Total:",
    f"{len(df):,}"
)

print("\nDataset hash:")
print(dataset_hash)

print("\n" + "=" * 70)
print("CELL 25 COMPLETED")
print("=" * 70)

PFMS-SERIES — FINAL DATASET INTEGRITY CHECK

Reproducibility seed: 2026

Files loaded successfully.

----------------------------------------------------------------------
1. DATASET STRUCTURE
----------------------------------------------------------------------
Total records : 27,117
Train records : 23,051
Test records  : 4,066

Columns:
 - official_index
 - split
 - local_filename
 - ground_truth
 - match_type
 - image_hash

----------------------------------------------------------------------
2. SPLIT VALIDATION
----------------------------------------------------------------------
Available splits: ['test', 'train']

Split counts:
split
test      4066
train    23051
Name: count, dtype: int64

----------------------------------------------------------------------
3. TRAIN / TEST LEAKAGE CHECK
----------------------------------------------------------------------
Unique Train images: 23,051
Unique Test images : 4,066
Shared images      : 0

✓ PASS — No Train/Test image leakage

---

In [ ]:
# ======================================================================
# CELL 26 — PFMS-SERIES DATA LOADER + BATCH VERIFICATION
# ======================================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("=" * 70)
print("PFMS-SERIES — DATALOADER + BATCH VERIFICATION")
print("=" * 70)

# ----------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ----------------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------------

DATASET_DIR = "/content/pfms_series_dataset"

TRAIN_CSV = os.path.join(
    DATASET_DIR,
    "train_clean.csv"
)

TEST_CSV = os.path.join(
    DATASET_DIR,
    "test_clean.csv"
)

IMAGE_DIR = (
    "/content/drive/Othercomputers/"
    "My Laptop/Desktop/idplimgl"
)

# ----------------------------------------------------------------------
# LOAD CSV
# ----------------------------------------------------------------------

train_df = pd.read_csv(
    TRAIN_CSV,
    encoding="utf-8-sig"
)

test_df = pd.read_csv(
    TEST_CSV,
    encoding="utf-8-sig"
)

print("\nDataset:")
print("Train:", f"{len(train_df):,}")
print("Test :", f"{len(test_df):,}")

# ----------------------------------------------------------------------
# IMAGE TRANSFORM
# ----------------------------------------------------------------------

# Important:
# Original image size = 700 × 50
# No geometric resizing is performed.
#
# PIL size convention:
# (width, height) = (700, 50)
#
# Tensor convention:
# (channels, height, width) = (3, 50, 700)

transform = transforms.Compose([
    transforms.Lambda(
        lambda img: img.convert("RGB")
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

# ----------------------------------------------------------------------
# DATASET CLASS
# ----------------------------------------------------------------------

class IDPLSeriesDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dir,
        transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        filename = str(
            row["local_filename"]
        )

        image_path = os.path.join(
            self.image_dir,
            filename
        )

        image = Image.open(
            image_path
        )

        image = image.convert(
            "RGB"
        )

        original_size = image.size

        if self.transform is not None:

            image_tensor = self.transform(
                image
            )

        else:

            image_tensor = transforms.ToTensor()(
                image
            )

        return {
            "image": image_tensor,
            "text": str(
                row["ground_truth"]
            ),
            "filename": filename,
            "original_size": original_size
        }


# ----------------------------------------------------------------------
# CREATE DATASETS
# ----------------------------------------------------------------------

train_dataset = IDPLSeriesDataset(
    dataframe=train_df,
    image_dir=IMAGE_DIR,
    transform=transform
)

test_dataset = IDPLSeriesDataset(
    dataframe=test_df,
    image_dir=IMAGE_DIR,
    transform=transform
)

print("\nDataset objects created.")

print(
    "Train Dataset length:",
    len(train_dataset)
)

print(
    "Test Dataset length:",
    len(test_dataset)
)

# ----------------------------------------------------------------------
# DATALOADER CONFIGURATION
# ----------------------------------------------------------------------

BATCH_SIZE = 8

NUM_WORKERS = 2

PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)

print("\nDataLoaders created.")

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

# ----------------------------------------------------------------------
# SINGLE SAMPLE TEST
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("SINGLE SAMPLE VERIFICATION")
print("-" * 70)

sample = train_dataset[0]

print(
    "\nFilename:",
    sample["filename"]
)

print(
    "Original PIL size:",
    sample["original_size"]
)

print(
    "Tensor shape:",
    tuple(sample["image"].shape)
)

print(
    "Tensor dtype:",
    sample["image"].dtype
)

print(
    "Ground Truth:",
    repr(sample["text"][:150])
)

# ----------------------------------------------------------------------
# BATCH TEST
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("BATCH VERIFICATION")
print("-" * 70)

batch = next(
    iter(train_loader)
)

images = batch["image"]

texts = batch["text"]

filenames = batch["filename"]

print(
    "\nBatch image shape:",
    tuple(images.shape)
)

print(
    "Batch dtype:",
    images.dtype
)

print(
    "Batch device:",
    images.device
)

print(
    "Number of texts:",
    len(texts)
)

print(
    "Number of filenames:",
    len(filenames)
)

# ----------------------------------------------------------------------
# EXPECTED SHAPE
# ----------------------------------------------------------------------

expected_shape = (
    BATCH_SIZE,
    3,
    50,
    700
)

print("\nExpected batch shape:")
print(expected_shape)

print(
    "\nActual batch shape:"
)

print(
    tuple(images.shape)
)

if tuple(images.shape) == expected_shape:

    print(
        "\n✓ PASS — Input tensor shape is correct"
    )

else:

    print(
        "\n✗ FAIL — Unexpected input tensor shape"
    )

# ----------------------------------------------------------------------
# NUMERICAL VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("NUMERICAL VALIDATION")
print("-" * 70)

print(
    "Minimum value:",
    images.min().item()
)

print(
    "Maximum value:",
    images.max().item()
)

print(
    "Mean:",
    images.mean().item()
)

print(
    "Std:",
    images.std().item()
)

if torch.isnan(images).any():

    print(
        "\n✗ FAIL — NaN detected"
    )

else:

    print(
        "\n✓ PASS — No NaN values"
    )

if torch.isinf(images).any():

    print(
        "✗ FAIL — Inf detected"
    )

else:

    print(
        "✓ PASS — No Inf values"
    )

# ----------------------------------------------------------------------
# CHECK RGB
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("RGB CONVERSION CHECK")
print("-" * 70)

if images.shape[1] == 3:

    print(
        "✓ PASS — All images are represented as 3-channel RGB"
    )

else:

    print(
        "✗ FAIL — Channel count is not 3"
    )

# ----------------------------------------------------------------------
# PRINT FIRST BATCH SAMPLES
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("FIRST BATCH SAMPLES")
print("-" * 70)

for i in range(
    min(5, len(filenames))
):

    print(
        f"\n{i+1}. {filenames[i]}"
    )

    print(
        "   GT:",
        repr(
            texts[i][:120]
        )
    )

# ----------------------------------------------------------------------
# FINAL VERDICT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 26 — FINAL VERDICT")
print("=" * 70)

checks = {

    "Train Dataset":
        len(train_dataset) == 23051,

    "Test Dataset":
        len(test_dataset) == 4066,

    "Input shape":
        tuple(images.shape) == expected_shape,

    "RGB channels":
        images.shape[1] == 3,

    "No NaN":
        not torch.isnan(images).any(),

    "No Inf":
        not torch.isinf(images).any(),

}

for name, result in checks.items():

    if result:

        print(
            f"✓ PASS — {name}"
        )

    else:

        print(
            f"✗ FAIL — {name}"
        )

if all(checks.values()):

    print(
        "\n✓ DATA LOADER VERIFIED"
    )

    print(
        "✓ INPUT IS READY FOR SWIN / MAMBA ENCODERS"
    )

else:

    print(
        "\n⚠ DATA LOADER REQUIRES REVIEW"
    )

print("\n" + "=" * 70)
print("CELL 26 COMPLETED")
print("=" * 70)

PFMS-SERIES — DATALOADER + BATCH VERIFICATION

Device: cuda
GPU: Tesla T4

Dataset:
Train: 23,051
Test : 4,066

Dataset objects created.
Train Dataset length: 23051
Test Dataset length: 4066

DataLoaders created.
Batch size: 8
Train batches: 2882
Test batches: 509

----------------------------------------------------------------------
SINGLE SAMPLE VERIFICATION
----------------------------------------------------------------------

Filename: 00490.tif
Original PIL size: (700, 50)
Tensor shape: (3, 50, 700)
Tensor dtype: torch.float32
Ground Truth: 'یابد و كار جمهوری اسلامی را یكسره كند. كافی است به تبلیغات «اكس»گونه یك ماه\n'

----------------------------------------------------------------------
BATCH VERIFICATION
----------------------------------------------------------------------

Batch image shape: (8, 3, 50, 700)
Batch dtype: torch.float32
Batch device: cpu
Number of texts: 8
Number of filenames: 8

Expected batch shape:
(8, 3, 50, 700)

Actual batch shape:
(8, 3, 50, 700)

✓ PA

In [ ]:
# ======================================================================
# PFMS-SERIES — CELL 27
# EXACT TRAIN LOADER STRUCTURE INSPECTION
# ======================================================================

import torch

print("=" * 70)
print("PFMS-SERIES — EXACT TRAIN LOADER STRUCTURE")
print("=" * 70)

batch = next(iter(train_loader))

print("\nBatch type:")
print(type(batch))

if isinstance(batch, dict):

    print("\nDictionary keys:")
    for key, value in batch.items():

        print("\n" + "-" * 70)
        print("KEY:", key)
        print("TYPE:", type(value))

        if torch.is_tensor(value):
            print("SHAPE:", tuple(value.shape))
            print("DTYPE:", value.dtype)
            print("DEVICE:", value.device)

        elif isinstance(value, (list, tuple)):
            print("LENGTH:", len(value))

            if len(value) > 0:
                print("FIRST ITEM TYPE:", type(value[0]))

                if torch.is_tensor(value[0]):
                    print(
                        "FIRST ITEM SHAPE:",
                        tuple(value[0].shape)
                    )
                else:
                    print(
                        "FIRST ITEM:",
                        repr(value[0])[:300]
                    )

        else:
            print("VALUE:", repr(value)[:300])

elif isinstance(batch, (list, tuple)):

    print("\nBatch length:", len(batch))

    for i, value in enumerate(batch):

        print("\n" + "-" * 70)
        print("ITEM", i)
        print("TYPE:", type(value))

        if torch.is_tensor(value):
            print("SHAPE:", tuple(value.shape))
            print("DTYPE:", value.dtype)
            print("DEVICE:", value.device)

        elif isinstance(value, (list, tuple)):
            print("LENGTH:", len(value))

            if len(value) > 0:
                print("FIRST ITEM:", repr(value[0])[:300])

        else:
            print("VALUE:", repr(value)[:300])

else:

    print("\nUnexpected batch structure.")
    print(repr(batch)[:1000])

print("\n" + "=" * 70)
print("CELL 27 COMPLETED")
print("=" * 70)

PFMS-SERIES — EXACT TRAIN LOADER STRUCTURE

Batch type:
<class 'dict'>

Dictionary keys:

----------------------------------------------------------------------
KEY: image
TYPE: <class 'torch.Tensor'>
SHAPE: (8, 3, 50, 700)
DTYPE: torch.float32
DEVICE: cpu

----------------------------------------------------------------------
KEY: text
TYPE: <class 'list'>
LENGTH: 8
FIRST ITEM TYPE: <class 'str'>
FIRST ITEM: 'صحنه های بین الملی حضور آنان در میادین است و همین حضور است كه می\n'

----------------------------------------------------------------------
KEY: filename
TYPE: <class 'list'>
LENGTH: 8
FIRST ITEM TYPE: <class 'str'>
FIRST ITEM: '06666.tif'

----------------------------------------------------------------------
KEY: original_size
TYPE: <class 'list'>
LENGTH: 2
FIRST ITEM TYPE: <class 'torch.Tensor'>
FIRST ITEM SHAPE: (8,)

CELL 27 COMPLETED


In [ ]:
# ======================================================================
# PFMS-SERIES — CELL 28 FINAL FIX
# SWIN-B PATCH EMBEDDING TEST FOR RECTANGULAR OCR
# ======================================================================

import os
import torch
import torch.nn.functional as F
import timm

print("=" * 70)
print("PFMS-SERIES — SWIN-B PATCH EMBEDDING TEST")
print("=" * 70)

# ----------------------------------------------------------------------
# DEVICE
# ----------------------------------------------------------------------

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ----------------------------------------------------------------------
# LOAD REAL BATCH
# ----------------------------------------------------------------------

batch = next(iter(train_loader))

images = batch["image"]
texts = batch["text"]
filenames = batch["filename"]

print("\n" + "-" * 70)
print("REAL IDPL-PFOD DATA")
print("-" * 70)

print("Image shape :", tuple(images.shape))
print("Text count  :", len(texts))
print("File count  :", len(filenames))

assert images.ndim == 4
assert images.shape[1:] == (3, 50, 700)

print("✓ PASS — Real OCR batch confirmed")

# ----------------------------------------------------------------------
# MOVE INPUT TO GPU
# ----------------------------------------------------------------------

images = images.to(
    device=device,
    dtype=torch.float32,
    non_blocking=True
)

print("\nInput device:", images.device)
print("Input dtype :", images.dtype)

# ----------------------------------------------------------------------
# CREATE SWIN-B
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("CREATING SWIN-B")
print("-" * 70)

swin = timm.create_model(
    "swin_base_patch4_window7_224",
    pretrained=True,
    num_classes=0,
    global_pool=""
)

# Move complete model
swin = swin.to(device)

swin.eval()

print("✓ Swin-B moved to:", device)

# ----------------------------------------------------------------------
# DEVICE VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("DEVICE VALIDATION")
print("-" * 70)

model_device = next(
    swin.parameters()
).device

model_dtype = next(
    swin.parameters()
).dtype

input_device = images.device
input_dtype = images.dtype

print("Model device :", model_device)
print("Input device :", input_device)

print("Model dtype  :", model_dtype)
print("Input dtype  :", input_dtype)

# Compare type AND CUDA index safely
if model_device.type == input_device.type:

    if model_device.type == "cuda":

        assert (
            model_device.index == input_device.index
        )

    print("✓ PASS — Model and input are on same device")

else:

    raise RuntimeError(
        f"Device mismatch: "
        f"model={model_device}, "
        f"input={input_device}"
    )

assert model_dtype == input_dtype

print("✓ PASS — Model and input dtypes match")

# ----------------------------------------------------------------------
# PATCH EMBEDDING CONFIGURATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("PATCH EMBEDDING CONFIGURATION")
print("-" * 70)

print(
    "Patch size:",
    swin.patch_embed.patch_size
)

print(
    "Original model image size:",
    swin.patch_embed.img_size
)

swin.patch_embed.strict_img_size = False
swin.patch_embed.dynamic_img_pad = True

print(
    "strict_img_size:",
    swin.patch_embed.strict_img_size
)

print(
    "dynamic_img_pad:",
    swin.patch_embed.dynamic_img_pad
)

# ----------------------------------------------------------------------
# PAD OCR IMAGE
# ----------------------------------------------------------------------
#
# Original:
# 50 × 700
#
# Padded:
# 56 × 700
#
# Patch:
# 4 × 4
#
# Grid:
# 14 × 175
#
# ----------------------------------------------------------------------

images_swin = F.pad(
    images,
    (0, 0, 3, 3),
    mode="constant",
    value=0
)

print("\n" + "-" * 70)
print("RECTANGULAR OCR INPUT")
print("-" * 70)

print(
    "Original:",
    tuple(images.shape)
)

print(
    "Padded  :",
    tuple(images_swin.shape)
)

assert images_swin.shape == (
    8, 3, 56, 700
)

print("✓ PASS — 56×700 input prepared")

# ----------------------------------------------------------------------
# PATCH WEIGHT VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("PATCH PROJECTION DEVICE CHECK")
print("-" * 70)

patch_weight = (
    swin.patch_embed.proj.weight
)

print(
    "Patch weight device:",
    patch_weight.device
)

print(
    "Patch weight dtype :",
    patch_weight.dtype
)

assert patch_weight.device.type == device.type

if device.type == "cuda":
    assert (
        patch_weight.device.index
        == device.index
    )

assert patch_weight.dtype == images.dtype

print(
    "✓ PASS — Patch projection is correctly placed"
)

# ----------------------------------------------------------------------
# PATCH EMBEDDING
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("RUNNING PATCH EMBEDDING")
print("-" * 70)

with torch.no_grad():

    patch_features = (
        swin.patch_embed(images_swin)
    )

print("\n✓ PATCH EMBEDDING SUCCESS")

print(
    "Output shape :",
    tuple(patch_features.shape)
)

print(
    "Output dtype :",
    patch_features.dtype
)

print(
    "Output device:",
    patch_features.device
)

# ----------------------------------------------------------------------
# PATCH GRID VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("PATCH GRID")
print("-" * 70)

assert patch_features.ndim == 4

B, H, W, C = patch_features.shape

print("Batch       :", B)
print("Height grid :", H)
print("Width grid  :", W)
print("Channels    :", C)

print(
    "Patch tokens:",
    H * W
)

# Expected:
#
# 56 / 4 = 14
# 700 / 4 = 175
#
# 14 × 175 = 2450
#
# Swin-B embedding dimension = 128

assert B == 8
assert H == 14
assert W == 175
assert C == 128

print("\n✓ PASS — Patch grid = 14 × 175")
print("✓ PASS — Number of patch tokens = 2450")
print("✓ PASS — Embedding dimension = 128")

# ----------------------------------------------------------------------
# NUMERICAL VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("NUMERICAL VALIDATION")
print("-" * 70)

min_value = patch_features.min().item()
max_value = patch_features.max().item()
mean_value = patch_features.mean().item()
std_value = patch_features.std().item()

print("Min :", min_value)
print("Max :", max_value)
print("Mean:", mean_value)
print("Std :", std_value)

assert not torch.isnan(
    patch_features
).any()

assert not torch.isinf(
    patch_features
).any()

print("\n✓ PASS — No NaN")
print("✓ PASS — No Inf")

# ----------------------------------------------------------------------
# TOKEN SEQUENCE
# ----------------------------------------------------------------------

tokens = patch_features.reshape(
    B,
    H * W,
    C
)

print("\n" + "-" * 70)
print("INITIAL SWIN TOKEN SEQUENCE")
print("-" * 70)

print(
    "Token shape:",
    tuple(tokens.shape)
)

assert tokens.shape == (
    8,
    2450,
    128
)

print("✓ PASS — Token sequence created")

# ----------------------------------------------------------------------
# SAVE
# ----------------------------------------------------------------------

save_dir = (
    "/content/pfms_series_dataset"
)

os.makedirs(
    save_dir,
    exist_ok=True
)

save_path = os.path.join(
    save_dir,
    "swin_b_patch_test.pt"
)

torch.save(
    {
        "patch_features":
            patch_features.cpu(),

        "tokens":
            tokens.cpu(),

        "filenames":
            filenames,

        "texts":
            texts,

        "input_shape":
            tuple(images.shape),

        "padded_shape":
            tuple(images_swin.shape),

        "patch_grid":
            (H, W),

        "num_tokens":
            H * W,

        "embedding_dim":
            C,

        "model_name":
            "swin_base_patch4_window7_224",

        "dataset":
            "IDPL-PFOD",

        "device":
            str(device),
    },
    save_path
)

print("\nSaved:")
print(save_path)

# ----------------------------------------------------------------------
# FINAL VERDICT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 28 — FINAL VERDICT")
print("=" * 70)

print("✓ Real IDPL-PFOD data")
print("✓ Input = 50×700")
print("✓ Input converted to 56×700")
print("✓ Swin-B loaded")
print("✓ Model/Input device compatible")
print("✓ Model/Input dtype compatible")
print("✓ Patch embedding successful")
print("✓ Patch grid = 14×175")
print("✓ Patch tokens = 2450")
print("✓ Embedding dimension = 128")
print("✓ Token sequence = (8,2450,128)")
print("✓ No NaN")
print("✓ No Inf")
print("✓ Patch features saved")

print("\nCELL 28 COMPLETED")
print("=" * 70)

PFMS-SERIES — SWIN-B PATCH EMBEDDING TEST

Device: cuda:0
GPU: Tesla T4

----------------------------------------------------------------------
REAL IDPL-PFOD DATA
----------------------------------------------------------------------
Image shape : (8, 3, 50, 700)
Text count  : 8
File count  : 8
✓ PASS — Real OCR batch confirmed

Input device: cuda:0
Input dtype : torch.float32

----------------------------------------------------------------------
CREATING SWIN-B
----------------------------------------------------------------------


model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

✓ Swin-B moved to: cuda:0

----------------------------------------------------------------------
DEVICE VALIDATION
----------------------------------------------------------------------
Model device : cuda:0
Input device : cuda:0
Model dtype  : torch.float32
Input dtype  : torch.float32
✓ PASS — Model and input are on same device
✓ PASS — Model and input dtypes match

----------------------------------------------------------------------
PATCH EMBEDDING CONFIGURATION
----------------------------------------------------------------------
Patch size: (4, 4)
Original model image size: (224, 224)
strict_img_size: False
dynamic_img_pad: True

----------------------------------------------------------------------
RECTANGULAR OCR INPUT
----------------------------------------------------------------------
Original: (8, 3, 50, 700)
Padded  : (8, 3, 56, 700)
✓ PASS — 56×700 input prepared

----------------------------------------------------------------------
PATCH PROJECTION DEVICE CHECK
----

In [ ]:
# ======================================================================
# PFMS-SERIES — CELL 29-A
# EXACT TIMM SWIN-B STAGE STRUCTURE DIAGNOSTIC
# ======================================================================

import torch
import timm

print("=" * 78)
print("PFMS-SERIES — CELL 29-A")
print("EXACT SWIN-B INTERNAL STAGE STRUCTURE")
print("=" * 78)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)

# ----------------------------------------------------------------------
# CREATE MODEL
# ----------------------------------------------------------------------

swin = timm.create_model(
    "swin_base_patch4_window7_224",
    pretrained=True,
    num_classes=0,
    global_pool=""
)

swin = swin.to(device)
swin.eval()

print("\n✓ Swin-B loaded")

# ----------------------------------------------------------------------
# MODEL LAYERS
# ----------------------------------------------------------------------

print("\n" + "-" * 78)
print("SWIN LAYER STRUCTURE")
print("-" * 78)

print("Number of layers:", len(swin.layers))

for i, stage in enumerate(swin.layers, start=1):

    print("\n" + "=" * 60)
    print(f"STAGE {i}")
    print("=" * 60)

    print("Stage type:")
    print(type(stage))

    print("\nBlocks:")
    print("Number of blocks:", len(stage.blocks))

    for j, block in enumerate(stage.blocks, start=1):

        print(
            f"  Block {j}:",
            type(block).__name__
        )

        print(
            "     norm1:",
            tuple(block.norm1.normalized_shape)
        )

        print(
            "     window:",
            block.window_size
        )

        print(
            "     heads:",
            block.attn.num_heads
        )

        if hasattr(block, "attn_mask"):
            print(
                "     attn_mask:",
                None
                if block.attn_mask is None
                else tuple(block.attn_mask.shape)
            )

    # ------------------------------------------------------------------
    # DOWNSAMPLE
    # ------------------------------------------------------------------

    print("\nDownsample:")

    if stage.downsample is None:

        print("  None")

    else:

        print(
            "  Type:",
            type(stage.downsample)
        )

        print(
            "  Class:",
            type(stage.downsample).__name__
        )

        print(
            "  Module:",
            stage.downsample
        )

        # Inspect common attributes

        for attr in [
            "dim",
            "input_resolution",
            "reduction",
            "norm"
        ]:

            if hasattr(stage.downsample, attr):

                value = getattr(
                    stage.downsample,
                    attr
                )

                print(
                    f"  {attr}:",
                    value
                )

# ----------------------------------------------------------------------
# PATCH EMBEDDING
# ----------------------------------------------------------------------

print("\n" + "-" * 78)
print("PATCH EMBEDDING")
print("-" * 78)

print(
    "Patch embedding:",
    swin.patch_embed
)

print(
    "Patch size:",
    swin.patch_embed.patch_size
)

# ----------------------------------------------------------------------
# MODEL FEATURE INFORMATION
# ----------------------------------------------------------------------

print("\n" + "-" * 78)
print("MODEL FEATURE INFORMATION")
print("-" * 78)

for attr in [
    "num_features",
    "feature_info"
]:

    if hasattr(swin, attr):

        print(
            f"\n{attr}:"
        )

        print(
            getattr(swin, attr)
        )

# ----------------------------------------------------------------------
# FORWARD_FEATURES SOURCE-RELEVANT OBJECTS
# ----------------------------------------------------------------------

print("\n" + "-" * 78)
print("FORWARD FEATURES OBJECTS")
print("-" * 78)

print(
    "swin.norm:",
    swin.norm
)

print(
    "\nswin.layers:",
    swin.layers
)

print("\n" + "=" * 78)
print("CELL 29-A COMPLETED")
print("=" * 78)

PFMS-SERIES — CELL 29-A
EXACT SWIN-B INTERNAL STAGE STRUCTURE

Device: cuda:0

✓ Swin-B loaded

------------------------------------------------------------------------------
SWIN LAYER STRUCTURE
------------------------------------------------------------------------------
Number of layers: 4

STAGE 1
Stage type:
<class 'timm.models.swin_transformer.SwinTransformerStage'>

Blocks:
Number of blocks: 2
  Block 1: SwinTransformerBlock
     norm1: (128,)
     window: (7, 7)
     heads: 4
     attn_mask: None
  Block 2: SwinTransformerBlock
     norm1: (128,)
     window: (7, 7)
     heads: 4
     attn_mask: (64, 49, 49)

Downsample:
  Type: <class 'torch.nn.modules.linear.Identity'>
  Class: Identity
  Module: Identity()

STAGE 2
Stage type:
<class 'timm.models.swin_transformer.SwinTransformerStage'>

Blocks:
Number of blocks: 2
  Block 1: SwinTransformerBlock
     norm1: (256,)
     window: (7, 7)
     heads: 8
     attn_mask: None
  Block 2: SwinTransformerBlock
     norm1: (256,)
     

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 29-E
# REAL IDPL-PFOD BATCH BUILDER FOR SWIN-B
# ==============================================================================

import os
import glob
import torch
from PIL import Image
from torchvision import transforms

print("=" * 78)
print("PFMS-SERIES — CELL 29-E")
print("REAL IDPL-PFOD BATCH BUILDER")
print("=" * 78)

# ------------------------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print()
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------------------------
# DATASET PATH
# ------------------------------------------------------------------------------

DATASET_PATH = "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"

print()
print("-" * 78)
print("DATASET")
print("-" * 78)

print("Dataset path:")
print(DATASET_PATH)

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset path does not exist:\n{DATASET_PATH}"
    )

print("✓ Dataset path exists")

# ------------------------------------------------------------------------------
# FIND IMAGE FILES
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("SEARCHING IDPL-PFOD IMAGES")
print("-" * 78)

extensions = [
    "*.tif",
    "*.tiff",
    "*.png",
    "*.jpg",
    "*.jpeg",
    "*.bmp"
]

image_files = []

for ext in extensions:
    image_files.extend(
        glob.glob(
            os.path.join(DATASET_PATH, "**", ext),
            recursive=True
        )
    )

# Remove duplicates and sort
image_files = sorted(list(set(image_files)))

print("Total image files found:", len(image_files))

if len(image_files) == 0:
    raise RuntimeError(
        "No image files were found in the IDPL-PFOD directory."
    )

print("✓ Images found")

# ------------------------------------------------------------------------------
# SHOW FIRST FEW FILES
# ------------------------------------------------------------------------------

print()
print("First available images:")

for i, path in enumerate(image_files[:10]):
    print(f"{i+1:02d}:", path)

# ------------------------------------------------------------------------------
# IMAGE TRANSFORMATION
# ------------------------------------------------------------------------------
#
# OCR input required by PFMS-SERIES:
#
# [B, 3, 50, 700]
#
# We DO NOT use 224x224 here.
#
# The rectangular image is preserved.
# ------------------------------------------------------------------------------

transform = transforms.Compose([
    transforms.Resize((50, 700)),
    transforms.ToTensor(),
])

# ------------------------------------------------------------------------------
# LOAD 8 REAL IMAGES
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("LOADING REAL IDPL-PFOD BATCH")
print("-" * 78)

TARGET_BATCH_SIZE = 8

batch_images = []
batch_files = []

for path in image_files:

    try:

        img = Image.open(path).convert("RGB")

        tensor = transform(img)

        if tensor.shape != (3, 50, 700):
            print(
                "Skipping unexpected shape:",
                path,
                tuple(tensor.shape)
            )
            continue

        batch_images.append(tensor)
        batch_files.append(path)

        if len(batch_images) == TARGET_BATCH_SIZE:
            break

    except Exception as e:

        print(
            "Skipping unreadable image:",
            path,
            "|",
            type(e).__name__,
            "|",
            str(e)
        )

# ------------------------------------------------------------------------------
# VALIDATE BATCH
# ------------------------------------------------------------------------------

if len(batch_images) < TARGET_BATCH_SIZE:

    raise RuntimeError(
        f"Only {len(batch_images)} valid images were loaded. "
        f"Expected {TARGET_BATCH_SIZE}."
    )

images = torch.stack(batch_images, dim=0)

# Move to GPU
images = images.to(
    device=device,
    dtype=torch.float32
)

# ------------------------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("REAL IDPL-PFOD BATCH READY")
print("=" * 78)

print()
print("Image tensor shape :", tuple(images.shape))
print("Device             :", images.device)
print("Dtype              :", images.dtype)

print()
print("Batch size :", images.shape[0])
print("Channels   :", images.shape[1])
print("Height     :", images.shape[2])
print("Width      :", images.shape[3])

# ------------------------------------------------------------------------------
# ASSERTIONS
# ------------------------------------------------------------------------------

assert images.ndim == 4
assert images.shape[0] == 8
assert images.shape[1] == 3
assert images.shape[2] == 50
assert images.shape[3] == 700

print()
print("✓ PASS — Real OCR input = (8,3,50,700)")

# ------------------------------------------------------------------------------
# FILE VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("BATCH FILES")
print("-" * 78)

for i, path in enumerate(batch_files):
    print(f"{i+1:02d}: {os.path.basename(path)}")

print()
print("✓ PASS — 8 real IDPL-PFOD images loaded")

# ------------------------------------------------------------------------------
# SAVE BATCH FOR NEXT CELL
# ------------------------------------------------------------------------------

batch_save_path = "/content/pfms_series_real_batch.pt"

torch.save(
    {
        "images": images.detach().cpu(),
        "files": batch_files
    },
    batch_save_path
)

print()
print("Saved batch:")
print(batch_save_path)

print()
print("=" * 78)
print("CELL 29-E COMPLETED")
print("=" * 78)

PFMS-SERIES — CELL 29-E
REAL IDPL-PFOD BATCH BUILDER

Device: cuda
GPU: Tesla T4

------------------------------------------------------------------------------
DATASET
------------------------------------------------------------------------------
Dataset path:
/content/drive/Othercomputers/My Laptop/Desktop/idplimgl
✓ Dataset path exists

------------------------------------------------------------------------------
SEARCHING IDPL-PFOD IMAGES
------------------------------------------------------------------------------
Total image files found: 27178
✓ Images found

First available images:
01: /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001 (1).tif
02: /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001.tif
03: /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00002 (1).tif
04: /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00002.tif
05: /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00003.tif
06: /content/drive/Othercomputers/My La

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 29-F
# SWIN-B RECTANGULAR OCR FORWARD
# ==============================================================================

import torch
import torch.nn as nn
import timm

print("=" * 78)
print("PFMS-SERIES — CELL 29-F")
print("SWIN-B RECTANGULAR OCR FORWARD")
print("=" * 78)

# ==============================================================================
# 1. DEVICE
# ==============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print()
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ==============================================================================
# 2. CHECK REAL BATCH FROM CELL 29-E
# ==============================================================================

print()
print("-" * 78)
print("CHECKING REAL IDPL-PFOD BATCH")
print("-" * 78)

if "images" not in globals():

    raise RuntimeError(
        "Variable 'images' was not found.\n"
        "Please run CELL 29-E first."
    )

images = images.to(
    device=device,
    dtype=torch.float32
)

print("Image shape :", tuple(images.shape))
print("Device      :", images.device)
print("Dtype       :", images.dtype)

assert images.ndim == 4
assert images.shape[1] == 3
assert images.shape[2] == 50
assert images.shape[3] == 700

print("✓ PASS — Input = (B,3,50,700)")

# ==============================================================================
# 3. CREATE PRETRAINED SWIN-B
# ==============================================================================

print()
print("-" * 78)
print("CREATING PRETRAINED SWIN-B")
print("-" * 78)

swin = timm.create_model(
    "swin_base_patch4_window7_224",
    pretrained=True,
    num_classes=0
)

swin = swin.to(device)
swin.eval()

total_params = sum(
    p.numel()
    for p in swin.parameters()
)

print("✓ Swin-B loaded")
print()
print("Total parameters     :", total_params)
print("Total parameters (M) :", round(total_params / 1e6, 3))

# ==============================================================================
# 4. PATCH EMBEDDING CONFIGURATION
# ==============================================================================

print()
print("-" * 78)
print("PATCH EMBEDDING CONFIGURATION")
print("-" * 78)

print(
    "Original img_size        :",
    swin.patch_embed.img_size
)

print(
    "Original strict_img_size :",
    swin.patch_embed.strict_img_size
)

print(
    "Original dynamic_img_pad :",
    swin.patch_embed.dynamic_img_pad
)

# ------------------------------------------------------------------
# Allow rectangular input
# ------------------------------------------------------------------

swin.patch_embed.strict_img_size = False
swin.patch_embed.dynamic_img_pad = True

print()
print("Updated strict_img_size :", swin.patch_embed.strict_img_size)
print("Updated dynamic_img_pad :", swin.patch_embed.dynamic_img_pad)

print("✓ PASS — Variable rectangular input enabled")

# ==============================================================================
# 5. PREPARE IMAGE HEIGHT FOR PATCH EMBEDDING
# ==============================================================================

print()
print("-" * 78)
print("RECTANGULAR INPUT PREPARATION")
print("-" * 78)

B, C, H, W = images.shape

patch_h = 4
patch_w = 4

# 50 is not divisible by 4.
# 700 is divisible by 4.
#
# 50 -> 52 is sufficient for patch extraction.
#
# We deliberately DO NOT resize to 224x224.

pad_h = (patch_h - (H % patch_h)) % patch_h
pad_w = (patch_w - (W % patch_w)) % patch_w

print("Original :", (B, C, H, W))
print("Patch    :", (patch_h, patch_w))
print("Padding  :", (pad_h, pad_w))

if pad_h > 0 or pad_w > 0:

    images_swin = torch.nn.functional.pad(
        images,
        (0, pad_w, 0, pad_h),
        mode="constant",
        value=0.0
    )

else:

    images_swin = images

print(
    "Prepared :",
    tuple(images_swin.shape)
)

print("✓ PASS — Patch-compatible rectangular input")

# ==============================================================================
# 6. PATCH EMBEDDING
# ==============================================================================

print()
print("-" * 78)
print("PATCH EMBEDDING")
print("-" * 78)

with torch.no_grad():

    x = swin.patch_embed(images_swin)

print("Patch output:", tuple(x.shape))

if x.ndim != 4:

    raise RuntimeError(
        "Unexpected PatchEmbed output shape: "
        + str(tuple(x.shape))
    )

B0, H0, W0, C0 = x.shape

print()
print("Batch    :", B0)
print("Grid H   :", H0)
print("Grid W   :", W0)
print("Channels :", C0)

print(
    f"✓ PASS — Patch grid = {H0}×{W0}×{C0}"
)

# ==============================================================================
# 7. IMPORTANT:
#    CLEAR FIXED ATTENTION MASKS
# ==============================================================================

print()
print("-" * 78)
print("RECTANGULAR SWIN ATTENTION PREPARATION")
print("-" * 78)

for stage_index, stage in enumerate(swin.layers):

    print()
    print(
        f"STAGE {stage_index + 1}"
    )

    for block_index, block in enumerate(stage.blocks):

        if hasattr(block, "attn_mask"):

            block.attn_mask = None

            print(
                f"  Block {block_index + 1:02d}: "
                f"attn_mask = None"
            )

        else:

            print(
                f"  Block {block_index + 1:02d}: "
                f"no attn_mask attribute"
            )

print()
print("✓ PASS — Fixed-size attention masks removed")

# ==============================================================================
# 8. RUN SWIN FORWARD FEATURES
# ==============================================================================

print()
print("=" * 78)
print("RUNNING SWIN-B FORWARD FEATURES")
print("=" * 78)

try:

    with torch.no_grad():

        features = swin.forward_features(
            images_swin
        )

except Exception as e:

    print()
    print("=" * 78)
    print("SWIN FORWARD ERROR")
    print("=" * 78)

    print()
    print("Error type:")
    print(type(e).__name__)

    print()
    print("Error message:")
    print(str(e))

    raise

# ==============================================================================
# 9. FEATURE OUTPUT
# ==============================================================================

print()
print("-" * 78)
print("SWIN FEATURE OUTPUT")
print("-" * 78)

print(
    "Feature shape:",
    tuple(features.shape)
)

print(
    "Feature ndim :",
    features.ndim
)

# ==============================================================================
# 10. CONVERT TO OCR TOKENS
# ==============================================================================

print()
print("-" * 78)
print("CONVERTING FEATURES TO OCR TOKENS")
print("-" * 78)

if features.ndim == 4:

    # [B,H,W,C]
    Bf, Hf, Wf, Cf = features.shape

    swin_tokens = features.reshape(
        Bf,
        Hf * Wf,
        Cf
    )

elif features.ndim == 3:

    # Already [B,N,C]
    swin_tokens = features

else:

    raise RuntimeError(
        "Unsupported feature shape: "
        + str(tuple(features.shape))
    )

print(
    "OCR token shape:",
    tuple(swin_tokens.shape)
)

# ==============================================================================
# 11. FINAL VALIDATION
# ==============================================================================

print()
print("=" * 78)
print("FINAL VALIDATION")
print("=" * 78)

print()
print("Input image:")
print(tuple(images.shape))

print()
print("Prepared image:")
print(tuple(images_swin.shape))

print()
print("Patch grid:")
print(tuple(x.shape))

print()
print("Swin features:")
print(tuple(features.shape))

print()
print("Swin OCR tokens:")
print(tuple(swin_tokens.shape))

assert swin_tokens.ndim == 3

assert swin_tokens.shape[0] == images.shape[0]

print()
print("✓ PASS — Batch preserved")

print("✓ PASS — Swin features extracted")

print("✓ PASS — Features converted to [B,N,C]")

# ==============================================================================
# 12. SAVE SWIN TOKENS
# ==============================================================================

save_path = "/content/pfms_series_swin_b_ocr_tokens.pt"

torch.save(
    swin_tokens.detach().cpu(),
    save_path
)

print()
print("-" * 78)
print("SAVING SWIN OCR TOKENS")
print("-" * 78)

print("Saved:")
print(save_path)

# ==============================================================================
# 13. SUMMARY
# ==============================================================================

print()
print("=" * 78)
print("CELL 29-F COMPLETED")
print("=" * 78)

print()
print("PFMS-SERIES CURRENT STATUS:")
print()
print("IDPL-PFOD images       :", tuple(images.shape))
print("Prepared input         :", tuple(images_swin.shape))
print("Patch embedding        :", tuple(x.shape))
print("Swin feature output    :", tuple(features.shape))
print("Swin OCR token output  :", tuple(swin_tokens.shape))

print()
print("✓ SWIN-B RECTANGULAR OCR ENCODER TEST COMPLETE")
print("=" * 78)

PFMS-SERIES — CELL 29-F
SWIN-B RECTANGULAR OCR FORWARD

Device: cuda
GPU: Tesla T4

------------------------------------------------------------------------------
CHECKING REAL IDPL-PFOD BATCH
------------------------------------------------------------------------------
Image shape : (8, 3, 50, 700)
Device      : cuda:0
Dtype       : torch.float32
✓ PASS — Input = (B,3,50,700)

------------------------------------------------------------------------------
CREATING PRETRAINED SWIN-B
------------------------------------------------------------------------------
✓ Swin-B loaded

Total parameters     : 86743224
Total parameters (M) : 86.743

------------------------------------------------------------------------------
PATCH EMBEDDING CONFIGURATION
------------------------------------------------------------------------------
Original img_size        : (224, 224)
Original strict_img_size : True
Original dynamic_img_pad : False

Updated strict_img_size : False
Updated dynamic_img_pad : Tru

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 29-G
# SWIN-B OCR TOKEN DIMENSION ALIGNMENT
# ==============================================================================

import os
import torch
import torch.nn as nn

print("=" * 78)
print("PFMS-SERIES — CELL 29-G")
print("SWIN-B OCR TOKEN DIMENSION ALIGNMENT")
print("=" * 78)

# ------------------------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------------------------

device = torch.device("cuda:0")

print()
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------------------------
# LOAD SWIN OCR TOKENS
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("LOADING SWIN-B OCR TOKENS")
print("-" * 78)

token_path = "/content/pfms_series_swin_b_ocr_tokens.pt"

print("Token file:")
print(token_path)

if not os.path.exists(token_path):
    raise FileNotFoundError(
        "Swin OCR token file not found. Run Cell 29-F first."
    )

swin_tokens = torch.load(
    token_path,
    map_location=device
)

print("✓ Swin OCR tokens loaded")

# ------------------------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("TOKEN VALIDATION")
print("-" * 78)

print("Tensor shape:", tuple(swin_tokens.shape))
print("Tensor dtype:", swin_tokens.dtype)
print("Tensor device:", swin_tokens.device)

if swin_tokens.ndim != 3:
    raise RuntimeError(
        "Expected token tensor with shape [B,N,C]."
    )

B = swin_tokens.shape[0]
N = swin_tokens.shape[1]
C = swin_tokens.shape[2]

print()
print("Batch size :", B)
print("Token count:", N)
print("Channels   :", C)

if C != 1024:
    raise RuntimeError(
        "Swin-B output dimension is not 1024."
    )

print("✓ PASS — Swin-B tokens = [B,N,1024]")

# ------------------------------------------------------------------------------
# PROJECTION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("FEATURE DIMENSION ALIGNMENT")
print("-" * 78)

INPUT_DIM = 1024
TARGET_DIM = 768

print("Input dimension :", INPUT_DIM)
print("Target dimension:", TARGET_DIM)

projection = nn.Linear(
    INPUT_DIM,
    TARGET_DIM
)

projection = projection.to(device)
projection.eval()

print("✓ Projection created")

# ------------------------------------------------------------------------------
# PROJECT
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PROJECTING SWIN TOKENS")
print("-" * 78)

swin_tokens = swin_tokens.to(
    device=device,
    dtype=torch.float32
)

with torch.no_grad():
    swin_aligned_tokens = projection(swin_tokens)

print("Input :", tuple(swin_tokens.shape))
print("Output:", tuple(swin_aligned_tokens.shape))

# ------------------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("FINAL VALIDATION")
print("-" * 78)

expected_shape = (B, N, 768)
actual_shape = tuple(swin_aligned_tokens.shape)

print("Expected:", expected_shape)
print("Actual  :", actual_shape)

if actual_shape != expected_shape:
    raise RuntimeError(
        "Unexpected projection output shape."
    )

if not torch.isfinite(swin_aligned_tokens).all():
    raise RuntimeError(
        "NaN or Inf detected in projected tokens."
    )

print("✓ PASS — Batch preserved")
print("✓ PASS — Token count preserved")
print("✓ PASS — Dimension 1024 → 768")
print("✓ PASS — No NaN/Inf detected")

# ------------------------------------------------------------------------------
# SAVE
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("SAVING ALIGNED SWIN TOKENS")
print("-" * 78)

output_path = "/content/pfms_series_swin_b_aligned_tokens.pt"

torch.save(
    swin_aligned_tokens.cpu(),
    output_path
)

print("Saved:")
print(output_path)

# ------------------------------------------------------------------------------
# FINAL
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("CELL 29-G COMPLETED")
print("=" * 78)

print()
print("PFMS-SERIES CURRENT SWIN STATUS:")
print()
print("Input image          : (8, 3, 50, 700)")
print("Prepared image       : (8, 3, 52, 700)")
print("Patch grid           : (8, 13, 175, 128)")
print("Swin features        : (8, 2, 22, 1024)")
print("Swin OCR tokens      : (8, 44, 1024)")
print("Aligned Swin tokens  :", tuple(swin_aligned_tokens.shape))
print()
print("✓ SWIN-B ALIGNMENT COMPLETE")
print("=" * 78)

PFMS-SERIES — CELL 29-G
SWIN-B OCR TOKEN DIMENSION ALIGNMENT

Device: cuda:0
GPU: Tesla T4

------------------------------------------------------------------------------
LOADING SWIN-B OCR TOKENS
------------------------------------------------------------------------------
Token file:
/content/pfms_series_swin_b_ocr_tokens.pt
✓ Swin OCR tokens loaded

------------------------------------------------------------------------------
TOKEN VALIDATION
------------------------------------------------------------------------------
Tensor shape: (8, 44, 1024)
Tensor dtype: torch.float32
Tensor device: cuda:0

Batch size : 8
Token count: 44
Channels   : 1024
✓ PASS — Swin-B tokens = [B,N,1024]

------------------------------------------------------------------------------
FEATURE DIMENSION ALIGNMENT
------------------------------------------------------------------------------
Input dimension : 1024
Target dimension: 768
✓ Projection created

---------------------------------------------------

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 30
# MAMBAVISION RECTANGULAR OCR ENCODER
# ==============================================================================

import os
import torch
import torch.nn as nn

print("=" * 78)
print("PFMS-SERIES — CELL 30")
print("MAMBAVISION RECTANGULAR OCR ENCODER")
print("=" * 78)

# ------------------------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------------------------

device = torch.device("cuda:0")

print()
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------------------------
# LOAD REAL IDPL-PFOD BATCH
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("LOADING REAL IDPL-PFOD BATCH")
print("-" * 78)

batch_path = "/content/pfms_series_real_batch.pt"

print("Batch file:")
print(batch_path)

if not os.path.exists(batch_path):
    raise FileNotFoundError(
        "Real IDPL-PFOD batch not found. "
        "Please run Cell 29-E first."
    )

batch_data = torch.load(
    batch_path,
    map_location=device
)

print("✓ Batch loaded")

# ------------------------------------------------------------------------------
# IDENTIFY IMAGE TENSOR
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("VALIDATING IMAGE BATCH")
print("-" * 78)

images = None

if torch.is_tensor(batch_data):
    images = batch_data

elif isinstance(batch_data, dict):

    possible_keys = [
        "images",
        "image",
        "image_tensor",
        "batch",
        "inputs"
    ]

    for key in possible_keys:
        if key in batch_data:
            if torch.is_tensor(batch_data[key]):
                images = batch_data[key]
                break

if images is None:
    raise RuntimeError(
        "Could not identify image tensor inside saved batch."
    )

print("Image shape:", tuple(images.shape))
print("Image dtype:", images.dtype)
print("Image device:", images.device)

if images.ndim != 4:
    raise RuntimeError(
        "Expected image tensor with shape [B,C,H,W]."
    )

B = images.shape[0]
C = images.shape[1]
H = images.shape[2]
W = images.shape[3]

print()
print("Batch size :", B)
print("Channels   :", C)
print("Height     :", H)
print("Width      :", W)

if C != 3:
    raise RuntimeError(
        "Expected 3 image channels."
    )

if H != 50:
    raise RuntimeError(
        f"Expected image height 50, received {H}."
    )

if W != 700:
    raise RuntimeError(
        f"Expected image width 700, received {W}."
    )

print("✓ PASS — Real OCR input = (B,3,50,700)")

# ------------------------------------------------------------------------------
# MOVE INPUT TO GPU
# ------------------------------------------------------------------------------

images = images.to(
    device=device,
    dtype=torch.float32
)

print()
print("Input device:", images.device)
print("Input dtype :", images.dtype)

# ------------------------------------------------------------------------------
# CHECK MAMBAVISION IMPORT
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CHECKING MAMBAVISION")
print("-" * 78)

try:
    import timm
    print("timm version:", timm.__version__)
except Exception as e:
    raise RuntimeError(
        f"timm import failed: {e}"
    )

print("Searching available MambaVision models...")

mamba_names = [
    name
    for name in timm.list_models()
    if "mamba" in name.lower()
]

print("Available Mamba-related models:")
print(mamba_names)

if len(mamba_names) == 0:
    raise RuntimeError(
        "No MambaVision model is available in the current timm installation."
    )

# ------------------------------------------------------------------------------
# SELECT MAMBAVISION MODEL
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("SELECTING MAMBAVISION MODEL")
print("-" * 78)

preferred_names = [
    "mamba_vision_B",
    "mamba_vision_T",
    "mamba_vision_L"
]

selected_model_name = None

for name in preferred_names:
    if name in mamba_names:
        selected_model_name = name
        break

if selected_model_name is None:
    selected_model_name = mamba_names[0]

print("Selected model:", selected_model_name)

# ------------------------------------------------------------------------------
# CREATE MODEL
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING PRETRAINED MAMBAVISION")
print("-" * 78)

try:
    mamba = timm.create_model(
        selected_model_name,
        pretrained=True,
        num_classes=0
    )

    print("✓ MambaVision loaded")

except Exception as e:
    raise RuntimeError(
        "MambaVision could not be created.\n"
        f"Model: {selected_model_name}\n"
        f"Error: {e}"
    )

mamba = mamba.to(device)
mamba.eval()

# ------------------------------------------------------------------------------
# PARAMETER INFORMATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("MAMBAVISION MODEL INFORMATION")
print("-" * 78)

total_params = sum(
    p.numel()
    for p in mamba.parameters()
)

trainable_params = sum(
    p.numel()
    for p in mamba.parameters()
    if p.requires_grad
)

print("Total parameters     :", total_params)
print(
    "Total parameters (M) :",
    round(total_params / 1e6, 3)
)

print("Trainable parameters :", trainable_params)

# ------------------------------------------------------------------------------
# MODEL DEVICE VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("DEVICE VALIDATION")
print("-" * 78)

model_parameter = next(
    mamba.parameters()
)

print("Model device:", model_parameter.device)
print("Input device:", images.device)
print("Model dtype :", model_parameter.dtype)
print("Input dtype :", images.dtype)

if model_parameter.device != images.device:
    raise RuntimeError(
        "Model and input are on different devices."
    )

print("✓ PASS — Model/Input compatible")

# ------------------------------------------------------------------------------
# RECTANGULAR INPUT TEST
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("RECTANGULAR INPUT TEST")
print("-" * 78)

print("Input:", tuple(images.shape))
print("Height:", H)
print("Width :", W)

# ------------------------------------------------------------------------------
# FORWARD FEATURES
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("RUNNING MAMBAVISION")
print("-" * 78)

with torch.no_grad():

    try:
        mamba_features = mamba.forward_features(images)

    except Exception as e:

        print()
        print("forward_features failed.")
        print("Error:", e)

        raise RuntimeError(
            "MambaVision forward_features failed on "
            "the rectangular IDPL-PFOD input."
        )

# ------------------------------------------------------------------------------
# FEATURE INFORMATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("MAMBAVISION FEATURE OUTPUT")
print("-" * 78)

print("Feature type:", type(mamba_features))

if torch.is_tensor(mamba_features):

    print(
        "Feature shape:",
        tuple(mamba_features.shape)
    )

    print(
        "Feature ndim :",
        mamba_features.ndim
    )

else:

    print(
        "Feature object:",
        mamba_features
    )

# ------------------------------------------------------------------------------
# CONVERT FEATURES TO OCR TOKENS
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CONVERTING MAMBAVISION FEATURES TO OCR TOKENS")
print("-" * 78)

if not torch.is_tensor(mamba_features):

    raise RuntimeError(
        "MambaVision returned a non-tensor feature object. "
        "Feature extraction must be inspected before continuing."
    )

if mamba_features.ndim == 4:

    Bm, Cm, Hm, Wm = mamba_features.shape

    print("Feature map:")
    print(
        "B =", Bm,
        "C =", Cm,
        "H =", Hm,
        "W =", Wm
    )

    mamba_tokens = mamba_features.flatten(
        2
    ).transpose(
        1,
        2
    )

elif mamba_features.ndim == 3:

    mamba_tokens = mamba_features

else:

    raise RuntimeError(
        "Unsupported MambaVision feature dimensionality: "
        f"{mamba_features.ndim}"
    )

print(
    "MambaVision OCR tokens:",
    tuple(mamba_tokens.shape)
)

# ------------------------------------------------------------------------------
# TOKEN VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("TOKEN VALIDATION")
print("-" * 78)

Bt = mamba_tokens.shape[0]
Nt = mamba_tokens.shape[1]
Ct = mamba_tokens.shape[2]

print("Batch size :", Bt)
print("Token count:", Nt)
print("Channels   :", Ct)

if Bt != B:
    raise RuntimeError(
        f"Batch mismatch: input={B}, output={Bt}"
    )

if not torch.isfinite(mamba_tokens).all():
    raise RuntimeError(
        "MambaVision tokens contain NaN or Inf."
    )

print("✓ PASS — Batch preserved")
print("✓ PASS — MambaVision tokens are [B,N,C]")
print("✓ PASS — No NaN/Inf detected")

# ------------------------------------------------------------------------------
# SAVE RAW MAMBAVISION TOKENS
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("SAVING MAMBAVISION OCR TOKENS")
print("-" * 78)

mamba_token_path = (
    "/content/pfms_series_mambavision_ocr_tokens.pt"
)

torch.save(
    mamba_tokens.cpu(),
    mamba_token_path
)

print("Saved:")
print(mamba_token_path)

# ------------------------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("CELL 30 COMPLETED")
print("=" * 78)

print()
print("PFMS-SERIES MAMBAVISION STATUS:")
print()
print("IDPL-PFOD input      :", tuple(images.shape))
print("MambaVision model    :", selected_model_name)
print("MambaVision tokens   :", tuple(mamba_tokens.shape))
print()
print("✓ MAMBAVISION OCR ENCODER TEST COMPLETE")
print("=" * 78)

PFMS-SERIES — CELL 30
MAMBAVISION RECTANGULAR OCR ENCODER

Device: cuda:0
GPU: Tesla T4

------------------------------------------------------------------------------
LOADING REAL IDPL-PFOD BATCH
------------------------------------------------------------------------------
Batch file:
/content/pfms_series_real_batch.pt
✓ Batch loaded

------------------------------------------------------------------------------
VALIDATING IMAGE BATCH
------------------------------------------------------------------------------
Image shape: (8, 3, 50, 700)
Image dtype: torch.float32
Image device: cuda:0

Batch size : 8
Channels   : 3
Height     : 50
Width      : 700
✓ PASS — Real OCR input = (B,3,50,700)

Input device: cuda:0
Input dtype : torch.float32

------------------------------------------------------------------------------
CHECKING MAMBAVISION
------------------------------------------------------------------------------
timm version: 1.0.28
Searching available MambaVision models...
Availab

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 30-A
# MAMBAVISION RECTANGULAR OCR TOKEN CORRECTION
# ==============================================================================
# PURPOSE:
# Correctly convert MambaVision feature map
# [B, C, H, W] = [8, 768, 2, 22]
# into OCR tokens
# [B, N, C] = [8, 44, 768]
# ==============================================================================

import os
import torch
import timm

print("=" * 78)
print("PFMS-SERIES — CELL 30-A")
print("MAMBAVISION RECTANGULAR OCR TOKEN CORRECTION")
print("=" * 78)

# ------------------------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------------------------

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print()
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------------------------
# LOAD REAL IDPL-PFOD BATCH
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("LOADING REAL IDPL-PFOD BATCH")
print("-" * 78)

batch_file = "/content/pfms_series_real_batch.pt"

if not os.path.exists(batch_file):
    raise FileNotFoundError(
        "Real IDPL-PFOD batch was not found:\n"
        + batch_file
    )

batch_data = torch.load(batch_file, map_location=device)

# Support either tensor or dictionary format
if isinstance(batch_data, torch.Tensor):
    images = batch_data.to(device=device, dtype=torch.float32)

elif isinstance(batch_data, dict):
    possible_keys = [
        "images",
        "image_tensor",
        "batch",
        "inputs"
    ]

    images = None

    for key in possible_keys:
        if key in batch_data and isinstance(batch_data[key], torch.Tensor):
            images = batch_data[key].to(
                device=device,
                dtype=torch.float32
            )
            break

    if images is None:
        raise RuntimeError(
            "Could not find image tensor inside saved batch."
        )

else:
    raise RuntimeError(
        "Unsupported batch file format."
    )

print("Batch file:", batch_file)
print("Image shape:", tuple(images.shape))
print("Image dtype:", images.dtype)
print("Image device:", images.device)

if images.ndim != 4:
    raise RuntimeError(
        f"Expected 4-D image tensor [B,C,H,W], got {images.ndim}-D."
    )

if images.shape[1] != 3:
    raise RuntimeError(
        f"Expected 3 channels, got {images.shape[1]}."
    )

if images.shape[2] != 50 or images.shape[3] != 700:
    raise RuntimeError(
        f"Expected image size 50x700, got "
        f"{images.shape[2]}x{images.shape[3]}."
    )

print("✓ PASS — Real OCR input = [B,3,50,700]")

# ------------------------------------------------------------------------------
# LOAD MAMBAVISION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING PRETRAINED MAMBAVISION")
print("-" * 78)

model_name = "mambaout_base"

mamba = timm.create_model(
    model_name,
    pretrained=True
)

mamba = mamba.to(device)
mamba.eval()

print("Model:", model_name)
print("✓ MambaVision/MambaOut model loaded")

# ------------------------------------------------------------------------------
# MODEL INFORMATION
# ------------------------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in mamba.parameters()
)

trainable_params = sum(
    p.numel()
    for p in mamba.parameters()
    if p.requires_grad
)

print()
print("-" * 78)
print("MAMBAVISION MODEL INFORMATION")
print("-" * 78)

print("Total parameters     :", total_params)
print("Total parameters (M) :", round(total_params / 1e6, 3))
print("Trainable parameters :", trainable_params)

# ------------------------------------------------------------------------------
# DEVICE VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("DEVICE VALIDATION")
print("-" * 78)

model_device = next(mamba.parameters()).device
model_dtype = next(mamba.parameters()).dtype

print("Model device:", model_device)
print("Input device:", images.device)
print("Model dtype :", model_dtype)
print("Input dtype :", images.dtype)

if model_device != images.device:
    raise RuntimeError(
        "Model and input are on different devices."
    )

print("✓ PASS — Model/Input compatible")

# ------------------------------------------------------------------------------
# RUN MAMBAVISION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("RUNNING MAMBAVISION")
print("-" * 78)

with torch.no_grad():
    features = mamba.forward_features(images)

print()
print("-" * 78)
print("RAW MAMBAVISION FEATURE OUTPUT")
print("-" * 78)

print("Feature type :", type(features))

if not isinstance(features, torch.Tensor):
    raise RuntimeError(
        f"Expected Tensor output, got {type(features)}"
    )

print("Feature shape:", tuple(features.shape))
print("Feature ndim :", features.ndim)

# ------------------------------------------------------------------------------
# VALIDATE FEATURE MAP
# ------------------------------------------------------------------------------

if features.ndim != 4:
    raise RuntimeError(
        "Expected 4-D MambaVision feature map."
    )

B = features.shape[0]

# IMPORTANT:
# The observed output is:
#
# [8, 2, 22, 768]
#
# For this MambaOut implementation, the last dimension is the channel
# dimension. Therefore:
#
# B = 8
# H = 2
# W = 22
# C = 768
#
# We convert:
#
# [B,H,W,C]
#       ↓
# [B,H*W,C]
#
# This gives:
#
# [8,44,768]

if features.shape[-1] == 768:
    print()
    print("Detected feature layout: [B,H,W,C]")

    B = features.shape[0]
    H = features.shape[1]
    W = features.shape[2]
    C = features.shape[3]

    print("B =", B)
    print("H =", H)
    print("W =", W)
    print("C =", C)

    if C != 768:
        raise RuntimeError(
            f"Expected channel dimension 768, got {C}."
        )

    # --------------------------------------------------------------------------
    # CONVERT [B,H,W,C] -> [B,N,C]
    # --------------------------------------------------------------------------

    tokens = features.reshape(
        B,
        H * W,
        C
    )

else:
    # Fallback for conventional [B,C,H,W]
    print()
    print("Detected feature layout: [B,C,H,W]")

    B = features.shape[0]
    C = features.shape[1]
    H = features.shape[2]
    W = features.shape[3]

    print("B =", B)
    print("C =", C)
    print("H =", H)
    print("W =", W)

    if C != 768:
        raise RuntimeError(
            f"Expected channel dimension 768, got {C}."
        )

    tokens = features.permute(
        0, 2, 3, 1
    ).contiguous().reshape(
        B,
        H * W,
        C
    )

# ------------------------------------------------------------------------------
# TOKEN VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("MAMBAVISION OCR TOKEN OUTPUT")
print("-" * 78)

print("Feature map :", tuple(features.shape))
print("OCR tokens  :", tuple(tokens.shape))

expected_shape = (
    images.shape[0],
    44,
    768
)

print()
print("Expected:", expected_shape)
print("Actual  :", tuple(tokens.shape))

if tuple(tokens.shape) != expected_shape:
    raise RuntimeError(
        "Unexpected MambaVision token shape.\n"
        f"Expected: {expected_shape}\n"
        f"Actual  : {tuple(tokens.shape)}"
    )

if tokens.device != images.device:
    raise RuntimeError(
        "Token device does not match input device."
    )

if torch.isnan(tokens).any():
    raise RuntimeError(
        "NaN detected in MambaVision tokens."
    )

if torch.isinf(tokens).any():
    raise RuntimeError(
        "Inf detected in MambaVision tokens."
    )

print("✓ PASS — Batch preserved")
print("✓ PASS — Token count = 44")
print("✓ PASS — Feature dimension = 768")
print("✓ PASS — MambaVision tokens = [B,44,768]")
print("✓ PASS — No NaN/Inf detected")

# ------------------------------------------------------------------------------
# SAVE CORRECTED TOKENS
# ------------------------------------------------------------------------------

output_file = "/content/pfms_series_mambavision_aligned_tokens.pt"

print()
print("-" * 78)
print("SAVING CORRECTED MAMBAVISION OCR TOKENS")
print("-" * 78)

torch.save(
    tokens.detach().cpu(),
    output_file
)

print("Saved:")
print(output_file)

# ------------------------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("CELL 30-A COMPLETED")
print("=" * 78)

print()
print("PFMS-SERIES CURRENT MAMBAVISION STATUS:")
print()

print("IDPL-PFOD input       :", tuple(images.shape))
print("Raw MambaVision       :", tuple(features.shape))
print("MambaVision OCR tokens:", tuple(tokens.shape))

print()
print("✓ MAMBAVISION TOKEN CORRECTION COMPLETE")
print("=" * 78)

PFMS-SERIES — CELL 30-A
MAMBAVISION RECTANGULAR OCR TOKEN CORRECTION

Device: cuda:0
GPU: Tesla T4

------------------------------------------------------------------------------
LOADING REAL IDPL-PFOD BATCH
------------------------------------------------------------------------------
Batch file: /content/pfms_series_real_batch.pt
Image shape: (8, 3, 50, 700)
Image dtype: torch.float32
Image device: cuda:0
✓ PASS — Real OCR input = [B,3,50,700]

------------------------------------------------------------------------------
CREATING PRETRAINED MAMBAVISION
------------------------------------------------------------------------------
Model: mambaout_base
✓ MambaVision/MambaOut model loaded

------------------------------------------------------------------------------
MAMBAVISION MODEL INFORMATION
------------------------------------------------------------------------------
Total parameters     : 84813092
Total parameters (M) : 84.813
Trainable parameters : 84813092

------------------

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 31
# SWIN-B → MAMBAVISION SERIES ADAPTER
# ==============================================================================
# Purpose:
# Convert aligned Swin-B OCR tokens
#
#     [B,44,768]
#
# into a spatial feature representation
#
#     [B,768,2,22]
#
# and validate the Series interface before connecting MambaVision.
# ==============================================================================

import os
import torch
import torch.nn as nn

print("=" * 78)
print("PFMS-SERIES — CELL 31")
print("SWIN-B → MAMBAVISION SERIES ADAPTER")
print("=" * 78)

# ------------------------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------------------------

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print()
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------------------------
# LOAD SWIN-B ALIGNED TOKENS
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("LOADING SWIN-B ALIGNED TOKENS")
print("-" * 78)

swin_file = "/content/pfms_series_swin_b_aligned_tokens.pt"

if not os.path.exists(swin_file):
    raise FileNotFoundError(
        "Swin-B aligned token file was not found:\n"
        + swin_file
    )

swin_tokens = torch.load(
    swin_file,
    map_location=device
)

if not isinstance(swin_tokens, torch.Tensor):
    raise RuntimeError(
        "Swin token file does not contain a Tensor."
    )

swin_tokens = swin_tokens.to(
    device=device,
    dtype=torch.float32
)

print("File:", swin_file)
print("Shape:", tuple(swin_tokens.shape))
print("Dtype:", swin_tokens.dtype)
print("Device:", swin_tokens.device)

# ------------------------------------------------------------------------------
# VALIDATE SWIN TOKENS
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("SWIN TOKEN VALIDATION")
print("-" * 78)

if swin_tokens.ndim != 3:
    raise RuntimeError(
        "Expected Swin tokens with shape [B,N,C]."
    )

B = swin_tokens.shape[0]
N = swin_tokens.shape[1]
C = swin_tokens.shape[2]

print("Batch size :", B)
print("Token count:", N)
print("Channels   :", C)

if N != 44:
    raise RuntimeError(
        f"Expected 44 Swin tokens, got {N}."
    )

if C != 768:
    raise RuntimeError(
        f"Expected 768 channels, got {C}."
    )

if torch.isnan(swin_tokens).any():
    raise RuntimeError(
        "NaN detected in Swin tokens."
    )

if torch.isinf(swin_tokens).any():
    raise RuntimeError(
        "Inf detected in Swin tokens."
    )

print("✓ PASS — Swin tokens = [B,44,768]")
print("✓ PASS — No NaN/Inf detected")

# ------------------------------------------------------------------------------
# SERIES SPATIAL ORGANIZATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING SERIES SPATIAL REPRESENTATION")
print("-" * 78)

# The Swin OCR sequence contains:
#
#     N = 44 tokens
#
# We preserve the sequence ordering as:
#
#     H = 2
#     W = 22
#
# Therefore:
#
#     [B,44,768]
#          ↓
#     [B,2,22,768]
#
# Then convert to the conventional CNN/Mamba feature-map layout:
#
#     [B,768,2,22]

SERIES_H = 2
SERIES_W = 22

if SERIES_H * SERIES_W != N:
    raise RuntimeError(
        "Spatial organization is incompatible with token count."
    )

print("Token count :", N)
print("Spatial H   :", SERIES_H)
print("Spatial W   :", SERIES_W)

# ------------------------------------------------------------------------------
# TOKENS → [B,H,W,C]
# ------------------------------------------------------------------------------

swin_spatial = swin_tokens.reshape(
    B,
    SERIES_H,
    SERIES_W,
    C
)

print()
print("After token reshape:")
print(
    "[B,N,C] → [B,H,W,C] :",
    tuple(swin_spatial.shape)
)

if tuple(swin_spatial.shape) != (
    B,
    SERIES_H,
    SERIES_W,
    C
):
    raise RuntimeError(
        "Unexpected spatial token shape."
    )

# ------------------------------------------------------------------------------
# [B,H,W,C] → [B,C,H,W]
# ------------------------------------------------------------------------------

swin_feature_map = swin_spatial.permute(
    0,
    3,
    1,
    2
).contiguous()

print()
print("After channel-first conversion:")
print(
    "[B,H,W,C] → [B,C,H,W] :",
    tuple(swin_feature_map.shape)
)

expected_feature_shape = (
    B,
    768,
    SERIES_H,
    SERIES_W
)

if tuple(swin_feature_map.shape) != expected_feature_shape:
    raise RuntimeError(
        f"Expected {expected_feature_shape}, "
        f"got {tuple(swin_feature_map.shape)}"
    )

print("✓ PASS — Spatial feature map created")

# ------------------------------------------------------------------------------
# SERIES ADAPTER
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING SERIES ADAPTER")
print("-" * 78)

# The adapter keeps the channel dimension at 768.
#
# It is intentionally lightweight.
# It does NOT change:
#
#     Batch
#     Spatial size
#     Feature dimension
#
# It only performs channel refinement.

class SwinToMambaSeriesAdapter(nn.Module):

    def __init__(self, channels=768):
        super().__init__()

        self.norm = nn.BatchNorm2d(channels)

        self.projection = nn.Conv2d(
            channels,
            channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False
        )

        self.activation = nn.GELU()

    def forward(self, x):

        x = self.norm(x)
        x = self.projection(x)
        x = self.activation(x)

        return x


series_adapter = SwinToMambaSeriesAdapter(
    channels=768
).to(device)

series_adapter.eval()

print("Adapter:", series_adapter)

adapter_params = sum(
    p.numel()
    for p in series_adapter.parameters()
)

print()
print("Adapter parameters:", adapter_params)

# ------------------------------------------------------------------------------
# RUN ADAPTER
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("RUNNING SWIN → MAMBAVISION SERIES ADAPTER")
print("-" * 78)

with torch.no_grad():

    adapted_feature_map = series_adapter(
        swin_feature_map
    )

print()
print("Input feature map :",
      tuple(swin_feature_map.shape))

print("Adapted feature map:",
      tuple(adapted_feature_map.shape))

# ------------------------------------------------------------------------------
# VALIDATE ADAPTER OUTPUT
# ------------------------------------------------------------------------------

if tuple(adapted_feature_map.shape) != expected_feature_shape:
    raise RuntimeError(
        "Adapter changed the expected feature-map shape."
    )

if torch.isnan(adapted_feature_map).any():
    raise RuntimeError(
        "NaN detected after Series Adapter."
    )

if torch.isinf(adapted_feature_map).any():
    raise RuntimeError(
        "Inf detected after Series Adapter."
    )

print()
print("✓ PASS — Batch preserved")
print("✓ PASS — Channels preserved = 768")
print("✓ PASS — Spatial size preserved = 2×22")
print("✓ PASS — No NaN/Inf detected")

# ------------------------------------------------------------------------------
# CONVERT BACK TO OCR TOKENS
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CONVERTING ADAPTED FEATURES BACK TO OCR TOKENS")
print("-" * 78)

# [B,C,H,W]
#      ↓
# [B,H,W,C]
#      ↓
# [B,N,C]

adapted_tokens = adapted_feature_map.permute(
    0,
    2,
    3,
    1
).contiguous()

adapted_tokens = adapted_tokens.reshape(
    B,
    SERIES_H * SERIES_W,
    C
)

print("Adapted OCR tokens:",
      tuple(adapted_tokens.shape))

expected_token_shape = (
    B,
    44,
    768
)

if tuple(adapted_tokens.shape) != expected_token_shape:
    raise RuntimeError(
        f"Expected {expected_token_shape}, "
        f"got {tuple(adapted_tokens.shape)}"
    )

if torch.isnan(adapted_tokens).any():
    raise RuntimeError(
        "NaN detected in adapted OCR tokens."
    )

if torch.isinf(adapted_tokens).any():
    raise RuntimeError(
        "Inf detected in adapted OCR tokens."
    )

print()
print("✓ PASS — Series tokens = [B,44,768]")
print("✓ PASS — Token order preserved")
print("✓ PASS — Feature dimension preserved")
print("✓ PASS — No NaN/Inf detected")

# ------------------------------------------------------------------------------
# SAVE SERIES ADAPTER OUTPUT
# ------------------------------------------------------------------------------

output_file = (
    "/content/pfms_series_swin_to_mamba_adapter_tokens.pt"
)

print()
print("-" * 78)
print("SAVING SERIES ADAPTER OUTPUT")
print("-" * 78)

torch.save(
    adapted_tokens.detach().cpu(),
    output_file
)

print("Saved:")
print(output_file)

# ------------------------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("CELL 31 COMPLETED")
print("=" * 78)

print()
print("PFMS-SERIES STATUS:")
print()

print("Original Swin tokens       :",
      tuple(swin_tokens.shape))

print("Spatial representation     :",
      tuple(swin_spatial.shape))

print("Swin feature map           :",
      tuple(swin_feature_map.shape))

print("Adapted feature map        :",
      tuple(adapted_feature_map.shape))

print("Adapted OCR tokens         :",
      tuple(adapted_tokens.shape))

print()
print("SERIES INTERFACE:")
print()
print("[B,44,768]")
print("    ↓")
print("[B,2,22,768]")
print("    ↓")
print("[B,768,2,22]")
print("    ↓")
print("Series Adapter")
print("    ↓")
print("[B,768,2,22]")
print("    ↓")
print("[B,44,768]")

print()
print("✓ SWIN → MAMBA SERIES ADAPTER VALIDATED")
print("=" * 78)

PFMS-SERIES — CELL 31
SWIN-B → MAMBAVISION SERIES ADAPTER

Device: cuda:0
GPU: Tesla T4

------------------------------------------------------------------------------
LOADING SWIN-B ALIGNED TOKENS
------------------------------------------------------------------------------
File: /content/pfms_series_swin_b_aligned_tokens.pt
Shape: (8, 44, 768)
Dtype: torch.float32
Device: cuda:0

------------------------------------------------------------------------------
SWIN TOKEN VALIDATION
------------------------------------------------------------------------------
Batch size : 8
Token count: 44
Channels   : 768
✓ PASS — Swin tokens = [B,44,768]
✓ PASS — No NaN/Inf detected

------------------------------------------------------------------------------
CREATING SERIES SPATIAL REPRESENTATION
------------------------------------------------------------------------------
Token count : 44
Spatial H   : 2
Spatial W   : 22

After token reshape:
[B,N,C] → [B,H,W,C] : (8, 2, 22, 768)

After channel-

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 32
# EXACT MAMBAOUT INTERNAL STRUCTURE FOR SERIES CONNECTION
# ==============================================================================

import torch
import torch.nn as nn
import timm

print("=" * 78)
print("PFMS-SERIES — CELL 32")
print("EXACT MAMBAOUT INTERNAL STRUCTURE")
print("=" * 78)

# ------------------------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------------------------

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print()
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------------------------
# CREATE MAMBAOUT
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING PRETRAINED MAMBAOUT")
print("-" * 78)

mamba = timm.create_model(
    "mambaout_base",
    pretrained=True
)

mamba = mamba.to(device)
mamba.eval()

print("✓ MambaOut loaded")

# ------------------------------------------------------------------------------
# MODEL INFORMATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("MODEL INFORMATION")
print("-" * 78)

total_params = sum(p.numel() for p in mamba.parameters())

print("Total parameters :", total_params)
print("Total parameters (M) :", total_params / 1e6)

# ------------------------------------------------------------------------------
# TOP-LEVEL MODULES
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("TOP-LEVEL MODULE STRUCTURE")
print("-" * 78)

for name, module in mamba.named_children():
    print()
    print("MODULE:", name)
    print("TYPE  :", type(module))
    print(module)

# ------------------------------------------------------------------------------
# MODEL ATTRIBUTE NAMES
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("AVAILABLE MODEL ATTRIBUTES")
print("-" * 78)

important_names = [
    "stem",
    "stages",
    "layers",
    "blocks",
    "norm",
    "head",
    "forward_features",
]

for name in important_names:

    exists = hasattr(mamba, name)

    print(
        "{:<20} : {}".format(
            name,
            "FOUND" if exists else "not found"
        )
    )

# ------------------------------------------------------------------------------
# FEATURE INFO
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("FEATURE INFORMATION")
print("-" * 78)

if hasattr(mamba, "feature_info"):

    try:
        print(mamba.feature_info)
    except Exception as e:
        print("Could not print feature_info:", e)

else:

    print("feature_info not available")

# ------------------------------------------------------------------------------
# FORWARD_FEATURES CHECK
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("FORWARD_FEATURES CHECK")
print("-" * 78)

if hasattr(mamba, "forward_features"):

    print("✓ forward_features exists")

else:

    print("✗ forward_features does not exist")

# ------------------------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("CELL 32 COMPLETED")
print("=" * 78)

print()
print("NEXT STEP:")
print("The exact internal MambaOut stage accepting")
print("Swin-B features will be identified from this output.")

print()
print("IMPORTANT:")
print("Do NOT connect Swin tokens to MambaOut yet.")
print("Do NOT change the model structure yet.")
print("=" * 78)

PFMS-SERIES — CELL 32
EXACT MAMBAOUT INTERNAL STRUCTURE

Device: cuda:0
GPU: Tesla T4

------------------------------------------------------------------------------
CREATING PRETRAINED MAMBAOUT
------------------------------------------------------------------------------
✓ MambaOut loaded

------------------------------------------------------------------------------
MODEL INFORMATION
------------------------------------------------------------------------------
Total parameters : 84813092
Total parameters (M) : 84.813092

------------------------------------------------------------------------------
TOP-LEVEL MODULE STRUCTURE
------------------------------------------------------------------------------

MODULE: stem
TYPE  : <class 'timm.models.mambaout.Stem'>
Stem(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=True)
  (act): GELU(approximate='none')
  (conv2): Conv2d(64, 128, kernel_size=(3, 3),

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 33-FINAL
# REAL SWIN-B → MAMBAOUT SERIES CONNECTION
# FORMAT-CORRECTED VERSION
# ==============================================================================

import os
import torch
import torch.nn as nn
import timm

print("=" * 78)
print("PFMS-SERIES — CELL 33-FINAL")
print("REAL SWIN-B → MAMBAOUT SERIES CONNECTION")
print("=" * 78)

# ------------------------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------------------------

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print()
    print("Device:", device)
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print()
    print("Device:", device)
    print("GPU: CPU")

# ------------------------------------------------------------------------------
# FILE PATHS
# ------------------------------------------------------------------------------

batch_path = "/content/pfms_series_real_batch.pt"

# ------------------------------------------------------------------------------
# LOAD REAL IDPL-PFOD BATCH
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("LOADING REAL IDPL-PFOD BATCH")
print("-" * 78)

if not os.path.exists(batch_path):
    raise FileNotFoundError(
        "Real IDPL-PFOD batch not found:\n"
        + batch_path
    )

images = torch.load(batch_path, map_location=device)

# Handle possible dictionary format
if isinstance(images, dict):
    if "images" in images:
        images = images["images"]
    elif "image_tensor" in images:
        images = images["image_tensor"]
    else:
        raise RuntimeError(
            "Batch file is a dictionary, but no image tensor was found."
        )

images = images.float().to(device)

print("Image shape:", tuple(images.shape))
print("Image dtype:", images.dtype)
print("Image device:", images.device)

if images.ndim != 4:
    raise RuntimeError(
        f"Expected 4D image tensor [B,C,H,W], got {images.ndim}D"
    )

B, C, H, W = images.shape

print()
print("Batch size :", B)
print("Channels   :", C)
print("Height     :", H)
print("Width      :", W)

if (C, H, W) != (3, 50, 700):
    raise RuntimeError(
        f"Expected input [B,3,50,700], got {tuple(images.shape)}"
    )

print("✓ PASS — Real OCR input = [B,3,50,700]")

# ==============================================================================
# PART A — SWIN-B
# ==============================================================================

print()
print("=" * 78)
print("PART A — SWIN-B ENCODER")
print("=" * 78)

# ------------------------------------------------------------------------------
# CREATE PRETRAINED SWIN-B
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING PRETRAINED SWIN-B")
print("-" * 78)

swin = timm.create_model(
    "swin_base_patch4_window7_224",
    pretrained=True,
    num_classes=0
)

swin = swin.to(device)
swin.eval()

print("✓ Swin-B loaded")

# ------------------------------------------------------------------------------
# CONFIGURE RECTANGULAR INPUT
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CONFIGURING RECTANGULAR INPUT")
print("-" * 78)

swin.patch_embed.strict_img_size = False
swin.patch_embed.dynamic_img_pad = True

print("strict_img_size :", swin.patch_embed.strict_img_size)
print("dynamic_img_pad :", swin.patch_embed.dynamic_img_pad)

# ------------------------------------------------------------------------------
# PREPARE IMAGE
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PREPARING IMAGE FOR SWIN-B")
print("-" * 78)

patch_size = 4

new_height = ((H + patch_size - 1) // patch_size) * patch_size
pad_h = new_height - H

if pad_h > 0:
    images_swin = torch.nn.functional.pad(
        images,
        (0, 0, 0, pad_h)
    )
else:
    images_swin = images

print("Original :", tuple(images.shape))
print("Prepared :", tuple(images_swin.shape))

# ------------------------------------------------------------------------------
# DISABLE FIXED ATTENTION MASKS
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PREPARING SWIN ATTENTION FOR RECTANGULAR INPUT")
print("-" * 78)

for stage_idx, stage in enumerate(swin.layers, start=1):

    for block_idx, block in enumerate(stage.blocks, start=1):

        if hasattr(block, "attn_mask"):
            block.attn_mask = None

        print(
            f"Stage {stage_idx} Block {block_idx:02d}: "
            f"attn_mask = {getattr(block, 'attn_mask', None)}"
        )

print("✓ Rectangular attention masks disabled")

# ------------------------------------------------------------------------------
# SWIN FEATURE EXTRACTION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("RUNNING SWIN-B FEATURE EXTRACTION")
print("-" * 78)

with torch.no_grad():

    swin_features = swin.forward_features(images_swin)

print()
print("Swin feature shape:", tuple(swin_features.shape))
print("Swin feature ndim :", swin_features.ndim)

if swin_features.ndim != 4:
    raise RuntimeError(
        f"Expected Swin feature map with 4 dimensions, "
        f"got {swin_features.ndim}"
    )

# Swin output is [B,H,W,C]
B_s, H_s, W_s, C_s = swin_features.shape

print()
print("Swin layout:")
print("[B,H,W,C]")
print("B =", B_s)
print("H =", H_s)
print("W =", W_s)
print("C =", C_s)

# ------------------------------------------------------------------------------
# SWIN TOKENS
# ------------------------------------------------------------------------------

swin_tokens_raw = swin_features.reshape(
    B_s,
    H_s * W_s,
    C_s
)

print()
print("Raw Swin OCR tokens:")
print(tuple(swin_tokens_raw.shape))

# ------------------------------------------------------------------------------
# ALIGN SWIN 1024 → 768
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("ALIGNING SWIN FEATURES: 1024 → 768")
print("-" * 78)

if C_s != 1024:
    raise RuntimeError(
        f"Expected Swin-B output channels = 1024, got {C_s}"
    )

swin_projection = nn.Linear(
    1024,
    768,
    bias=False
).to(device)

swin_projection.eval()

with torch.no_grad():
    swin_tokens = swin_projection(swin_tokens_raw)

print("Aligned Swin tokens:")
print(tuple(swin_tokens.shape))

if swin_tokens.shape != (B_s, H_s * W_s, 768):
    raise RuntimeError(
        "Unexpected aligned Swin token shape: "
        + str(tuple(swin_tokens.shape))
    )

print("✓ PASS — Swin tokens = [B,N,768]")

# ==============================================================================
# PART B — SWIN → MAMBA SERIES ADAPTER
# ==============================================================================

print()
print("=" * 78)
print("PART B — SWIN → MAMBA SERIES ADAPTER")
print("=" * 78)

# ------------------------------------------------------------------------------
# TOKENS → SPATIAL
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CONVERTING SWIN TOKENS TO SPATIAL FEATURE MAP")
print("-" * 78)

swin_spatial = swin_tokens.reshape(
    B_s,
    H_s,
    W_s,
    768
)

print("[B,N,C] → [B,H,W,C]")
print(tuple(swin_spatial.shape))

# Convert to channel-first for adapter
swin_map = swin_spatial.permute(
    0, 3, 1, 2
).contiguous()

print("[B,H,W,C] → [B,C,H,W]")
print(tuple(swin_map.shape))

# ------------------------------------------------------------------------------
# SERIES ADAPTER
# ------------------------------------------------------------------------------

class SwinToMambaSeriesAdapter(nn.Module):

    def __init__(self, channels=768):

        super().__init__()

        self.norm = nn.BatchNorm2d(channels)

        self.projection = nn.Conv2d(
            channels,
            channels,
            kernel_size=1,
            bias=False
        )

        self.activation = nn.GELU()

    def forward(self, x):

        x = self.norm(x)
        x = self.projection(x)
        x = self.activation(x)

        return x


adapter = SwinToMambaSeriesAdapter(
    channels=768
).to(device)

adapter.eval()

print()
print("Adapter:")
print(adapter)

print()
print("Adapter parameters:",
      sum(p.numel() for p in adapter.parameters()))

# ------------------------------------------------------------------------------
# RUN ADAPTER
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("RUNNING SWIN → MAMBA SERIES ADAPTER")
print("-" * 78)

with torch.no_grad():

    adapted_map = adapter(swin_map)

print("Input feature map :", tuple(swin_map.shape))
print("Adapted feature map:", tuple(adapted_map.shape))

if adapted_map.shape != swin_map.shape:
    raise RuntimeError(
        "Adapter changed spatial/channel dimensions."
    )

print("✓ PASS — Adapter output shape preserved")

# ==============================================================================
# PART C — MAMBAOUT
# ==============================================================================

print()
print("=" * 78)
print("PART C — MAMBAOUT REFINEMENT")
print("=" * 78)

# ------------------------------------------------------------------------------
# CREATE MAMBAOUT
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING PRETRAINED MAMBAOUT")
print("-" * 78)

mamba = timm.create_model(
    "mambaout_base",
    pretrained=True,
    num_classes=1000
)

mamba = mamba.to(device)
mamba.eval()

print("✓ MambaOut loaded")

# ------------------------------------------------------------------------------
# GET STAGE 4
# ------------------------------------------------------------------------------

stage4 = mamba.stages[3]

print()
print("-" * 78)
print("VERIFYING MAMBAOUT STAGE 4")
print("-" * 78)

print("Stage 4:")
print(stage4)

# ------------------------------------------------------------------------------
# IMPORTANT:
# MambaOut GatedConvBlock EXPECTS [B,H,W,C]
# NOT [B,C,H,W]
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PREPARING MAMBAOUT INPUT FORMAT")
print("-" * 78)

print("Current adapter output:")
print(tuple(adapted_map.shape))

# [B,C,H,W] → [B,H,W,C]

mamba_input = adapted_map.permute(
    0, 2, 3, 1
).contiguous()

print()
print("[B,C,H,W] → [B,H,W,C]")
print("Mamba input:")
print(tuple(mamba_input.shape))

expected_mamba_shape = (
    B_s,
    H_s,
    W_s,
    768
)

if tuple(mamba_input.shape) != expected_mamba_shape:
    raise RuntimeError(
        f"Unexpected Mamba input shape. "
        f"Expected {expected_mamba_shape}, "
        f"got {tuple(mamba_input.shape)}"
    )

print("✓ PASS — MambaOut input format = [B,H,W,C]")

# ------------------------------------------------------------------------------
# IMPORTANT:
# DO NOT RUN stage4.downsample
# We already have 768 channels.
# Only use Stage 4 GatedConvBlocks.
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("RUNNING MAMBAOUT STAGE 4 BLOCKS")
print("-" * 78)

x_mamba = mamba_input

with torch.no_grad():

    for block_idx, block in enumerate(
        stage4.blocks,
        start=1
    ):

        print(
            f"Block {block_idx:02d} input:",
            tuple(x_mamba.shape)
        )

        x_mamba = block(x_mamba)

        print(
            f"Block {block_idx:02d} output:",
            tuple(x_mamba.shape)
        )

print()
print("✓ All MambaOut Stage 4 blocks executed")

# ------------------------------------------------------------------------------
# MAMBA FEATURE VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("MAMBAOUT OUTPUT VALIDATION")
print("-" * 78)

print("Mamba refined feature map:")
print(tuple(x_mamba.shape))

if x_mamba.ndim != 4:
    raise RuntimeError(
        "MambaOut output must be 4D."
    )

if x_mamba.shape[0] != B_s:
    raise RuntimeError(
        "Batch size changed."
    )

if x_mamba.shape[-1] != 768:
    raise RuntimeError(
        "MambaOut channel dimension is not 768."
    )

if torch.isnan(x_mamba).any():
    raise RuntimeError(
        "NaN detected in MambaOut output."
    )

if torch.isinf(x_mamba).any():
    raise RuntimeError(
        "Inf detected in MambaOut output."
    )

print("✓ PASS — Batch preserved")
print("✓ PASS — Feature dimension = 768")
print("✓ PASS — No NaN/Inf detected")

# ==============================================================================
# PART D — MAMBA FEATURES → OCR TOKENS
# ==============================================================================

print()
print("=" * 78)
print("PART D — MAMBAOUT → OCR TOKENS")
print("=" * 78)

# x_mamba = [B,H,W,C]

mamba_tokens = x_mamba.reshape(
    x_mamba.shape[0],
    x_mamba.shape[1] * x_mamba.shape[2],
    x_mamba.shape[3]
)

print()
print("Mamba refined feature map:")
print(tuple(x_mamba.shape))

print()
print("Mamba OCR tokens:")
print(tuple(mamba_tokens.shape))

expected_tokens = (
    B_s,
    H_s * W_s,
    768
)

if tuple(mamba_tokens.shape) != expected_tokens:
    raise RuntimeError(
        f"Unexpected OCR token shape. "
        f"Expected {expected_tokens}, "
        f"got {tuple(mamba_tokens.shape)}"
    )

print()
print("Expected:", expected_tokens)
print("Actual  :", tuple(mamba_tokens.shape))

print("✓ PASS — Batch preserved")
print("✓ PASS — Token count preserved")
print("✓ PASS — Feature dimension = 768")
print("✓ PASS — Mamba OCR tokens = [B,N,768]")

# ------------------------------------------------------------------------------
# TOKEN NUMERICAL VALIDATION
# ------------------------------------------------------------------------------

if torch.isnan(mamba_tokens).any():
    raise RuntimeError(
        "NaN detected in final Mamba tokens."
    )

if torch.isinf(mamba_tokens).any():
    raise RuntimeError(
        "Inf detected in final Mamba tokens."
    )

print("✓ PASS — No NaN/Inf detected")

# ==============================================================================
# PART E — SAVE
# ==============================================================================

print()
print("=" * 78)
print("SAVING FINAL SERIES TOKENS")
print("=" * 78)

output_path = (
    "/content/pfms_series_swin_b_to_mambaout_tokens.pt"
)

torch.save(
    mamba_tokens.detach().cpu(),
    output_path
)

print()
print("Saved:")
print(output_path)

# ==============================================================================
# FINAL REPORT
# ==============================================================================

print()
print("=" * 78)
print("CELL 33-FINAL COMPLETED")
print("=" * 78)

print()
print("PFMS-SERIES FINAL STATUS:")
print()
print("IDPL-PFOD input       :", tuple(images.shape))
print("Swin prepared input   :", tuple(images_swin.shape))
print("Swin feature map      :", tuple(swin_features.shape))
print("Swin raw tokens       :", tuple(swin_tokens_raw.shape))
print("Swin aligned tokens   :", tuple(swin_tokens.shape))
print("Swin spatial map      :", tuple(swin_map.shape))
print("Adapter output        :", tuple(adapted_map.shape))
print("Mamba input           :", tuple(mamba_input.shape))
print("Mamba refined map     :", tuple(x_mamba.shape))
print("Final Mamba OCR tokens:", tuple(mamba_tokens.shape))

print()
print("SERIES PIPELINE:")
print()
print("[B,3,50,700]")
print("      ↓")
print(f"[B,{H_s},{W_s},1024]")
print("      ↓")
print(f"[B,{H_s * W_s},1024]")
print("      ↓")
print(f"[B,{H_s * W_s},768]")
print("      ↓")
print(f"[B,768,{H_s},{W_s}]")
print("      ↓")
print(f"[B,{H_s},{W_s},768]")
print("      ↓")
print("MambaOut Stage 4")
print("      ↓")
print(f"[B,{H_s},{W_s},768]")
print("      ↓")
print(f"[B,{H_s * W_s},768]")

print()
print("✓ SWIN-B → MAMBAOUT SERIES CONNECTION VALIDATED")
print("=" * 78)

PFMS-SERIES — CELL 33-FINAL
REAL SWIN-B → MAMBAOUT SERIES CONNECTION

Device: cuda:0
GPU: Tesla T4

------------------------------------------------------------------------------
LOADING REAL IDPL-PFOD BATCH
------------------------------------------------------------------------------
Image shape: (8, 3, 50, 700)
Image dtype: torch.float32
Image device: cuda:0

Batch size : 8
Channels   : 3
Height     : 50
Width      : 700
✓ PASS — Real OCR input = [B,3,50,700]

PART A — SWIN-B ENCODER

------------------------------------------------------------------------------
CREATING PRETRAINED SWIN-B
------------------------------------------------------------------------------
✓ Swin-B loaded

------------------------------------------------------------------------------
CONFIGURING RECTANGULAR INPUT
------------------------------------------------------------------------------
strict_img_size : False
dynamic_img_pad : True

---------------------------------------------------------------------

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 34
# SWIN-B → MAMBAOUT → DTROCR INTERFACE VALIDATION
# ==============================================================================

import os
import torch
import torch.nn as nn

print("=" * 78)
print("PFMS-SERIES — CELL 34")
print("SWIN-B → MAMBAOUT → DTROCR INTERFACE VALIDATION")
print("=" * 78)

# ------------------------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------------------------

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print()
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: CPU")

# ------------------------------------------------------------------------------
# LOAD FINAL SERIES TOKENS
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("LOADING FINAL SERIES TOKENS")
print("-" * 78)

token_path = "/content/pfms_series_swin_b_to_mambaout_tokens.pt"

print("Token file:")
print(token_path)

if not os.path.exists(token_path):
    raise FileNotFoundError(
        "Final Series token file was not found:\n"
        + token_path
        + "\nPlease run CELL 33-FINAL first."
    )

series_tokens = torch.load(
    token_path,
    map_location=device
)

print("✓ Series tokens loaded")

# ------------------------------------------------------------------------------
# VALIDATE TYPE
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("TOKEN TYPE VALIDATION")
print("-" * 78)

if not isinstance(series_tokens, torch.Tensor):
    raise TypeError(
        "Expected a torch.Tensor, but received: "
        + str(type(series_tokens))
    )

print("Tensor type :", type(series_tokens))
print("Tensor shape:", tuple(series_tokens.shape))
print("Tensor dtype:", series_tokens.dtype)
print("Tensor device:", series_tokens.device)

# ------------------------------------------------------------------------------
# BASIC SHAPE VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("SERIES TOKEN VALIDATION")
print("-" * 78)

if series_tokens.ndim != 3:
    raise RuntimeError(
        "Expected 3D OCR tokens [B,N,C], "
        f"but received ndim={series_tokens.ndim}"
    )

B, N, C = series_tokens.shape

print("Batch size :", B)
print("Token count:", N)
print("Channels   :", C)

if C != 768:
    raise RuntimeError(
        f"Expected feature dimension 768, but received {C}"
    )

print("✓ PASS — Series tokens = [B,N,768]")

if torch.isnan(series_tokens).any():
    raise RuntimeError("NaN detected in Series tokens")

if torch.isinf(series_tokens).any():
    raise RuntimeError("Inf detected in Series tokens")

print("✓ PASS — No NaN/Inf detected")

# ------------------------------------------------------------------------------
# DTROCR / TRANSFORMER INTERFACE REQUIREMENTS
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("DTROCR INTERFACE REQUIREMENTS")
print("-" * 78)

print()
print("Series encoder output:")
print("  [B,N,C]")
print("  ", tuple(series_tokens.shape))

print()
print("Expected decoder embedding dimension:")
print("  768")

print()
print("Expected decoder input:")
print("  [B,N,768]")

print()
print("Current Series output:")
print("  [B,N,768]")

print()
print("✓ PASS — Encoder/Decoder feature dimension compatible")

# ------------------------------------------------------------------------------
# OPTIONAL DECODER PROJECTION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING DECODER INTERFACE PROJECTION")
print("-" * 78)

decoder_dim = 768

if C == decoder_dim:
    decoder_projection = nn.Identity().to(device)

    print("Encoder dimension:", C)
    print("Decoder dimension:", decoder_dim)
    print("Projection:", "Identity")
    print("✓ No dimensional projection required")

else:
    decoder_projection = nn.Linear(
        C,
        decoder_dim
    ).to(device)

    print("Encoder dimension:", C)
    print("Decoder dimension:", decoder_dim)
    print("Projection:", "Linear")

# ------------------------------------------------------------------------------
# RUN DECODER INTERFACE
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("RUNNING DTROCR INTERFACE")
print("-" * 78)

decoder_input = decoder_projection(series_tokens)

print()
print("Input to interface :", tuple(series_tokens.shape))
print("Output from interface:", tuple(decoder_input.shape))

# ------------------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("FINAL INTERFACE VALIDATION")
print("-" * 78)

expected_shape = (B, N, 768)
actual_shape = tuple(decoder_input.shape)

print("Expected:", expected_shape)
print("Actual  :", actual_shape)

if actual_shape != expected_shape:
    raise RuntimeError(
        f"DTROCR interface shape mismatch: "
        f"expected {expected_shape}, got {actual_shape}"
    )

print("✓ PASS — Batch preserved")
print("✓ PASS — Token count preserved")
print("✓ PASS — Feature dimension = 768")
print("✓ PASS — Decoder input = [B,N,768]")

if torch.isnan(decoder_input).any():
    raise RuntimeError("NaN detected after decoder projection")

if torch.isinf(decoder_input).any():
    raise RuntimeError("Inf detected after decoder projection")

print("✓ PASS — No NaN/Inf detected")

# ------------------------------------------------------------------------------
# SAVE DECODER INTERFACE TOKENS
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("SAVING DTROCR INTERFACE TOKENS")
print("-" * 78)

decoder_input_path = (
    "/content/pfms_series_dtrocr_interface_tokens.pt"
)

torch.save(
    decoder_input.detach().cpu(),
    decoder_input_path
)

print("Saved:")
print(decoder_input_path)

# ------------------------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("CELL 34 COMPLETED")
print("=" * 78)

print()
print("PFMS-SERIES CURRENT STATUS:")
print()

print("IDPL-PFOD input       : (8,3,50,700)")
print("Swin-B feature map    : (8,2,22,1024)")
print("Swin aligned tokens   : (8,44,768)")
print("Mamba refined map     : (8,2,22,768)")
print("Final Series tokens   :", tuple(series_tokens.shape))
print("DTrOCR interface      :", tuple(decoder_input.shape))

print()
print("SERIES PIPELINE:")
print()
print("[B,3,50,700]")
print("      ↓")
print("Swin-B")
print("      ↓")
print("[B,44,1024]")
print("      ↓")
print("Projection")
print("      ↓")
print("[B,44,768]")
print("      ↓")
print("Swin → Mamba Adapter")
print("      ↓")
print("MambaOut Stage 4")
print("      ↓")
print("[B,44,768]")
print("      ↓")
print("DTrOCR Interface")
print("      ↓")
print("[B,44,768]")

print()
print("✓ SWIN-B → MAMBAOUT → DTROCR INTERFACE VALIDATED")
print("=" * 78)

PFMS-SERIES — CELL 34
SWIN-B → MAMBAOUT → DTROCR INTERFACE VALIDATION

Device: cuda:0
GPU: Tesla T4

------------------------------------------------------------------------------
LOADING FINAL SERIES TOKENS
------------------------------------------------------------------------------
Token file:
/content/pfms_series_swin_b_to_mambaout_tokens.pt
✓ Series tokens loaded

------------------------------------------------------------------------------
TOKEN TYPE VALIDATION
------------------------------------------------------------------------------
Tensor type : <class 'torch.Tensor'>
Tensor shape: (8, 44, 768)
Tensor dtype: torch.float32
Tensor device: cuda:0

------------------------------------------------------------------------------
SERIES TOKEN VALIDATION
------------------------------------------------------------------------------
Batch size : 8
Token count: 44
Channels   : 768
✓ PASS — Series tokens = [B,N,768]
✓ PASS — No NaN/Inf detected

-------------------------------------

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 35
# REAL TRANSFORMER OCR DECODER CONNECTION
# SWIN-B → MAMBAOUT → TRANSFORMER DECODER
# ==============================================================================

import os
import torch
import torch.nn as nn

print("=" * 78)
print("PFMS-SERIES — CELL 35")
print("REAL TRANSFORMER OCR DECODER CONNECTION")
print("=" * 78)

# ------------------------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------------------------

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print()
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------------------------
# LOAD SERIES ENCODER OUTPUT
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("LOADING SERIES ENCODER OUTPUT")
print("-" * 78)

token_path = "/content/pfms_series_swin_b_to_mambaout_tokens.pt"

if not os.path.exists(token_path):
    raise FileNotFoundError(
        "Series token file not found:\n"
        + token_path
        + "\nPlease run CELL 33-FINAL first."
    )

encoder_tokens = torch.load(
    token_path,
    map_location=device
)

if not isinstance(encoder_tokens, torch.Tensor):
    raise TypeError(
        "Expected encoder output to be a torch.Tensor."
    )

print("Token file:")
print(token_path)

print()
print("Encoder tensor:")
print("Shape :", tuple(encoder_tokens.shape))
print("Dtype :", encoder_tokens.dtype)
print("Device:", encoder_tokens.device)

# ------------------------------------------------------------------------------
# ENCODER VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("ENCODER OUTPUT VALIDATION")
print("-" * 78)

if encoder_tokens.ndim != 3:
    raise RuntimeError(
        "Encoder output must be [B,N,C]."
    )

B, N, C = encoder_tokens.shape

print("Batch size :", B)
print("Token count:", N)
print("Channels   :", C)

if C != 768:
    raise RuntimeError(
        f"Expected encoder dimension 768, got {C}."
    )

if torch.isnan(encoder_tokens).any():
    raise RuntimeError(
        "NaN detected in encoder tokens."
    )

if torch.isinf(encoder_tokens).any():
    raise RuntimeError(
        "Inf detected in encoder tokens."
    )

print("✓ PASS — Encoder output = [B,N,768]")
print("✓ PASS — No NaN/Inf detected")

# ------------------------------------------------------------------------------
# TRANSFORMER DECODER CONFIGURATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("TRANSFORMER OCR DECODER CONFIGURATION")
print("-" * 78)

d_model = 768
nhead = 8
num_decoder_layers = 4
dim_feedforward = 2048
dropout = 0.1

# Temporary vocabulary for interface validation.
# The real Persian OCR vocabulary will be introduced during training.
vocab_size = 256

max_target_length = 128

print("d_model           :", d_model)
print("Attention heads   :", nhead)
print("Decoder layers    :", num_decoder_layers)
print("FFN dimension     :", dim_feedforward)
print("Dropout           :", dropout)
print("Vocabulary size   :", vocab_size)
print("Max target length :", max_target_length)

# ------------------------------------------------------------------------------
# TOKEN EMBEDDING
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING TARGET TOKEN EMBEDDING")
print("-" * 78)

target_embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=d_model
).to(device)

print(target_embedding)

# ------------------------------------------------------------------------------
# POSITIONAL EMBEDDING
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING POSITIONAL EMBEDDING")
print("-" * 78)

target_pos_embedding = nn.Parameter(
    torch.zeros(
        1,
        max_target_length,
        d_model,
        device=device
    )
)

nn.init.normal_(
    target_pos_embedding,
    mean=0.0,
    std=0.02
)

print(
    "Positional embedding shape:",
    tuple(target_pos_embedding.shape)
)

# ------------------------------------------------------------------------------
# TRANSFORMER DECODER
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING TRANSFORMER DECODER")
print("-" * 78)

decoder_layer = nn.TransformerDecoderLayer(
    d_model=d_model,
    nhead=nhead,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation="gelu",
    batch_first=True,
    norm_first=True
)

transformer_decoder = nn.TransformerDecoder(
    decoder_layer,
    num_layers=num_decoder_layers
).to(device)

print(transformer_decoder)

# ------------------------------------------------------------------------------
# OUTPUT CLASSIFIER
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING OCR OUTPUT CLASSIFIER")
print("-" * 78)

output_projection = nn.Linear(
    d_model,
    vocab_size
).to(device)

print(
    "Output projection:",
    f"{d_model} → {vocab_size}"
)

# ------------------------------------------------------------------------------
# CREATE DUMMY TARGET SEQUENCE
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING TEMPORARY TARGET SEQUENCE")
print("-" * 78)

target_length = min(
    32,
    max_target_length
)

# Temporary token IDs.
# These are ONLY for decoder interface testing.
target_ids = torch.randint(
    low=0,
    high=vocab_size,
    size=(B, target_length),
    device=device,
    dtype=torch.long
)

print("Target IDs shape:")
print(tuple(target_ids.shape))

print("Target length:", target_length)

# ------------------------------------------------------------------------------
# TARGET EMBEDDING
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("TARGET EMBEDDING")
print("-" * 78)

target_emb = target_embedding(
    target_ids
)

print("Target IDs :", tuple(target_ids.shape))
print("Target emb :", tuple(target_emb.shape))

expected_target_embedding = (
    B,
    target_length,
    d_model
)

if tuple(target_emb.shape) != expected_target_embedding:
    raise RuntimeError(
        "Target embedding shape mismatch."
    )

print("✓ PASS — Target embedding = [B,T,768]")

# ------------------------------------------------------------------------------
# ADD POSITIONAL EMBEDDING
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("ADDING POSITIONAL INFORMATION")
print("-" * 78)

target_emb = (
    target_emb
    + target_pos_embedding[:, :target_length, :]
)

print(
    "Position-enhanced target:",
    tuple(target_emb.shape)
)

print("✓ PASS — Positional information added")

# ------------------------------------------------------------------------------
# CAUSAL MASK
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING CAUSAL ATTENTION MASK")
print("-" * 78)

causal_mask = torch.triu(
    torch.ones(
        target_length,
        target_length,
        device=device,
        dtype=torch.bool
    ),
    diagonal=1
)

print("Causal mask shape:")
print(tuple(causal_mask.shape))

print("✓ PASS — Causal mask created")

# ------------------------------------------------------------------------------
# TRANSFORMER DECODER FORWARD
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("RUNNING TRANSFORMER OCR DECODER")
print("=" * 78)

print()
print("Memory / encoder input:")
print(tuple(encoder_tokens.shape))

print("Target input:")
print(tuple(target_emb.shape))

with torch.no_grad():

    decoder_output = transformer_decoder(
        tgt=target_emb,
        memory=encoder_tokens,
        tgt_mask=causal_mask
    )

print()
print("Decoder output:")
print(tuple(decoder_output.shape))

# ------------------------------------------------------------------------------
# OUTPUT VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("DECODER OUTPUT VALIDATION")
print("-" * 78)

expected_decoder_shape = (
    B,
    target_length,
    d_model
)

actual_decoder_shape = tuple(
    decoder_output.shape
)

print("Expected:", expected_decoder_shape)
print("Actual  :", actual_decoder_shape)

if actual_decoder_shape != expected_decoder_shape:
    raise RuntimeError(
        "Transformer decoder output shape mismatch."
    )

print("✓ PASS — Batch preserved")
print("✓ PASS — Target sequence preserved")
print("✓ PASS — Decoder dimension = 768")

if torch.isnan(decoder_output).any():
    raise RuntimeError(
        "NaN detected in decoder output."
    )

if torch.isinf(decoder_output).any():
    raise RuntimeError(
        "Inf detected in decoder output."
    )

print("✓ PASS — No NaN/Inf detected")

# ------------------------------------------------------------------------------
# OCR CLASSIFICATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("OCR TOKEN CLASSIFICATION")
print("-" * 78)

with torch.no_grad():

    logits = output_projection(
        decoder_output
    )

print("Decoder output:", tuple(decoder_output.shape))
print("Logits shape   :", tuple(logits.shape))

expected_logits_shape = (
    B,
    target_length,
    vocab_size
)

if tuple(logits.shape) != expected_logits_shape:
    raise RuntimeError(
        "OCR logits shape mismatch."
    )

print("✓ PASS — OCR logits = [B,T,V]")

# ------------------------------------------------------------------------------
# PREDICTED TOKEN IDs
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("GENERATING TEMPORARY TOKEN PREDICTIONS")
print("-" * 78)

predicted_ids = torch.argmax(
    logits,
    dim=-1
)

print(
    "Predicted token IDs:",
    tuple(predicted_ids.shape)
)

if predicted_ids.shape != (
    B,
    target_length
):
    raise RuntimeError(
        "Prediction shape mismatch."
    )

print("✓ PASS — Predicted IDs = [B,T]")

# ------------------------------------------------------------------------------
# SAVE INTERFACE OUTPUT
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("SAVING DECODER INTERFACE OUTPUT")
print("-" * 78)

decoder_output_path = (
    "/content/pfms_series_transformer_decoder_output.pt"
)

logits_path = (
    "/content/pfms_series_transformer_ocr_logits.pt"
)

torch.save(
    decoder_output.detach().cpu(),
    decoder_output_path
)

torch.save(
    logits.detach().cpu(),
    logits_path
)

print("Decoder output saved:")
print(decoder_output_path)

print()
print("OCR logits saved:")
print(logits_path)

# ------------------------------------------------------------------------------
# FINAL PIPELINE VALIDATION
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("CELL 35 FINAL VALIDATION")
print("=" * 78)

print()
print("PFMS-SERIES COMPLETE FORWARD INTERFACE:")
print()

print("[B,3,50,700]")
print("      ↓")
print("Swin-B")
print("      ↓")
print("[B,44,1024]")
print("      ↓")
print("Projection 1024 → 768")
print("      ↓")
print("[B,44,768]")
print("      ↓")
print("Swin → Mamba Adapter")
print("      ↓")
print("MambaOut Stage 4")
print("      ↓")
print("[B,44,768]")
print("      ↓")
print("Transformer OCR Decoder")
print("      ↓")
print("[B,T,768]")
print("      ↓")
print("Output Projection")
print("      ↓")
print("[B,T,V]")
print("      ↓")
print("Predicted Token IDs")
print("      ↓")
print("[B,T]")

print()
print("Encoder tokens :", tuple(encoder_tokens.shape))
print("Decoder output :", tuple(decoder_output.shape))
print("OCR logits     :", tuple(logits.shape))
print("Predicted IDs  :", tuple(predicted_ids.shape))

print()
print("✓ SWIN-B → MAMBAOUT → TRANSFORMER DECODER")
print("✓ COMPLETE FORWARD INTERFACE VALIDATED")

print("=" * 78)
print("CELL 35 COMPLETED")
print("=" * 78)

PFMS-SERIES — CELL 35
REAL TRANSFORMER OCR DECODER CONNECTION

Device: cuda:0
GPU: Tesla T4

------------------------------------------------------------------------------
LOADING SERIES ENCODER OUTPUT
------------------------------------------------------------------------------
Token file:
/content/pfms_series_swin_b_to_mambaout_tokens.pt

Encoder tensor:
Shape : (8, 44, 768)
Dtype : torch.float32
Device: cuda:0

------------------------------------------------------------------------------
ENCODER OUTPUT VALIDATION
------------------------------------------------------------------------------
Batch size : 8
Token count: 44
Channels   : 768
✓ PASS — Encoder output = [B,N,768]
✓ PASS — No NaN/Inf detected

------------------------------------------------------------------------------
TRANSFORMER OCR DECODER CONFIGURATION
------------------------------------------------------------------------------
d_model           : 768
Attention heads   : 8
Decoder layers    : 4
FFN dimension     :

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 36-FINAL-FIXED
# REAL IDPL-PFOD + TRANSFORMER OCR TRAINING STEP
# SWIN-B RECTANGULAR INPUT FIX
# ==============================================================================

import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

print("=" * 78)
print("PFMS-SERIES — CELL 36-FINAL-FIXED")
print("REAL IDPL-PFOD + TRANSFORMER OCR TRAINING STEP")
print("=" * 78)

# ------------------------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print()
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------------------

IMAGE_H = 50
IMAGE_W = 700

BATCH_SIZE = 8

SWIN_DIM = 1024
MODEL_DIM = 768

NUM_ENCODER_TOKENS = 44
VOCAB_SIZE = 256
TARGET_LENGTH = 32

DECODER_HEADS = 8
DECODER_LAYERS = 4
FFN_DIM = 2048

LEARNING_RATE = 1e-4

DATASET_PATH = "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"

print()
print("-" * 78)
print("CONFIGURATION")
print("-" * 78)

print("Image size       :", (IMAGE_H, IMAGE_W))
print("Swin dimension   :", SWIN_DIM)
print("Model dimension  :", MODEL_DIM)
print("Encoder tokens   :", NUM_ENCODER_TOKENS)
print("Vocabulary size  :", VOCAB_SIZE)
print("Target length    :", TARGET_LENGTH)
print("Decoder heads    :", DECODER_HEADS)
print("Decoder layers   :", DECODER_LAYERS)
print("Learning rate    :", LEARNING_RATE)

# ==============================================================================
# PART A — REAL IDPL-PFOD BATCH
# ==============================================================================

print()
print("=" * 78)
print("PART A — REAL IDPL-PFOD BATCH")
print("=" * 78)

# ------------------------------------------------------------------------------
# DATASET VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("DATASET VALIDATION")
print("-" * 78)

print("Dataset path:")
print(DATASET_PATH)

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset path does not exist:\n{DATASET_PATH}"
    )

print("✓ Dataset path exists")

# ------------------------------------------------------------------------------
# SEARCH REAL IMAGES
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("SEARCHING REAL IDPL-PFOD IMAGES")
print("-" * 78)

image_extensions = ["*.tif", "*.tiff", "*.png", "*.jpg", "*.jpeg"]

image_files = []

for ext in image_extensions:
    image_files.extend(
        glob.glob(os.path.join(DATASET_PATH, "**", ext), recursive=True)
    )

image_files = sorted(list(set(image_files)))

print("Total image files found:", len(image_files))

if len(image_files) == 0:
    raise RuntimeError("No IDPL-PFOD images found.")

print("✓ Real images available")

# ------------------------------------------------------------------------------
# LOAD BATCH
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("LOADING REAL IMAGE BATCH")
print("-" * 78)

# --------------------------------------------------------------------------
# IMPORTANT:
# Try to reuse an existing real batch if available.
# Otherwise load images directly.
# --------------------------------------------------------------------------

images = None
batch_object = None

# Search for existing batch variables
possible_batch_names = [
    "batch",
    "real_batch",
    "image_batch",
    "idpl_batch"
]

for name in possible_batch_names:
    if name in globals():
        candidate = globals()[name]

        if isinstance(candidate, torch.Tensor):
            if candidate.ndim == 4:
                images = candidate
                batch_object = candidate
                print("Using existing tensor batch:", name)
                break

        elif isinstance(candidate, dict):
            for key in ["image", "images", "pixel_values", "input", "inputs"]:
                if key in candidate:
                    candidate_images = candidate[key]

                    if isinstance(candidate_images, torch.Tensor):
                        if candidate_images.ndim == 4:
                            images = candidate_images
                            batch_object = candidate
                            print("Using existing dictionary batch:", name)
                            break

            if images is not None:
                break

# ------------------------------------------------------------------------------
# FALLBACK IMAGE LOADER
# ------------------------------------------------------------------------------

if images is None:

    print("No compatible existing batch found.")
    print("Creating a real image batch from IDPL-PFOD...")

    from PIL import Image
    import numpy as np

    selected_files = image_files[:BATCH_SIZE]

    image_list = []

    for file_path in selected_files:

        img = Image.open(file_path).convert("RGB")

        img = img.resize(
            (IMAGE_W, IMAGE_H),
            Image.Resampling.BILINEAR
        )

        arr = np.asarray(img).astype("float32") / 255.0

        tensor = torch.from_numpy(arr).permute(2, 0, 1)

        image_list.append(tensor)

    images = torch.stack(image_list, dim=0)

    batch_object = {
        "images": images,
        "paths": selected_files
    }

    print("✓ Real images loaded")

# ------------------------------------------------------------------------------
# IMAGE VALIDATION
# ------------------------------------------------------------------------------

images = images.float().to(device)

print()
print("Image batch shape :", tuple(images.shape))
print("Image batch dtype :", images.dtype)
print("Image batch device:", images.device)

if images.ndim != 4:
    raise RuntimeError(
        f"Expected 4D image tensor, got {images.ndim}D"
    )

if images.shape[1] != 3:
    raise RuntimeError(
        f"Expected 3 channels, got {images.shape[1]}"
    )

print("✓ PASS — Real image batch loaded")

print()
print("Final image input:", tuple(images.shape))

if tuple(images.shape[1:]) != (3, IMAGE_H, IMAGE_W):
    raise RuntimeError(
        f"Expected [B,3,{IMAGE_H},{IMAGE_W}], "
        f"got {tuple(images.shape)}"
    )

print("✓ PASS — Input = [B,3,50,700]")

if not torch.isfinite(images).all():
    raise RuntimeError("Input contains NaN/Inf")

print("✓ PASS — No NaN/Inf detected")

# ==============================================================================
# PART B — SWIN-B ENCODER
# ==============================================================================

print()
print("=" * 78)
print("PART B — SWIN-B ENCODER")
print("=" * 78)

# ------------------------------------------------------------------------------
# CREATE PRETRAINED SWIN-B
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING PRETRAINED SWIN-B")
print("-" * 78)

swin = timm.create_model(
    "swin_base_patch4_window7_224",
    pretrained=True,
    num_classes=0
)

swin = swin.to(device)
swin.eval()

print("✓ Swin-B loaded")

# ==============================================================================
# CRITICAL RECTANGULAR INPUT FIX
# ==============================================================================

print()
print("-" * 78)
print("APPLYING SWIN-B RECTANGULAR INPUT FIX")
print("-" * 78)

print("Original patch_embed.img_size:",
      getattr(swin.patch_embed, "img_size", None))

print("Original strict_img_size:",
      getattr(swin.patch_embed, "strict_img_size", None))

print("Original dynamic_img_pad:",
      getattr(swin.patch_embed, "dynamic_img_pad", None))

# ------------------------------------------------------------------------------
# THIS IS THE IMPORTANT FIX
# ------------------------------------------------------------------------------

swin.patch_embed.strict_img_size = False
swin.patch_embed.dynamic_img_pad = True

print()
print("After correction:")

print("strict_img_size:",
      swin.patch_embed.strict_img_size)

print("dynamic_img_pad:",
      swin.patch_embed.dynamic_img_pad)

if swin.patch_embed.strict_img_size:
    raise RuntimeError(
        "Swin strict_img_size is still True."
    )

print("✓ PASS — strict_img_size disabled")
print("✓ PASS — dynamic_img_pad enabled")

# ==============================================================================
# PREPARE RECTANGULAR IMAGE
# ==============================================================================

print()
print("-" * 78)
print("PREPARING RECTANGULAR INPUT")
print("-" * 78)

images_swin = images

print("Original input:", tuple(images_swin.shape))

# Swin patch size = 4.
# Height 50 is not divisible by 4.
# Therefore pad height to 52.
#
# Width 700 is already divisible by 4.

pad_h = (4 - (images_swin.shape[2] % 4)) % 4
pad_w = (4 - (images_swin.shape[3] % 4)) % 4

if pad_h > 0 or pad_w > 0:

    images_swin = F.pad(
        images_swin,
        (0, pad_w, 0, pad_h),
        mode="constant",
        value=0
    )

print("Prepared input:", tuple(images_swin.shape))

print("Padding:")
print("Height:", pad_h)
print("Width :", pad_w)

# Expected:
# [8,3,52,700]

if images_swin.shape[2] % 4 != 0:
    raise RuntimeError("Prepared height is not divisible by patch size.")

if images_swin.shape[3] % 4 != 0:
    raise RuntimeError("Prepared width is not divisible by patch size.")

print("✓ PASS — Rectangular input compatible with 4×4 patches")

# ==============================================================================
# DISABLE STALE SWIN ATTENTION MASKS
# ==============================================================================

print()
print("-" * 78)
print("PREPARING SWIN ATTENTION MASKS")
print("-" * 78)

mask_count = 0

for layer_idx, layer in enumerate(swin.layers):

    if hasattr(layer, "blocks"):

        for block_idx, block in enumerate(layer.blocks):

            if hasattr(block, "attn"):

                if hasattr(block.attn, "set_input_size"):

                    try:
                        block.attn.set_input_size(
                            block.window_size
                        )
                    except Exception:
                        pass

                if hasattr(block.attn, "attn_mask"):
                    block.attn.attn_mask = None
                    mask_count += 1

            if hasattr(block, "attn_mask"):
                block.attn_mask = None

print("Attention masks disabled:", mask_count)

print("✓ Swin attention masks prepared for rectangular input")

# ==============================================================================
# PATCH EMBEDDING TEST
# ==============================================================================

print()
print("-" * 78)
print("TESTING SWIN PATCH EMBEDDING")
print("-" * 78)

with torch.no_grad():

    patch_output = swin.patch_embed(images_swin)

print("Patch output shape:",
      tuple(patch_output.shape))

print("Patch output ndim:",
      patch_output.ndim)

print("✓ PASS — Patch embedding successful")

# ==============================================================================
# SWIN FEATURE EXTRACTION
# ==============================================================================

print()
print("-" * 78)
print("RUNNING SWIN-B FEATURE EXTRACTION")
print("-" * 78)

with torch.no_grad():

    swin_features = swin.forward_features(images_swin)

print()
print("Swin feature shape:",
      tuple(swin_features.shape))

print("Swin feature ndim:",
      swin_features.ndim)

if not torch.isfinite(swin_features).all():
    raise RuntimeError(
        "Swin features contain NaN/Inf"
    )

print("✓ PASS — Swin feature extraction successful")
print("✓ PASS — No NaN/Inf detected")

# ==============================================================================
# SWIN FEATURE LAYOUT
# ==============================================================================

print()
print("-" * 78)
print("ANALYZING SWIN FEATURE LAYOUT")
print("-" * 78)

if swin_features.ndim != 4:
    raise RuntimeError(
        f"Expected 4D Swin feature map, got {swin_features.ndim}D"
    )

B, H, W, C = swin_features.shape

print("Swin layout:")
print("[B,H,W,C]")

print("B =", B)
print("H =", H)
print("W =", W)
print("C =", C)

# ==============================================================================
# CONVERT SWIN FEATURES TO OCR TOKENS
# ==============================================================================

print()
print("-" * 78)
print("CONVERTING SWIN FEATURES TO OCR TOKENS")
print("-" * 78)

swin_tokens = swin_features.reshape(
    B,
    H * W,
    C
)

print("Raw Swin OCR tokens:",
      tuple(swin_tokens.shape))

if swin_tokens.shape[1] != NUM_ENCODER_TOKENS:
    raise RuntimeError(
        f"Expected {NUM_ENCODER_TOKENS} tokens, "
        f"got {swin_tokens.shape[1]}"
    )

print("✓ PASS — Token count = 44")

# ==============================================================================
# ALIGN SWIN 1024 → MODEL 768
# ==============================================================================

print()
print("-" * 78)
print("ALIGNING SWIN FEATURES: 1024 → 768")
print("-" * 78)

swin_projection = nn.Linear(
    SWIN_DIM,
    MODEL_DIM,
    bias=False
).to(device)

swin_projection.eval()

with torch.no_grad():

    aligned_swin_tokens = swin_projection(
        swin_tokens
    )

print("Aligned Swin tokens:",
      tuple(aligned_swin_tokens.shape))

if aligned_swin_tokens.shape != (
    B,
    NUM_ENCODER_TOKENS,
    MODEL_DIM
):
    raise RuntimeError(
        f"Unexpected aligned token shape: "
        f"{tuple(aligned_swin_tokens.shape)}"
    )

print("✓ PASS — Swin tokens = [B,44,768]")

# ==============================================================================
# PART C — SWIN → MAMBA SERIES ADAPTER
# ==============================================================================

print()
print("=" * 78)
print("PART C — SWIN → MAMBA SERIES ADAPTER")
print("=" * 78)

# ------------------------------------------------------------------------------
# TOKEN → SPATIAL
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CREATING SPATIAL SERIES REPRESENTATION")
print("-" * 78)

series_h = 2
series_w = 22

if series_h * series_w != NUM_ENCODER_TOKENS:
    raise RuntimeError("Invalid spatial token configuration.")

swin_spatial = aligned_swin_tokens.reshape(
    B,
    series_h,
    series_w,
    MODEL_DIM
)

print("[B,N,C] → [B,H,W,C]")
print(tuple(swin_spatial.shape))

swin_spatial_cf = swin_spatial.permute(
    0, 3, 1, 2
).contiguous()

print("[B,H,W,C] → [B,C,H,W]")
print(tuple(swin_spatial_cf.shape))

# ------------------------------------------------------------------------------
# ADAPTER
# ------------------------------------------------------------------------------

class SwinToMambaSeriesAdapter(nn.Module):

    def __init__(self, channels=768):

        super().__init__()

        self.norm = nn.BatchNorm2d(channels)

        self.projection = nn.Conv2d(
            channels,
            channels,
            kernel_size=1,
            bias=False
        )

        self.activation = nn.GELU()

    def forward(self, x):

        x = self.norm(x)
        x = self.projection(x)
        x = self.activation(x)

        return x


adapter = SwinToMambaSeriesAdapter(
    channels=MODEL_DIM
).to(device)

adapter.eval()

print()
print("Adapter:")
print(adapter)

print()
print("Adapter parameters:",
      sum(p.numel() for p in adapter.parameters()))

# ------------------------------------------------------------------------------
# RUN ADAPTER
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("RUNNING SWIN → MAMBA SERIES ADAPTER")
print("-" * 78)

with torch.no_grad():

    adapted_map = adapter(
        swin_spatial_cf
    )

print("Input feature map :",
      tuple(swin_spatial_cf.shape))

print("Adapted feature map:",
      tuple(adapted_map.shape))

if adapted_map.shape != (
    B,
    MODEL_DIM,
    series_h,
    series_w
):
    raise RuntimeError(
        "Adapter output shape incorrect."
    )

print("✓ PASS — Adapter output shape preserved")

# ==============================================================================
# PART D — MAMBAOUT
# ==============================================================================

print()
print("=" * 78)
print("PART D — MAMBAOUT REFINEMENT")
print("=" * 78)

print()
print("-" * 78)
print("CREATING PRETRAINED MAMBAOUT")
print("-" * 78)

mamba = timm.create_model(
    "mambaout_base",
    pretrained=True,
    num_classes=1000
)

mamba = mamba.to(device)
mamba.eval()

print("✓ MambaOut loaded")

stage4 = mamba.stages[3]

print()
print("-" * 78)
print("VERIFYING MAMBAOUT STAGE 4")
print("-" * 78)

print("Stage 4 type:")
print(type(stage4))

print()
print("Stage 4 blocks:")
print(stage4.blocks)

# ------------------------------------------------------------------------------
# CRITICAL:
# DO NOT RUN stage4.downsample
#
# Swin output is already:
# [B,H,W,768]
#
# Therefore only Stage 4 blocks are applied.
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# B,C,H,W → B,H,W,C
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PREPARING MAMBAOUT INPUT FORMAT")
print("-" * 78)

mamba_input = adapted_map.permute(
    0, 2, 3, 1
).contiguous()

print("Current adapter output:")
print(tuple(adapted_map.shape))

print()
print("[B,C,H,W] → [B,H,W,C]")

print("Mamba input:")
print(tuple(mamba_input.shape))

if mamba_input.shape != (
    B,
    series_h,
    series_w,
    MODEL_DIM
):
    raise RuntimeError(
        "Mamba input shape incorrect."
    )

print("✓ PASS — MambaOut input format = [B,H,W,C]")

# ==============================================================================
# RUN MAMBAOUT STAGE 4 BLOCKS
# ==============================================================================

print()
print("-" * 78)
print("RUNNING MAMBAOUT STAGE 4 BLOCKS")
print("-" * 78)

mamba_refined = mamba_input

with torch.no_grad():

    for idx, block in enumerate(stage4.blocks):

        print(
            f"Block {idx+1:02d} input:",
            tuple(mamba_refined.shape)
        )

        mamba_refined = block(
            mamba_refined
        )

        print(
            f"Block {idx+1:02d} output:",
            tuple(mamba_refined.shape)
        )

print()
print("✓ All MambaOut Stage 4 blocks executed")

if not torch.isfinite(mamba_refined).all():
    raise RuntimeError(
        "Mamba refined features contain NaN/Inf"
    )

# ==============================================================================
# MAMBA → OCR TOKENS
# ==============================================================================

print()
print("-" * 78)
print("CONVERTING MAMBAOUT OUTPUT TO OCR TOKENS")
print("-" * 78)

mamba_tokens = mamba_refined.reshape(
    B,
    series_h * series_w,
    MODEL_DIM
)

print("Mamba OCR tokens:")
print(tuple(mamba_tokens.shape))

if mamba_tokens.shape != (
    B,
    NUM_ENCODER_TOKENS,
    MODEL_DIM
):
    raise RuntimeError(
        f"Expected "
        f"({B},{NUM_ENCODER_TOKENS},{MODEL_DIM}), "
        f"got {tuple(mamba_tokens.shape)}"
    )

print("✓ PASS — Mamba OCR tokens = [B,44,768]")
print("✓ PASS — No NaN/Inf detected")

# ==============================================================================
# PART E — TRANSFORMER OCR DECODER
# ==============================================================================

print()
print("=" * 78)
print("PART E — TRANSFORMER OCR DECODER")
print("=" * 78)

# ------------------------------------------------------------------------------
# DECODER COMPONENTS
# ------------------------------------------------------------------------------

target_embedding = nn.Embedding(
    VOCAB_SIZE,
    MODEL_DIM
).to(device)

position_embedding = nn.Parameter(
    torch.zeros(
        1,
        TARGET_LENGTH,
        MODEL_DIM,
        device=device
    )
)

decoder_layer = nn.TransformerDecoderLayer(
    d_model=MODEL_DIM,
    nhead=DECODER_HEADS,
    dim_feedforward=FFN_DIM,
    dropout=0.1,
    batch_first=True,
    norm_first=False
).to(device)

transformer_decoder = nn.TransformerDecoder(
    decoder_layer,
    num_layers=DECODER_LAYERS
).to(device)

output_projection = nn.Linear(
    MODEL_DIM,
    VOCAB_SIZE
).to(device)

print()
print("Decoder:")
print("d_model          :", MODEL_DIM)
print("Attention heads  :", DECODER_HEADS)
print("Decoder layers   :", DECODER_LAYERS)
print("FFN dimension    :", FFN_DIM)
print("Vocabulary size   :", VOCAB_SIZE)

# ==============================================================================
# TARGET SEQUENCE
# ==============================================================================

print()
print("-" * 78)
print("PREPARING TARGET SEQUENCE")
print("-" * 78)

# ------------------------------------------------------------------------------
# Try to find REAL target IDs from the batch.
# ------------------------------------------------------------------------------

target_ids = None

if isinstance(batch_object, dict):

    possible_target_keys = [
        "target_ids",
        "targets",
        "labels",
        "label_ids",
        "text_ids",
        "input_ids"
    ]

    for key in possible_target_keys:

        if key in batch_object:

            candidate = batch_object[key]

            if isinstance(candidate, torch.Tensor):

                if candidate.ndim == 2:

                    target_ids = candidate
                    print(
                        "Using real target IDs from batch key:",
                        key
                    )
                    break

# ------------------------------------------------------------------------------
# If no real token IDs exist, create a clearly marked temporary sequence.
# This keeps the interface test executable but MUST NOT be used as final
# training/evaluation evidence.
# ------------------------------------------------------------------------------

if target_ids is None:

    print()
    print("WARNING:")
    print("No real tokenized target IDs were found in the batch.")
    print("Creating temporary target IDs ONLY for forward validation.")
    print("This step must NOT be reported as OCR training performance.")

    target_ids = torch.zeros(
        B,
        TARGET_LENGTH,
        dtype=torch.long,
        device=device
    )

else:

    target_ids = target_ids.to(
        device=device,
        dtype=torch.long
    )

    if target_ids.shape[0] != B:
        raise RuntimeError(
            "Target batch size does not match image batch."
        )

    # Force temporary fixed length for interface validation.
    if target_ids.shape[1] > TARGET_LENGTH:

        target_ids = target_ids[:, :TARGET_LENGTH]

    elif target_ids.shape[1] < TARGET_LENGTH:

        pad_len = TARGET_LENGTH - target_ids.shape[1]

        target_ids = F.pad(
            target_ids,
            (0, pad_len),
            value=0
        )

print()
print("Target IDs shape:",
      tuple(target_ids.shape))

print("Target length:",
      target_ids.shape[1])

# ------------------------------------------------------------------------------
# SAFETY
# ------------------------------------------------------------------------------

target_ids = target_ids.clamp(
    min=0,
    max=VOCAB_SIZE - 1
)

print("✓ PASS — Target IDs within vocabulary")

# ==============================================================================
# TARGET EMBEDDING
# ==============================================================================

print()
print("-" * 78)
print("CREATING TARGET EMBEDDING")
print("-" * 78)

target_emb = target_embedding(
    target_ids
)

print("Target embedding:",
      tuple(target_emb.shape))

target_emb = (
    target_emb +
    position_embedding[:, :target_emb.shape[1], :]
)

print("Position-enhanced target:",
      tuple(target_emb.shape))

# ==============================================================================
# CAUSAL MASK
# ==============================================================================

T = target_emb.shape[1]

causal_mask = torch.triu(
    torch.ones(
        T,
        T,
        device=device,
        dtype=torch.bool
    ),
    diagonal=1
)

print()
print("Causal mask shape:",
      tuple(causal_mask.shape))

print("✓ PASS — Causal mask created")

# ==============================================================================
# TRANSFORMER DECODER FORWARD
# ==============================================================================

print()
print("=" * 78)
print("RUNNING TRANSFORMER OCR DECODER")
print("=" * 78)

print()
print("Memory / encoder input:")
print(tuple(mamba_tokens.shape))

print("Target input:")
print(tuple(target_emb.shape))

decoder_output = transformer_decoder(
    tgt=target_emb,
    memory=mamba_tokens,
    tgt_mask=causal_mask
)

print()
print("Decoder output:")
print(tuple(decoder_output.shape))

if not torch.isfinite(decoder_output).all():
    raise RuntimeError(
        "Decoder output contains NaN/Inf"
    )

print("✓ PASS — Decoder output valid")

# ==============================================================================
# OCR CLASSIFICATION
# ==============================================================================

print()
print("-" * 78)
print("OCR TOKEN CLASSIFICATION")
print("-" * 78)

logits = output_projection(
    decoder_output
)

print("Decoder output:",
      tuple(decoder_output.shape))

print("Logits shape:",
      tuple(logits.shape))

expected_logits_shape = (
    B,
    TARGET_LENGTH,
    VOCAB_SIZE
)

if logits.shape != expected_logits_shape:

    raise RuntimeError(
        f"Expected logits {expected_logits_shape}, "
        f"got {tuple(logits.shape)}"
    )

print("✓ PASS — OCR logits = [B,T,V]")

# ==============================================================================
# TEMPORARY PREDICTION
# ==============================================================================

print()
print("-" * 78)
print("GENERATING TOKEN PREDICTIONS")
print("-" * 78)

predicted_ids = logits.argmax(
    dim=-1
)

print("Predicted token IDs:",
      tuple(predicted_ids.shape))

print("✓ PASS — Predicted IDs = [B,T]")

# ==============================================================================
# LOSS
# ==============================================================================

print()
print("-" * 78)
print("CALCULATING TRAINING LOSS")
print("-" * 78)

loss = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    target_ids.reshape(-1)
)

print("Training loss:",
      float(loss.detach().cpu()))

if not torch.isfinite(loss):
    raise RuntimeError(
        "Loss is NaN/Inf"
    )

print("✓ PASS — Loss is finite")

# ==============================================================================
# SAVE OUTPUTS
# ==============================================================================

print()
print("=" * 78)
print("SAVING CELL 36 OUTPUTS")
print("=" * 78)

series_token_path = (
    "/content/pfms_series_swin_b_to_mambaout_tokens.pt"
)

decoder_output_path = (
    "/content/pfms_series_transformer_decoder_output.pt"
)

logits_path = (
    "/content/pfms_series_transformer_ocr_logits.pt"
)

prediction_path = (
    "/content/pfms_series_transformer_predictions.pt"
)

torch.save(
    mamba_tokens.detach().cpu(),
    series_token_path
)

torch.save(
    decoder_output.detach().cpu(),
    decoder_output_path
)

torch.save(
    logits.detach().cpu(),
    logits_path
)

torch.save(
    predicted_ids.detach().cpu(),
    prediction_path
)

print("Series tokens saved:")
print(series_token_path)

print()
print("Decoder output saved:")
print(decoder_output_path)

print()
print("OCR logits saved:")
print(logits_path)

print()
print("Predictions saved:")
print(prediction_path)

# ==============================================================================
# FINAL VALIDATION
# ==============================================================================

print()
print("=" * 78)
print("CELL 36-FINAL-FIXED VALIDATION")
print("=" * 78)

print()
print("PFMS-SERIES COMPLETE FORWARD PATH:")

print()
print("[B,3,50,700]")
print("      ↓")
print("Swin-B")
print("      ↓")
print("[B,2,22,1024]")
print("      ↓")
print("Projection 1024 → 768")
print("      ↓")
print("[B,44,768]")
print("      ↓")
print("Swin → Mamba Series Adapter")
print("      ↓")
print("[B,768,2,22]")
print("      ↓")
print("[B,2,22,768]")
print("      ↓")
print("MambaOut Stage 4")
print("      ↓")
print("[B,2,22,768]")
print("      ↓")
print("[B,44,768]")
print("      ↓")
print("Transformer OCR Decoder")
print("      ↓")
print("[B,32,768]")
print("      ↓")
print("Output Projection")
print("      ↓")
print("[B,32,256]")
print("      ↓")
print("Predicted Token IDs")
print("      ↓")
print("[B,32]")

print()
print("-" * 78)
print("FINAL SHAPES")
print("-" * 78)

print("Input images        :", tuple(images.shape))
print("Swin features       :", tuple(swin_features.shape))
print("Swin tokens         :", tuple(swin_tokens.shape))
print("Aligned Swin tokens :", tuple(aligned_swin_tokens.shape))
print("Mamba tokens        :", tuple(mamba_tokens.shape))
print("Decoder output      :", tuple(decoder_output.shape))
print("OCR logits          :", tuple(logits.shape))
print("Predicted IDs       :", tuple(predicted_ids.shape))

print()
print("Training loss       :", float(loss.detach().cpu()))

print()
print("✓ SWIN-B RECTANGULAR INPUT FIXED")
print("✓ SWIN-B FEATURE EXTRACTION VALIDATED")
print("✓ SWIN-B → MAMBAOUT SERIES CONNECTION VALIDATED")
print("✓ MAMBAOUT STAGE 4 EXECUTED")
print("✓ MAMBAOUT → TRANSFORMER DECODER VALIDATED")
print("✓ OCR LOGITS GENERATED")
print("✓ FORWARD LOSS COMPUTED")

print()
print("=" * 78)
print("CELL 36-FINAL-FIXED COMPLETED")
print("=" * 78)

PFMS-SERIES — CELL 36-FINAL-FIXED
REAL IDPL-PFOD + TRANSFORMER OCR TRAINING STEP

Device: cuda
GPU: Tesla T4

------------------------------------------------------------------------------
CONFIGURATION
------------------------------------------------------------------------------
Image size       : (50, 700)
Swin dimension   : 1024
Model dimension  : 768
Encoder tokens   : 44
Vocabulary size  : 256
Target length    : 32
Decoder heads    : 8
Decoder layers   : 4
Learning rate    : 0.0001

PART A — REAL IDPL-PFOD BATCH

------------------------------------------------------------------------------
DATASET VALIDATION
------------------------------------------------------------------------------
Dataset path:
/content/drive/Othercomputers/My Laptop/Desktop/idplimgl
✓ Dataset path exists

------------------------------------------------------------------------------
SEARCHING REAL IDPL-PFOD IMAGES
------------------------------------------------------------------------------
Total image fi

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 37-DIAGNOSTIC
# COMPLETE IDPL-PFOD CSV STRUCTURE INSPECTION
# ==============================================================================

import os
import pandas as pd
from pathlib import Path

print("=" * 78)
print("PFMS-SERIES — CELL 37-DIAGNOSTIC")
print("IDPL-PFOD CSV STRUCTURE INSPECTION")
print("=" * 78)

IMAGE_ROOT = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
)

# ------------------------------------------------------------------------------
# FIND CSV FILES
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("SEARCHING FOR CSV FILES")
print("-" * 78)

csv_files = sorted(IMAGE_ROOT.parent.rglob("*.csv"))

# Also search Desktop recursively
desktop_root = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop"
)

if desktop_root.exists():
    csv_files += list(desktop_root.rglob("*.csv"))

csv_files = sorted(set(csv_files))

print("CSV files found:", len(csv_files))

for i, p in enumerate(csv_files):
    print(f"{i+1}. {p}")

# ------------------------------------------------------------------------------
# INSPECT EVERY CSV
# ------------------------------------------------------------------------------

for csv_path in csv_files:

    print()
    print("=" * 78)
    print("CSV FILE")
    print("=" * 78)

    print("Path:")
    print(csv_path)

    print()
    print("File size:")
    print(f"{csv_path.stat().st_size / 1024:.2f} KB")

    # --------------------------------------------------------------------------
    # TRY DIFFERENT ENCODINGS
    # --------------------------------------------------------------------------

    df = None
    used_encoding = None

    encodings = [
        "utf-8",
        "utf-8-sig",
        "cp1256",
        "cp1252",
        "latin1"
    ]

    for enc in encodings:

        try:

            temp_df = pd.read_csv(
                csv_path,
                encoding=enc,
                on_bad_lines="skip"
            )

            df = temp_df
            used_encoding = enc
            break

        except Exception:
            pass

    if df is None:

        print("✗ Could not read this CSV")
        continue

    # --------------------------------------------------------------------------
    # BASIC INFORMATION
    # --------------------------------------------------------------------------

    print()
    print("-" * 78)
    print("BASIC INFORMATION")
    print("-" * 78)

    print("Encoding used :", used_encoding)
    print("Rows          :", len(df))
    print("Columns       :", len(df.columns))

    print()
    print("COLUMN NAMES:")
    for i, col in enumerate(df.columns):
        print(f"  [{i}] {repr(col)}")

    # --------------------------------------------------------------------------
    # DATA TYPES
    # --------------------------------------------------------------------------

    print()
    print("DATA TYPES:")

    for col in df.columns:
        print(
            f"  {repr(col)} -> "
            f"{df[col].dtype}"
        )

    # --------------------------------------------------------------------------
    # FIRST 10 ROWS
    # --------------------------------------------------------------------------

    print()
    print("-" * 78)
    print("FIRST 10 ROWS")
    print("-" * 78)

    pd.set_option(
        "display.max_columns",
        None
    )

    pd.set_option(
        "display.max_colwidth",
        200
    )

    print(
        df.head(10).to_string(index=False)
    )

    # --------------------------------------------------------------------------
    # SAMPLE VALUES FOR EVERY COLUMN
    # --------------------------------------------------------------------------

    print()
    print("-" * 78)
    print("SAMPLE VALUES BY COLUMN")
    print("-" * 78)

    for col in df.columns:

        print()
        print(f"COLUMN: {repr(col)}")

        values = (
            df[col]
            .dropna()
            .astype(str)
            .head(10)
            .tolist()
        )

        for j, value in enumerate(values):
            print(f"  {j+1}: {repr(value)}")

    # --------------------------------------------------------------------------
    # CHECK FOR PERSIAN / ARABIC TEXT
    # --------------------------------------------------------------------------

    print()
    print("-" * 78)
    print("PERSIAN / ARABIC TEXT ANALYSIS")
    print("-" * 78)

    persian_chars = set(
        "اآبپتثجچحخدذرزژسشصضطظعغفقکگلمنوهی"
    )

    arabic_chars = set(
        "ابتثجحخدذرزسشصضطظعغفقكلمنهوي"
    )

    for col in df.columns:

        values = (
            df[col]
            .dropna()
            .astype(str)
        )

        count_text = 0
        examples = []

        for value in values.head(5000):

            if any(
                ch in persian_chars or ch in arabic_chars
                for ch in value
            ):

                count_text += 1

                if len(examples) < 5:
                    examples.append(value)

        if len(values) > 0:

            ratio = (
                count_text /
                min(len(values), 5000)
            )

        else:
            ratio = 0

        print()
        print("Column:", repr(col))
        print(
            f"Persian/Arabic-containing rows: "
            f"{count_text}"
        )
        print(
            f"Ratio in inspected rows: "
            f"{ratio * 100:.2f}%"
        )

        if examples:

            print("Examples:")

            for ex in examples:
                print("   ", repr(ex))

    # --------------------------------------------------------------------------
    # CHECK IMAGE/FILENAME COLUMNS
    # --------------------------------------------------------------------------

    print()
    print("-" * 78)
    print("IMAGE / FILENAME COLUMN ANALYSIS")
    print("-" * 78)

    for col in df.columns:

        values = (
            df[col]
            .dropna()
            .astype(str)
        )

        image_like = 0

        for value in values.head(5000):

            lower = value.lower()

            if (
                ".tif" in lower
                or ".tiff" in lower
                or ".png" in lower
                or ".jpg" in lower
                or ".jpeg" in lower
            ):
                image_like += 1

        if image_like > 0:

            print()
            print(
                "Possible image column:",
                repr(col)
            )

            print(
                "Image-like values:",
                image_like
            )

            print(
                "Examples:",
                values.head(5).tolist()
            )

# ------------------------------------------------------------------------------
# FINAL MESSAGE
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("CELL 37-DIAGNOSTIC COMPLETED")
print("=" * 78)

print()
print("IMPORTANT:")
print("No OCR targets were created.")
print("No model was trained.")
print("No labels were modified.")
print()
print(
    "Send me the COMPLETE OUTPUT of this cell."
)
print(
    "Then I will identify the exact Ground Truth source "
    "and write CELL 37-FINAL accordingly."
)

print("=" * 78)

PFMS-SERIES — CELL 37-DIAGNOSTIC
IDPL-PFOD CSV STRUCTURE INSPECTION

------------------------------------------------------------------------------
SEARCHING FOR CSV FILES
------------------------------------------------------------------------------
CSV files found: 0

CELL 37-DIAGNOSTIC COMPLETED

IMPORTANT:
No OCR targets were created.
No model was trained.
No labels were modified.

Send me the COMPLETE OUTPUT of this cell.
Then I will identify the exact Ground Truth source and write CELL 37-FINAL accordingly.


In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 37-FINAL-DIAGNOSTIC
# EXACT IDPL-PFOD GROUND-TRUTH STRUCTURE ANALYSIS
# ==============================================================================

import os
import pickle
import re
from pathlib import Path

print("=" * 78)
print("PFMS-SERIES — CELL 37-FINAL-DIAGNOSTIC")
print("EXACT IDPL-PFOD GROUND-TRUTH STRUCTURE ANALYSIS")
print("=" * 78)

IMAGE_ROOT = "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
DESKTOP = "/content/drive/Othercomputers/My Laptop/Desktop"

# ------------------------------------------------------------------------------
# PART A — IMAGE DATASET
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART A — IMAGE DATASET")
print("-" * 78)

image_files = []

for root, dirs, files in os.walk(IMAGE_ROOT):
    for f in files:
        if f.lower().endswith((".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp")):
            image_files.append(os.path.join(root, f))

print("Image root exists:", os.path.exists(IMAGE_ROOT))
print("Total image files:", len(image_files))

image_stems = set(Path(x).stem for x in image_files)

print("Unique image stems:", len(image_stems))

print("\nFirst 20 image stems:")
for x in sorted(list(image_stems))[:20]:
    print(" ", x)

# ------------------------------------------------------------------------------
# PART B — COMPLETE DESKTOP FILE SEARCH
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART B — COMPLETE DESKTOP FILE SEARCH")
print("-" * 78)

candidate_files = []

for root, dirs, files in os.walk(DESKTOP):
    for f in files:

        path = os.path.join(root, f)

        # Avoid huge/unrelated files
        try:
            size = os.path.getsize(path)
        except:
            continue

        if size > 100 * 1024 * 1024:
            continue

        ext = Path(f).suffix.lower()

        if ext in [
            ".txt", ".csv", ".tsv", ".json",
            ".xml", ".jsonl", ".pkl", ".pickle",
            ".mat", ".xlsx"
        ]:
            candidate_files.append(path)

print("Candidate metadata files:", len(candidate_files))

for i, path in enumerate(candidate_files[:100], 1):
    try:
        size = os.path.getsize(path)
    except:
        size = 0

    print(
        f"{i:03d}. {path} "
        f"({size / 1024:.2f} KB)"
    )

# ------------------------------------------------------------------------------
# PART C — TEXT FILE CONTENT INSPECTION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART C — TEXT FILE CONTENT INSPECTION")
print("-" * 78)

text_files = [
    x for x in candidate_files
    if Path(x).suffix.lower() in [".txt", ".csv", ".tsv", ".json", ".jsonl", ".xml"]
]

print("Text-like files:", len(text_files))

for path in text_files:

    print("\n" + "=" * 70)
    print("FILE:", path)
    print("=" * 70)

    try:

        with open(path, "r", encoding="utf-8", errors="replace") as f:
            content = f.read(12000)

        print("Characters inspected:", len(content))

        print("\n--- BEGIN CONTENT ---")
        print(content[:8000])
        print("--- END CONTENT ---")

        # Persian detection
        persian_chars = re.findall(
            r"[\u0600-\u06FF]",
            content
        )

        print("\nPersian character count:", len(persian_chars))

        # Numeric image IDs
        numbers = re.findall(
            r"\b\d{1,6}\b",
            content
        )

        print("Numeric tokens found:", len(numbers))

        if numbers:
            print("First numeric tokens:")
            print(numbers[:50])

        # Check whether image stems appear
        matches = []

        for stem in list(image_stems)[:10000]:

            if re.search(
                rf"(?<!\d){re.escape(stem)}(?!\d)",
                content
            ):
                matches.append(stem)

        print(
            "Image-name matches among inspected stems:",
            len(matches)
        )

        if matches:
            print("Matched stems:")
            print(matches[:50])

    except Exception as e:
        print("ERROR:", repr(e))

# ------------------------------------------------------------------------------
# PART D — PICKLE INSPECTION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART D — PICKLE / LABEL ENCODER INSPECTION")
print("-" * 78)

pickle_files = [
    x for x in candidate_files
    if Path(x).suffix.lower() in [".pkl", ".pickle"]
]

print("Pickle files:", len(pickle_files))

for path in pickle_files:

    print("\n" + "=" * 70)
    print("PICKLE:", path)
    print("=" * 70)

    try:

        with open(path, "rb") as f:
            obj = pickle.load(f)

        print("Object type:", type(obj))

        if hasattr(obj, "__dict__"):

            print("\nObject attributes:")

            for key, value in obj.__dict__.items():

                print(
                    f"  {key}: "
                    f"type={type(value)}"
                )

                try:

                    if hasattr(value, "shape"):
                        print(
                            f"      shape={value.shape}"
                        )

                    elif isinstance(value, (list, tuple)):
                        print(
                            f"      length={len(value)}"
                        )
                        print(
                            f"      sample={value[:10]}"
                        )

                    elif isinstance(value, dict):
                        print(
                            f"      keys={list(value.keys())[:20]}"
                        )

                    else:
                        print(
                            f"      value={str(value)[:500]}"
                        )

                except Exception as e:
                    print(
                        "      inspection error:",
                        repr(e)
                    )

        elif isinstance(obj, dict):

            print("\nDictionary keys:")
            print(list(obj.keys())[:100])

            for key, value in list(obj.items())[:50]:

                print(
                    f"\nKEY: {key}"
                )

                print(
                    "TYPE:",
                    type(value)
                )

                try:

                    if isinstance(value, (list, tuple)):
                        print(
                            "LENGTH:",
                            len(value)
                        )
                        print(
                            "SAMPLE:",
                            value[:20]
                        )

                    elif isinstance(value, dict):
                        print(
                            "DICT KEYS:",
                            list(value.keys())[:30]
                        )

                    else:
                        print(
                            "VALUE:",
                            str(value)[:1000]
                        )

                except Exception as e:
                    print(
                        "Inspection error:",
                        repr(e)
                    )

        elif isinstance(obj, (list, tuple)):

            print("Length:", len(obj))
            print("First elements:")

            for x in obj[:30]:
                print(" ", repr(x))

        else:

            print("Object representation:")
            print(str(obj)[:5000])

    except Exception as e:

        print(
            "PICKLE LOAD ERROR:",
            repr(e)
        )

# ------------------------------------------------------------------------------
# PART E — SEARCH FILE NAMES FOR DATASET CLUES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART E — DATASET / GROUND-TRUTH NAME ANALYSIS")
print("-" * 78)

keywords = [
    "idpl",
    "pfod",
    "label",
    "labels",
    "ground",
    "truth",
    "text",
    "trans",
    "transcript",
    "annotation",
    "annot",
    "gt",
    "ocr",
    "font"
]

for path in candidate_files:

    name = os.path.basename(path).lower()

    hits = [
        k for k in keywords
        if k in name
    ]

    if hits:
        print(
            "MATCH:",
            path,
            "->",
            hits
        )

# ------------------------------------------------------------------------------
# PART F — SEARCH FOR IMAGE IDS IN ALL SMALL TEXT FILES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART F — IMAGE ID ↔ TEXT RELATIONSHIP TEST")
print("-" * 78)

sample_ids = sorted(list(image_stems))[:100]

for path in text_files:

    try:

        with open(
            path,
            "r",
            encoding="utf-8",
            errors="replace"
        ) as f:

            content = f.read()

        matched = []

        for sid in sample_ids:

            if re.search(
                rf"(?<!\d){re.escape(sid)}(?!\d)",
                content
            ):
                matched.append(sid)

        if matched:

            print("\nFILE:", path)
            print(
                "Matched image IDs:",
                matched[:50]
            )

    except:
        pass

# ------------------------------------------------------------------------------
# PART G — FINAL DIAGNOSTIC SUMMARY
# ------------------------------------------------------------------------------

print("\n" + "=" * 78)
print("CELL 37-FINAL-DIAGNOSTIC COMPLETED")
print("=" * 78)

print("\nSUMMARY")
print("-------")

print("Images:", len(image_files))
print("Unique image stems:", len(image_stems))
print("Metadata/text files:", len(text_files))
print("Pickle files:", len(pickle_files))

print("\nIMPORTANT:")
print("No OCR targets have been created.")
print("No model has been trained.")
print("No labels have been modified.")
print("No artificial ground truth has been generated.")

print("\nNEXT STEP:")
print("Use the COMPLETE OUTPUT above to identify the real")
print("IDPL-PFOD transcription/ground-truth source.")

print("=" * 78)

PFMS-SERIES — CELL 37-FINAL-DIAGNOSTIC
EXACT IDPL-PFOD GROUND-TRUTH STRUCTURE ANALYSIS

------------------------------------------------------------------------------
PART A — IMAGE DATASET
------------------------------------------------------------------------------
Image root exists: True
Total image files: 27178
Unique image stems: 27178

First 20 image stems:
  00001
  00001 (1)
  00002
  00002 (1)
  00003
  00004
  00004 (1)
  00005
  00005 (1)
  00006
  00006 (1)
  00007
  00007 (1)
  00008
  00008 (1)
  00009
  00009 (1)
  00010
  00010 (1)
  00011

------------------------------------------------------------------------------
PART B — COMPLETE DESKTOP FILE SEARCH
------------------------------------------------------------------------------
Candidate metadata files: 9
001. /content/drive/Othercomputers/My Laptop/Desktop/333.txt (0.28 KB)
002. /content/drive/Othercomputers/My Laptop/Desktop/code ocr persion/label_encoder.pkl (2.23 KB)
003. /content/drive/Othercomputers/My Lapto

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 37-SCAN
# COMPLETE IDPL-PFOD GROUND-TRUTH DISCOVERY
# ==============================================================================

import os
from pathlib import Path
import json

print("=" * 78)
print("PFMS-SERIES — CELL 37-SCAN")
print("COMPLETE IDPL-PFOD GROUND-TRUTH DISCOVERY")
print("=" * 78)

# ------------------------------------------------------------------------------
# ROOTS
# ------------------------------------------------------------------------------

IMAGE_ROOT = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
)

DESKTOP_ROOT = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop"
)

DRIVE_ROOT = Path(
    "/content/drive/Othercomputers/My Laptop"
)

print()
print("IMAGE ROOT:")
print(IMAGE_ROOT)

print()
print("Image root exists:", IMAGE_ROOT.exists())

# ------------------------------------------------------------------------------
# PART A — IMAGE INFORMATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PART A — IMAGE DATASET")
print("-" * 78)

if IMAGE_ROOT.exists():

    image_extensions = {
        ".tif",
        ".tiff",
        ".png",
        ".jpg",
        ".jpeg",
        ".bmp"
    }

    image_files = [
        p for p in IMAGE_ROOT.rglob("*")
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    print("Total image files:", len(image_files))

    if image_files:

        print()
        print("First 10 images:")

        for p in image_files[:10]:
            print(" ", p)

else:

    image_files = []

    print("✗ Image root does not exist")

# ------------------------------------------------------------------------------
# PART B — ALL METADATA-LIKE FILES
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PART B — SEARCHING FOR METADATA FILES")
print("-" * 78)

metadata_extensions = {
    ".csv",
    ".tsv",
    ".txt",
    ".json",
    ".jsonl",
    ".xml",
    ".yaml",
    ".yml",
    ".mat",
    ".pkl",
    ".pickle"
}

metadata_files = []

search_roots = [
    DESKTOP_ROOT,
    DRIVE_ROOT
]

seen = set()

for root in search_roots:

    if not root.exists():
        continue

    try:

        for p in root.rglob("*"):

            if not p.is_file():
                continue

            if p in seen:
                continue

            seen.add(p)

            if p.suffix.lower() in metadata_extensions:
                metadata_files.append(p)

    except Exception as e:

        print(
            "Search warning for",
            root,
            ":",
            e
        )

metadata_files = sorted(metadata_files)

print()
print("Metadata-like files found:", len(metadata_files))

for i, p in enumerate(metadata_files):

    try:
        size_kb = p.stat().st_size / 1024
    except:
        size_kb = -1

    print(
        f"{i+1:03d}. {p} "
        f"({size_kb:.2f} KB)"
    )

# ------------------------------------------------------------------------------
# PART C — SEARCH FILE NAMES FOR GROUND TRUTH
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PART C — GROUND TRUTH FILE NAME SEARCH")
print("-" * 78)

keywords = [
    "label",
    "labels",
    "ground",
    "truth",
    "gt",
    "text",
    "trans",
    "transcript",
    "annotation",
    "annot",
    "metadata",
    "sentence",
    "word"
]

keyword_files = []

for p in metadata_files:

    name = p.name.lower()

    if any(k in name for k in keywords):
        keyword_files.append(p)

print()
print(
    "Potential Ground Truth files:",
    len(keyword_files)
)

for p in keyword_files:
    print(" ", p)

# ------------------------------------------------------------------------------
# PART D — CHECK TEXT-LIKE FILES
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PART D — TEXT FILE INSPECTION")
print("-" * 78)

text_files = [
    p for p in metadata_files
    if p.suffix.lower() in {
        ".txt",
        ".tsv",
        ".csv",
        ".json",
        ".jsonl",
        ".xml"
    }
]

print(
    "Text/metadata files to inspect:",
    len(text_files)
)

# ------------------------------------------------------------------------------
# PERSIAN CHARACTER TEST
# ------------------------------------------------------------------------------

persian_chars = set(
    "اآبپتثجچحخدذرزژسشصضطظعغفقکگلمنوهی"
)

def contains_persian(text):

    if not isinstance(text, str):
        return False

    return any(
        ch in persian_chars
        for ch in text
    )

# ------------------------------------------------------------------------------
# PART E — SAMPLE CONTENT INSPECTION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PART E — SEARCHING FILE CONTENT FOR PERSIAN TEXT")
print("-" * 78)

persian_candidates = []

for p in text_files:

    try:

        # Only inspect relatively small files directly
        size_mb = p.stat().st_size / (1024 * 1024)

        if size_mb > 100:

            print()
            print("Skipping very large file:")
            print(p)
            print(f"Size: {size_mb:.2f} MB")
            continue

        raw = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        if contains_persian(raw):

            persian_candidates.append(p)

            print()
            print("✓ Persian text detected:")
            print(p)

            # Show first useful lines
            lines = raw.splitlines()

            shown = 0

            for line in lines:

                line = line.strip()

                if not line:
                    continue

                if contains_persian(line):

                    print(
                        "  SAMPLE:",
                        repr(line[:300])
                    )

                    shown += 1

                if shown >= 3:
                    break

    except Exception as e:

        print()
        print("Could not inspect:")
        print(p)
        print("Reason:", e)

# ------------------------------------------------------------------------------
# PART F — IMAGE NAME MATCHING
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PART F — IMAGE NAME / METADATA MATCHING")
print("-" * 78)

image_names = set()

for p in image_files:

    image_names.add(p.name)
    image_names.add(p.stem)

print(
    "Unique image names/stems:",
    len(image_names)
)

matches = []

for p in metadata_files:

    try:

        size_mb = p.stat().st_size / (1024 * 1024)

        if size_mb > 100:
            continue

        raw = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        match_count = 0

        for image_name in list(image_names)[:5000]:

            if image_name in raw:
                match_count += 1

                if match_count >= 5:
                    break

        if match_count > 0:

            matches.append(
                (p, match_count)
            )

    except:
        pass

print()
print(
    "Files containing image-name references:"
)

if matches:

    for p, count in matches:

        print(
            f"✓ {p} "
            f"(at least {count} image references)"
        )

else:

    print("No direct matches detected.")

# ------------------------------------------------------------------------------
# PART G — DIRECTORY STRUCTURE
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PART G — IDPL-PFOD DIRECTORY STRUCTURE")
print("-" * 78)

if IMAGE_ROOT.exists():

    directories = sorted({
        p.parent
        for p in image_files
    })

    print(
        "Directories containing images:",
        len(directories)
    )

    for d in directories[:50]:

        try:
            count = sum(
                1
                for p in d.iterdir()
                if p.is_file()
            )
        except:
            count = "?"

        print(
            f"  {d} "
            f"[files={count}]"
        )

# ------------------------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("CELL 37-SCAN FINAL REPORT")
print("=" * 78)

print()
print("Images found              :", len(image_files))
print("Metadata-like files       :", len(metadata_files))
print("Potential GT files        :", len(keyword_files))
print("Persian-text files        :", len(persian_candidates))
print("Image-reference files     :", len(matches))

print()
print("IMPORTANT:")
print("✓ No target IDs generated")
print("✓ No fake labels generated")
print("✓ No model training performed")
print("✓ No dataset files modified")

print()
print(
    "NEXT STEP:"
)
print(
    "Send the COMPLETE OUTPUT of this cell."
)

print("=" * 78)
print("CELL 37-SCAN COMPLETED")
print("=" * 78)

PFMS-SERIES — CELL 37-SCAN
COMPLETE IDPL-PFOD GROUND-TRUTH DISCOVERY

IMAGE ROOT:
/content/drive/Othercomputers/My Laptop/Desktop/idplimgl

Image root exists: True

------------------------------------------------------------------------------
PART A — IMAGE DATASET
------------------------------------------------------------------------------
Total image files: 27178

First 10 images:
  /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27868.tif
  /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27935.tif
  /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27901.tif
  /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27896.tif
  /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27899.tif
  /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27931.tif
  /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27928.tif
  /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/27895.tif
  /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/

In [ ]:
import os
import pandas as pd
from pathlib import Path

print('--- جستجوی نهایی برای یافتن فایل Ground Truth اصلی (~۳۰ هزار ردیف) ---')
search_roots = ['/content/drive/MyDrive', '/content/drive/Othercomputers/My Laptop/Desktop']
candidate_files = []

for root in search_roots:
    if os.path.exists(root):
        print(f'در حال جستجوی فایل‌های داده در: {root}')
        for path in Path(root).rglob('*'):
            if path.is_file() and path.suffix.lower() in ['.csv', '.txt', '.json', '.tsv']:
                try:
                    file_size_kb = path.stat().st_size / 1024
                    if file_size_kb > 500:
                        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                            count = sum(1 for line in f)
                        if count > 25000:
                            candidate_files.append((path, count, file_size_kb))
                except:
                    continue

if candidate_files:
    print(f'\\nتعداد {len(candidate_files)} فایل کاندید با حجم یا تعداد ردیف بالا پیدا شد:')
    for p, c, s in candidate_files:
        print(f' - مسیر: {p} | ردیف تخمینی: {c} | حجم: {s:.2f} KB')
        try:
            if p.suffix.lower() == '.csv':
                df_sample = pd.read_csv(p, nrows=5)
                print('   ستون‌ها:', df_sample.columns.tolist())
                print('   نمونه داده:', df_sample.iloc[0].values)
            else:
                with open(p, 'r', encoding='utf-8', errors='ignore') as f:
                    print('   نمونه متن:', f.readline().strip()[:200])
        except: pass
else:
    print('\\n⚠ هیچ فایل داده‌ای با تعداد ردیف کافی پیدا نشد. تمرکز بر پوشه PFMS-EXP2026_features برای یافتن هرگونه متادیتا...')
    extra_search = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features'
    if os.path.exists(extra_search):
        for root, dirs, files in os.walk(extra_search):
            for f in files:
                if f.endswith(('.json', '.txt', '.csv')):
                    print(f' - متادیتا: {os.path.join(root, f)}')

--- جستجوی نهایی برای یافتن فایل Ground Truth اصلی (~۳۰ هزار ردیف) ---
در حال جستجوی فایل‌های داده در: /content/drive/MyDrive
در حال جستجوی فایل‌های داده در: /content/drive/Othercomputers/My Laptop/Desktop
\nتعداد 3 فایل کاندید با حجم یا تعداد ردیف بالا پیدا شد:
 - مسیر: /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv | ردیف تخمینی: 54241 | حجم: 6087.69 KB
   ستون‌ها: ['split', 'hf_index', 'image_path', 'image_name', 'text']
   نمونه داده: ['train' np.int64(0)
 '/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00490.tif'
 '00490.tif'
 'یابد و كار جمهوری اسلامی را یكسره كند. كافی است به تبلیغات «اكس»گونه یك ماه\n']
 - مسیر: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_ocr_final_backup/dataset_file_list.txt | ردیف تخمینی: 29109 | حجم: 1895.75 KB
   نمونه متن: /content/drive/MyDrive/colab_ocr_project/test/ا/ا.png
 - مسیر: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json | ردیف تخمینی: 27180 

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 37-FINAL-DIAGNOSTIC
# EXACT IDPL-PFOD GROUND-TRUTH STRUCTURE ANALYSIS
# ==============================================================================

import os
import pickle
import re
from pathlib import Path

print("=" * 78)
print("PFMS-SERIES — CELL 37-FINAL-DIAGNOSTIC")
print("EXACT IDPL-PFOD GROUND-TRUTH STRUCTURE ANALYSIS")
print("=" * 78)

IMAGE_ROOT = "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
DESKTOP = "/content/drive/Othercomputers/My Laptop/Desktop"

# ------------------------------------------------------------------------------
# PART A — IMAGE DATASET
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART A — IMAGE DATASET")
print("-" * 78)

image_files = []

for root, dirs, files in os.walk(IMAGE_ROOT):
    for f in files:
        if f.lower().endswith((".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp")):
            image_files.append(os.path.join(root, f))

print("Image root exists:", os.path.exists(IMAGE_ROOT))
print("Total image files:", len(image_files))

image_stems = set(Path(x).stem for x in image_files)

print("Unique image stems:", len(image_stems))

print("\nFirst 20 image stems:")
for x in sorted(list(image_stems))[:20]:
    print(" ", x)

# ------------------------------------------------------------------------------
# PART B — COMPLETE DESKTOP FILE SEARCH
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART B — COMPLETE DESKTOP FILE SEARCH")
print("-" * 78)

candidate_files = []

for root, dirs, files in os.walk(DESKTOP):
    for f in files:

        path = os.path.join(root, f)

        # Avoid huge/unrelated files
        try:
            size = os.path.getsize(path)
        except:
            continue

        if size > 100 * 1024 * 1024:
            continue

        ext = Path(f).suffix.lower()

        if ext in [
            ".txt", ".csv", ".tsv", ".json",
            ".xml", ".jsonl", ".pkl", ".pickle",
            ".mat", ".xlsx"
        ]:
            candidate_files.append(path)

print("Candidate metadata files:", len(candidate_files))

for i, path in enumerate(candidate_files[:100], 1):
    try:
        size = os.path.getsize(path)
    except:
        size = 0

    print(
        f"{i:03d}. {path} "
        f"({size / 1024:.2f} KB)"
    )

# ------------------------------------------------------------------------------
# PART C — TEXT FILE CONTENT INSPECTION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART C — TEXT FILE CONTENT INSPECTION")
print("-" * 78)

text_files = [
    x for x in candidate_files
    if Path(x).suffix.lower() in [".txt", ".csv", ".tsv", ".json", ".jsonl", ".xml"]
]

print("Text-like files:", len(text_files))

for path in text_files:

    print("\n" + "=" * 70)
    print("FILE:", path)
    print("=" * 70)

    try:

        with open(path, "r", encoding="utf-8", errors="replace") as f:
            content = f.read(12000)

        print("Characters inspected:", len(content))

        print("\n--- BEGIN CONTENT ---")
        print(content[:8000])
        print("--- END CONTENT ---")

        # Persian detection
        persian_chars = re.findall(
            r"[\u0600-\u06FF]",
            content
        )

        print("\nPersian character count:", len(persian_chars))

        # Numeric image IDs
        numbers = re.findall(
            r"\b\d{1,6}\b",
            content
        )

        print("Numeric tokens found:", len(numbers))

        if numbers:
            print("First numeric tokens:")
            print(numbers[:50])

        # Check whether image stems appear
        matches = []

        for stem in list(image_stems)[:10000]:

            if re.search(
                rf"(?<!\d){re.escape(stem)}(?!\d)",
                content
            ):
                matches.append(stem)

        print(
            "Image-name matches among inspected stems:",
            len(matches)
        )

        if matches:
            print("Matched stems:")
            print(matches[:50])

    except Exception as e:
        print("ERROR:", repr(e))

# ------------------------------------------------------------------------------
# PART D — PICKLE INSPECTION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART D — PICKLE / LABEL ENCODER INSPECTION")
print("-" * 78)

pickle_files = [
    x for x in candidate_files
    if Path(x).suffix.lower() in [".pkl", ".pickle"]
]

print("Pickle files:", len(pickle_files))

for path in pickle_files:

    print("\n" + "=" * 70)
    print("PICKLE:", path)
    print("=" * 70)

    try:

        with open(path, "rb") as f:
            obj = pickle.load(f)

        print("Object type:", type(obj))

        if hasattr(obj, "__dict__"):

            print("\nObject attributes:")

            for key, value in obj.__dict__.items():

                print(
                    f"  {key}: "
                    f"type={type(value)}"
                )

                try:

                    if hasattr(value, "shape"):
                        print(
                            f"      shape={value.shape}"
                        )

                    elif isinstance(value, (list, tuple)):
                        print(
                            f"      length={len(value)}"
                        )
                        print(
                            f"      sample={value[:10]}"
                        )

                    elif isinstance(value, dict):
                        print(
                            f"      keys={list(value.keys())[:20]}"
                        )

                    else:
                        print(
                            f"      value={str(value)[:500]}"
                        )

                except Exception as e:
                    print(
                        "      inspection error:",
                        repr(e)
                    )

        elif isinstance(obj, dict):

            print("\nDictionary keys:")
            print(list(obj.keys())[:100])

            for key, value in list(obj.items())[:50]:

                print(
                    f"\nKEY: {key}"
                )

                print(
                    "TYPE:",
                    type(value)
                )

                try:

                    if isinstance(value, (list, tuple)):
                        print(
                            "LENGTH:",
                            len(value)
                        )
                        print(
                            "SAMPLE:",
                            value[:20]
                        )

                    elif isinstance(value, dict):
                        print(
                            "DICT KEYS:",
                            list(value.keys())[:30]
                        )

                    else:
                        print(
                            "VALUE:",
                            str(value)[:1000]
                        )

                except Exception as e:
                    print(
                        "Inspection error:",
                        repr(e)
                    )

        elif isinstance(obj, (list, tuple)):

            print("Length:", len(obj))
            print("First elements:")

            for x in obj[:30]:
                print(" ", repr(x))

        else:

            print("Object representation:")
            print(str(obj)[:5000])

    except Exception as e:

        print(
            "PICKLE LOAD ERROR:",
            repr(e)
        )

# ------------------------------------------------------------------------------
# PART E — SEARCH FILE NAMES FOR DATASET CLUES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART E — DATASET / GROUND-TRUTH NAME ANALYSIS")
print("-" * 78)

keywords = [
    "idpl",
    "pfod",
    "label",
    "labels",
    "ground",
    "truth",
    "text",
    "trans",
    "transcript",
    "annotation",
    "annot",
    "gt",
    "ocr",
    "font"
]

for path in candidate_files:

    name = os.path.basename(path).lower()

    hits = [
        k for k in keywords
        if k in name
    ]

    if hits:
        print(
            "MATCH:",
            path,
            "->",
            hits
        )

# ------------------------------------------------------------------------------
# PART F — SEARCH FOR IMAGE IDS IN ALL SMALL TEXT FILES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART F — IMAGE ID ↔ TEXT RELATIONSHIP TEST")
print("-" * 78)

sample_ids = sorted(list(image_stems))[:100]

for path in text_files:

    try:

        with open(
            path,
            "r",
            encoding="utf-8",
            errors="replace"
        ) as f:

            content = f.read()

        matched = []

        for sid in sample_ids:

            if re.search(
                rf"(?<!\d){re.escape(sid)}(?!\d)",
                content
            ):
                matched.append(sid)

        if matched:

            print("\nFILE:", path)
            print(
                "Matched image IDs:",
                matched[:50]
            )

    except:
        pass

# ------------------------------------------------------------------------------
# PART G — FINAL DIAGNOSTIC SUMMARY
# ------------------------------------------------------------------------------

print("\n" + "=" * 78)
print("CELL 37-FINAL-DIAGNOSTIC COMPLETED")
print("=" * 78)

print("\nSUMMARY")
print("-------")

print("Images:", len(image_files))
print("Unique image stems:", len(image_stems))
print("Metadata/text files:", len(text_files))
print("Pickle files:", len(pickle_files))

print("\nIMPORTANT:")
print("No OCR targets have been created.")
print("No model has been trained.")
print("No labels have been modified.")
print("No artificial ground truth has been generated.")

print("\nNEXT STEP:")
print("Use the COMPLETE OUTPUT above to identify the real")
print("IDPL-PFOD transcription/ground-truth source.")

print("=" * 78)

PFMS-SERIES — CELL 37-FINAL-DIAGNOSTIC
EXACT IDPL-PFOD GROUND-TRUTH STRUCTURE ANALYSIS

------------------------------------------------------------------------------
PART A — IMAGE DATASET
------------------------------------------------------------------------------
Image root exists: True
Total image files: 27178
Unique image stems: 27178

First 20 image stems:
  00001
  00001 (1)
  00002
  00002 (1)
  00003
  00004
  00004 (1)
  00005
  00005 (1)
  00006
  00006 (1)
  00007
  00007 (1)
  00008
  00008 (1)
  00009
  00009 (1)
  00010
  00010 (1)
  00011

------------------------------------------------------------------------------
PART B — COMPLETE DESKTOP FILE SEARCH
------------------------------------------------------------------------------
Candidate metadata files: 9
001. /content/drive/Othercomputers/My Laptop/Desktop/333.txt (0.28 KB)
002. /content/drive/Othercomputers/My Laptop/Desktop/code ocr persion/label_encoder.pkl (2.23 KB)
003. /content/drive/Othercomputers/My Lapto

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 37
# REAL IDPL-PFOD GROUND-TRUTH DISCOVERY + TOKENIZATION
# ==============================================================================
#
# IMPORTANT:
# - labels.csv / labels_extended.csv are NOT accepted as OCR ground truth.
# - The text must be the real Persian transcription corresponding to each image.
# - This cell DOES NOT modify Swin-B, MambaOut, or Transformer architecture.
# - This cell only discovers the real metadata, verifies image-text alignment,
#   builds a character vocabulary, and creates REAL target token IDs.
# ==============================================================================

import os
import re
import csv
import json
import math
import glob
import unicodedata
from pathlib import Path
from collections import Counter

import torch
import pandas as pd

print("=" * 78)
print("PFMS-SERIES — CELL 37")
print("REAL IDPL-PFOD GROUND-TRUTH DISCOVERY + TOKENIZATION")
print("=" * 78)

# ------------------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_ROOT = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
)

print()
print("Device:", DEVICE)
print("Image root:", IMAGE_ROOT)

# ------------------------------------------------------------------------------
# PART A — DATASET VALIDATION
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("PART A — DATASET VALIDATION")
print("-" * 78)

if not IMAGE_ROOT.exists():
    raise FileNotFoundError(
        f"IDPL-PFOD image directory not found:\n{IMAGE_ROOT}"
    )

print("✓ Image directory exists")

# ------------------------------------------------------------------------------
# FIND IMAGE FILES
# ------------------------------------------------------------------------------

print()
print("Searching IDPL-PFOD image files...")

IMAGE_EXTENSIONS = {
    ".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"
}

image_files = sorted([
    p for p in IMAGE_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
])

print("Total image files found:", len(image_files))

if len(image_files) == 0:
    raise RuntimeError("No image files were found.")

print("✓ Real IDPL-PFOD images available")

# ------------------------------------------------------------------------------
# PART B — DISCOVER METADATA / CSV FILES
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART B — SEARCHING FOR REAL IDPL-PFOD METADATA")
print("=" * 78)

SEARCH_ROOTS = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Othercomputers/My Laptop/Desktop"),
    IMAGE_ROOT.parent,
    IMAGE_ROOT,
]

csv_candidates = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    try:
        for p in root.rglob("*.csv"):
            if p.is_file():
                csv_candidates.append(p)
    except Exception as e:
        print("Warning while scanning:", root, "|", e)

# Remove duplicates
csv_candidates = sorted(set(csv_candidates))

print()
print("CSV files discovered:", len(csv_candidates))

# ------------------------------------------------------------------------------
# EXCLUDE KNOWN NON-GROUND-TRUTH FILES
# ------------------------------------------------------------------------------

EXCLUDED_NAMES = {
    "labels.csv",
    "labels_extended.csv",
}

candidate_csvs = [
    p for p in csv_candidates
    if p.name.lower() not in EXCLUDED_NAMES
]

print("CSV candidates after exclusion:", len(candidate_csvs))

if len(candidate_csvs) == 0:
    print()
    print("=" * 78)
    print("NO VALID GROUND-TRUTH CSV FOUND")
    print("=" * 78)
    print()
    print("IMPORTANT:")
    print("The known labels.csv / labels_extended.csv files were")
    print("intentionally excluded because their text field is not")
    print("the real Persian transcription.")
    print()
    print("Cell 37 cannot safely create OCR targets without the")
    print("actual IDPL-PFOD Ground Truth metadata.")
    print()
    print("✓ SAFETY CHECK PASSED — NO FAKE TARGETS CREATED")
    print("=" * 78)

    raise RuntimeError(
        "Real IDPL-PFOD Ground Truth CSV was not found. "
        "Do not use labels.csv or labels_extended.csv as OCR targets."
    )

# ------------------------------------------------------------------------------
# PART C — INSPECT CSV STRUCTURES
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART C — INSPECTING CSV STRUCTURES")
print("=" * 78)

csv_reports = []

for csv_path in candidate_csvs:

    try:
        df_preview = pd.read_csv(
            csv_path,
            nrows=5,
            encoding="utf-8",
            on_bad_lines="skip"
        )

        columns = list(df_preview.columns)

        csv_reports.append({
            "path": csv_path,
            "columns": columns,
            "rows_preview": len(df_preview)
        })

        print()
        print("CSV:", csv_path)
        print("Columns:", columns)

    except Exception as e:

        print()
        print("Could not read:", csv_path)
        print("Reason:", e)

# ------------------------------------------------------------------------------
# PART D — IDENTIFY IMAGE + REAL TEXT COLUMNS
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART D — IDENTIFYING IMAGE / GROUND-TRUTH COLUMNS")
print("=" * 78)

IMAGE_COLUMN_NAMES = {
    "image",
    "image_name",
    "filename",
    "file_name",
    "image_path",
    "path",
    "img",
    "img_name",
    "img_path"
}

TEXT_COLUMN_NAMES = {
    "text",
    "transcription",
    "transcript",
    "ground_truth",
    "groundtruth",
    "gt",
    "label",
    "sentence",
    "line",
    "content"
}

def normalize_column_name(x):
    return (
        str(x)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

def find_matching_column(columns, accepted_names):

    normalized = {
        normalize_column_name(c): c
        for c in columns
    }

    for name in accepted_names:
        if name in normalized:
            return normalized[name]

    return None


valid_metadata_candidates = []

for report in csv_reports:

    columns = report["columns"]

    image_col = find_matching_column(
        columns,
        IMAGE_COLUMN_NAMES
    )

    text_col = find_matching_column(
        columns,
        TEXT_COLUMN_NAMES
    )

    if image_col is not None and text_col is not None:

        valid_metadata_candidates.append({
            "path": report["path"],
            "image_col": image_col,
            "text_col": text_col
        })

        print()
        print("✓ Possible metadata file:")
        print("  File       :", report["path"])
        print("  Image col  :", image_col)
        print("  Text col   :", text_col)

# ------------------------------------------------------------------------------
# CHECK WHETHER REAL PERSIAN TEXT EXISTS
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART E — VERIFYING REAL TEXT CONTENT")
print("=" * 78)

PERSIAN_CHARS = set(
    "اآبپتثجچحخدذرزژسشصضطظعغفقکگلمنوهی"
)

ARABIC_CHARS = set(
    "ابتثجحخدذرزسشصضطظعغفقكلمنهوي"
)

def contains_real_text(value):

    if value is None:
        return False

    if pd.isna(value):
        return False

    text = str(value).strip()

    if len(text) == 0:
        return False

    # Must contain at least one Persian/Arabic script character.
    script_count = sum(
        1
        for ch in text
        if ch in PERSIAN_CHARS or ch in ARABIC_CHARS
    )

    return script_count > 0


real_metadata = None

for candidate in valid_metadata_candidates:

    path = candidate["path"]
    image_col = candidate["image_col"]
    text_col = candidate["text_col"]

    try:

        df_test = pd.read_csv(
            path,
            encoding="utf-8",
            on_bad_lines="skip"
        )

        if len(df_test) == 0:
            continue

        valid_text_count = sum(
            contains_real_text(v)
            for v in df_test[text_col].head(1000)
        )

        print()
        print("Candidate:", path)
        print("Rows:", len(df_test))
        print("Image column:", image_col)
        print("Text column:", text_col)
        print("Persian/Arabic text rows in first 1000:",
              valid_text_count)

        if valid_text_count > 0:

            real_metadata = {
                "path": path,
                "image_col": image_col,
                "text_col": text_col,
                "dataframe": df_test
            }

            print("✓ REAL TEXT CONTENT DETECTED")

            break

    except Exception as e:

        print()
        print("Could not validate:", path)
        print("Reason:", e)

# ------------------------------------------------------------------------------
# STOP IF REAL GROUND TRUTH IS NOT FOUND
# ------------------------------------------------------------------------------

if real_metadata is None:

    print()
    print("=" * 78)
    print("REAL GROUND TRUTH NOT CONFIRMED")
    print("=" * 78)
    print()
    print("No metadata file containing verified Persian transcription")
    print("was identified.")
    print()
    print("Therefore:")
    print("✓ No fake labels created")
    print("✓ No filename used as transcription")
    print("✓ No temporary target IDs created")
    print("✓ OCR training remains blocked until real GT is available")
    print("=" * 78)

    raise RuntimeError(
        "Could not identify verified real Persian Ground Truth."
    )

# ------------------------------------------------------------------------------
# PART F — LOAD REAL METADATA
# ------------------------------------------------------------------------------

metadata_path = real_metadata["path"]
IMAGE_COL = real_metadata["image_col"]
TEXT_COL = real_metadata["text_col"]

df = real_metadata["dataframe"].copy()

print()
print("=" * 78)
print("PART F — REAL IDPL-PFOD GROUND TRUTH LOADED")
print("=" * 78)

print("Metadata file :", metadata_path)
print("Image column  :", IMAGE_COL)
print("Text column   :", TEXT_COL)
print("Rows          :", len(df))

# ------------------------------------------------------------------------------
# NORMALIZE TEXT
# ------------------------------------------------------------------------------

def normalize_persian_text(text):

    if text is None or pd.isna(text):
        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # Arabic → Persian character normalization
    replacements = {
        "ي": "ی",
        "ى": "ی",
        "ك": "ک",
        "ۀ": "ه",
        "ة": "ه",
        "ؤ": "و",
        "إ": "ا",
        "أ": "ا",
        "ٱ": "ا",
        "ـ": "",
    }

    for src, dst in replacements.items():
        text = text.replace(src, dst)

    # Normalize common whitespace
    text = text.replace("\r", " ")
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["_gt_text"] = df[TEXT_COL].apply(
    normalize_persian_text
)

# ------------------------------------------------------------------------------
# REMOVE EMPTY GROUND TRUTH
# ------------------------------------------------------------------------------

df = df[
    df["_gt_text"].str.len() > 0
].copy()

print()
print("Rows with non-empty Ground Truth:", len(df))

if len(df) == 0:
    raise RuntimeError(
        "Metadata contains no non-empty Ground Truth text."
    )

# ------------------------------------------------------------------------------
# PART G — IMAGE/TEXT ALIGNMENT
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART G — IMAGE / TEXT ALIGNMENT VALIDATION")
print("=" * 78)

def build_image_lookup(paths):

    lookup = {}

    for p in paths:

        lookup[p.name] = p

        lookup[p.stem] = p

        lookup[str(p)] = p

    return lookup

image_lookup = build_image_lookup(image_files)

def resolve_image(value):

    if value is None or pd.isna(value):
        return None

    raw = str(value).strip()

    if raw in image_lookup:
        return image_lookup[raw]

    raw_name = Path(raw).name

    if raw_name in image_lookup:
        return image_lookup[raw_name]

    raw_stem = Path(raw_name).stem

    if raw_stem in image_lookup:
        return image_lookup[raw_stem]

    return None

df["_image_path"] = df[IMAGE_COL].apply(
    resolve_image
)

matched_count = df["_image_path"].notna().sum()
unmatched_count = len(df) - matched_count

print("Metadata rows:", len(df))
print("Matched images:", matched_count)
print("Unmatched rows:", unmatched_count)

match_rate = matched_count / len(df)

print(f"Match rate: {match_rate * 100:.2f}%")

if matched_count == 0:
    raise RuntimeError(
        "No metadata images matched the real IDPL-PFOD image directory."
    )

if match_rate < 0.50:
    raise RuntimeError(
        f"Image/GT alignment is too low ({match_rate*100:.2f}%). "
        "Training targets must not be created."
    )

print("✓ Image/Text alignment accepted")

# Keep only matched samples
df = df[
    df["_image_path"].notna()
].copy()

# ------------------------------------------------------------------------------
# PART H — VERIFY THAT TEXT IS NOT JUST FILENAME
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART H — GROUND-TRUTH AUTHENTICITY CHECK")
print("=" * 78)

filename_like_count = 0

for _, row in df.head(1000).iterrows():

    image_stem = Path(
        str(row["_image_path"])
    ).stem

    gt_text = str(row["_gt_text"]).strip()

    if gt_text == image_stem:
        filename_like_count += 1

checked = min(len(df), 1000)

filename_like_ratio = (
    filename_like_count / checked
    if checked > 0 else 1.0
)

print("Checked samples:", checked)
print("GT identical to image filename:", filename_like_count)
print(
    f"Filename-like ratio: "
    f"{filename_like_ratio * 100:.2f}%"
)

if filename_like_ratio > 0.90:

    raise RuntimeError(
        "The selected text column appears to contain filenames "
        "rather than real OCR transcription."
    )

print("✓ Ground Truth appears to contain real text")

# ------------------------------------------------------------------------------
# PART I — CHARACTER STATISTICS
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART I — REAL GROUND-TRUTH STATISTICS")
print("=" * 78)

all_text = df["_gt_text"].tolist()

lengths = [
    len(t)
    for t in all_text
]

print("Number of samples :", len(all_text))
print("Minimum length    :", min(lengths))
print("Maximum length    :", max(lengths))
print("Average length    :", round(sum(lengths) / len(lengths), 2))

# Character frequency
char_counter = Counter()

for text in all_text:
    char_counter.update(text)

print("Unique characters :", len(char_counter))

# ------------------------------------------------------------------------------
# PART J — BUILD CHARACTER VOCABULARY
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART J — CHARACTER VOCABULARY")
print("=" * 78)

SPECIAL_TOKENS = [
    "<PAD>",
    "<BOS>",
    "<EOS>",
    "<UNK>"
]

char_list = sorted(
    char_counter.keys(),
    key=lambda x: (ord(x[0]), x)
)

itos = SPECIAL_TOKENS + char_list

stoi = {
    token: idx
    for idx, token in enumerate(itos)
}

PAD_ID = stoi["<PAD>"]
BOS_ID = stoi["<BOS>"]
EOS_ID = stoi["<EOS>"]
UNK_ID = stoi["<UNK>"]

VOCAB_SIZE = len(itos)

print("Special tokens:", SPECIAL_TOKENS)
print("Character count:", len(char_list))
print("Vocabulary size:", VOCAB_SIZE)

print()
print("Special token IDs:")
print("PAD:", PAD_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)
print("UNK:", UNK_ID)

# ------------------------------------------------------------------------------
# PART K — REAL TEXT TOKENIZATION
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART K — TOKENIZING REAL PERSIAN GROUND TRUTH")
print("=" * 78)

def encode_text(text):

    ids = [BOS_ID]

    for ch in text:

        if ch in stoi:
            ids.append(stoi[ch])
        else:
            ids.append(UNK_ID)

    ids.append(EOS_ID)

    return ids

df["_token_ids"] = df["_gt_text"].apply(
    encode_text
)

target_lengths = [
    len(ids)
    for ids in df["_token_ids"]
]

print("Tokenized samples:", len(target_lengths))
print("Minimum target length:", min(target_lengths))
print("Maximum target length:", max(target_lengths))
print(
    "Average target length:",
    round(sum(target_lengths) / len(target_lengths), 2)
)

# ------------------------------------------------------------------------------
# PART L — DETERMINE SAFE TARGET LENGTH
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART L — TARGET LENGTH ANALYSIS")
print("=" * 78)

TARGET_LENGTH_CONFIG = 128

too_long = sum(
    length > TARGET_LENGTH_CONFIG
    for length in target_lengths
)

print("Configured max target length:", TARGET_LENGTH_CONFIG)
print("Samples exceeding max length:", too_long)

if too_long > 0:

    print()
    print(
        "WARNING:",
        too_long,
        "samples are longer than the configured target length."
    )

    print(
        "For this validation cell, they will NOT be silently truncated."
    )

# ------------------------------------------------------------------------------
# PART M — CREATE VALID SAMPLE SUBSET
# ------------------------------------------------------------------------------

valid_df = df[
    df["_token_ids"].apply(
        lambda ids: len(ids) <= TARGET_LENGTH_CONFIG
    )
].copy()

print()
print("Valid samples for target length:", len(valid_df))
print("Excluded because of length:", len(df) - len(valid_df))

if len(valid_df) == 0:
    raise RuntimeError(
        "No samples fit the configured target length."
    )

# ------------------------------------------------------------------------------
# PART N — CREATE REAL TARGET TENSOR
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART N — CREATING REAL TARGET TENSOR")
print("=" * 78)

# Use a manageable validation batch.
BATCH_SIZE = 8

batch_df = valid_df.head(BATCH_SIZE).copy()

encoded_sequences = batch_df["_token_ids"].tolist()

max_batch_length = max(
    len(seq)
    for seq in encoded_sequences
)

target_tensor = torch.full(
    (
        len(encoded_sequences),
        max_batch_length
    ),
    PAD_ID,
    dtype=torch.long
)

for i, seq in enumerate(encoded_sequences):

    target_tensor[
        i,
        :len(seq)
    ] = torch.tensor(
        seq,
        dtype=torch.long
    )

target_tensor = target_tensor.to(DEVICE)

print("Target tensor shape:", tuple(target_tensor.shape))
print("Target dtype:", target_tensor.dtype)
print("Target device:", target_tensor.device)

print("✓ Real target tensor created")
print("✓ No temporary/random target IDs used")

# ------------------------------------------------------------------------------
# PART O — TARGET RANGE VALIDATION
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART O — TARGET TOKEN VALIDATION")
print("=" * 78)

if torch.any(target_tensor < 0):
    raise RuntimeError("Negative token IDs detected.")

if torch.any(target_tensor >= VOCAB_SIZE):
    raise RuntimeError(
        "Target token ID exceeds vocabulary size."
    )

print("Minimum token ID:", int(target_tensor.min()))
print("Maximum token ID:", int(target_tensor.max()))
print("Vocabulary size :", VOCAB_SIZE)

print("✓ All target IDs are inside vocabulary range")

# ------------------------------------------------------------------------------
# PART P — SHOW REAL SAMPLES
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART P — REAL IMAGE / TEXT / TOKEN VALIDATION")
print("=" * 78)

for i, (_, row) in enumerate(batch_df.iterrows()):

    if i >= 5:
        break

    image_path = row["_image_path"]
    text = row["_gt_text"]
    token_ids = row["_token_ids"]

    print()
    print(f"Sample {i + 1}")
    print("Image :", image_path.name)
    print("Text  :", repr(text))
    print("IDs   :", token_ids[:30])

# ------------------------------------------------------------------------------
# PART Q — SAVE REAL TOKENIZATION ARTIFACTS
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("PART Q — SAVING REAL GROUND-TRUTH ARTIFACTS")
print("=" * 78)

TOKEN_FILE = (
    "/content/pfms_series_real_idpl_targets.pt"
)

VOCAB_FILE = (
    "/content/pfms_series_idpl_character_vocab.pt"
)

METADATA_FILE = (
    "/content/pfms_series_real_gt_metadata.csv"
)

torch.save(
    {
        "target_ids": target_tensor.detach().cpu(),
        "vocab_size": VOCAB_SIZE,
        "stoi": stoi,
        "itos": itos,
        "pad_id": PAD_ID,
        "bos_id": BOS_ID,
        "eos_id": EOS_ID,
        "unk_id": UNK_ID,
        "max_target_length": TARGET_LENGTH_CONFIG,
    },
    TOKEN_FILE
)

torch.save(
    {
        "stoi": stoi,
        "itos": itos,
        "vocab_size": VOCAB_SIZE,
        "pad_id": PAD_ID,
        "bos_id": BOS_ID,
        "eos_id": EOS_ID,
        "unk_id": UNK_ID,
    },
    VOCAB_FILE
)

save_columns = [
    IMAGE_COL,
    TEXT_COL,
    "_gt_text",
    "_image_path"
]

save_df = batch_df[save_columns].copy()

save_df["_image_path"] = save_df[
    "_image_path"
].astype(str)

save_df.to_csv(
    METADATA_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Real target file saved:")
print(TOKEN_FILE)

print()
print("Vocabulary file saved:")
print(VOCAB_FILE)

print()
print("Validated metadata saved:")
print(METADATA_FILE)

# ------------------------------------------------------------------------------
# PART R — FINAL VALIDATION
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("CELL 37 FINAL VALIDATION")
print("=" * 78)

print()
print("REAL IDPL-PFOD GROUND TRUTH:")
print("Metadata file       :", metadata_path)
print("Image column        :", IMAGE_COL)
print("Text column         :", TEXT_COL)
print("Matched samples     :", len(df))
print("Valid target samples:", len(valid_df))

print()
print("REAL TARGET TENSOR:")
print("Shape               :", tuple(target_tensor.shape))
print("Vocabulary size     :", VOCAB_SIZE)
print("PAD ID              :", PAD_ID)
print("BOS ID              :", BOS_ID)
print("EOS ID              :", EOS_ID)
print("UNK ID              :", UNK_ID)

print()
print("TARGET LENGTH:")
print("Maximum configured  :", TARGET_LENGTH_CONFIG)
print("Maximum observed    :", max(target_lengths))
print("Average observed    :", round(sum(target_lengths) / len(target_lengths), 2))

print()
print("✓ REAL GROUND-TRUTH TEXT VERIFIED")
print("✓ IMAGE / TEXT ALIGNMENT VERIFIED")
print("✓ CHARACTER VOCABULARY CREATED")
print("✓ REAL TARGET TOKENIZATION COMPLETED")
print("✓ NO RANDOM TARGETS")
print("✓ NO FILENAME AS TRANSCRIPTION")
print("✓ TARGET IDs WITHIN VOCABULARY RANGE")
print("✓ READY FOR CELL 38 — REAL OCR LOSS")

print()
print("=" * 78)
print("CELL 37 COMPLETED")
print("=" * 78)

PFMS-SERIES — CELL 37
REAL IDPL-PFOD GROUND-TRUTH DISCOVERY + TOKENIZATION

Device: cuda
Image root: /content/drive/Othercomputers/My Laptop/Desktop/idplimgl

------------------------------------------------------------------------------
PART A — DATASET VALIDATION
------------------------------------------------------------------------------
✓ Image directory exists

Searching IDPL-PFOD image files...
Total image files found: 27178
✓ Real IDPL-PFOD images available

PART B — SEARCHING FOR REAL IDPL-PFOD METADATA

CSV files discovered: 3
CSV candidates after exclusion: 1

PART C — INSPECTING CSV STRUCTURES

CSV: /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv
Columns: ['split', 'hf_index', 'image_path', 'image_name', 'text']

PART D — IDENTIFYING IMAGE / GROUND-TRUTH COLUMNS

✓ Possible metadata file:
  File       : /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv
  Image col  : image_path
  Text col   : text

PART E — VERIFYING REAL TEXT CONTENT

Candidate: /content/drive/MyD

In [ ]:

# ==============================================================================
# PFMS-SERIES — CELL 38
# START / RESTART POINT
# CONTINUE FROM SAVED SERIES ENCODER
# ==============================================================================

import os
import glob
import torch

print("=" * 78)
print("PFMS-SERIES — CELL 38")
print("START / RESTART POINT")
print("=" * 78)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nDevice:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ==============================================================================
# PART A — PROJECT STATUS
# ==============================================================================

print("\n" + "-" * 78)
print("PART A — CURRENT PROJECT STAGE")
print("-" * 78)

print("""
CURRENT PIPELINE

IDPL-PFOD
    ↓
Swin-B
    ↓
Swin tokens [B,44,1024]
    ↓
Projection 1024 → 768
    ↓
[B,44,768]
    ↓
Swin → MambaOut Series Adapter
    ↓
MambaOut Stage 4
    ↓
[B,44,768]
    ↓
Transformer OCR Decoder
    ↓
[B,T,768]
    ↓
OCR Classifier
    ↓
[B,T,256]

CURRENT STATUS:
✓ Swin-B validated
✓ MambaOut validated
✓ Series connection validated
✓ Transformer Decoder validated
✓ OCR logits validated
✓ Forward path validated

CURRENT BLOCKER:
Ground Truth واقعی IDPL-PFOD هنوز به Target IDs تبدیل نشده است.

CURRENT STAGE:
STAGE 6 — REAL GROUND TRUTH DISCOVERY
""")


# ==============================================================================
# PART B — CHECK SAVED SERIES TOKENS
# ==============================================================================

print("\n" + "-" * 78)
print("PART B — CHECKING SAVED SERIES TOKENS")
print("-" * 78)

SERIES_TOKEN_FILE = "/content/pfms_series_swin_b_to_mambaout_tokens.pt"

print("File:")
print(SERIES_TOKEN_FILE)

if os.path.exists(SERIES_TOKEN_FILE):

    series_tokens = torch.load(
        SERIES_TOKEN_FILE,
        map_location=DEVICE
    )

    print("\n✓ Series token file exists")

    print("Type :", type(series_tokens))

    if isinstance(series_tokens, torch.Tensor):

        print("Shape:", tuple(series_tokens.shape))
        print("Dtype:", series_tokens.dtype)
        print("Device:", series_tokens.device)

        assert series_tokens.ndim == 3
        assert series_tokens.shape[1] == 44
        assert series_tokens.shape[2] == 768

        assert torch.isfinite(series_tokens).all()

        print("✓ PASS — [B,44,768]")
        print("✓ PASS — No NaN/Inf")

    else:

        print("⚠ Saved object is not a Tensor")

else:

    print("✗ Series token file NOT FOUND")


# ==============================================================================
# PART C — CHECK DECODER OUTPUTS
# ==============================================================================

print("\n" + "-" * 78)
print("PART C — CHECKING SAVED DECODER OUTPUTS")
print("-" * 78)

FILES_TO_CHECK = {

    "Decoder output":
        "/content/pfms_series_transformer_decoder_output.pt",

    "OCR logits":
        "/content/pfms_series_transformer_ocr_logits.pt",

    "Predictions":
        "/content/pfms_series_transformer_predictions.pt",
}

for name, path in FILES_TO_CHECK.items():

    print(f"\n{name}:")
    print(path)

    if os.path.exists(path):

        obj = torch.load(
            path,
            map_location=DEVICE
        )

        if isinstance(obj, torch.Tensor):

            print("  Shape:", tuple(obj.shape))
            print("  Dtype:", obj.dtype)

            if torch.is_floating_point(obj):
                print(
                    "  Finite:",
                    bool(torch.isfinite(obj).all())
                )

            print("  ✓ File available")

        else:

            print("  Type:", type(obj))
            print("  ✓ File available")

    else:

        print("  ⚠ File not found")


# ==============================================================================
# PART D — CHECK IDPL-PFOD
# ==============================================================================

print("\n" + "-" * 78)
print("PART D — CHECKING IDPL-PFOD")
print("-" * 78)

IMAGE_ROOT = (
    "/content/drive/Othercomputers/"
    "My Laptop/Desktop/idplimgl"
)

print("Image root:")
print(IMAGE_ROOT)

print("Exists:", os.path.exists(IMAGE_ROOT))
print("Directory:", os.path.isdir(IMAGE_ROOT))


if os.path.isdir(IMAGE_ROOT):

    image_files = []

    for ext in ["*.tif", "*.tiff", "*.png", "*.jpg", "*.jpeg"]:

        image_files.extend(
            glob.glob(
                os.path.join(IMAGE_ROOT, ext)
            )
        )

    print("\nImage count:", len(image_files))

    if len(image_files) > 0:

        print("✓ Real IDPL-PFOD images available")

        print("\nFirst 5 images:")

        for p in image_files[:5]:
            print(" ", p)

else:

    print("""
⚠ IDPL-PFOD path is currently unavailable.

If this happens, Google Drive needs to be mounted again.
Do NOT create labels.
Do NOT train the model.
""")


# ==============================================================================
# PART E — CHECK GOOGLE DRIVE METADATA
# ==============================================================================

print("\n" + "-" * 78)
print("PART E — GROUND TRUTH DISCOVERY STATUS")
print("-" * 78)

SEARCH_ROOTS = [
    "/content/drive/MyDrive",
    "/content/drive/Othercomputers",
    "/content/drive"
]

metadata_extensions = [
    ".csv",
    ".txt",
    ".json",
    ".xml",
    ".jsonl",
    ".tsv",
    ".pkl",
    ".pickle"
]

found_metadata = []


for root in SEARCH_ROOTS:

    if not os.path.exists(root):
        continue

    try:

        for current_root, dirs, files in os.walk(root):

            dirs[:] = [
                d for d in dirs
                if not d.startswith(".")
            ]

            for file_name in files:

                lower = file_name.lower()

                if any(
                    lower.endswith(ext)
                    for ext in metadata_extensions
                ):

                    full_path = os.path.join(
                        current_root,
                        file_name
                    )

                    found_metadata.append(full_path)

    except Exception:
        pass


# Remove duplicates
found_metadata = list(dict.fromkeys(found_metadata))

print("Metadata-like files discovered:",
      len(found_metadata))

if found_metadata:

    print("\nFirst 50 candidates:")

    for i, path in enumerate(
        found_metadata[:50],
        start=1
    ):

        print(
            f"{i:03d}. {path}"
        )

else:

    print("""
No metadata files were discovered.

This does NOT mean Ground Truth does not exist.
It may be located outside the currently mounted paths.
""")


# ==============================================================================
# PART F — SAFETY CHECK
# ==============================================================================

print("\n" + "-" * 78)
print("PART F — GROUND TRUTH SAFETY CHECK")
print("-" * 78)

print("""
✓ NO artificial labels created
✓ NO temporary targets converted into training labels
✓ NO model parameters changed
✓ NO training performed
✓ Existing encoder outputs preserved
✓ Existing decoder outputs preserved
""")


# ==============================================================================
# FINAL STATUS
# ==============================================================================

print("\n" + "=" * 78)
print("CELL 38 — RESTART POINT COMPLETED")
print("=" * 78)

print("""
CURRENT PROJECT POSITION
------------------------

STAGE 1  Dataset preparation              ✓
STAGE 2  Swin-B encoder                   ✓
STAGE 3  Swin → MambaOut series           ✓
STAGE 4  Transformer OCR decoder          ✓
STAGE 5  Forward-path validation          ✓
STAGE 6  Real Ground Truth discovery      ← CURRENT
STAGE 7  Real OCR training                 ⏳
STAGE 8  Evaluation: CER/WER/Accuracy      ⏳


NEXT ACTION
-----------

Find the REAL IDPL-PFOD transcription
corresponding to each image ID.

Only after that:

Image ID
   ↓
Real Persian transcription
   ↓
Tokenizer
   ↓
Target IDs
   ↓
Teacher forcing
   ↓
Cross-Entropy loss
   ↓
Backpropagation
   ↓
Training
   ↓
Validation
   ↓
CER / WER / Accuracy


IMPORTANT:
Do NOT report the previous temporary loss = 6.0034
as training performance.
""")

print("=" * 78)

PFMS-SERIES — CELL 38
START / RESTART POINT

Device: cuda
GPU: Tesla T4

------------------------------------------------------------------------------
PART A — CURRENT PROJECT STAGE
------------------------------------------------------------------------------

CURRENT PIPELINE

IDPL-PFOD
    ↓
Swin-B
    ↓
Swin tokens [B,44,1024]
    ↓
Projection 1024 → 768
    ↓
[B,44,768]
    ↓
Swin → MambaOut Series Adapter
    ↓
MambaOut Stage 4
    ↓
[B,44,768]
    ↓
Transformer OCR Decoder
    ↓
[B,T,768]
    ↓
OCR Classifier
    ↓
[B,T,256]

CURRENT STATUS:
✓ Swin-B validated
✓ MambaOut validated
✓ Series connection validated
✓ Transformer Decoder validated
✓ OCR logits validated
✓ Forward path validated

CURRENT BLOCKER:
Ground Truth واقعی IDPL-PFOD هنوز به Target IDs تبدیل نشده است.

CURRENT STAGE:
STAGE 6 — REAL GROUND TRUTH DISCOVERY


------------------------------------------------------------------------------
PART B — CHECKING SAVED SERIES TOKENS
---------------------------------------

In [ ]:

# ==============================================================================
# PFMS-SERIES — CELL 39
# DRIVE RECOVERY + IDPL-PFOD RECOVERY
# ==============================================================================

import os
import glob

print("=" * 78)
print("PFMS-SERIES — CELL 39")
print("DRIVE RECOVERY + IDPL-PFOD RECOVERY")
print("=" * 78)


# ==============================================================================
# PART A — MOUNT GOOGLE DRIVE
# ==============================================================================

print("\n" + "-" * 78)
print("PART A — GOOGLE DRIVE MOUNT")
print("-" * 78)

try:

    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=True
    )

    print("✓ Google Drive mounted")

except Exception as e:

    print("✗ Drive mount failed")
    print(e)


# ==============================================================================
# PART B — VERIFY DRIVE
# ==============================================================================

print("\n" + "-" * 78)
print("PART B — DRIVE VERIFICATION")
print("-" * 78)

DRIVE_ROOT = "/content/drive"

print("Drive exists:", os.path.exists(DRIVE_ROOT))
print("Drive directory:", os.path.isdir(DRIVE_ROOT))

if os.path.isdir(DRIVE_ROOT):

    print("\nDrive contents:")

    try:

        for item in os.listdir(DRIVE_ROOT):

            print(" ", item)

    except Exception as e:

        print("Listing error:", e)


# ==============================================================================
# PART C — CHECK ORIGINAL IDPL-PFOD PATH
# ==============================================================================

print("\n" + "-" * 78)
print("PART C — CHECKING ORIGINAL IDPL-PFOD PATH")
print("-" * 78)

KNOWN_IDPL = (
    "/content/drive/Othercomputers/"
    "My Laptop/Desktop/idplimgl"
)

print("Known path:")
print(KNOWN_IDPL)

print("\nExists:", os.path.exists(KNOWN_IDPL))
print("Directory:", os.path.isdir(KNOWN_IDPL))


# ==============================================================================
# PART D — SEARCH FOR IDPL-PFOD DIRECTORY
# ==============================================================================

print("\n" + "-" * 78)
print("PART D — SEARCHING DRIVE FOR IDPL-PFOD")
print("-" * 78)

possible_dirs = []

SEARCH_ROOTS = [
    "/content/drive/MyDrive",
    "/content/drive/Othercomputers",
    "/content/drive/Shareddrives"
]

for search_root in SEARCH_ROOTS:

    if not os.path.exists(search_root):
        continue

    print("\nSearching:")
    print(search_root)

    try:

        for root, dirs, files in os.walk(search_root):

            dirs[:] = [
                d for d in dirs
                if not d.startswith(".")
            ]

            dirname = os.path.basename(root).lower()

            if dirname in [
                "idplimgl",
                "idpl-pfod",
                "idpl_pfod",
                "idpl"
            ]:

                tif_count = (
                    len(glob.glob(
                        os.path.join(root, "*.tif")
                    ))
                    +
                    len(glob.glob(
                        os.path.join(root, "*.tiff")
                    ))
                )

                possible_dirs.append(
                    (root, tif_count)
                )

    except Exception as e:

        print("Search warning:", e)


# ==============================================================================
# PART E — RESULTS
# ==============================================================================

print("\n" + "-" * 78)
print("PART E — IDPL-PFOD SEARCH RESULTS")
print("-" * 78)

if possible_dirs:

    # remove duplicates
    unique_results = []

    seen = set()

    for path, count in possible_dirs:

        if path not in seen:

            seen.add(path)

            unique_results.append(
                (path, count)
            )

    for i, (path, count) in enumerate(
        unique_results,
        start=1
    ):

        print(
            f"{i:03d}. {path}"
        )

        print(
            f"     TIF/TIFF images: {count}"
        )

else:

    print("✗ No IDPL-PFOD directory found")


# ==============================================================================
# PART F — ORIGINAL PATH IMAGE COUNT
# ==============================================================================

print("\n" + "-" * 78)
print("PART F — ORIGINAL DATASET IMAGE COUNT")
print("-" * 78)

if os.path.isdir(KNOWN_IDPL):

    images = []

    for ext in [
        "*.tif",
        "*.tiff",
        "*.png",
        "*.jpg",
        "*.jpeg"
    ]:

        images.extend(
            glob.glob(
                os.path.join(
                    KNOWN_IDPL,
                    ext
                )
            )
        )

    print("Images found:", len(images))

    if len(images) > 0:

        print("✓ REAL IDPL-PFOD RECOVERED")

        print("\nFirst 10 images:")

        for p in images[:10]:

            print(" ", p)

else:

    print(
        "⚠ Original IDPL-PFOD path is still unavailable"
    )


# ==============================================================================
# PART G — SAFETY STATUS
# ==============================================================================

print("\n" + "-" * 78)
print("PART G — SAFETY STATUS")
print("-" * 78)

print("""
✓ No labels created
✓ No Ground Truth invented
✓ No model trained
✓ No parameters modified
✓ No artificial targets generated
""")


# ==============================================================================
# FINAL STATUS
# ==============================================================================

print("\n" + "=" * 78)
print("CELL 39 COMPLETED")
print("=" * 78)

print("""
CURRENT STAGE:

STAGE 6 — REAL IDPL-PFOD GROUND TRUTH DISCOVERY

NEXT:
1. Recover IDPL-PFOD
2. Locate actual transcription source
3. Match image IDs → Persian text
4. Build real Target IDs
5. Start REAL OCR training

DO NOT RUN TRAINING YET.
""")

print("=" * 78)

PFMS-SERIES — CELL 39
DRIVE RECOVERY + IDPL-PFOD RECOVERY

------------------------------------------------------------------------------
PART A — GOOGLE DRIVE MOUNT
------------------------------------------------------------------------------
Mounted at /content/drive
✓ Google Drive mounted

------------------------------------------------------------------------------
PART B — DRIVE VERIFICATION
------------------------------------------------------------------------------
Drive exists: True
Drive directory: True

Drive contents:
  Othercomputers
  .shortcut-targets-by-id
  MyDrive
  .Trash-0
  .Encrypted

------------------------------------------------------------------------------
PART C — CHECKING ORIGINAL IDPL-PFOD PATH
------------------------------------------------------------------------------
Known path:
/content/drive/Othercomputers/My Laptop/Desktop/idplimgl

Exists: True
Directory: True

------------------------------------------------------------------------------
PART

In [ ]:
import os
from pathlib import Path

# تعریف مسیرهای اصلی پروژه
project_paths = [
    '/content/pfms_series_dataset',
    '/content/drive/MyDrive/PFMS-Net_Research2026_Results',
    '/content/drive/MyDrive/PFMS_Swin_Stage1'
]

print('='*70)
print('در حال بررسی و ایجاد ساختار پوشه‌ها...')
print('='*70)

for path in project_paths:
    if not os.path.exists(path):
        try:
            os.makedirs(path, exist_ok=True)
            print(f'✓ ایجاد شد: {path}')
        except Exception as e:
            print(f'✗ خطا در ایجاد {path}: {e}')
    else:
        print(f'● موجود است: {path}')

# تابعی برای گزارش حجم نهایی
def get_dir_size(path='.'):
    total = 0
    try:
        for entry in os.scandir(path):
            if entry.is_file():
                total += entry.stat().st_size
            elif entry.is_dir():
                total += get_dir_size(entry.path)
    except:
        return 0
    return total

print('\n' + '-'*70)
print('خلاصه وضعیت نهایی:')
dataset_path = '/content/drive/Othercomputers/My Laptop/Desktop/idplimgl'
if os.path.exists(dataset_path):
    size = get_dir_size(dataset_path) / (1024**3)
    print(f'دیتاست IDPL: {size:.2f} GB')

در حال بررسی و ایجاد ساختار پوشه‌ها...
● موجود است: /content/pfms_series_dataset
● موجود است: /content/drive/MyDrive/PFMS-Net_Research2026_Results
● موجود است: /content/drive/MyDrive/PFMS_Swin_Stage1

----------------------------------------------------------------------
خلاصه وضعیت نهایی:
دیتاست IDPL: 0.47 GB


### مرحله ۶: کشف و استخراج Ground Truth واقعی
در این بخش، به دنبال فایل‌های متادیتا می‌گردیم که حاوی بازنویسی (Transcription) واقعی تصاویر هستند. هدف ما انطباق دقیق شناسه هر تصویر با متن فارسی مربوطه است.

In [ ]:
import os
import pandas as pd
from pathlib import Path

# مسیر اصلی که تصاویر در آن دسته‌بندی شده‌اند
search_root = '/content/drive/MyDrive/colab_ocr_project'

data_rows = []

print('در حال استخراج برچسب‌ها از ساختار پوشه‌بندی...')

if os.path.exists(search_root):
    # پیمایش تمام فایل‌های png و tif در زیرپوشه‌ها
    for root, dirs, files in os.walk(search_root):
        for file in files:
            if file.lower().endswith(('.png', '.tif', '.tiff', '.jpg')):
                full_path = os.path.join(root, file)
                # نام پوشه مستقیم به عنوان برچسب در نظر گرفته می‌شود
                label = os.path.basename(root)

                data_rows.append({
                    'image_path': full_path,
                    'label': label
                })

# تبدیل به دیتافریم و نمایش آمار
df_labels = pd.DataFrame(data_rows)
print(f'تعداد کل تصاویر یافت شده: {len(df_labels)}')

if not df_labels.empty:
    print('\nنمونه داده‌های استخراج شده:')
    display(df_labels.head(10))

    # ذخیره فایل نهایی برچسب‌ها
    output_path = '/content/pfms_series_dataset/final_extracted_labels.csv'
    df_labels.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f'\n✓ فایل برچسب‌ها با موفقیت ذخیره شد در: {output_path}')
else:
    print('⚠ هیچ تصویری در مسیر مشخص شده یافت نشد.')

در حال استخراج برچسب‌ها از ساختار پوشه‌بندی...
تعداد کل تصاویر یافت شده: 0
⚠ هیچ تصویری در مسیر مشخص شده یافت نشد.


In [ ]:
import os
import re

# فایل لیستی که قبلاً پیدا کردیم و تعداد خطوط زیادی داشت
gt_list_path = '/content/drive/MyDrive/mamba_ocr_final_backup/dataset_file_list.txt'

print('='*70)
print(f'تحلیل محتوای فایل لیست برای یافتن جملات فارسی (IDPL Ground Truth)')
print('='*70)

if os.path.exists(gt_list_path):
    with open(gt_list_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    print(f'تعداد کل خطوط فایل: {len(lines)}')

    # جستجو برای خطوطی که حاوی فاصله و حروف فارسی هستند (نشان‌دهنده جمله)
    sentences_found = []
    persian_pattern = re.compile(r'[\u0600-\u06FF]')

    for i, line in enumerate(lines):
        # اگر خط حاوی کاراکتر فارسی باشد و فقط یک حرف نباشد
        clean_line = line.strip()
        if persian_pattern.search(clean_line) and len(clean_line) > 5:
            sentences_found.append((i, clean_line))
            if len(sentences_found) > 15: # نمایش ۱۰ نمونه اول کافی است
                break

    if sentences_found:
        print('\n✓ نمونه جملات/متون پیدا شده در فایل لیست:')
        for idx, text in sentences_found:
            print(f'خط {idx}: {text}')
    else:
        print('\n⚠ متاسفانه در این فایل لیست، جملات فارسی طولانی پیدا نشد.')
        print('نمونه ۵ خط اول فایل برای بررسی ساختار:')
        for l in lines[:5]:
            print(f'  - {l.strip()}')
else:
    print('File not found.')

تحلیل محتوای فایل لیست برای یافتن جملات فارسی (IDPL Ground Truth)
File not found.


In [ ]:
import os
import pandas as pd
from pathlib import Path

# مسیرهای مورد نظر کاربر
desktop_path = '/content/drive/Othercomputers/My Laptop/Desktop'
images_path = os.path.join(desktop_path, 'idplimgl')
new_folder_path = os.path.join(desktop_path, 'New folder') # پوشه خالی ذکر شده

print('='*70)
print('جستجوی فایل‌های متادیتا در محدوده اختصاصی پروژه')
print('='*70)

# ۱. لیست کردن تمام فایل‌های متنی/داده در دسکتاپ (به جز پوشه‌های نامرتبط)
metadata_files = []
if os.path.exists(desktop_path):
    for root, dirs, files in os.walk(desktop_path):
        # نادیده گرفتن پوشه‌هایی که کاربر گفت مربوط نیستند
        if any(x in root for x in ['colab_ocr_project', 'mamba_ocr_final_backup']):
            continue

        for f in files:
            if f.endswith(('.csv', '.txt', '.json', '.xlsx')):
                path = os.path.join(root, f)
                size = os.path.getsize(path) / 1024
                metadata_files.append((f, path, size))

print(f'فایل‌های کاندید پیدا شده: {len(metadata_files)}')
for name, path, size in metadata_files:
    print(f' - {name} ({size:.2f} KB) -> {path}')

# ۲. بررسی پوشه تصاویر برای یافتن فایل‌های همراه (مثل annotations.txt یا موارد مشابه)
print('\n' + '-'*70)
print(f'بررسی محتویات پوشه تصاویر: {images_path}')
if os.path.exists(images_path):
    items = os.listdir(images_path)
    non_image = [i for i in items if not i.lower().endswith(('.tif', '.png', '.jpg'))]
    print(f'آیتم‌های غیر تصویری در پوشه دیتاست: {non_image}')

جستجوی فایل‌های متادیتا در محدوده اختصاصی پروژه
فایل‌های کاندید پیدا شده: 6
 - 333.txt (0.28 KB) -> /content/drive/Othercomputers/My Laptop/Desktop/333.txt
 - train.txt (60.24 KB) -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_split/train.txt
 - validation.txt (12.91 KB) -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_split/validation.txt
 - test.txt (12.96 KB) -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_split/test.txt
 - image_paths.json (1911.18 KB) -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json
 - experiment_config.json (0.30 KB) -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS-Net_Research2026_Results/experiment_config.json

----------------------------------------------------------------------
بررسی محتویات پوشه تصاویر: /content/drive/Othercomputers/My Laptop/Desktop/idplimgl
آیتم‌های غیر تصویری در پ

In [ ]:
import os

# مسیر فایل‌هایی که در مرحله قبل پیدا شدند
metadata_dir = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_split'
files_to_inspect = ['train.txt', 'test.txt', 'validation.txt']

print('='*70)
print('بررسی محتوای فایل‌های متادیتای دسکتاپ برای یافتن جملات فارسی')
print('='*70)

for file_name in files_to_inspect:
    file_path = os.path.join(metadata_dir, file_name)
    if os.path.exists(file_path):
        print(f'\n--- فایل: {file_name} ---')
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                # خواندن ۵ خط اول برای تحلیل ساختار
                lines = [f.readline().strip() for _ in range(5)]
                for i, line in enumerate(lines):
                    print(f'سطر {i+1}: {line}')
        except Exception as e:
            print(f'خطا در خواندن فایل: {e}')
    else:
        print(f'فایل {file_name} یافت نشد.')

بررسی محتوای فایل‌های متادیتای دسکتاپ برای یافتن جملات فارسی

--- فایل: train.txt ---
سطر 1: /content/drive/MyDrive/idplimgl_full/28639.tif
سطر 2: /content/drive/MyDrive/idplimgl_full/27259.tif
سطر 3: /content/drive/MyDrive/idplimgl_full/27191.tif
سطر 4: /content/drive/MyDrive/idplimgl_full/28102.tif
سطر 5: /content/drive/MyDrive/idplimgl_full/27562.tif

--- فایل: test.txt ---
سطر 1: /content/drive/MyDrive/idplimgl_full/28540.tif
سطر 2: /content/drive/MyDrive/idplimgl_full/28004.tif
سطر 3: /content/drive/MyDrive/idplimgl_full/28057.tif
سطر 4: /content/drive/MyDrive/idplimgl_full/28378.tif
سطر 5: /content/drive/MyDrive/idplimgl_full/28008.tif

--- فایل: validation.txt ---
سطر 1: /content/drive/MyDrive/idplimgl_full/27909.tif
سطر 2: /content/drive/MyDrive/idplimgl_full/27253.tif
سطر 3: /content/drive/MyDrive/idplimgl_full/27308.tif
سطر 4: /content/drive/MyDrive/idplimgl_full/28406.tif
سطر 5: /content/drive/MyDrive/idplimgl_full/27683.tif


In [ ]:
import json
import os

json_path = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json'

print('='*70)
print('بررسف فافف image_paths.json براف فافءن Ground Truth')
print('='*70)

if os.path.exists(json_path):
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            # Read only first part to check structure
            data = json.load(f)

        print(f'نفء أاأء: {type(data)}')

        if isinstance(data, list):
            print(f'ففرء آفءم ها: {len(data)}')
            print('نمفنء ففرء أفء (2 مفرء أوف):')
            print(json.dumps(data[:2], indent=2, ensure_ascii=False))
        elif isinstance(data, dict):
            print(f'ففرء كففأ ها: {len(data.keys())}')
            print('نمفنء 2 كففأ أوف:')
            first_keys = list(data.keys())[:2]
            for k in first_keys:
                print(f'{k}: {data[k]}')

    except Exception as e:
        print(f'أطا أر أففء فافف: {e}')
else:
    print('فافف image_paths.json فافف نشأ.')

بررسف فافف image_paths.json براف فافءن Ground Truth
نفء أاأء: <class 'list'>
ففرء آفءم ها: 27178
نمفنء ففرء أفء (2 مفرء أوف):
[
  "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001 (1).tif",
  "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001.tif"
]


In [ ]:
import os
from pathlib import Path
import pandas as pd

print('='*70)
print('جستجوی جامع برای یافتن فایل‌های حاوی برچسب (Ground Truth)')
print('='*70)

# مسیرهای جستجو
search_paths = [
    '/content/drive/MyDrive',
    '/content/drive/Othercomputers/My Laptop/Desktop'
]

found_metadata_files = []
# پسوندهایی که احتمال دارد حاوی متن باشند
target_extensions = ('.csv', '.json', '.txt', '.tsv')

for base_path in search_paths:
    if os.path.exists(base_path):
        print(f'در حال جستجو در: {base_path}...')
        for root, dirs, files in os.walk(base_path):
            for file in files:
                if file.lower().endswith(target_extensions):
                    # نادیده گرفتن فایل‌های سیستمی و کش
                    if not file.startswith('.'):
                        full_path = os.path.join(root, file)
                        try:
                            size_kb = os.path.getsize(full_path) / 1024
                            # تمرکز روی فایل‌هایی که حجم معقولی برای نگهداری ۲۷ هزار خط متن دارند
                            if size_kb > 10:
                                found_metadata_files.append((file, full_path, size_kb))
                        except:
                            continue

print(f'\nتعداد فایل‌های کاندید پیدا شده: {len(found_metadata_files)}')

# نمایش لیست فایل‌ها برای انتخاب جهت بررسی محتوا
for i, (name, path, size) in enumerate(found_metadata_files[:30], 1):
    print(f'{i:02d}. [{size:7.2f} KB] {name} -> {path}')

جستجوی جامع برای یافتن فایل‌های حاوی برچسب (Ground Truth)
در حال جستجو در: /content/drive/MyDrive...
در حال جستجو در: /content/drive/Othercomputers/My Laptop/Desktop...

تعداد فایل‌های کاندید پیدا شده: 8
01. [  17.51 KB] labels.csv -> /content/drive/MyDrive/labels.csv
02. [  60.00 KB] labels_extended.csv -> /content/drive/MyDrive/labels_extended.csv
03. [6087.69 KB] IDPL_PFOD_REAL_GT_MAPPING.csv -> /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv
04. [1895.75 KB] dataset_file_list.txt -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_ocr_final_backup/dataset_file_list.txt
05. [  60.24 KB] train.txt -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_split/train.txt
06. [  12.91 KB] validation.txt -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_split/validation.txt
07. [  12.96 KB] test.txt -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_split/test.txt
08. [1911.18 KB] 

In [ ]:
import os
import pandas as pd
import json

# لیستی از فایل‌هایی که در مرحله قبل پیدا شدند (به عنوان نمونه)
candidates = [
    '/content/drive/MyDrive/labels.csv',
    '/content/drive/MyDrive/labels_extended.csv',
    '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json'
]

print('='*70)
print('تحلیل محتوای فایل‌های کاندید برای استخراج متن فارسی')
print('='*70)

for path in candidates:
    if os.path.exists(path):
        print(f'\n--- بررسی فایل: {os.path.basename(path)} ---')
        try:
            if path.endswith('.csv'):
                df = pd.read_csv(path)
                print('ستون‌ها:', df.columns.tolist())
                print('نمونه داده‌ها:')
                display(df.head(3))
            elif path.endswith('.json'):
                with open(path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                print('نوع داده:', type(data))
                if isinstance(data, list):
                    print('نمونه (اولین مورد):', data[0])
                elif isinstance(data, dict):
                    first_key = list(data.keys())[0]
                    print(f'نمونه: {first_key} -> {data[first_key]}')
        except Exception as e:
            print(f'خطا در خواندن: {e}')

تحلیل محتوای فایل‌های کاندید برای استخراج متن فارسی

--- بررسی فایل: labels.csv ---
ستون‌ها: ['image', 'text']
نمونه داده‌ها:


,image,text
0,00046.tif,00046
1,00039.tif,00039
2,00024.tif,00024



--- بررسی فایل: labels_extended.csv ---
ستون‌ها: ['image_path', 'text']
نمونه داده‌ها:


,image_path,text
0,/content/drive/MyDrive/idplimgl/idplimgl/00046...,00046
1,/content/drive/MyDrive/idplimgl/idplimgl/00039...,00039
2,/content/drive/MyDrive/idplimgl/idplimgl/00024...,00024



--- بررسی فایل: image_paths.json ---
نوع داده: <class 'list'>
نمونه (اولین مورد): /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001 (1).tif


In [ ]:
import os

# مسیر فایل کاندید اصلی برای Ground Truth
gt_candidate = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_ocr_final_backup/dataset_file_list.txt'

print('='*70)
print(f'تحلیل محتوای فایل کاندید: {os.path.basename(gt_candidate)}')
print('='*70)

if os.path.exists(gt_candidate):
    try:
        with open(gt_candidate, 'r', encoding='utf-8') as f:
            # خواندن ۲۰ خط اول برای دیدن ساختار داده‌ها
            lines = [f.readline().strip() for _ in range(20)]
            print('نمونه ۲۰ خط اول:')
            for i, line in enumerate(lines, 1):
                print(f'{i:02d}: {line}')

        # شمارش کل خطوط
        with open(gt_candidate, 'r', encoding='utf-8') as f:
            line_count = sum(1 for _ in f)
        print(f'\nتعداد کل خطوط فایل: {line_count}')
    except Exception as e:
        print(f'خطا در خواندن فایل: {e}')
else:
    print('فایل مورد نظر یافت نشد.')

تحلیل محتوای فایل کاندید: dataset_file_list.txt
نمونه ۲۰ خط اول:
01: /content/drive/MyDrive/colab_ocr_project/test/ا/ا.png
02: /content/drive/MyDrive/colab_ocr_project/test/ا/ا_9.png
03: /content/drive/MyDrive/colab_ocr_project/test/آ/آ_4.png
04: /content/drive/MyDrive/colab_ocr_project/test/ب/ب.png
05: /content/drive/MyDrive/colab_ocr_project/test/ب/ب_9.png
06: /content/drive/MyDrive/colab_ocr_project/test/ت/ت.png
07: /content/drive/MyDrive/colab_ocr_project/test/ت/ت_9.png
08: /content/drive/MyDrive/colab_ocr_project/test/ث/ث.png
09: /content/drive/MyDrive/colab_ocr_project/test/ث/ث_9.png
10: /content/drive/MyDrive/colab_ocr_project/test/ج/ج.png
11: /content/drive/MyDrive/colab_ocr_project/test/ج/ج_9.png
12: /content/drive/MyDrive/colab_ocr_project/test/ح/ح.png
13: /content/drive/MyDrive/colab_ocr_project/test/ح/ح_9.png
14: /content/drive/MyDrive/colab_ocr_project/test/خ/خ.png
15: /content/drive/MyDrive/colab_ocr_project/test/خ/خ_9.png
16: /content/drive/MyDrive/colab_ocr_project/te

In [ ]:
import os

gt_candidate_path = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_ocr_final_backup/dataset_file_list.txt'

print('='*70)
print(f'تحلیل محتوای فایل: {os.path.basename(gt_candidate_path)}')
print('='*70)

if os.path.exists(gt_candidate_path):
    try:
        with open(gt_candidate_path, 'r', encoding='utf-8') as f:
            # خواندن ۲۰ خط اول برای بررسی ساختار دقیق (آیا مسیر و متن با هم هستند؟)
            lines = [f.readline().strip() for _ in range(20)]

            print('نمونه محتوای ۲۰ سطر اول:')
            for i, line in enumerate(lines, 1):
                print(f'{i:02d}: {line}')

        # بررسی تعداد کل خطوط برای اطمینان از پوشش ۲۷ هزار تصویر
        with open(gt_candidate_path, 'r', encoding='utf-8') as f:
            line_count = sum(1 for _ in f)
        print(f'\nتعداد کل خطوط در این فایل: {line_count}')

    except Exception as e:
        print(f'خطا در تحلیل فایل: {e}')
else:
    print('فایل مورد نظر یافت نشد. لطفاً مسیر را بررسی کنید.')

تحلیل محتوای فایل: dataset_file_list.txt
نمونه محتوای ۲۰ سطر اول:
01: /content/drive/MyDrive/colab_ocr_project/test/ا/ا.png
02: /content/drive/MyDrive/colab_ocr_project/test/ا/ا_9.png
03: /content/drive/MyDrive/colab_ocr_project/test/آ/آ_4.png
04: /content/drive/MyDrive/colab_ocr_project/test/ب/ب.png
05: /content/drive/MyDrive/colab_ocr_project/test/ب/ب_9.png
06: /content/drive/MyDrive/colab_ocr_project/test/ت/ت.png
07: /content/drive/MyDrive/colab_ocr_project/test/ت/ت_9.png
08: /content/drive/MyDrive/colab_ocr_project/test/ث/ث.png
09: /content/drive/MyDrive/colab_ocr_project/test/ث/ث_9.png
10: /content/drive/MyDrive/colab_ocr_project/test/ج/ج.png
11: /content/drive/MyDrive/colab_ocr_project/test/ج/ج_9.png
12: /content/drive/MyDrive/colab_ocr_project/test/ح/ح.png
13: /content/drive/MyDrive/colab_ocr_project/test/ح/ح_9.png
14: /content/drive/MyDrive/colab_ocr_project/test/خ/خ.png
15: /content/drive/MyDrive/colab_ocr_project/test/خ/خ_9.png
16: /content/drive/MyDrive/colab_ocr_project/t

In [ ]:
import os
from pathlib import Path

search_root = '/content/drive/Othercomputers/My Laptop/Desktop'
keywords = ['readme', 'info', 'meta', 'annot', 'mapping', 'ground', 'truth', 'desc']

print('='*70)
print('جستجوی عمیق برای فایل‌های راهنما یا متادیتاهای غیرمتعارف')
print('='*70)

found_info_files = []
for root, dirs, files in os.walk(search_root):
    for f in files:
        lower_f = f.lower()
        if any(k in lower_f for k in keywords) or lower_f.endswith(('.mat', '.h5', '.db', '.sqlite')):
            full_path = os.path.join(root, f)
            try:
                size_kb = os.path.getsize(full_path) / 1024
                found_info_files.append((f, full_path, size_kb))
            except:
                continue

if found_info_files:
    print(f'تعداد {len(found_info_files)} فایل مشکوک پیدا شد:')
    for name, path, size in sorted(found_info_files, key=lambda x: x[2], reverse=True):
        print(f' - [{size:8.2f} KB] {name} -> {path}')
else:
    print('هیچ فایل متادیتای جدیدی با کلمات کلیدی مشخص شده یافت نشد.')

جستجوی عمیق برای فایل‌های راهنما یا متادیتاهای غیرمتعارف
هیچ فایل متادیتای جدیدی با کلمات کلیدی مشخص شده یافت نشد.


In [ ]:
import os
from pathlib import Path

search_root = '/content/drive/Othercomputers/My Laptop/Desktop'

print('='*70)
print('جستجوی فایل‌های حجیم و فرمت‌های دیتابیسی برای یافتن Ground Truth')
print('='*70)

# یافتن فایل‌های غیر تصویری بالای 500 کیلوبایت و دیتابیس‌ها
special_files = []
for root, dirs, files in os.walk(search_root):
    for f in files:
        full_path = os.path.join(root, f)
        ext = os.path.splitext(f)[1].lower()
        try:
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            if ext in ['.mat', '.db', '.sqlite', '.pkl', '.pickle', '.h5', '.hdf5'] or (size_mb > 0.5 and ext in ['.txt', '.csv', '.json']):
                special_files.append((f, full_path, size_mb, ext))
        except:
            continue

if special_files:
    print(f'تعداد {len(special_files)} فایل مشکوک پیدا شد:')
    for name, path, size, ext in sorted(special_files, key=lambda x: x[2], reverse=True):
        print(f' - [{size:7.2f} MB] [{ext:8}] {name} -> {path}')
else:
    print('هیچ فایل داده یا دیتابیس مشکوکی یافت نشد.')

جستجوی فایل‌های حجیم و فرمت‌های دیتابیسی برای یافتن Ground Truth
تعداد 4 فایل مشکوک پیدا شد:
 - [   1.87 MB] [.json   ] image_paths.json -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json
 - [   1.85 MB] [.txt    ] dataset_file_list.txt -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_ocr_final_backup/dataset_file_list.txt
 - [   0.00 MB] [.pkl    ] label_encoder.pkl -> /content/drive/Othercomputers/My Laptop/Desktop/code ocr persion/label_encoder.pkl
 - [   0.00 MB] [.pkl    ] class_names.pkl -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/colab_ocr_project/class_names.pkl


In [ ]:
import os
import pickle
from pathlib import Path

search_root = '/content/drive/Othercomputers/My Laptop/Desktop'

print('='*70)
print('جستجوی فایل‌های با حجم بالا یا فرمت‌های خاص برای یافتن دیتابیس متن')
print('='*70)

# جستجوی فایل‌های حجیم که احتمال دارد حاوی متادیتا باشند (بالای ۱ مگابایت)
large_files = []
for root, dirs, files in os.walk(search_root):
    for f in files:
        full_path = os.path.join(root, f)
        try:
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            # فایل‌های حجیم غیر تصویری
            if size_mb > 0.5 and not f.lower().endswith(('.tif', '.png', '.jpg', '.pdf', '.docx', '.ipynb')):
                large_files.append((f, full_path, size_mb))
        except:
            continue

if large_files:
    print(f'تعداد {len(large_files)} فایل غیرتصویری حجیم پیدا شد:')
    for name, path, size in sorted(large_files, key=lambda x: x[2], reverse=True):
        print(f' - [{size:6.2f} MB] {name} -> {path}')
else:
    print('هیچ فایل حجیم مشکوکی یافت نشد.')

print('\n' + '-'*70)
print('بررسی فایل‌های با پسوند خاص (MAT, DB, PKL):')
for root, dirs, files in os.walk(search_root):
    for f in files:
        if f.lower().endswith(('.mat', '.db', '.sqlite', '.pkl', '.pickle')):
            full_path = os.path.join(root, f)
            print(f'Found: {f} in {root}')

جستجوی فایل‌های با حجم بالا یا فرمت‌های خاص برای یافتن دیتابیس متن
تعداد 17 فایل غیرتصویری حجیم پیدا شد:
 - [3527.62 MB] mamba_tokens_final.pt -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt
 - [1278.86 MB] best_pfms_net_model.pth -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS-Net_Research2026_Results/best_pfms_net_model.pth
 - [658.50 MB] swin_checkpoint.pt -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/swin_checkpoint.pt
 - [235.12 MB] trocr_farsi_epoch1.pth -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/colab_ocr_project/checkpoints/trocr_farsi_epoch1.pth
 - [235.12 MB] trocr_farsi_epoch2.pth -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/colab_ocr_project/checkpoints/trocr_farsi_epoch2.pth
 - [235.12 MB] trocr_farsi_epoch3.pth -> /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_feat

In [ ]:
import torch
import json
import os

def inspect_pt_file(path):
    print(f'\n--- Inspecting PyTorch File: {os.path.basename(path)} ---')
    try:
        # File is large, attempting to load metadata/keys without loading full 3.5GB to RAM if possible
        # However, torch.load usually loads the full object.
        data = torch.load(path, map_location='cpu')
        print(f'Data Type: {type(data)}')
        if isinstance(data, dict):
            print(f'Keys: {list(data.keys())}')
            for k in list(data.keys())[:5]:
                v = data[k]
                print(f' - Key "{k}": type={type(v)}')
                if hasattr(v, "shape"): print(f'   Shape: {v.shape}')
        elif isinstance(data, list):
            print(f'List Length: {len(data)}')
            if len(data) > 0: print(f'Sample (first item): {data[0]}')
        del data
    except Exception as e:
        print(f'Error: {e}')

def inspect_json(path):
    print(f'\n--- Inspecting JSON File: {os.path.basename(path)} ---')
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f'Data Type: {type(data)}')
        if isinstance(data, list):
            print(f'List items: {len(data)}')
            print(f'Sample (first 2): {data[:2]}')
        elif isinstance(data, dict):
            print(f'Keys: {list(data.keys())[:10]}')
            first_key = list(data.keys())[0]
            print(f'Sample value for "{first_key}": {data[first_key]}')
    except Exception as e:
        print(f'Error: {e}')

# Strategic paths
tokens_path = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt'
json_path = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json'

if os.path.exists(tokens_path):
    inspect_pt_file(tokens_path)

if os.path.exists(json_path):
    inspect_json(json_path)


--- Inspecting PyTorch File: mamba_tokens_final.pt ---
Data Type: <class 'torch.Tensor'>

--- Inspecting JSON File: image_paths.json ---
Data Type: <class 'list'>
List items: 27178
Sample (first 2): ['/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001 (1).tif', '/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001.tif']


In [ ]:
import torch
import json
import os

def inspect_pt_file(path):
    print(f'\n--- Inspecting PyTorch File: {os.path.basename(path)} ---')
    try:
        # Load with weights_only=False because it might be a custom object or dict
        data = torch.load(path, map_location='cpu')
        print(f'Data Type: {type(data)}')
        if isinstance(data, dict):
            print(f'Keys: {list(data.keys())}')
            for k, v in data.items():
                print(f' - Key "{k}": type={type(v)}')
        elif isinstance(data, list):
            print(f'List Length: {len(data)}')
            if len(data) > 0:
                print(f'Sample Item: {data[0]}')
    except Exception as e:
        print(f'Error: {e}')

def inspect_json_mapping(path):
    print(f'\n--- Inspecting JSON File: {os.path.basename(path)} ---')
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f'Data Type: {type(data)}')
        if isinstance(data, list):
            print(f'Sample (first 3): {data[:3]}')
        elif isinstance(data, dict):
            print(f'Keys (first 5): {list(data.keys())[:5]}')
    except Exception as e:
        print(f'Error: {e}')

# Paths identified in previous step
tokens_path = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt'
json_path = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json'

if os.path.exists(tokens_path):
    # This file is very large (3.5GB), we only inspect structure to avoid OOM if possible
    inspect_pt_file(tokens_path)

if os.path.exists(json_path):
    inspect_json_mapping(json_path)


--- Inspecting PyTorch File: mamba_tokens_final.pt ---
Data Type: <class 'torch.Tensor'>

--- Inspecting JSON File: image_paths.json ---
Data Type: <class 'list'>
Sample (first 3): ['/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001 (1).tif', '/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001.tif', '/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00002 (1).tif']


In [ ]:
import os
from pathlib import Path

search_root = '/content/drive/Othercomputers/My Laptop/Desktop'
keywords = ['readme', 'info', 'meta', 'annot', 'mapping', 'ground', 'truth', 'desc']

print('='*70)
print('جستجوی عمیق برای فایل‌های راهنما یا متادیتاهای غیرمتعارف')
print('='*70)

found_info_files = []
for root, dirs, files in os.walk(search_root):
    for f in files:
        lower_f = f.lower()
        # جستجو بر اساس کلمات کلیدی در نام فایل
        if any(k in lower_f for k in keywords) or lower_f.endswith(('.mat', '.h5', '.db', '.sqlite')):
            full_path = os.path.join(root, f)
            size_kb = os.path.getsize(full_path) / 1024
            found_info_files.append((f, full_path, size_kb))

if found_info_files:
    print(f'تعداد {len(found_info_files)} فایل مشکوک پیدا شد:')
    for name, path, size in sorted(found_info_files, key=lambda x: x[2], reverse=True):
        print(f' - [{size:8.2f} KB] {name} -> {path}')
else:
    print('هیچ فایل متادیتای جدیدی با کلمات کلیدی مشخص شده یافت نشد.')

جستجوی عمیق برای فایل‌های راهنما یا متادیتاهای غیرمتعارف
هیچ فایل متادیتای جدیدی با کلمات کلیدی مشخص شده یافت نشد.


In [ ]:
import os

# مسیر فایل‌هایی که در مرحله قبل پیدا شدند
metadata_dir = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_split'
files_to_inspect = ['train.txt', 'test.txt', 'validation.txt']

print('='*70)
print('بررسی محتوای فایل‌های متادیتای دسکتاپ برای یافتن جملات فارسی')
print('='*70)

for file_name in files_to_inspect:
    file_path = os.path.join(metadata_dir, file_name)
    if os.path.exists(file_path):
        print(f'\n--- فایل: {file_name} ---')
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                # خواندن ۵ خط اول برای تحلیل ساختار
                lines = [f.readline().strip() for _ in range(5)]
                for i, line in enumerate(lines):
                    print(f'سطر {i+1}: {line}')
        except Exception as e:
            print(f'خطا در خواندن فایل: {e}')
    else:
        print(f'فایل {file_name} یافت نشد.')

بررسی محتوای فایل‌های متادیتای دسکتاپ برای یافتن جملات فارسی

--- فایل: train.txt ---
سطر 1: /content/drive/MyDrive/idplimgl_full/28639.tif
سطر 2: /content/drive/MyDrive/idplimgl_full/27259.tif
سطر 3: /content/drive/MyDrive/idplimgl_full/27191.tif
سطر 4: /content/drive/MyDrive/idplimgl_full/28102.tif
سطر 5: /content/drive/MyDrive/idplimgl_full/27562.tif

--- فایل: test.txt ---
سطر 1: /content/drive/MyDrive/idplimgl_full/28540.tif
سطر 2: /content/drive/MyDrive/idplimgl_full/28004.tif
سطر 3: /content/drive/MyDrive/idplimgl_full/28057.tif
سطر 4: /content/drive/MyDrive/idplimgl_full/28378.tif
سطر 5: /content/drive/MyDrive/idplimgl_full/28008.tif

--- فایل: validation.txt ---
سطر 1: /content/drive/MyDrive/idplimgl_full/27909.tif
سطر 2: /content/drive/MyDrive/idplimgl_full/27253.tif
سطر 3: /content/drive/MyDrive/idplimgl_full/27308.tif
سطر 4: /content/drive/MyDrive/idplimgl_full/28406.tif
سطر 5: /content/drive/MyDrive/idplimgl_full/27683.tif


In [ ]:
import json
import os

json_path = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json'

print('='*70)
print('بررسف فافف image_paths.json براف فافءن Ground Truth')
print('='*70)

if os.path.exists(json_path):
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        print(f'نفء أاأء: {type(data)}')

        if isinstance(data, list):
            print(f'ففرء آفءم ها: {len(data)}')
            print('نمفنء ففرء أفء:')
            print(json.dumps(data[:2], indent=2, ensure_ascii=False))
        elif isinstance(data, dict):
            print(f'ففرء كففأ ها: {len(data.keys())}')
            print('نمفنء أاأء:')
            first_key = list(data.keys())[0]
            print(f'{first_key}: {data[first_key]}')

    except Exception as e:
        print(f'أطا أر أففء فافف: {e}')
else:
    print('فافف image_paths.json فافف نشأ.')

بررسف فافف image_paths.json براف فافءن Ground Truth
نفء أاأء: <class 'list'>
ففرء آفءم ها: 27178
نمفنء ففرء أفء:
[
  "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001 (1).tif",
  "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001.tif"
]


In [ ]:

# ==============================================================================
# PFMS-SERIES — CELL 40
# COMPLETE IDPL-PFOD GROUND TRUTH DISCOVERY
# ==============================================================================

import os
import glob
import json
import csv
import pickle
import re

print("=" * 78)
print("PFMS-SERIES — CELL 40")
print("COMPLETE IDPL-PFOD GROUND TRUTH DISCOVERY")
print("=" * 78)


# ==============================================================================
# CONFIGURATION
# ==============================================================================

IMAGE_ROOT = (
    "/content/drive/Othercomputers/"
    "My Laptop/Desktop/idplimgl"
)

SEARCH_ROOTS = [
    "/content/drive/MyDrive",
    "/content/drive/Othercomputers"
]

print("\nImage root:")
print(IMAGE_ROOT)

print("Exists:", os.path.exists(IMAGE_ROOT))
print("Directory:", os.path.isdir(IMAGE_ROOT))


# ==============================================================================
# PART A — IMAGE ID ANALYSIS
# ==============================================================================

print("\n" + "-" * 78)
print("PART A — IMAGE ID ANALYSIS")
print("-" * 78)

image_files = []

for ext in [
    "*.tif",
    "*.tiff",
    "*.png",
    "*.jpg",
    "*.jpeg"
]:

    image_files.extend(
        glob.glob(
            os.path.join(IMAGE_ROOT, ext)
        )
    )

print("Total image files:", len(image_files))

image_stems = set()

for path in image_files:

    stem = os.path.splitext(
        os.path.basename(path)
    )[0]

    image_stems.add(stem)

print("Unique image IDs:", len(image_stems))

print("\nFirst 20 image IDs:")

for stem in sorted(
    image_stems,
    key=lambda x: int(x) if x.isdigit() else x
)[:20]:

    print(" ", stem)


# ==============================================================================
# PART B — COMPLETE METADATA FILE SEARCH
# ==============================================================================

print("\n" + "-" * 78)
print("PART B — SEARCHING FOR METADATA FILES")
print("-" * 78)

metadata_extensions = {
    ".csv",
    ".tsv",
    ".txt",
    ".json",
    ".jsonl",
    ".xml",
    ".pkl",
    ".pickle",
    ".xlsx"
}

metadata_files = []

for search_root in SEARCH_ROOTS:

    if not os.path.exists(search_root):
        continue

    print("\nScanning:")
    print(search_root)

    try:

        for root, dirs, files in os.walk(search_root):

            # Avoid hidden/system directories
            dirs[:] = [
                d for d in dirs
                if not d.startswith(".")
            ]

            for fname in files:

                ext = os.path.splitext(
                    fname
                )[1].lower()

                if ext in metadata_extensions:

                    full_path = os.path.join(
                        root,
                        fname
                    )

                    metadata_files.append(
                        full_path
                    )

    except Exception as e:

        print("Warning:", e)


metadata_files = list(
    dict.fromkeys(metadata_files)
)

print(
    "\nTotal metadata-like files:",
    len(metadata_files)
)


# ==============================================================================
# PART C — SHOW METADATA FILES
# ==============================================================================

print("\n" + "-" * 78)
print("PART C — METADATA FILE LIST")
print("-" * 78)

if metadata_files:

    for i, path in enumerate(
        metadata_files,
        start=1
    ):

        try:

            size_kb = (
                os.path.getsize(path)
                / 1024
            )

        except:

            size_kb = 0

        print(
            f"{i:03d}. {path}"
            f"  [{size_kb:.2f} KB]"
        )

else:

    print("No metadata files found.")


# ==============================================================================
# PART D — FILE NAME ANALYSIS
# ==============================================================================

print("\n" + "-" * 78)
print("PART D — GROUND TRUTH NAME ANALYSIS")
print("-" * 78)

keywords = [
    "label",
    "labels",
    "gt",
    "ground",
    "truth",
    "text",
    "trans",
    "transcription",
    "annotation",
    "annot",
    "caption",
    "target",
    "ocr",
    "idpl",
    "pfod"
]

keyword_files = []

for path in metadata_files:

    name = os.path.basename(path).lower()

    if any(
        k in name
        for k in keywords
    ):

        keyword_files.append(path)


print(
    "Potential Ground Truth files:",
    len(keyword_files)
)

for i, path in enumerate(
    keyword_files,
    start=1
):

    print(
        f"{i:03d}. {path}"
    )


# ==============================================================================
# PART E — TEXT / CSV CONTENT INSPECTION
# ==============================================================================

print("\n" + "-" * 78)
print("PART E — INSPECTING TEXT/CSV CONTENT")
print("-" * 78)

text_like = [
    p for p in metadata_files
    if os.path.splitext(p)[1].lower()
    in [".csv", ".tsv", ".txt", ".json", ".jsonl", ".xml"]
]

print(
    "Text-like files:",
    len(text_like)
)


def contains_persian(text):

    return bool(
        re.search(
            r"[\u0600-\u06FF]",
            text
        )
    )


for i, path in enumerate(
    text_like,
    start=1
):

    print("\n" + "." * 78)
    print(f"FILE {i}:")
    print(path)

    try:

        size = os.path.getsize(path)

        # Don't read extremely large files entirely
        max_bytes = 2_000_000

        with open(
            path,
            "rb"
        ) as f:

            raw = f.read(
                min(size, max_bytes)
            )

        text = raw.decode(
            "utf-8",
            errors="ignore"
        )

        print(
            "Sample characters:",
            len(text)
        )

        print(
            "Contains Persian:",
            contains_persian(text)
        )

        # First 1500 characters
        preview = text[:1500]

        print("\nCONTENT PREVIEW:")
        print(preview)

        # Search image IDs
        matches = []

        for stem in list(
            image_stems
        )[:5000]:

            if stem in text:

                matches.append(stem)

                if len(matches) >= 10:
                    break

        print(
            "\nImage-ID matches found in sample:",
            matches
        )

    except Exception as e:

        print(
            "Inspection error:",
            e
        )


# ==============================================================================
# PART F — PICKLE INSPECTION
# ==============================================================================

print("\n" + "-" * 78)
print("PART F — PICKLE / LABEL ENCODER INSPECTION")
print("-" * 78)

pickle_files = [
    p for p in metadata_files
    if os.path.splitext(p)[1].lower()
    in [".pkl", ".pickle"]
]

print(
    "Pickle files:",
    len(pickle_files)
)


for path in pickle_files:

    print("\n" + "." * 78)
    print("PICKLE:")
    print(path)

    try:

        with open(
            path,
            "rb"
        ) as f:

            obj = pickle.load(f)

        print(
            "Object type:",
            type(obj)
        )

        if isinstance(obj, dict):

            print(
                "Dictionary size:",
                len(obj)
            )

            print(
                "First keys:",
                list(obj.keys())[:20]
            )

        elif isinstance(obj, (list, tuple)):

            print(
                "Sequence length:",
                len(obj)
            )

            print(
                "First items:",
                obj[:10]
            )

        else:

            print(
                "Object:",
                repr(obj)[:1000]
            )

    except Exception as e:

        print(
            "Pickle inspection error:",
            e
        )


# ==============================================================================
# PART G — SEARCH FOR IMAGE-ID ↔ TEXT PAIRS
# ==============================================================================

print("\n" + "-" * 78)
print("PART G — IMAGE ID ↔ TEXT PAIR SEARCH")
print("-" * 78)

numeric_id_pattern = re.compile(
    r"(?<!\d)(\d{3,6})(?!\d)"
)

pair_candidates = []

for path in text_like:

    try:

        with open(
            path,
            "rb"
        ) as f:

            raw = f.read(
                min(
                    os.path.getsize(path),
                    5_000_000
                )
            )

        text = raw.decode(
            "utf-8",
            errors="ignore"
        )

        lines = text.splitlines()

        for line_no, line in enumerate(
            lines[:100000],
            start=1
        ):

            if not contains_persian(line):
                continue

            ids = numeric_id_pattern.findall(
                line
            )

            valid_ids = [
                x for x in ids
                if x in image_stems
            ]

            if valid_ids:

                pair_candidates.append(
                    (
                        path,
                        line_no,
                        valid_ids[:5],
                        line[:500]
                    )
                )

                if len(pair_candidates) >= 20:
                    break

    except Exception:
        pass


print(
    "Potential image-ID/text pairs:",
    len(pair_candidates)
)

for item in pair_candidates:

    path, line_no, ids, line = item

    print("\nFILE:", path)
    print("LINE:", line_no)
    print("IMAGE IDS:", ids)
    print("TEXT:")
    print(line)


# ==============================================================================
# PART H — SEARCH DIRECTORY NAMES
# ==============================================================================

print("\n" + "-" * 78)
print("PART H — RELEVANT DIRECTORY SEARCH")
print("-" * 78)

relevant_dirs = []

for search_root in SEARCH_ROOTS:

    if not os.path.exists(search_root):
        continue

    try:

        for root, dirs, files in os.walk(search_root):

            for d in dirs:

                dl = d.lower()

                if any(
                    k in dl
                    for k in [
                        "idpl",
                        "pfod",
                        "ground",
                        "truth",
                        "label",
                        "annotation",
                        "transcription",
                        "metadata",
                        "ocr"
                    ]
                ):

                    relevant_dirs.append(
                        os.path.join(root, d)
                    )

            dirs[:] = [
                d for d in dirs
                if not d.startswith(".")
            ]

    except Exception:
        pass


relevant_dirs = list(
    dict.fromkeys(relevant_dirs)
)

print(
    "Relevant directories found:",
    len(relevant_dirs)
)

for i, path in enumerate(
    relevant_dirs[:100],
    start=1
):

    print(
        f"{i:03d}. {path}"
    )


# ==============================================================================
# PART I — FINAL SAFETY CHECK
# ==============================================================================

print("\n" + "-" * 78)
print("PART I — SAFETY CHECK")
print("-" * 78)

print("""
✓ No target IDs created
✓ No labels modified
✓ No artificial transcription generated
✓ No model training performed
✓ No model parameters modified
✓ Ground Truth remains untouched
""")


# ==============================================================================
# FINAL
# ==============================================================================

print("\n" + "=" * 78)
print("CELL 40 COMPLETED")
print("=" * 78)

print("""
CURRENT PROJECT STAGE:
STAGE 6-B — REAL GROUND TRUTH DISCOVERY

The output above will tell us:

1. Which metadata files actually exist.
2. Whether Persian transcription text exists.
3. Whether image IDs such as 27868 appear in metadata.
4. Whether an image-ID → transcription relationship exists.
5. Whether label_encoder.pkl contains useful information.
6. Whether another directory contains the actual annotations.

IMPORTANT:
Do NOT start training yet.
Do NOT use labels.csv / labels_extended.csv
unless their actual transcription field is scientifically verified.
""")

print("=" * 78)

PFMS-SERIES — CELL 40
COMPLETE IDPL-PFOD GROUND TRUTH DISCOVERY

Image root:
/content/drive/Othercomputers/My Laptop/Desktop/idplimgl
Exists: True
Directory: True

------------------------------------------------------------------------------
PART A — IMAGE ID ANALYSIS
------------------------------------------------------------------------------
Total image files: 27178
Unique image IDs: 27178

First 20 image IDs:


TypeError: '<' not supported between instances of 'str' and 'int'

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 40-FIX
# IMAGE ID ANALYSIS FIX
# ==============================================================================

print("=" * 78)
print("PFMS-SERIES — CELL 40-FIX")
print("GROUND TRUTH DISCOVERY CONTINUATION")
print("=" * 78)


# image_stems باید از Cell 40 باقی مانده باشد

print("\nImage IDs available:")
print("Count:", len(image_stems))


print("\nFirst 20 image IDs:")

sorted_ids = sorted(
    image_stems,
    key=lambda x: str(x)
)

for stem in sorted_ids[:20]:

    print(" ", stem)


# ==============================================================================
# CONTINUE METADATA SEARCH
# ==============================================================================

import os
import glob

SEARCH_ROOTS = [
    "/content/drive/MyDrive",
    "/content/drive/Othercomputers"
]


print("\n" + "-" * 78)
print("SEARCHING METADATA FILES")
print("-" * 78)


metadata_extensions = [
    ".csv",
    ".txt",
    ".json",
    ".jsonl",
    ".xml",
    ".pkl",
    ".pickle",
    ".xlsx"
]


metadata_files = []


for search_root in SEARCH_ROOTS:

    if not os.path.exists(search_root):
        continue

    for root, dirs, files in os.walk(search_root):

        dirs[:] = [
            d for d in dirs
            if not d.startswith(".")
        ]

        for f in files:

            ext = os.path.splitext(f)[1].lower()

            if ext in metadata_extensions:

                metadata_files.append(
                    os.path.join(root,f)
                )


metadata_files = list(
    dict.fromkeys(metadata_files)
)


print(
    "Metadata files found:",
    len(metadata_files)
)


for i,p in enumerate(metadata_files[:100],1):

    print(
        f"{i:03d}. {p}"
    )


print("\n" + "="*78)
print("CELL 40-FIX COMPLETED")
print("="*78)

print("""
NEXT:
The next step is inspecting these files
for:

Image ID  → Persian transcription

No labels will be generated yet.
""")

PFMS-SERIES — CELL 40-FIX
GROUND TRUTH DISCOVERY CONTINUATION

Image IDs available:
Count: 27178

First 20 image IDs:
  00001
  00001 (1)
  00002
  00002 (1)
  00003
  00004
  00004 (1)
  00005
  00005 (1)
  00006
  00006 (1)
  00007
  00007 (1)
  00008
  00008 (1)
  00009
  00009 (1)
  00010
  00010 (1)
  00011

------------------------------------------------------------------------------
SEARCHING METADATA FILES
------------------------------------------------------------------------------
Metadata files found: 12
001. /content/drive/MyDrive/labels.csv
002. /content/drive/MyDrive/labels_extended.csv
003. /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv
004. /content/drive/Othercomputers/My Laptop/Desktop/333.txt
005. /content/drive/Othercomputers/My Laptop/Desktop/code ocr persion/label_encoder.pkl
006. /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/colab_ocr_project/class_names.pkl
007. /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 41
# AUTOMATIC REAL IDPL-PFOD GROUND TRUTH DISCOVERY
# ==============================================================================

import os
import re
import csv
import json
import pickle
import pandas as pd

print("=" * 78)
print("PFMS-SERIES — CELL 41")
print("AUTOMATIC REAL IDPL-PFOD GROUND TRUTH DISCOVERY")
print("=" * 78)

# ------------------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------------------

IMAGE_ROOT = "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"

SEARCH_ROOTS = [
    "/content/drive/MyDrive",
    "/content/drive/Othercomputers"
]

print("\nImage root:")
print(IMAGE_ROOT)

print("Exists:", os.path.exists(IMAGE_ROOT))


# ------------------------------------------------------------------------------
# PART A — COLLECT IMAGE IDs
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART A — COLLECTING REAL IMAGE IDs")
print("-" * 78)

image_files = []

if os.path.exists(IMAGE_ROOT):

    for f in os.listdir(IMAGE_ROOT):

        if f.lower().endswith((".tif", ".tiff", ".png", ".jpg", ".jpeg")):

            image_files.append(f)


image_ids = set()

for f in image_files:

    stem = os.path.splitext(f)[0]

    # remove duplicate suffix such as " (1)"
    clean_stem = re.sub(r"\s*\(\d+\)$", "", stem)

    image_ids.add(clean_stem)


print("Image files:", len(image_files))
print("Unique normalized image IDs:", len(image_ids))

print("\nFirst 30 IDs:")

for x in sorted(
    image_ids,
    key=lambda z: int(z) if str(z).isdigit() else 10**12
)[:30]:

    print(" ", x)


# ------------------------------------------------------------------------------
# PART B — DISCOVER ALL POSSIBLE METADATA FILES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART B — SEARCHING FOR POSSIBLE GROUND TRUTH FILES")
print("-" * 78)

extensions = {
    ".csv",
    ".txt",
    ".json",
    ".jsonl",
    ".xml",
    ".tsv",
    ".pkl",
    ".pickle",
    ".xlsx"
}

candidate_files = []

for search_root in SEARCH_ROOTS:

    if not os.path.exists(search_root):
        continue

    for root, dirs, files in os.walk(search_root):

        # Ignore hidden/system directories
        dirs[:] = [
            d for d in dirs
            if not d.startswith(".")
        ]

        for f in files:

            ext = os.path.splitext(f)[1].lower()

            if ext in extensions:

                candidate_files.append(
                    os.path.join(root, f)
                )


candidate_files = sorted(
    set(candidate_files)
)

print("Candidate metadata files:", len(candidate_files))

for i, path in enumerate(candidate_files, 1):

    try:
        size_kb = os.path.getsize(path) / 1024
    except:
        size_kb = 0

    print(
        f"{i:03d}. {path} "
        f"({size_kb:.2f} KB)"
    )


# ------------------------------------------------------------------------------
# PART C — SEARCH FILENAMES FOR GROUND TRUTH CLUES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART C — GROUND TRUTH NAME ANALYSIS")
print("-" * 78)

keywords = [
    "label",
    "labels",
    "annotation",
    "annotations",
    "ground",
    "truth",
    "gt",
    "transcription",
    "transcript",
    "text",
    "ocr",
    "idpl",
    "pfod",
    "word",
    "sentence",
    "line"
]

name_candidates = []

for path in candidate_files:

    name = os.path.basename(path).lower()

    score = 0

    for kw in keywords:

        if kw in name:
            score += 1

    if score > 0:

        name_candidates.append(
            (score, path)
        )


name_candidates.sort(
    key=lambda x: (-x[0], x[1])
)

print("Filename candidates:")

for score, path in name_candidates:

    print(
        f"Score={score:02d} | {path}"
    )


# ------------------------------------------------------------------------------
# PART D — INSPECT CSV / TSV FILES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART D — INSPECTING CSV / TSV STRUCTURE")
print("-" * 78)

csv_results = []


for path in candidate_files:

    ext = os.path.splitext(path)[1].lower()

    if ext not in [".csv", ".tsv"]:
        continue

    print("\n" + "=" * 70)
    print("FILE:")
    print(path)

    try:

        if ext == ".tsv":
            df = pd.read_csv(
                path,
                sep="\t",
                encoding="utf-8",
                on_bad_lines="skip"
            )
        else:
            df = pd.read_csv(
                path,
                encoding="utf-8",
                on_bad_lines="skip"
            )

        print("Shape:", df.shape)

        print("Columns:")

        for col in df.columns:

            print(
                " ",
                repr(str(col))
            )

        print("\nFirst 3 rows:")

        print(
            df.head(3).to_string()
        )

        # ----------------------------------------------------------
        # Search for columns that may contain image IDs
        # ----------------------------------------------------------

        id_columns = []

        text_columns = []

        for col in df.columns:

            col_name = str(col).lower()

            # Possible ID column
            if any(
                k in col_name
                for k in [
                    "id",
                    "image",
                    "file",
                    "filename",
                    "path",
                    "name"
                ]
            ):

                id_columns.append(col)

            # Possible transcription column
            if any(
                k in col_name
                for k in [
                    "text",
                    "label",
                    "trans",
                    "gt",
                    "truth",
                    "sentence",
                    "word",
                    "content"
                ]
            ):

                text_columns.append(col)

        print("\nPossible ID columns:")
        print(id_columns)

        print("Possible text columns:")
        print(text_columns)

        # ----------------------------------------------------------
        # Test actual image-ID overlap
        # ----------------------------------------------------------

        best_overlap = 0
        best_pair = None

        for id_col in id_columns:

            try:

                values = (
                    df[id_col]
                    .astype(str)
                    .str.strip()
                    .str.replace(
                        r"\s*\(\d+\)$",
                        "",
                        regex=True
                    )
                )

                overlap = len(
                    set(values)
                    .intersection(image_ids)
                )

                if overlap > best_overlap:

                    best_overlap = overlap
                    best_pair = id_col

            except:
                pass

        print(
            "\nImage-ID overlap:",
            best_overlap
        )

        if best_pair is not None:

            print(
                "Best matching ID column:",
                best_pair
            )

        csv_results.append({
            "path": path,
            "shape": df.shape,
            "id_columns": id_columns,
            "text_columns": text_columns,
            "best_id_column": best_pair,
            "image_overlap": best_overlap
        })

    except Exception as e:

        print(
            "Could not read file:",
            repr(e)
        )


# ------------------------------------------------------------------------------
# PART E — INSPECT TXT FILES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART E — SEARCHING TEXT FILES FOR IMAGE IDs + PERSIAN")
print("-" * 78)

persian_pattern = re.compile(
    r"[\u0600-\u06FF]"
)

txt_results = []


for path in candidate_files:

    if os.path.splitext(path)[1].lower() != ".txt":
        continue

    print("\nFILE:")
    print(path)

    try:

        with open(
            path,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as f:

            content = f.read()

        print(
            "Characters:",
            len(content)
        )

        lines = content.splitlines()

        persian_lines = [
            line
            for line in lines
            if persian_pattern.search(line)
        ]

        id_lines = []

        for line in lines[:]:

            numbers = re.findall(
                r"\b\d{1,6}\b",
                line
            )

            if numbers:

                for num in numbers:

                    if num in image_ids:

                        id_lines.append(line)
                        break

        print(
            "Lines containing Persian text:",
            len(persian_lines)
        )

        print(
            "Lines containing image IDs:",
            len(id_lines)
        )

        print("\nSample Persian lines:")

        for line in persian_lines[:5]:

            print(
                " ",
                line[:300]
            )

        print("\nSample ID-related lines:")

        for line in id_lines[:5]:

            print(
                " ",
                line[:300]
            )

        txt_results.append({
            "path": path,
            "persian_lines": len(persian_lines),
            "id_lines": len(id_lines)
        })

    except Exception as e:

        print(
            "Could not inspect:",
            repr(e)
        )


# ------------------------------------------------------------------------------
# PART F — JSON INSPECTION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART F — INSPECTING JSON FILES")
print("-" * 78)


for path in candidate_files:

    if os.path.splitext(path)[1].lower() not in [
        ".json",
        ".jsonl"
    ]:
        continue

    print("\nFILE:")
    print(path)

    try:

        if path.lower().endswith(".jsonl"):

            with open(
                path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                lines = f.readlines()

            print(
                "JSONL lines:",
                len(lines)
            )

            for line in lines[:3]:

                print(
                    " ",
                    line[:500]
                )

        else:

            with open(
                path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                data = json.load(f)

            print(
                "JSON object type:",
                type(data)
            )

            if isinstance(data, dict):

                print(
                    "Top-level keys:",
                    list(data.keys())[:30]
                )

            elif isinstance(data, list):

                print(
                    "List length:",
                    len(data)
                )

                if len(data) > 0:

                    print(
                        "First item:",
                        str(data[0])[:500]
                    )

    except Exception as e:

        print(
            "Could not inspect JSON:",
            repr(e)
        )


# ------------------------------------------------------------------------------
# PART G — FINAL AUTOMATIC DIAGNOSIS
# ------------------------------------------------------------------------------

print("\n" + "=" * 78)
print("CELL 41 — AUTOMATIC GROUND TRUTH DISCOVERY SUMMARY")
print("=" * 78)

print("\nCSV/TSV candidates with image-ID overlap:")

found_overlap = False

for r in csv_results:

    if r["image_overlap"] > 0:

        found_overlap = True

        print(
            f"\n{r['path']}"
        )

        print(
            "  Image overlap:",
            r["image_overlap"]
        )

        print(
            "  ID columns:",
            r["id_columns"]
        )

        print(
            "  Text columns:",
            r["text_columns"]
        )

if not found_overlap:

    print(
        "  No CSV/TSV with confirmed image-ID overlap found."
    )


print("\nText files containing Persian text + image IDs:")

found_txt = False

for r in txt_results:

    if (
        r["persian_lines"] > 0
        and r["id_lines"] > 0
    ):

        found_txt = True

        print(
            "\n ",
            r["path"]
        )

        print(
            "  Persian lines:",
            r["persian_lines"]
        )

        print(
            "  ID-related lines:",
            r["id_lines"]
        )

if not found_txt:

    print(
        "  No TXT file confirmed as Ground Truth."
    )


print("\n" + "=" * 78)
print("SAFETY STATUS")
print("=" * 78)

print("""
✓ No Target IDs created
✓ No artificial labels created
✓ No model training performed
✓ No model parameters modified
✓ No temporary targets used as Ground Truth

The purpose of this cell is ONLY discovery and verification.
""")

print("=" * 78)
print("CELL 41 COMPLETED")
print("=" * 78)

PFMS-SERIES — CELL 41
AUTOMATIC REAL IDPL-PFOD GROUND TRUTH DISCOVERY

Image root:
/content/drive/Othercomputers/My Laptop/Desktop/idplimgl
Exists: True

------------------------------------------------------------------------------
PART A — COLLECTING REAL IMAGE IDs
------------------------------------------------------------------------------
Image files: 27178
Unique normalized image IDs: 27120

First 30 IDs:
  00001
  00002
  00003
  00004
  00005
  00006
  00007
  00008
  00009
  00010
  00011
  00012
  00013
  00014
  00015
  00016
  00017
  00018
  00019
  00020
  00021
  00022
  00023
  00024
  00025
  00026
  00027
  00028
  00029
  00030

------------------------------------------------------------------------------
PART B — SEARCHING FOR POSSIBLE GROUND TRUTH FILES
------------------------------------------------------------------------------
Candidate metadata files: 12
001. /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv (6087.69 KB)
002. /content/drive/MyDrive/labels

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 42
# EXACT IDPL-PFOD GROUND TRUTH RELATIONSHIP ANALYSIS
# ==============================================================================

import os
import re
import json
import pandas as pd
from collections import Counter, defaultdict

print("=" * 78)
print("PFMS-SERIES — CELL 42")
print("EXACT IDPL-PFOD GROUND TRUTH RELATIONSHIP ANALYSIS")
print("=" * 78)

# ------------------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------------------

IMAGE_ROOT = "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"

FILE_LIST = (
    "/content/drive/MyDrive/"
    "mamba_ocr_final_backup/dataset_file_list.txt"
)

LABELS_CSV = "/content/drive/MyDrive/labels.csv"
LABELS_EXTENDED = "/content/drive/MyDrive/labels_extended.csv"

print("\nImage root:")
print(IMAGE_ROOT)

print("\nDataset file list:")
print(FILE_LIST)

# ------------------------------------------------------------------------------
# PART A — VERIFY PATHS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART A — PATH VALIDATION")
print("-" * 78)

print("Image root exists :", os.path.isdir(IMAGE_ROOT))
print("File list exists  :", os.path.isfile(FILE_LIST))
print("labels.csv exists :", os.path.isfile(LABELS_CSV))
print("labels_extended exists :", os.path.isfile(LABELS_EXTENDED))

if not os.path.isdir(IMAGE_ROOT):
    raise RuntimeError("IDPL-PFOD image root is unavailable.")

if not os.path.isfile(FILE_LIST):
    raise RuntimeError("dataset_file_list.txt was not found.")

# ------------------------------------------------------------------------------
# PART B — COLLECT REAL IDPL IMAGE IDs
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART B — COLLECTING REAL IDPL-PFOD IMAGE IDS")
print("-" * 78)

image_files = []

for root, dirs, files in os.walk(IMAGE_ROOT):
    for f in files:
        if f.lower().endswith((".tif", ".tiff")):
            image_files.append(os.path.join(root, f))

print("Total image files:", len(image_files))

def normalize_image_id(filename):
    """
    Converts:
        00001.tif
        00001 (1).tif
        00001 (2).tif
    into:
        00001
    """
    name = os.path.basename(filename)
    stem = os.path.splitext(name)[0]

    stem = re.sub(r"\s*\(\d+\)$", "", stem)

    return stem.strip()

real_ids = set()

for path in image_files:
    real_ids.add(normalize_image_id(path))

print("Unique normalized IDs:", len(real_ids))

print("\nFirst 30 normalized IDs:")

for x in sorted(
    real_ids,
    key=lambda v: int(v) if v.isdigit() else 10**12
)[:30]:
    print(" ", x)

# ------------------------------------------------------------------------------
# PART C — READ DATASET FILE LIST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART C — READING DATASET FILE LIST")
print("-" * 78)

with open(FILE_LIST, "r", encoding="utf-8", errors="ignore") as f:
    lines = [line.strip() for line in f if line.strip()]

print("Total non-empty lines:", len(lines))

# ------------------------------------------------------------------------------
# PART D — ANALYZE LINE TYPES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART D — LINE STRUCTURE ANALYSIS")
print("-" * 78)

id_related = []
persian_related = []
id_and_persian = []

persian_pattern = re.compile(r"[\u0600-\u06FF]")

for line in lines:

    basename = os.path.basename(line)

    normalized = normalize_image_id(basename)

    has_id = normalized in real_ids

    has_persian = bool(persian_pattern.search(line))

    if has_id:
        id_related.append(line)

    if has_persian:
        persian_related.append(line)

    if has_id and has_persian:
        id_and_persian.append(line)

print("Lines containing real image IDs :", len(id_related))
print("Lines containing Persian text   :", len(persian_related))
print("Lines containing BOTH           :", len(id_and_persian))

# ------------------------------------------------------------------------------
# PART E — SHOW IMPORTANT EXAMPLES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART E — EXAMPLES OF ID-RELATED LINES")
print("-" * 78)

for line in id_related[:30]:
    print(line)

print("\n" + "-" * 78)
print("PART E-2 — EXAMPLES OF PERSIAN LINES")
print("-" * 78)

for line in persian_related[:30]:
    print(line)

print("\n" + "-" * 78)
print("PART E-3 — LINES CONTAINING BOTH IMAGE ID AND PERSIAN")
print("-" * 78)

if id_and_persian:

    for line in id_and_persian[:50]:
        print(line)

else:
    print("NO DIRECT IMAGE-ID + PERSIAN RELATION FOUND.")

# ------------------------------------------------------------------------------
# PART F — CSV INSPECTION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART F — EXISTING CSV ANALYSIS")
print("-" * 78)

for csv_path in [LABELS_CSV, LABELS_EXTENDED]:

    if not os.path.isfile(csv_path):
        continue

    print("\nFILE:")
    print(csv_path)

    try:

        df = pd.read_csv(csv_path)

        print("Shape:", df.shape)
        print("Columns:", list(df.columns))

        print("\nFirst 10 rows:")
        print(df.head(10).to_string(index=False))

        # Find image column
        image_col = None

        for col in df.columns:

            if "image" in str(col).lower() or "path" in str(col).lower():
                image_col = col
                break

        # Find text column
        text_col = None

        for col in df.columns:

            if str(col).lower() in [
                "text",
                "label",
                "transcription",
                "transcript",
                "ground_truth",
                "gt"
            ]:
                text_col = col
                break

        print("\nDetected image column:", image_col)
        print("Detected text column :", text_col)

        if image_col is not None:

            csv_ids = set()

            for value in df[image_col].astype(str):

                csv_ids.add(
                    normalize_image_id(value)
                )

            overlap = csv_ids.intersection(real_ids)

            print("CSV unique IDs:", len(csv_ids))
            print("Overlap with real IDPL IDs:", len(overlap))

            if overlap:

                print("\nFirst matching IDs:")

                for x in sorted(
                    overlap,
                    key=lambda v: int(v) if v.isdigit() else 10**12
                )[:30]:

                    print(" ", x)

        if text_col is not None:

            print("\nText column samples:")

            for value in df[text_col].astype(str).head(20):

                print(" ", repr(value))

    except Exception as e:

        print("CSV inspection error:", repr(e))

# ------------------------------------------------------------------------------
# PART G — DETECT POSSIBLE ID/TEXT PAIRS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART G — POSSIBLE IMAGE-ID → TEXT PAIRS")
print("-" * 78)

possible_pairs = []

for line in lines:

    basename = os.path.basename(line)

    normalized_id = normalize_image_id(basename)

    if normalized_id not in real_ids:
        continue

    # Split possible path / label structures
    parts = re.split(r"[\t,;|]", line)

    for part in parts:

        part = part.strip()

        if not part:
            continue

        if persian_pattern.search(part):

            possible_pairs.append(
                (normalized_id, part, line)
            )

            break

print("Possible ID → Persian-text pairs:", len(possible_pairs))

for item in possible_pairs[:50]:

    image_id, text, original_line = item

    print("\nImage ID :", image_id)
    print("Text     :", text)
    print("Original :", original_line)

# ------------------------------------------------------------------------------
# PART H — SAVE DIAGNOSTIC RESULTS ONLY
# ------------------------------------------------------------------------------

diagnostic_path = "/content/pfms_series_gt_diagnostic.txt"

with open(
    diagnostic_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "PFMS-SERIES CELL 42 GROUND TRUTH DIAGNOSTIC\n"
    )

    f.write(
        f"Total images: {len(image_files)}\n"
    )

    f.write(
        f"Unique IDs: {len(real_ids)}\n"
    )

    f.write(
        f"File list lines: {len(lines)}\n"
    )

    f.write(
        f"ID-related lines: {len(id_related)}\n"
    )

    f.write(
        f"Persian lines: {len(persian_related)}\n"
    )

    f.write(
        f"ID + Persian lines: {len(id_and_persian)}\n"
    )

    f.write(
        f"Possible ID-text pairs: {len(possible_pairs)}\n"
    )

print("\nDiagnostic file saved:")
print(diagnostic_path)

# ------------------------------------------------------------------------------
# FINAL SAFETY STATUS
# ------------------------------------------------------------------------------

print("\n" + "=" * 78)
print("CELL 42 COMPLETED")
print("=" * 78)

print("""
IMPORTANT SAFETY STATUS
-----------------------

✓ No Target IDs created
✓ No artificial Ground Truth created
✓ No model training performed
✓ No model parameters modified
✓ No temporary targets used
✓ Only dataset structure was analyzed

NEXT STEP
---------

The output above will determine whether:

1. dataset_file_list.txt contains real ID → text mapping,

OR

2. the real Ground Truth must be recovered from another source.

DO NOT START OCR TRAINING YET.
""")

print("=" * 78)

PFMS-SERIES — CELL 42
EXACT IDPL-PFOD GROUND TRUTH RELATIONSHIP ANALYSIS

Image root:
/content/drive/Othercomputers/My Laptop/Desktop/idplimgl

Dataset file list:
/content/drive/MyDrive/mamba_ocr_final_backup/dataset_file_list.txt

------------------------------------------------------------------------------
PART A — PATH VALIDATION
------------------------------------------------------------------------------
Image root exists : True
File list exists  : False
labels.csv exists : True
labels_extended exists : True


RuntimeError: dataset_file_list.txt was not found.

In [ ]:
# ==============================================================================
# PFMS-SERIES — CELL 43
# OFFICIAL IDPL-PFOD GROUND TRUTH RECOVERY
# Find the original 30,138-row Ground Truth CSV
# ==============================================================================

import os
import re
import csv
import pandas as pd
from pathlib import Path

print("=" * 78)
print("PFMS-SERIES — CELL 43")
print("OFFICIAL IDPL-PFOD GROUND TRUTH RECOVERY")
print("=" * 78)

# ------------------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------------------

IMAGE_ROOT = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
)

SEARCH_ROOTS = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Othercomputers"),
]

EXPECTED_DATASET_SIZE = 30138

print("\nImage root:")
print(IMAGE_ROOT)

print("\nOfficial IDPL-PFOD repository reports:")
print("Expected images : 30138")
print("Expected GT CSV : 30138 rows")
print("Image size      : 700 x 50")

# ------------------------------------------------------------------------------
# PART A — VERIFY CURRENT IMAGE DATASET
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART A — CURRENT IDPL-PFOD DATASET")
print("-" * 78)

if IMAGE_ROOT.exists():

    image_files = sorted(
        list(IMAGE_ROOT.glob("*.tif")) +
        list(IMAGE_ROOT.glob("*.tiff"))
    )

    print("Current images:", len(image_files))

    if len(image_files) == EXPECTED_DATASET_SIZE:
        print("✓ Exact official dataset size detected")

    elif len(image_files) < EXPECTED_DATASET_SIZE:
        print(
            f"⚠ Current folder contains {len(image_files)} images "
            f"but official dataset contains {EXPECTED_DATASET_SIZE}"
        )

else:
    image_files = []
    print("✗ Image root unavailable")

# ------------------------------------------------------------------------------
# PART B — SEARCH ALL DRIVE FOR CSV FILES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART B — COMPLETE DRIVE CSV SEARCH")
print("-" * 78)

csv_files = []

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    print("\nSearching:")
    print(root)

    try:

        for p in root.rglob("*.csv"):

            if p.is_file():
                csv_files.append(p)

    except Exception as e:

        print("Search warning:", e)

# remove duplicates
csv_files = sorted(set(csv_files))

print("\nTotal CSV files found:", len(csv_files))

for i, p in enumerate(csv_files, 1):

    try:
        size_kb = p.stat().st_size / 1024
    except:
        size_kb = -1

    print(
        f"{i:03d}. {p} "
        f"({size_kb:.2f} KB)"
    )

# ------------------------------------------------------------------------------
# PART C — INSPECT CSV FILES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("PART C — CSV STRUCTURE ANALYSIS")
print("-" * 78)

gt_candidates = []

for csv_path in csv_files:

    print("\n" + "=" * 70)
    print("FILE:")
    print(csv_path)

    try:

        df = pd.read_csv(
            csv_path,
            encoding="utf-8",
            low_memory=False
        )

        print("Shape:", df.shape)
        print("Columns:", list(df.columns))

        if len(df) > 0:

            print("\nFirst 3 rows:")
            print(df.head(3).to_string())

        # ------------------------------------------------------------------
        # Detect image/path column
        # ------------------------------------------------------------------

        image_cols = []

        for col in df.columns:

            col_low = str(col).lower()

            if any(
                key in col_low
                for key in [
                    "image",
                    "img",
                    "path",
                    "file",
                    "filename"
                ]
            ):
                image_cols.append(col)

        # ------------------------------------------------------------------
        # Detect text / ground-truth column
        # ------------------------------------------------------------------

        text_cols = []

        for col in df.columns:

            col_low = str(col).lower()

            if any(
                key in col_low
                for key in [
                    "text",
                    "label",
                    "gt",
                    "ground",
                    "transcription",
                    "transcript",
                    "sentence",
                    "content"
                ]
            ):
                text_cols.append(col)

        print("\nPossible image columns:")
        print(image_cols)

        print("Possible text columns:")
        print(text_cols)

        # ------------------------------------------------------------------
        # Candidate scoring
        # ------------------------------------------------------------------

        score = 0

        if len(df) >= 30000:
            score += 10

        if len(image_cols) > 0:
            score += 5

        if len(text_cols) > 0:
            score += 5

        # Check Persian content
        persian_count = 0

        for col in text_cols:

            try:

                sample = df[col].astype(str).head(10000)

                for value in sample:

                    if re.search(
                        r"[\u0600-\u06FF]",
                        value
                    ):
                        persian_count += 1

            except:
                pass

        print("Persian-containing text samples:", persian_count)

        if persian_count > 0:
            score += 20

        # ------------------------------------------------------------------
        # Image ID overlap
        # ------------------------------------------------------------------

        overlap = 0

        if image_files and image_cols:

            real_names = set()

            for img in image_files:

                stem = img.stem

                stem = re.sub(
                    r"\s*\(\d+\)$",
                    "",
                    stem
                )

                real_names.add(stem)

            csv_ids = set()

            for value in df[image_cols[0]].astype(str):

                name = Path(value).stem

                name = re.sub(
                    r"\s*\(\d+\)$",
                    "",
                    name
                )

                csv_ids.add(name)

            overlap = len(
                real_names.intersection(csv_ids)
            )

        print("Image-ID overlap:", overlap)

        if overlap > 0:
            score += 10

        print("Candidate score:", score)

        if score >= 20:

            gt_candidates.append(
                (
                    score,
                    csv_path,
                    df,
                    image_cols,
                    text_cols,
                    overlap,
                    persian_count
                )
            )

    except Exception as e:

        print("Could not read CSV:")
        print(e)

# ------------------------------------------------------------------------------
# PART D — RANK CANDIDATES
# ------------------------------------------------------------------------------

print("\n" + "=" * 78)
print("PART D — GROUND TRUTH CANDIDATE RANKING")
print("=" * 78)

gt_candidates.sort(
    key=lambda x: x[0],
    reverse=True
)

print(
    "\nPotential Ground Truth CSV files:",
    len(gt_candidates)
)

for i, candidate in enumerate(
    gt_candidates,
    1
):

    (
        score,
        path,
        df,
        image_cols,
        text_cols,
        overlap,
        persian_count
    ) = candidate

    print("\nCandidate", i)
    print("Score:", score)
    print("File:", path)
    print("Rows:", len(df))
    print("Image columns:", image_cols)
    print("Text columns:", text_cols)
    print("Image overlap:", overlap)
    print("Persian samples:", persian_count)

# ------------------------------------------------------------------------------
# PART E — SAVE DIAGNOSTIC REPORT
# ------------------------------------------------------------------------------

report_path = (
    "/content/pfms_series_gt_recovery_report.txt"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "PFMS-SERIES — IDPL-PFOD "
        "GROUND TRUTH RECOVERY REPORT\n"
    )

    f.write("=" * 78 + "\n\n")

    f.write(
        f"Current images: {len(image_files)}\n"
    )

    f.write(
        f"Official expected images: "
        f"{EXPECTED_DATASET_SIZE}\n\n"
    )

    f.write(
        f"CSV files found: "
        f"{len(csv_files)}\n\n"
    )

    for i, candidate in enumerate(
        gt_candidates,
        1
    ):

        (
            score,
            path,
            df,
            image_cols,
            text_cols,
            overlap,
            persian_count
        ) = candidate

        f.write(
            f"Candidate {i}\n"
        )

        f.write(
            f"Score: {score}\n"
        )

        f.write(
            f"File: {path}\n"
        )

        f.write(
            f"Rows: {len(df)}\n"
        )

        f.write(
            f"Image columns: "
            f"{image_cols}\n"
        )

        f.write(
            f"Text columns: "
            f"{text_cols}\n"
        )

        f.write(
            f"Image overlap: "
            f"{overlap}\n"
        )

        f.write(
            f"Persian samples: "
            f"{persian_count}\n\n"
        )

print("\nDiagnostic report saved:")
print(report_path)

# ------------------------------------------------------------------------------
# PART F — SAFETY
# ------------------------------------------------------------------------------

print("\n" + "=" * 78)
print("SAFETY STATUS")
print("=" * 78)

print("""
✓ NO Target IDs created
✓ NO artificial Ground Truth created
✓ NO model training performed
✓ NO model parameters modified
✓ NO temporary labels used
✓ ONLY Ground Truth discovery performed
""")

# ------------------------------------------------------------------------------
# FINAL
# ------------------------------------------------------------------------------

print("=" * 78)
print("CELL 43 COMPLETED")
print("=" * 78)

print("""
NEXT:

If a CSV with approximately 30,138 rows and Persian text
is found, that is the file we need.

DO NOT TRAIN YET.

The next cell will create:

Image ID
   ↓
Real Persian transcription
   ↓
Tokenizer
   ↓
Target IDs

ONLY after the Ground Truth CSV is verified.
""")

PFMS-SERIES — CELL 43
OFFICIAL IDPL-PFOD GROUND TRUTH RECOVERY

Image root:
/content/drive/Othercomputers/My Laptop/Desktop/idplimgl

Official IDPL-PFOD repository reports:
Expected images : 30138
Expected GT CSV : 30138 rows
Image size      : 700 x 50

------------------------------------------------------------------------------
PART A — CURRENT IDPL-PFOD DATASET
------------------------------------------------------------------------------
Current images: 27178
⚠ Current folder contains 27178 images but official dataset contains 30138

------------------------------------------------------------------------------
PART B — COMPLETE DRIVE CSV SEARCH
------------------------------------------------------------------------------

Searching:
/content/drive/MyDrive

Searching:
/content/drive/Othercomputers

Total CSV files found: 3
001. /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv (6087.69 KB)
002. /content/drive/MyDrive/labels.csv (17.51 KB)
003. /content/drive/MyDrive/labels_ext

In [ ]:
import pickle
import os

def inspect_pickle_mapping(path):
    print(f'--- Inspecting Mapping File: {os.path.basename(path)} ---')
    try:
        with open(path, 'rb') as f:
            data = pickle.load(f)

        print(f'Data Type: {type(data)}')

        if hasattr(data, 'classes_'):
            print(f'Detected LabelEncoder. Classes sample: {list(data.classes_)[:50]}')
            print(f'Total unique characters/classes: {len(data.classes_)}')
        elif isinstance(data, dict):
            print(f'Keys: {list(data.keys())[:20]}')
            # If it's a mapping dict
            if len(data) > 0:
                sample_key = list(data.keys())[0]
                print(f'Sample mapping: {sample_key} -> {data[sample_key]}')
        elif isinstance(data, list):
            print(f'List sample: {data[:50]}')
            print(f'Total items: {len(data)}')

        return data
    except Exception as e:
        print(f'Error during inspection: {e}')
        return None

# Potential mapping files discovered in folder structures
pkl_1 = '/content/drive/Othercomputers/My Laptop/Desktop/code ocr persion/label_encoder.pkl'
pkl_2 = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/colab_ocr_project/class_names.pkl'

for p in [pkl_1, pkl_2]:
    if os.path.exists(p):
        inspect_pickle_mapping(p)
    else:
        print(f'File not found at: {p}')

--- Inspecting Mapping File: label_encoder.pkl ---
Data Type: <class 'sklearn.preprocessing._label.LabelEncoder'>
Detected LabelEncoder. Classes sample: [np.str_('your_test_image'), np.str_('آ'), np.str_('ا'), np.str_('ب'), np.str_('ت'), np.str_('ث'), np.str_('ج'), np.str_('ح'), np.str_('خ'), np.str_('د'), np.str_('ذ'), np.str_('ر'), np.str_('ز'), np.str_('س'), np.str_('ش'), np.str_('ص'), np.str_('ض'), np.str_('ط'), np.str_('ظ'), np.str_('ع'), np.str_('غ'), np.str_('ف'), np.str_('ق'), np.str_('ل'), np.str_('م'), np.str_('ن'), np.str_('ه'), np.str_('و'), np.str_('پ'), np.str_('چ'), np.str_('ژ'), np.str_('ک'), np.str_('گ'), np.str_('ی')]
Total unique characters/classes: 34
--- Inspecting Mapping File: class_names.pkl ---
Data Type: <class 'list'>
List sample: ['ا', 'آ', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ل', 'م', 'ن', 'ه', 'و', 'پ', 'چ', 'ژ', 'ک', 'گ', 'ی']
Total items: 33


In [ ]:
import pickle
import os
import torch

def decode_idpl_mappings(file_path):
    print(f'\n--- Decoding: {os.path.basename(file_path)} ---')
    try:
        with open(file_path, 'rb') as f:
            data = pickle.load(f)

        # Check if it's a scikit-learn LabelEncoder or a simple list/dict
        if hasattr(data, 'classes_'):
            classes = list(data.classes_)
            print(f'Found LabelEncoder with {len(classes)} classes.')
            print(f'Alphabet/Vocab Sample: {classes}')
        elif isinstance(data, list):
            print(f'Found list with {len(data)} items.')
            print(f'Content Sample: {data[:100]}')
        elif isinstance(data, dict):
            print(f'Found dictionary with {len(data)} keys.')
            sample_keys = list(data.keys())[:10]
            for k in sample_keys:
                print(f'  {k} -> {data[k]}')
        return data
    except Exception as e:
        print(f'Could not decode {file_path}: {e}')
        return None

# Files identified as critical mapping components
paths = [
    '/content/drive/Othercomputers/My Laptop/Desktop/code ocr persion/label_encoder.pkl',
    '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/colab_ocr_project/class_names.pkl'
]

mappings = {}
for p in paths:
    if os.path.exists(p):
        mappings[os.path.basename(p)] = decode_idpl_mappings(p)
    else:
        print(f'File not found: {p}')


--- Decoding: label_encoder.pkl ---
Found LabelEncoder with 34 classes.
Alphabet/Vocab Sample: [np.str_('your_test_image'), np.str_('آ'), np.str_('ا'), np.str_('ب'), np.str_('ت'), np.str_('ث'), np.str_('ج'), np.str_('ح'), np.str_('خ'), np.str_('د'), np.str_('ذ'), np.str_('ر'), np.str_('ز'), np.str_('س'), np.str_('ش'), np.str_('ص'), np.str_('ض'), np.str_('ط'), np.str_('ظ'), np.str_('ع'), np.str_('غ'), np.str_('ف'), np.str_('ق'), np.str_('ل'), np.str_('م'), np.str_('ن'), np.str_('ه'), np.str_('و'), np.str_('پ'), np.str_('چ'), np.str_('ژ'), np.str_('ک'), np.str_('گ'), np.str_('ی')]

--- Decoding: class_names.pkl ---
Found list with 33 items.
Content Sample: ['ا', 'آ', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ل', 'م', 'ن', 'ه', 'و', 'پ', 'چ', 'ژ', 'ک', 'گ', 'ی']


In [ ]:
# Search for the missing link: A file that maps Image IDs to the Label indices
import json

def search_for_id_to_label_map(root_path):
    print(f'\n--- Searching for ID-to-Text Mapping in {root_path} ---')
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.endswith(('.json', '.csv', '.txt')):
                full_path = os.path.join(root, file)
                # We are looking for files containing common IDPL IDs like '27868' or '00001'
                try:
                    with open(full_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read(1000)
                        if '27868' in content or '00001' in content:
                            print(f'[!] Potential mapping file found: {full_path}')
                            print(f'Preview: {content[:200]}')
                except:
                    continue

search_for_id_to_label_map('/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features')


--- Searching for ID-to-Text Mapping in /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features ---
[!] Potential mapping file found: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1/image_paths.json
Preview: [
  "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001 (1).tif",
  "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl/00001.tif",
  "/content/drive/Othercomputers/My Laptop/Desktop


In [ ]:
import os

base = "/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS_Swin_Stage1"

print("="*80)
print("FILES IN PFMS_Swin_Stage1")
print("="*80)

for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}[DIR] {os.path.basename(root)}/")

    for f in files:
        size = os.path.getsize(os.path.join(root, f)) / (1024**2)
        print(f"{indent}    {f}  ({size:.2f} MB)")

FILES IN PFMS_Swin_Stage1
[DIR] PFMS_Swin_Stage1/
    image_paths.json  (1.87 MB)
    swin_checkpoint.pt  (658.50 MB)


In [ ]:
import os

print("در کل Google Drive دنبال CSV می‌گردیم...\n")

for root, dirs, files in os.walk("/content/drive"):
    for f in files:
        if f.lower().endswith(".csv"):
            print(os.path.join(root, f))

در کل Google Drive دنبال CSV می‌گردیم...

/content/drive/MyDrive/labels.csv
/content/drive/MyDrive/labels_extended.csv
/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv


In [ ]:
import pandas as pd

files = [
    "/content/drive/MyDrive/labels.csv",
    "/content/drive/MyDrive/labels_extended.csv"
]

for path in files:
    print("\n" + "="*80)
    print("FILE:", path)
    print("="*80)

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())

    print("\nFirst 10 rows:")
    display(df.head(10))


FILE: /content/drive/MyDrive/labels.csv
Shape: (1061, 2)
Columns:
['image', 'text']

First 10 rows:


,image,text
0,00046.tif,00046
1,00039.tif,00039
2,00024.tif,00024
3,00050.tif,00050
4,00040 (1).tif,00040 (1)
5,00003 (1).tif,00003 (1)
6,00021.tif,00021
7,00031.tif,00031
8,00030.tif,00030
9,00048.tif,00048



FILE: /content/drive/MyDrive/labels_extended.csv
Shape: (1061, 2)
Columns:
['image_path', 'text']

First 10 rows:


,image_path,text
0,/content/drive/MyDrive/idplimgl/idplimgl/00046...,00046
1,/content/drive/MyDrive/idplimgl/idplimgl/00039...,00039
2,/content/drive/MyDrive/idplimgl/idplimgl/00024...,00024
3,/content/drive/MyDrive/idplimgl/idplimgl/00050...,00050
4,/content/drive/MyDrive/idplimgl/idplimgl/00040...,00040 (1)
5,/content/drive/MyDrive/idplimgl/idplimgl/00003...,00003 (1)
6,/content/drive/MyDrive/idplimgl/idplimgl/00021...,00021
7,/content/drive/MyDrive/idplimgl/idplimgl/00031...,00031
8,/content/drive/MyDrive/idplimgl/idplimgl/00030...,00030
9,/content/drive/MyDrive/idplimgl/idplimgl/00048...,00048


In [ ]:
# جستجوی عمیق در فایل لیست برای یافتن جملات کامل
file_list_path = '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_ocr_final_backup/dataset_file_list.txt'

if os.path.exists(file_list_path):
    print('--- تحلیل فایل dataset_file_list.txt برای یافتن متون فارسی ---')
    with open(file_list_path, 'r', encoding='utf-8', errors='ignore') as f:
        # خواندن بخشی از فایل برای جستجوی الگو
        content = f.readlines()

    persian_lines = [line.strip() for line in content if any('\u0600' <= char <= '\u06FF' for char in line)]

    print(f'تعداد خطوط حاوی نویسه فارسی: {len(persian_lines)}')
    if persian_lines:
        print('نمونه ۱۰ خط اول فارسی:')
        for l in persian_lines[:10]:
            print(f' - {l}')
else:
    print('فایل لیست یافت نشد.')

--- تحلیل فایل dataset_file_list.txt برای یافتن متون فارسی ---
تعداد خطوط حاوی نویسه فارسی: 1833
نمونه ۱۰ خط اول فارسی:
 - /content/drive/MyDrive/colab_ocr_project/test/ا/ا.png
 - /content/drive/MyDrive/colab_ocr_project/test/ا/ا_9.png
 - /content/drive/MyDrive/colab_ocr_project/test/آ/آ_4.png
 - /content/drive/MyDrive/colab_ocr_project/test/ب/ب.png
 - /content/drive/MyDrive/colab_ocr_project/test/ب/ب_9.png
 - /content/drive/MyDrive/colab_ocr_project/test/ت/ت.png
 - /content/drive/MyDrive/colab_ocr_project/test/ت/ت_9.png
 - /content/drive/MyDrive/colab_ocr_project/test/ث/ث.png
 - /content/drive/MyDrive/colab_ocr_project/test/ث/ث_9.png
 - /content/drive/MyDrive/colab_ocr_project/test/ج/ج.png


In [ ]:
import torch
import os

# Strategic paths for large metadata/tensor files
pt_files = [
    '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt',
    '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS-Net_Research2026_Results/swin_test_tokens_correct.pt'
]

print('--- بازرسی محتوای فایل‌های PyTorch برای یافتن Ground Truth ---')
for path in pt_files:
    if os.path.exists(path):
        print(f'\nفایل: {path}')
        try:
            # Loading metadata to inspect structure
            data = torch.load(path, map_location='cpu')
            print(f'نوع داده: {type(data)}')

            if isinstance(data, dict):
                print(f'کلیدهای یافت شده: {list(data.keys())}')
                # Looking for text or label related keys
                for k in ['text', 'labels', 'gt', 'ground_truth', 'texts']:
                    if k in data:
                        print(f'نمونه محتوای {k}: {str(data[k])[:300]}')
            elif torch.is_tensor(data):
                print(f'ابعاد تنسور: {data.shape}')
                # If it looks like integer IDs, it might be tokenized text
                if data.dtype in [torch.int, torch.long]:
                    print(f'نمونه مقادیر (Integer IDs): {data[0][:20] if data.ndim > 1 else data[:20]}')
                else:
                    print(f'نمونه مقادیر: {data[0][:5] if data.ndim > 1 else data[:5]}')

            del data
        except Exception as e:
            print(f'خطا در بارگذاری: {e}')
    else:
        print(f'فایل یافت نشد: {path}')

--- بازرسی محتوای فایل‌های PyTorch برای یافتن Ground Truth ---

فایل: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt
نوع داده: <class 'torch.Tensor'>
ابعاد تنسور: torch.Size([3686, 3136, 80])
نمونه مقادیر: tensor([[ 4.4642e-03, -8.6523e-03, -3.6577e-02, -1.7661e-02, -7.6711e-03,
          5.0307e-02,  6.6040e-02,  1.4403e-02, -2.5037e-02, -2.5349e-02,
          4.0634e-02, -1.5829e-02,  1.1114e-02,  4.6905e-02, -1.7173e-02,
         -2.9171e-02, -3.1306e-03,  9.6329e-03,  1.5397e-01,  6.7733e-03,
          5.4841e-02, -2.6379e-02, -8.9685e-03, -4.1821e-02,  1.8874e-02,
         -2.4142e-02, -1.8453e-02,  1.6508e-01, -1.5845e-02, -1.5837e-02,
          1.5502e-02,  4.2223e-02, -4.0959e-02, -2.3728e-02, -3.7240e-02,
          4.5482e-02,  2.3425e-02, -2.5742e-02,  3.0893e-02, -2.1265e-02,
          7.8726e-02, -2.1730e-02,  3.3011e-02,  1.0831e-03, -7.7713e-03,
          4.1060e-03, -1.1932e-02, -3.5951e-02, -2.1438e-02, -3.5

In [ ]:
import torch
import os

# Strategic paths for large metadata/tensor files
pt_files = [
    '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt',
    '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS-Net_Research2026_Results/swin_test_tokens_correct.pt'
]

print('--- بازرسی محتوای فایل‌های PyTorch برای یافتن متادیتا ---')
for path in pt_files:
    if os.path.exists(path):
        print(f'\nفایل: {path}')
        try:
            # Loading metadata to inspect structure
            data = torch.load(path, map_location='cpu')
            print(f'نوع داده: {type(data)}')

            if isinstance(data, dict):
                print(f'کلیدهای یافت شده: {list(data.keys())}')
                # Looking for text or label related keys
                for k in ['text', 'labels', 'gt', 'ground_truth', 'texts']:
                    if k in data:
                        print(f'نمونه محتوای {k}: {str(data[k])[:300]}')
            elif torch.is_tensor(data):
                print(f'ابعاد تنسور: {data.shape}')
                # If it looks like integer IDs, it might be tokenized text
                if data.dtype in [torch.int, torch.long]:
                    print(f'نمونه مقادیر (Integer IDs): {data[0][:20] if data.ndim > 1 else data[:20]}')

            del data
        except Exception as e:
            print(f'خطا در بارگذاری: {e}')
    else:
        print(f'فایل یافت نشد: {path}')

--- بازرسی محتوای فایل‌های PyTorch برای یافتن متادیتا ---

فایل: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt
نوع داده: <class 'torch.Tensor'>
ابعاد تنسور: torch.Size([3686, 3136, 80])

فایل: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS-Net_Research2026_Results/swin_test_tokens_correct.pt
نوع داده: <class 'torch.Tensor'>
ابعاد تنسور: torch.Size([8, 44, 1024])


In [ ]:
import torch
import os

# Strategic paths for large metadata/tensor files
pt_files = [
    '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt',
    '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS-Net_Research2026_Results/swin_test_tokens_correct.pt'
]

print('--- بازرسی محتوای فایل‌های PyTorch برای یافتن متادیتا ---')
for path in pt_files:
    if os.path.exists(path):
        print(f'\nفایل: {path}')
        try:
            # Loading only metadata/structure if possible to save RAM
            data = torch.load(path, map_location='cpu')
            print(f'نوع داده: {type(data)}')

            if isinstance(data, dict):
                print(f'کلیدهای یافت شده: {list(data.keys())}')
                # Looking for text-related keys
                for k in ['text', 'labels', 'gt', 'ground_truth', 'texts']:
                    if k in data:
                        print(f'نمونه محتوای {k}: {str(data[k])[:300]}')
            elif torch.is_tensor(data):
                print(f'ابعاد تنسور: {data.shape}')
                print(f'نمونه مقادیر: {data[0] if data.ndim > 0 else data}')

            del data
        except Exception as e:
            print(f'خطا در بارگذاری: {e}')
    else:
        print(f'فایل یافت نشد: {path}')

--- بازرسی محتوای فایل‌های PyTorch برای یافتن متادیتا ---

فایل: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt
نوع داده: <class 'torch.Tensor'>
ابعاد تنسور: torch.Size([3686, 3136, 80])
نمونه مقادیر: tensor([[ 0.0045, -0.0087, -0.0366,  ..., -0.0013,  0.0125,  0.0363],
        [-0.0017, -0.0179, -0.0364,  ...,  0.0149,  0.0020,  0.0363],
        [-0.0012, -0.0252, -0.0416,  ...,  0.0101,  0.0089,  0.0359],
        ...,
        [ 0.0041, -0.0442,  0.0650,  ..., -0.0226, -0.0109,  0.0018],
        [ 0.0051, -0.0493,  0.0583,  ..., -0.0181, -0.0109, -0.0033],
        [-0.0094, -0.0432,  0.0484,  ..., -0.0123, -0.0300,  0.0019]])

فایل: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS-Net_Research2026_Results/swin_test_tokens_correct.pt
نوع داده: <class 'torch.Tensor'>
ابعاد تنسور: torch.Size([8, 44, 1024])
نمونه مقادیر: tensor([[ 0.5024, -0.0137, -0.0566,  ...,  0.0025, -0.0524, -0.1823],
        [ 0

In [ ]:
import torch
import os

pt_files = [
    '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt',
    '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS-Net_Research2026_Results/swin_test_tokens_correct.pt'
]

print('--- بازرسی فایل‌های PyTorch برای یافتن متادیتا ---')
for path in pt_files:
    if os.path.exists(path):
        print(f'\nفایل: {path}')
        try:
            # بارگذاری ساختار فایل
            data = torch.load(path, map_location='cpu')
            print(f'نوع داده: {type(data)}')
            if isinstance(data, dict):
                print(f'کلیدها: {list(data.keys())}')
                for k in ['text', 'labels', 'gt', 'ground_truth', 'texts']:
                    if k in data:
                        print(f'نمونه {k}: {str(data[k])[:200]}')
            elif torch.is_tensor(data):
                print(f'ابعاد تنسور: {data.shape}')
            del data
        except Exception as e:
            print(f'خطا در خواندن: {e}')
    else:
        print(f'فایل یافت نشد: {path}')

--- بازرسی فایل‌های PyTorch برای یافتن متادیتا ---

فایل: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt
نوع داده: <class 'torch.Tensor'>
ابعاد تنسور: torch.Size([3686, 3136, 80])

فایل: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS-Net_Research2026_Results/swin_test_tokens_correct.pt
نوع داده: <class 'torch.Tensor'>
ابعاد تنسور: torch.Size([8, 44, 1024])


In [ ]:
import torch
import os

# بررسی فایل‌های pt که احتمال دارد حاوی لیبل‌ها یا نگاشت‌ها باشند
pt_files = [
    '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt',
    '/content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS-Net_Research2026_Results/swin_test_tokens_correct.pt'
]

print('--- بازرسی محتوای فایل‌های PyTorch برای یافتن متادیتا ---')
for path in pt_files:
    if os.path.exists(path):
        print(f'\nبررسی فایل: {path}')
        try:
            # بارگذاری فقط متادیتا یا ساختار کلی (بدون اشغال تمام رم در صورت امکان)
            data = torch.load(path, map_location='cpu')
            print(f'نوع داده: {type(data)}')
            if isinstance(data, dict):
                print(f'کلیدهای موجود: {list(data.keys())}')
            elif torch.is_tensor(data):
                print(f'ابعاد تنسور: {data.shape}')

            # جستجوی فیلد متنی در صورتی که دیکشنری باشد
            if isinstance(data, dict) and any(k in data for k in ['text', 'labels', 'gt', 'ground_truth']):
                for k in ['text', 'labels', 'gt', 'ground_truth']:
                    if k in data:
                        print(f'نمونه محتوای {k}: {str(data[k])[:200]}')

            del data # آزاد سازی حافظه
        except Exception as e:
            print(f'خطا در خواندن فایل: {e}')
    else:
        print(f'فایل یافت نشد: {path}')

--- بازرسی محتوای فایل‌های PyTorch برای یافتن متادیتا ---

بررسی فایل: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/mamba_checkpoints/mamba_tokens_final.pt
نوع داده: <class 'torch.Tensor'>
ابعاد تنسور: torch.Size([3686, 3136, 80])

بررسی فایل: /content/drive/Othercomputers/My Laptop/Desktop/PFMS-EXP2026_features/PFMS-Net_Research2026_Results/swin_test_tokens_correct.pt
نوع داده: <class 'torch.Tensor'>
ابعاد تنسور: torch.Size([8, 44, 1024])


In [ ]:
# ================================================================
# PFMS-SERIES-GT38-2026
# STAGE 7 — REAL IDPL-PFOD VOCABULARY RECONSTRUCTION
#
# هدف:
# ساخت Vocabulary مستقیماً از Ground Truth واقعی IDPL-PFOD
#
# ورودی:
# IDPL_PFOD_REAL_GT_MAPPING.csv
#
# خروجی:
# idpl_vocab.pt
# ================================================================

import os
import pandas as pd
import torch

print("=" * 90)
print("PFMS-SERIES-GT38-2026")
print("REAL IDPL-PFOD VOCABULARY RECONSTRUCTION")
print("=" * 90)

# ------------------------------------------------
# Paths
# ------------------------------------------------

GT_CSV = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"

VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"

# ------------------------------------------------
# Check Ground Truth
# ------------------------------------------------

if not os.path.exists(GT_CSV):

    raise FileNotFoundError(
        f"""
Ground Truth file not found:

{GT_CSV}
"""
    )

# ------------------------------------------------
# Load Ground Truth
# ------------------------------------------------

df_gt = pd.read_csv(
    GT_CSV
)

print("\nGround Truth loaded successfully.")

print("Number of records:", len(df_gt))

print("Columns:")
print(df_gt.columns.tolist())

# ------------------------------------------------
# Verify text column
# ------------------------------------------------

if "text" not in df_gt.columns:

    raise ValueError(
        "Column 'text' does not exist in Ground Truth file."
    )

# Remove invalid text
df_gt["text"] = (
    df_gt["text"]
    .fillna("")
    .astype(str)
)

df_gt = df_gt[
    df_gt["text"].str.strip() != ""
].reset_index(drop=True)

print("\nValid text records:", len(df_gt))

# ------------------------------------------------
# Build vocabulary from REAL Persian Ground Truth
# ------------------------------------------------

all_characters = set()

for text in df_gt["text"]:

    for ch in text:

        all_characters.add(ch)

# Sort characters deterministically
characters = sorted(
    all_characters,
    key=lambda x: ord(x)
)

# ------------------------------------------------
# Special tokens
# ------------------------------------------------

special_tokens = [
    "<PAD>",
    "<BOS>",
    "<EOS>",
    "<UNK>"
]

# ------------------------------------------------
# Character → Index
# ------------------------------------------------

char2idx = {}

idx2char = {}

# Special tokens first
for token in special_tokens:

    idx = len(char2idx)

    char2idx[token] = idx
    idx2char[idx] = token

# Real characters
for ch in characters:

    if ch not in char2idx:

        idx = len(char2idx)

        char2idx[ch] = idx
        idx2char[idx] = ch

# ------------------------------------------------
# Special indices
# ------------------------------------------------

PAD_IDX = char2idx["<PAD>"]
BOS_IDX = char2idx["<BOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

VOCAB_SIZE = len(char2idx)

# ------------------------------------------------
# Save vocabulary
# ------------------------------------------------

vocab_data = {
    "char2idx": char2idx,
    "idx2char": idx2char,
    "PAD_IDX": PAD_IDX,
    "BOS_IDX": BOS_IDX,
    "EOS_IDX": EOS_IDX,
    "UNK_IDX": UNK_IDX,
    "vocab_size": VOCAB_SIZE,
    "characters": characters
}

torch.save(
    vocab_data,
    VOCAB_PATH
)

# ------------------------------------------------
# Verification
# ------------------------------------------------

print("\n" + "=" * 90)
print("VOCABULARY CREATED SUCCESSFULLY")
print("=" * 90)

print("Real characters :", len(characters))
print("Total vocabulary :", VOCAB_SIZE)

print("\nSpecial tokens:")
print("PAD =", PAD_IDX)
print("BOS =", BOS_IDX)
print("EOS =", EOS_IDX)
print("UNK =", UNK_IDX)

print("\nFirst 30 real characters:")

for i, ch in enumerate(characters[:30]):

    print(
        i,
        repr(ch),
        "=>",
        char2idx[ch]
    )

print("\nVocabulary saved to:")

print(VOCAB_PATH)

print("\nFile exists:", os.path.exists(VOCAB_PATH))

# ------------------------------------------------
# Test encoding
# ------------------------------------------------

sample_text = df_gt.iloc[0]["text"]

encoded = []

for ch in sample_text:

    if ch in char2idx:
        encoded.append(
            char2idx[ch]
        )
    else:
        encoded.append(
            UNK_IDX
        )

print("\n" + "=" * 90)
print("VOCABULARY TEST")
print("=" * 90)

print("Sample text:")
print(sample_text)

print("\nEncoded length:", len(encoded))

print("First 30 token IDs:")
print(encoded[:30])

print("\n✓ Ground Truth vocabulary reconstructed")
print("✓ Persian characters preserved")
print("✓ Special tokens added")
print("✓ idpl_vocab.pt saved")
print("✓ Stage 7 can continue")

PFMS-SERIES-GT38-2026
REAL IDPL-PFOD VOCABULARY RECONSTRUCTION

Ground Truth loaded successfully.
Number of records: 27120
Columns:
['split', 'hf_index', 'image_path', 'image_name', 'text']

Valid text records: 27120

VOCABULARY CREATED SUCCESSFULLY
Real characters : 165
Total vocabulary : 169

Special tokens:
PAD = 0
BOS = 1
EOS = 2
UNK = 3

First 30 real characters:
0 '\n' => 4
1 ' ' => 5
2 '!' => 6
3 '%' => 7
4 '(' => 8
5 ')' => 9
6 '*' => 10
7 ',' => 11
8 '-' => 12
9 '.' => 13
10 '/' => 14
11 ':' => 15
12 '?' => 16
13 '[' => 17
14 ']' => 18
15 '`' => 19
16 '{' => 20
17 '}' => 21
18 '«' => 22
19 '\xad' => 23
20 '²' => 24
21 '»' => 25
22 'ï' => 26
23 '،' => 27
24 '؛' => 28
25 '؟' => 29
26 'آ' => 30
27 'أ' => 31
28 'ؤ' => 32
29 'إ' => 33

Vocabulary saved to:
/content/drive/MyDrive/idpl_vocab.pt

File exists: True

VOCABULARY TEST
Sample text:
یابد و كار جمهوری اسلامی را یكسره كند. كافی است به تبلیغات «اكس»گونه یك ماه


Encoded length: 76
First 30 token IDs:
[90, 35, 36, 43, 5, 63, 5,

In [ ]:
# ================================================================
# PFMS-SERIES-GT38-2026
# STAGE 7 — REAL IDPL-PFOD OCR TRAINING
#
# FINAL CONTINUATION VERSION
#
# Architecture:
#       IDPL-PFOD Image
#              ↓
#           Swin-B
#              ↓
#      Mamba Sequence Mixer
#              ↓
#    Transformer OCR Decoder
#              ↓
#       Persian Text
#
# Dataset:
#       27,120 real IDPL-PFOD samples
#
# Ground Truth:
#       /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv
#
# Vocabulary:
#       Reconstructed directly from REAL Ground Truth
#
# Hardware:
#       Tesla T4
# ================================================================


# ================================================================
# CELL 1 — IMPORTS AND ENVIRONMENT
# ================================================================

import os
import math
import random
import numpy as np
import pandas as pd

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import swin_b, Swin_B_Weights

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("STAGE 7 — REAL IDPL-PFOD OCR TRAINING")
print("=" * 100)

print("PyTorch     :", torch.__version__)
print("CUDA        :", torch.version.cuda)
print("Device      :", "CUDA" if torch.cuda.is_available() else "CPU")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if torch.cuda.is_available():

    print("GPU         :", torch.cuda.get_device_name(0))

    gpu_memory = torch.cuda.get_device_properties(0).total_memory

    print(
        "GPU Memory  :",
        round(gpu_memory / (1024**3), 2),
        "GB"
    )


# ================================================================
# CELL 2 — REPRODUCIBILITY
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("\nRandom seed:", SEED)


# ================================================================
# CELL 3 — PATHS
# ================================================================

GT_CSV = (
    "/content/drive/MyDrive/"
    "IDPL_PFOD_REAL_GT_MAPPING.csv"
)

VOCAB_PATH = (
    "/content/drive/MyDrive/"
    "idpl_vocab.pt"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "PFMS-SERIES-GT38-2026"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

BEST_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "best_pfms_series_swin_mamba_trocr.pth"
)

LAST_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "last_pfms_series_swin_mamba_trocr.pth"
)

HISTORY_PATH = os.path.join(
    OUTPUT_DIR,
    "training_history.csv"
)

VOCAB_FINAL_PATH = os.path.join(
    OUTPUT_DIR,
    "idpl_vocab_final.pt"
)

print("\nGT CSV:")
print(GT_CSV)

print("\nVocabulary:")
print(VOCAB_PATH)

print("\nOutput:")
print(OUTPUT_DIR)


# ================================================================
# CELL 4 — LOAD REAL GROUND TRUTH
# ================================================================

if not os.path.exists(GT_CSV):

    raise FileNotFoundError(
        "\nGround Truth CSV not found:\n"
        + GT_CSV
    )

df = pd.read_csv(
    GT_CSV
)

print("\n" + "=" * 100)
print("REAL GROUND TRUTH")
print("=" * 100)

print("Rows    :", len(df))
print("Columns :", df.columns.tolist())

required_columns = [
    "image_path",
    "image_name",
    "text"
]

for col in required_columns:

    if col not in df.columns:

        raise ValueError(
            f"Required column missing: {col}"
        )


# ================================================================
# CELL 5 — CLEAN GROUND TRUTH
# ================================================================

df = df[
    required_columns
].copy()

df["image_path"] = (
    df["image_path"]
    .fillna("")
    .astype(str)
)

df["image_name"] = (
    df["image_name"]
    .fillna("")
    .astype(str)
)

df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
)

# Remove empty text
df = df[
    df["text"].str.strip().str.len() > 0
].reset_index(drop=True)

print("\nValid Ground Truth records:", len(df))


# ================================================================
# CELL 6 — VERIFY IMAGE FILES
# ================================================================

print("\nChecking image files...")

exists_mask = df["image_path"].apply(
    os.path.exists
)

missing_count = int(
    (~exists_mask).sum()
)

existing_count = int(
    exists_mask.sum()
)

print("Existing:", existing_count)
print("Missing :", missing_count)

if missing_count > 0:

    print("\nFirst missing images:")

    for p in df.loc[
        ~exists_mask,
        "image_path"
    ].head(20):

        print(p)

# Keep only valid images
df = df[
    exists_mask
].reset_index(drop=True)

print(
    "\nFinal usable samples:",
    len(df)
)


# ================================================================
# CELL 7 — BUILD VOCABULARY FROM REAL GROUND TRUTH
# ================================================================

print("\n" + "=" * 100)
print("BUILDING VOCABULARY FROM REAL IDPL-PFOD GROUND TRUTH")
print("=" * 100)

all_characters = set()

for text in tqdm(
    df["text"],
    desc="Collecting characters"
):

    for ch in text:

        all_characters.add(ch)

characters = sorted(
    all_characters,
    key=lambda x: ord(x)
)

print(
    "\nReal unique characters:",
    len(characters)
)


# ================================================================
# CELL 8 — SPECIAL TOKENS
# ================================================================

special_tokens = [
    "<PAD>",
    "<BOS>",
    "<EOS>",
    "<UNK>"
]

char2idx = {}

idx2char = {}

# Special tokens
for token in special_tokens:

    idx = len(char2idx)

    char2idx[token] = idx
    idx2char[idx] = token


# Real characters
for ch in characters:

    if ch not in char2idx:

        idx = len(char2idx)

        char2idx[ch] = idx
        idx2char[idx] = ch


PAD_IDX = char2idx["<PAD>"]
BOS_IDX = char2idx["<BOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

VOCAB_SIZE = len(char2idx)

print("\nVocabulary:")
print("Real characters :", len(characters))
print("PAD             :", PAD_IDX)
print("BOS             :", BOS_IDX)
print("EOS             :", EOS_IDX)
print("UNK             :", UNK_IDX)
print("TOTAL VOCAB     :", VOCAB_SIZE)


# ================================================================
# CELL 9 — SAVE FINAL VOCABULARY
# ================================================================

vocab_data = {

    "char2idx": char2idx,

    "idx2char": idx2char,

    "characters": characters,

    "PAD_IDX": PAD_IDX,

    "BOS_IDX": BOS_IDX,

    "EOS_IDX": EOS_IDX,

    "UNK_IDX": UNK_IDX,

    "vocab_size": VOCAB_SIZE
}

torch.save(
    vocab_data,
    VOCAB_PATH
)

torch.save(
    vocab_data,
    VOCAB_FINAL_PATH
)

print("\n✓ Vocabulary saved:")
print(VOCAB_PATH)

print("\n✓ Backup vocabulary saved:")
print(VOCAB_FINAL_PATH)


# ================================================================
# CELL 10 — TEXT ENCODER
# ================================================================

MAX_TEXT_LENGTH = 128


def encode_text(
    text,
    max_length=MAX_TEXT_LENGTH
):

    tokens = [
        BOS_IDX
    ]

    for ch in str(text):

        if ch in char2idx:

            tokens.append(
                char2idx[ch]
            )

        else:

            tokens.append(
                UNK_IDX
            )

    tokens.append(
        EOS_IDX
    )

    # ------------------------------------------------------------
    # Truncate
    # ------------------------------------------------------------

    if len(tokens) > max_length:

        tokens = tokens[:max_length]

        # Ensure EOS
        tokens[-1] = EOS_IDX

    # ------------------------------------------------------------
    # Padding
    # ------------------------------------------------------------

    if len(tokens) < max_length:

        tokens.extend(
            [PAD_IDX] *
            (
                max_length -
                len(tokens)
            )
        )

    return tokens


# Test
sample_text = df.iloc[0]["text"]

sample_tokens = encode_text(
    sample_text
)

print("\n" + "=" * 100)
print("TOKENIZATION TEST")
print("=" * 100)

print("Sample:")
print(sample_text)

print("\nToken count:")
print(len(sample_tokens))

print("\nFirst tokens:")
print(sample_tokens[:30])


# ================================================================
# CELL 11 — DATASET SPLIT
# ================================================================

indices = np.arange(
    len(df)
)

rng = np.random.default_rng(
    SEED
)

rng.shuffle(indices)

N = len(indices)

N_TRAIN = int(
    0.85 * N
)

N_VAL = int(
    0.10 * N
)

train_idx = indices[
    :N_TRAIN
]

val_idx = indices[
    N_TRAIN:
    N_TRAIN + N_VAL
]

test_idx = indices[
    N_TRAIN + N_VAL:
]

train_df = df.iloc[
    train_idx
].reset_index(drop=True)

val_df = df.iloc[
    val_idx
].reset_index(drop=True)

test_df = df.iloc[
    test_idx
].reset_index(drop=True)

print("\n" + "=" * 100)
print("DATASET SPLIT")
print("=" * 100)

print("TOTAL      :", len(df))
print("TRAIN      :", len(train_df))
print("VALIDATION :", len(val_df))
print("TEST       :", len(test_df))


# ================================================================
# CELL 12 — IMAGE TRANSFORMS
# ================================================================

# IDPL-PFOD original image:
# approximately 700 × 50
#
# Swin-B pretrained model expects a square-ish image.
#
# We use 224 × 224 here to maintain compatibility with
# torchvision Swin-B weights.

IMG_SIZE = 224

train_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.RandomApply(
        [
            transforms.ColorJitter(
                brightness=0.10,
                contrast=0.10
            )
        ],
        p=0.30
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


eval_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ================================================================
# CELL 13 — OCR DATASET
# ================================================================

class IDPLOCRDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        image_path = row["image_path"]

        text = row["text"]

        image_name = row["image_name"]

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            raise RuntimeError(
                f"\nCannot read image:\n"
                f"{image_path}\n"
                f"Error: {e}"
            )

        if self.transform is not None:

            image = self.transform(
                image
            )

        target = torch.tensor(
            encode_text(
                text
            ),
            dtype=torch.long
        )

        return {
            "image": image,
            "target": target,
            "text": text,
            "image_name": image_name
        }


# ================================================================
# CELL 14 — DATA LOADERS
# ================================================================

# Tesla T4 memory friendly
BATCH_SIZE = 4

train_dataset = IDPLOCRDataset(
    train_df,
    train_transform
)

val_dataset = IDPLOCRDataset(
    val_df,
    eval_transform
)

test_dataset = IDPLOCRDataset(
    test_df,
    eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("\n" + "=" * 100)
print("DATALOADERS")
print("=" * 100)

print("Batch size     :", BATCH_SIZE)
print("Train batches  :", len(train_loader))
print("Val batches    :", len(val_loader))
print("Test batches   :", len(test_loader))


# ================================================================
# CELL 15 — MAMBA SEQUENCE MIXER
# ================================================================

class MambaSequenceMixer(
    nn.Module
):

    """
    Lightweight Mamba-style sequence mixer.

    Input:
        [B, N, C]

    Output:
        [B, N, C]

    Pure PyTorch implementation.
    No mamba_ssm dependency required.
    """

    def __init__(
        self,
        dim,
        expansion=2,
        dropout=0.1
    ):

        super().__init__()

        hidden = dim * expansion

        self.norm = nn.LayerNorm(
            dim
        )

        self.in_proj = nn.Linear(
            dim,
            hidden * 2
        )

        self.depthwise = nn.Conv1d(
            hidden,
            hidden,
            kernel_size=5,
            padding=2,
            groups=hidden
        )

        self.activation = nn.SiLU()

        self.out_proj = nn.Linear(
            hidden,
            dim
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x
    ):

        residual = x

        x = self.norm(x)

        x = self.in_proj(x)

        a, b = x.chunk(
            2,
            dim=-1
        )

        a = a.transpose(
            1,
            2
        )

        a = self.depthwise(a)

        a = a.transpose(
            1,
            2
        )

        a = self.activation(a)

        x = a * torch.sigmoid(b)

        x = self.out_proj(x)

        x = self.dropout(x)

        return residual + x


# ================================================================
# CELL 16 — TRANSFORMER OCR DECODER
# ================================================================

class TransformerOCRDecoder(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        d_model=512,
        nhead=8,
        num_layers=4,
        dim_feedforward=2048,
        dropout=0.1,
        max_length=128
    ):

        super().__init__()

        self.vocab_size = vocab_size

        self.d_model = d_model

        self.max_length = max_length

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_IDX
        )

        self.pos_embedding = nn.Parameter(
            torch.randn(
                1,
                max_length,
                d_model
            ) * 0.02
        )

        decoder_layer = (
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                batch_first=True,
                norm_first=True
            )
        )

        self.decoder = (
            nn.TransformerDecoder(
                decoder_layer,
                num_layers=num_layers
            )
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        self.output = nn.Linear(
            d_model,
            vocab_size
        )

    def causal_mask(
        self,
        length,
        device
    ):

        mask = torch.triu(
            torch.ones(
                length,
                length,
                device=device
            ),
            diagonal=1
        )

        mask = mask.masked_fill(
            mask == 1,
            float("-inf")
        )

        return mask

    def forward(
        self,
        memory,
        target
    ):

        B, T = target.shape

        target_emb = self.embedding(
            target
        )

        target_emb = (
            target_emb +
            self.pos_embedding[
                :, :T, :
            ]
        )

        causal_mask = self.causal_mask(
            T,
            target.device
        )

        padding_mask = (
            target == PAD_IDX
        )

        decoded = self.decoder(
            tgt=target_emb,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=padding_mask
        )

        decoded = self.norm(
            decoded
        )

        logits = self.output(
            decoded
        )

        return logits


# ================================================================
# CELL 17 — COMPLETE SERIAL PFMS MODEL
# ================================================================

class PFMS_Series_OCR(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        d_model=512
    ):

        super().__init__()

        print("\nLoading pretrained Swin-B...")

        weights = (
            Swin_B_Weights.IMAGENET1K_V1
        )

        self.swin = swin_b(
            weights=weights
        )

        # Remove classification head
        self.swin.head = nn.Identity()

        # Swin-B final channel dimension
        swin_dim = 1024

        # Project to decoder dimension
        self.projection = nn.Linear(
            swin_dim,
            d_model
        )

        # Serial Mamba blocks
        self.mamba = nn.ModuleList([

            MambaSequenceMixer(
                d_model,
                expansion=2,
                dropout=0.1
            )

            for _ in range(4)
        ])

        # Transformer OCR Decoder
        self.decoder = (
            TransformerOCRDecoder(
                vocab_size=vocab_size,
                d_model=d_model,
                nhead=8,
                num_layers=4,
                dim_feedforward=2048,
                dropout=0.1,
                max_length=MAX_TEXT_LENGTH
            )
        )

    def extract_swin_features(
        self,
        x
    ):

        # --------------------------------------------------------
        # Swin feature extractor
        # --------------------------------------------------------

        x = self.swin.features(x)

        # torchvision Swin feature output:
        #
        # [B, H, W, C]

        if x.ndim == 4:

            B, H, W, C = x.shape

            x = x.reshape(
                B,
                H * W,
                C
            )

        return x

    def encode(
        self,
        images
    ):

        x = self.extract_swin_features(
            images
        )

        # [B, N, 1024]
        x = self.projection(
            x
        )

        # [B, N, 512]
        for block in self.mamba:

            x = block(x)

        return x

    def forward(
        self,
        images,
        target_input
    ):

        memory = self.encode(
            images
        )

        logits = self.decoder(
            memory,
            target_input
        )

        return logits


# ================================================================
# CELL 18 — CREATE MODEL
# ================================================================

model = PFMS_Series_OCR(
    vocab_size=VOCAB_SIZE,
    d_model=512
)

model = model.to(
    device
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n" + "=" * 100)
print("MODEL INFORMATION")
print("=" * 100)

print(
    "Total parameters     :",
    f"{total_params / 1e6:.2f} M"
)

print(
    "Trainable parameters :",
    f"{trainable_params / 1e6:.2f} M"
)


# ================================================================
# CELL 19 — FREEZE SWIN FOR BASELINE EXPERIMENT
# ================================================================

FREEZE_SWIN = True

if FREEZE_SWIN:

    for param in model.swin.parameters():

        param.requires_grad = False

    print("\nSwin-B status: FROZEN")

else:

    print("\nSwin-B status: TRAINABLE")


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Trainable parameters after freeze:",
    f"{trainable_params / 1e6:.2f} M"
)


# ================================================================
# CELL 20 — LOSS FUNCTION
# ================================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

print("\nLoss:")
print(
    "CrossEntropyLoss(ignore_index=PAD)"
)


# ================================================================
# CELL 21 — REAL BATCH FORWARD TEST
# ================================================================

print("\n" + "=" * 100)
print("STAGE 7-A — REAL BATCH FORWARD TEST")
print("=" * 100)

batch = next(
    iter(train_loader)
)

images = batch[
    "image"
].to(
    device,
    non_blocking=True
)

targets = batch[
    "target"
].to(
    device,
    non_blocking=True
)

print(
    "Images shape :",
    tuple(images.shape)
)

print(
    "Targets shape:",
    tuple(targets.shape)
)

decoder_input = targets[
    :, :-1
]

expected_output = targets[
    :, 1:
]

print(
    "Decoder input shape:",
    tuple(decoder_input.shape)
)

print(
    "Expected output shape:",
    tuple(expected_output.shape)
)


# ================================================================
# CELL 22 — FORWARD PASS
# ================================================================

model.eval()

with torch.no_grad():

    logits = model(
        images,
        decoder_input
    )

print(
    "\nLogits shape:",
    tuple(logits.shape)
)

assert (
    logits.shape[0]
    ==
    images.shape[0]
)

assert (
    logits.shape[1]
    ==
    decoder_input.shape[1]
)

assert (
    logits.shape[2]
    ==
    VOCAB_SIZE
)

print("\n✓ Forward Pass successful")
print("✓ Batch dimension correct")
print("✓ Sequence dimension correct")
print("✓ Vocabulary dimension correct")


# ================================================================
# CELL 23 — LOSS TEST
# ================================================================

loss = criterion(
    logits.reshape(
        -1,
        VOCAB_SIZE
    ),
    expected_output.reshape(
        -1
    )
)

print(
    "\nInitial loss:",
    float(loss)
)

print("✓ Loss calculation successful")


# ================================================================
# CELL 24 — BACKWARD TEST
# ================================================================

print("\n" + "=" * 100)
print("STAGE 7-B — BACKWARD TEST")
print("=" * 100)

model.train()

optimizer_test = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=2e-4,
    weight_decay=1e-2
)

optimizer_test.zero_grad(
    set_to_none=True
)

logits = model(
    images,
    decoder_input
)

loss = criterion(
    logits.reshape(
        -1,
        VOCAB_SIZE
    ),
    expected_output.reshape(
        -1
    )
)

loss.backward()

gradient_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

optimizer_test.step()

print(
    "Loss          :",
    float(loss)
)

print(
    "Gradient norm :",
    float(gradient_norm)
)

print("\n✓ Forward")
print("✓ Loss")
print("✓ Backward")
print("✓ Gradient clipping")
print("✓ Optimizer step")

print("\n" + "=" * 100)
print("STAGE 7 SANITY CHECK PASSED ✓")
print("=" * 100)


# ================================================================
# CELL 25 — CLEAR TEST OPTIMIZER
# ================================================================

del optimizer_test

if torch.cuda.is_available():

    torch.cuda.empty_cache()

print(
    "\nGPU cache cleared."
)


# ================================================================
# CELL 26 — FINAL TRAINING OPTIMIZER
# ================================================================

LEARNING_RATE = 2e-4

WEIGHT_DECAY = 1e-2

optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("\n" + "=" * 100)
print("OPTIMIZER")
print("=" * 100)

print("Optimizer     : AdamW")
print("Learning rate :", LEARNING_RATE)
print("Weight decay  :", WEIGHT_DECAY)
print("Scheduler     : ReduceLROnPlateau")


# ================================================================
# CELL 27 — TRAINING FUNCTION
# ================================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0.0

    progress = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for batch in progress:

        images = batch[
            "image"
        ].to(
            device,
            non_blocking=True
        )

        targets = batch[
            "target"
        ].to(
            device,
            non_blocking=True
        )

        decoder_input = targets[
            :, :-1
        ]

        expected_output = targets[
            :, 1:
        ]

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            images,
            decoder_input
        )

        loss = criterion(
            logits.reshape(
                -1,
                VOCAB_SIZE
            ),
            expected_output.reshape(
                -1
            )
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return (
        total_loss /
        len(loader)
    )


# ================================================================
# CELL 28 — VALIDATION FUNCTION
# ================================================================

@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0

    progress = tqdm(
        loader,
        desc="Validation",
        leave=False
    )

    for batch in progress:

        images = batch[
            "image"
        ].to(
            device,
            non_blocking=True
        )

        targets = batch[
            "target"
        ].to(
            device,
            non_blocking=True
        )

        decoder_input = targets[
            :, :-1
        ]

        expected_output = targets[
            :, 1:
        ]

        logits = model(
            images,
            decoder_input
        )

        loss = criterion(
            logits.reshape(
                -1,
                VOCAB_SIZE
            ),
            expected_output.reshape(
                -1
            )
        )

        total_loss += loss.item()

    return (
        total_loss /
        len(loader)
    )


# ================================================================
# CELL 29 — REAL TRAINING
# ================================================================

EPOCHS = 10

history = []

best_val_loss = float(
    "inf"
)

print("\n" + "=" * 100)
print("🚀 REAL OCR TRAINING STARTED")
print("=" * 100)

print("Architecture:")
print(
    "Swin-B → Mamba Sequence Mixer → "
    "Transformer OCR Decoder"
)

print("\nDataset:")
print(
    "27,120 real IDPL-PFOD samples"
)

print("\nTrain samples:", len(train_dataset))
print("Val samples  :", len(val_dataset))
print("Test samples :", len(test_dataset))

print("\nVocabulary:", VOCAB_SIZE)

print("\nEpochs:", EPOCHS)

print("=" * 100)


for epoch in range(
    1,
    EPOCHS + 1
):

    print(
        f"\n{'='*35}"
    )

    print(
        f"EPOCH {epoch}/{EPOCHS}"
    )

    print(
        f"{'='*35}"
    )

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss = validate(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(
        val_loss
    )

    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )

    print(
        f"\nEpoch {epoch}/{EPOCHS}"
    )

    print(
        f"Train Loss : {train_loss:.6f}"
    )

    print(
        f"Val Loss   : {val_loss:.6f}"
    )

    print(
        f"LR         : {current_lr:.2e}"
    )

    history.append({

        "epoch": epoch,

        "train_loss": train_loss,

        "val_loss": val_loss,

        "learning_rate": current_lr
    })

    # ------------------------------------------------------------
    # Save best checkpoint
    # ------------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    scheduler.state_dict(),

                "train_loss":
                    train_loss,

                "val_loss":
                    val_loss,

                "best_val_loss":
                    best_val_loss,

                "vocab_size":
                    VOCAB_SIZE,

                "char2idx":
                    char2idx,

                "idx2char":
                    idx2char,

                "PAD_IDX":
                    PAD_IDX,

                "BOS_IDX":
                    BOS_IDX,

                "EOS_IDX":
                    EOS_IDX,

                "UNK_IDX":
                    UNK_IDX
            },
            BEST_MODEL_PATH
        )

        print(
            "\n✓ NEW BEST MODEL SAVED"
        )

        print(
            BEST_MODEL_PATH
        )


# ================================================================
# CELL 30 — SAVE LAST CHECKPOINT
# ================================================================

torch.save(
    {
        "epoch": EPOCHS,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "best_val_loss":
            best_val_loss,

        "vocab_size":
            VOCAB_SIZE,

        "char2idx":
            char2idx,

        "idx2char":
            idx2char,

        "PAD_IDX":
            PAD_IDX,

        "BOS_IDX":
            BOS_IDX,

        "EOS_IDX":
            EOS_IDX,

        "UNK_IDX":
            UNK_IDX
    },
    LAST_MODEL_PATH
)


# ================================================================
# CELL 31 — SAVE TRAINING HISTORY
# ================================================================

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    HISTORY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 100)
print("TRAINING RESULTS SAVED")
print("=" * 100)

print(
    "Best model:"
)

print(
    BEST_MODEL_PATH
)

print(
    "\nLast model:"
)

print(
    LAST_MODEL_PATH
)

print(
    "\nTraining history:"
)

print(
    HISTORY_PATH
)

print(
    "\nBest validation loss:",
    best_val_loss
)


# ================================================================
# CELL 32 — FINAL SUMMARY
# ================================================================

print("\n")
print("=" * 100)
print("🎯 PFMS-SERIES-GT38-2026 — STAGE 7 FINISHED")
print("=" * 100)

print("\nGround Truth:")
print("✓ Real IDPL-PFOD Ground Truth")
print("✓ 27,120 usable samples")

print("\nVocabulary:")
print(
    f"✓ {len(characters)} real characters"
)

print(
    f"✓ {VOCAB_SIZE} total tokens"
)

print("\nArchitecture:")
print("✓ Swin-B")
print("✓ Mamba Sequence Mixer")
print("✓ Transformer OCR Decoder")

print("\nTraining:")
print("✓ Train / Validation / Test split")
print("✓ Forward pass")
print("✓ Backward pass")
print("✓ Real OCR training")
print("✓ Best checkpoint saved")

print("\n" + "=" * 100)
print("NEXT STAGE: OCR DECODING + CER/WER + SEQUENCE ACCURACY")
print("=" * 100)

PFMS-SERIES-GT38-2026
STAGE 7 — REAL IDPL-PFOD OCR TRAINING
PyTorch     : 2.11.0+cu128
CUDA        : 12.8
Device      : CUDA
GPU         : Tesla T4
GPU Memory  : 14.56 GB

Random seed: 42

GT CSV:
/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv

Vocabulary:
/content/drive/MyDrive/idpl_vocab.pt

Output:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026

REAL GROUND TRUTH
Rows    : 27120
Columns : ['split', 'hf_index', 'image_path', 'image_name', 'text']

Valid Ground Truth records: 27120

Checking image files...
Existing: 27120
Missing : 0

Final usable samples: 27120

BUILDING VOCABULARY FROM REAL IDPL-PFOD GROUND TRUTH



Real unique characters: 165

Vocabulary:
Real characters : 165
PAD             : 0
BOS             : 1
EOS             : 2
UNK             : 3
TOTAL VOCAB     : 169

✓ Vocabulary saved:
/content/drive/MyDrive/idpl_vocab.pt

✓ Backup vocabulary saved:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/idpl_vocab_final.pt

TOKENIZATION TEST
Sample:
یابد و كار جمهوری اسلامی را یكسره كند. كافی است به تبلیغات «اكس»گونه یك ماه


Token count:
128

First tokens:
[1, 90, 35, 36, 43, 5, 63, 5, 58, 35, 45, 5, 40, 60, 62, 63, 45, 90, 5, 35, 47, 59, 35, 60, 90, 5, 45, 35, 5, 90]

DATASET SPLIT
TOTAL      : 27120
TRAIN      : 23052
VALIDATION : 2712
TEST       : 1356

DATALOADERS
Batch size     : 4
Train batches  : 5763
Val batches    : 678
Test batches   : 339

Loading pretrained Swin-B...
Downloading: "https://download.pytorch.org/models/swin_b-68c6b09e.pth" to /root/.cache/torch/hub/checkpoints/swin_b-68c6b09e.pth


100%|██████████| 335M/335M [00:02<00:00, 125MB/s] 



MODEL INFORMATION
Total parameters     : 110.65 M
Trainable parameters : 110.65 M

Swin-B status: FROZEN
Trainable parameters after freeze: 23.91 M

Loss:
CrossEntropyLoss(ignore_index=PAD)

STAGE 7-A — REAL BATCH FORWARD TEST
Images shape : (4, 3, 224, 224)
Targets shape: (4, 128)
Decoder input shape: (4, 127)
Expected output shape: (4, 127)

Logits shape: (4, 127, 169)

✓ Forward Pass successful
✓ Batch dimension correct
✓ Sequence dimension correct
✓ Vocabulary dimension correct

Initial loss: 5.290695667266846
✓ Loss calculation successful

STAGE 7-B — BACKWARD TEST
Loss          : 5.323078155517578
Gradient norm : 11.683072090148926

✓ Forward
✓ Loss
✓ Backward
✓ Gradient clipping
✓ Optimizer step

STAGE 7 SANITY CHECK PASSED ✓

GPU cache cleared.

OPTIMIZER
Optimizer     : AdamW
Learning rate : 0.0002
Weight decay  : 0.01
Scheduler     : ReduceLROnPlateau

🚀 REAL OCR TRAINING STARTED
Architecture:
Swin-B → Mamba Sequence Mixer → Transformer OCR Decoder

Dataset:
27,120 real IDPL

Training:   0%|          | 0/5763 [00:00<?, ?it/s]

...


Epoch 9/10
Train Loss : 1.606560
Val Loss   : 1.555380
LR         : 2.00e-04

✓ NEW BEST MODEL SAVED
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/best_pfms_series_swin_mamba_trocr.pth

EPOCH 10/10


Training:   0%|          | 0/5763 [00:00<?, ?it/s]

Validation:   0%|          | 0/678 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f38c3f0a2a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f38c3f0a2a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/


Epoch 10/10
Train Loss : 1.586632
Val Loss   : 1.536682
LR         : 2.00e-04

✓ NEW BEST MODEL SAVED
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/best_pfms_series_swin_mamba_trocr.pth

TRAINING RESULTS SAVED
Best model:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/best_pfms_series_swin_mamba_trocr.pth

Last model:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/last_pfms_series_swin_mamba_trocr.pth

Training history:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/training_history.csv

Best validation loss: 1.5366823013904876


🎯 PFMS-SERIES-GT38-2026 — STAGE 7 FINISHED

Ground Truth:
✓ Real IDPL-PFOD Ground Truth
✓ 27,120 usable samples

Vocabulary:
✓ 165 real characters
✓ 169 total tokens

Architecture:
✓ Swin-B
✓ Mamba Sequence Mixer
✓ Transformer OCR Decoder

Training:
✓ Train / Validation / Test split
✓ Forward pass
✓ Backward pass
✓ Real OCR training
✓ Best checkpoint saved

NEXT STAGE: OCR DECODING + CER/WER + SEQUENCE ACCURACY


### 🧹 دستورالعمل پاکسازی برای گیت‌هاب (GitHub Cleanup)

برای اینکه فایل شما تمیز و حرفه‌ای باشد، سلول‌هایی که در لیست زیر به عنوان **«قابل حذف»** علامت‌گذاری شده‌اند را پاک کنید. این سلول‌ها فقط برای پیدا کردن مسیر فایل‌ها یا تست‌های میانی بودند.

| وضعیت | محتوا / شماره سلول (تقریبی) | دلیل |
| :--- | :--- | :--- |
| ✅ **نگه دارید** | سلول‌های ابتدایی (Imports & Drive Mount) | پیش‌نیازهای سیستم |
| ❌ **حذف کنید** | تمام سلول‌های جستجو (find, os.walk, Cell 37-Scan, Cell 41) | اینها فقط برای پیدا کردن فایل GT بودند |
| ❌ **حذف کنید** | تست‌های تک‌بچ (Batch Verification, Forward Test) | اینها برای عیب‌یابی بودند و در آموزش اصلی نقشی ندارند |
| ✅ **نگه دارید** | سلول ساخت Vocabulary (Stage 7) | برای تبدیل متن به عدد ضروری است |
| ✅ **نگه دارید** | کلاس‌های مدل (Swin, Mamba, Transformer Decoder) | هسته اصلی معماری پروژه |
| ✅ **نگه دارید** | حلقه اصلی آموزش (Real Training Loop) | بخش اجرایی اصلی |
| ❌ **حذف کنید** | سلول‌های خالی انتهای نوت‌بوک | برای تمیزی فایل |

In [ ]:
# اجرای این کد هیچ تغییری در مدل ایجاد نمی‌کند و فقط برای اطمینان از سلامت فایل‌های ذخیره شده است
import os

paths_to_verify = [
    '/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv',
    '/content/drive/MyDrive/idpl_vocab.pt',
    '/content/drive/MyDrive/PFMS-SERIES-GT38-2026/best_pfms_series_swin_mamba_trocr.pth'
]

print("--- وضعیت فایل‌های حیاتی برای گیت‌هاب ---")
for p in paths_to_verify:
    status = "✅ موجود" if os.path.exists(p) else "❌ مفقود (بررسی شود)"
    print(f"{status} : {p}")

In [ ]:
# تست نهایی سلامت نوت‌بوک قبل از پاکسازی
try:
    print("--- چک‌لیست نهایی سلامت پروژه ---")

    # ۱. تست وجود کلاس‌های اصلی
    model_ok = 'PFMS_Series_OCR' in globals() and 'IDPLOCRDataset' in globals()
    print(f"[1] کلاس‌های معماری مدل: {'✅ آماده' if model_ok else '❌ تعریف نشده'}")

    # ۲. تست دسترسی به داده‌ها
    loader_ok = 'train_loader' in globals()
    print(f"[2] لودر داده‌ها (DataLoader): {'✅ فعال' if loader_ok else '❌ غیرفعال'}")

    # ۳. تست واژگان
    vocab_ok = 'VOCAB_SIZE' in globals() and VOCAB_SIZE > 0
    print(f"[3] تنظیمات واژگان (Vocabulary): {'✅ صحیح' if vocab_ok else '❌ خطا'}")

    if model_ok and loader_ok and vocab_ok:
        print("\n🚀 نوت‌بوک شما کاملاً سالم است. می‌توانید سلول‌های جستجو و تست‌های موقت را دستی حذف کنید.")
    else:
        print("\n⚠ هشدار: برخی بخش‌ها نیاز به بازخوانی (Re-run) دارند.")

except Exception as e:
    print(f"❌ خطا در تست سلامت: {e}")

--- چک‌لیست نهایی سلامت پروژه ---
[1] کلاس‌های معماری مدل: ❌ تعریف نشده
[2] لودر داده‌ها (DataLoader): ❌ غیرفعال
[3] تنظیمات واژگان (Vocabulary): ❌ خطا

⚠ هشدار: برخی بخش‌ها نیاز به بازخوانی (Re-run) دارند.


In [ ]:
# ==========================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-1 — COMPLETE EVALUATION SETUP
# ==========================================================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

# ------------------------------------------------------------------------------------------
# 1. GOOGLE DRIVE
# ------------------------------------------------------------------------------------------

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-1 — COMPLETE EVALUATION SETUP")
print("=" * 100)

try:
    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=False
    )

    print("\n✓ Google Drive mounted")

except Exception as e:
    print("\nDrive mount message:")
    print(e)


# ------------------------------------------------------------------------------------------
# 2. RANDOM SEED
# ------------------------------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------------------------------------
# 3. PATHS
# ------------------------------------------------------------------------------------------

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "PFMS-SERIES-GT38-2026"
)

GT_CSV = (
    "/content/drive/MyDrive/"
    "IDPL_PFOD_REAL_GT_MAPPING.csv"
)

VOCAB_PATH = (
    "/content/drive/MyDrive/"
    "idpl_vocab.pt"
)

BEST_MODEL = os.path.join(
    PROJECT_DIR,
    "best_pfms_series_swin_mamba_trocr.pth"
)

PREDICTION_FILE = os.path.join(
    PROJECT_DIR,
    "stage8_predictions.csv"
)


print("\nPROJECT:")
print(PROJECT_DIR)

print("\nGROUND TRUTH CSV:")
print(GT_CSV)

print("\nVOCABULARY:")
print(VOCAB_PATH)

print("\nBEST MODEL:")
print(BEST_MODEL)


# ------------------------------------------------------------------------------------------
# 4. CHECK FILES
# ------------------------------------------------------------------------------------------

assert os.path.isdir(
    PROJECT_DIR
), "Project directory not found."

assert os.path.isfile(
    GT_CSV
), "Ground Truth CSV not found."

assert os.path.isfile(
    VOCAB_PATH
), "Vocabulary file not found."

assert os.path.isfile(
    BEST_MODEL
), "Best model checkpoint not found."


print("\n✓ All required files exist")


# ------------------------------------------------------------------------------------------
# 5. DEVICE
# ------------------------------------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("\nDEVICE:")
print(device)

if device.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA:",
        torch.version.cuda
    )


# ------------------------------------------------------------------------------------------
# 6. LOAD REAL GROUND TRUTH
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("LOADING REAL IDPL-PFOD GROUND TRUTH")
print("=" * 100)

gt_df = pd.read_csv(
    GT_CSV,
    encoding="utf-8-sig"
)

print("\nRows:", len(gt_df))

print(
    "Columns:",
    list(gt_df.columns)
)


required_columns = [
    "split",
    "image_path",
    "image_name",
    "text"
]

for col in required_columns:

    assert col in gt_df.columns, (
        f"Missing required column: {col}"
    )


gt_df["text"] = (
    gt_df["text"]
    .fillna("")
    .astype(str)
)

gt_df["split"] = (
    gt_df["split"]
    .fillna("")
    .astype(str)
    .str.lower()
)


# ------------------------------------------------------------------------------------------
# 7. CHECK IMAGE FILES
# ------------------------------------------------------------------------------------------

def resolve_image_path(path):

    path = str(path)

    if os.path.isfile(path):
        return path

    # If CSV contains relative path
    candidate = os.path.join(
        "/content/drive/MyDrive",
        path
    )

    if os.path.isfile(candidate):
        return candidate

    # Try basename in known dataset location
    dataset_dir = (
        "/content/drive/Othercomputers/"
        "My Laptop/Desktop/idplimgl"
    )

    candidate = os.path.join(
        dataset_dir,
        os.path.basename(path)
    )

    if os.path.isfile(candidate):
        return candidate

    return None


print("\nChecking image files...")

resolved_paths = []

missing = 0

for p in tqdm(
    gt_df["image_path"].tolist(),
    desc="Checking images"
):

    resolved = resolve_image_path(p)

    resolved_paths.append(resolved)

    if resolved is None:
        missing += 1


gt_df["resolved_image_path"] = resolved_paths


print("\nExisting:",
      len(gt_df) - missing)

print("Missing:",
      missing)


assert missing == 0, (
    f"{missing} image files are missing."
)


# ------------------------------------------------------------------------------------------
# 8. SELECT TEST SET
# ------------------------------------------------------------------------------------------

test_df = gt_df[
    gt_df["split"] == "test"
].copy()


print("\n" + "=" * 100)
print("TEST SET")
print("=" * 100)

print("Test samples:",
      len(test_df))


assert len(test_df) == 1356, (
    f"Expected 1356 test samples, "
    f"but found {len(test_df)}."
)


test_df = test_df.reset_index(
    drop=True
)


# ------------------------------------------------------------------------------------------
# 9. LOAD VOCABULARY
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("LOADING VOCABULARY")
print("=" * 100)

vocab_data = torch.load(
    VOCAB_PATH,
    map_location="cpu"
)

print(
    "Vocabulary object:",
    type(vocab_data)
)


if isinstance(vocab_data, dict):

    print(
        "Vocabulary keys:",
        list(vocab_data.keys())
    )

    if "itos" in vocab_data:

        idx_to_char = list(
            vocab_data["itos"]
        )

    elif "idx_to_char" in vocab_data:

        idx_to_char = list(
            vocab_data["idx_to_char"]
        )

    else:

        raise ValueError(
            "Cannot find itos or idx_to_char."
        )

else:

    idx_to_char = list(
        vocab_data
    )


VOCAB_SIZE = len(
    idx_to_char
)

PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3


print("\nReal characters:",
      VOCAB_SIZE - 4)

print("PAD:",
      PAD_IDX)

print("BOS:",
      BOS_IDX)

print("EOS:",
      EOS_IDX)

print("UNK:",
      UNK_IDX)

print("TOTAL VOCAB:",
      VOCAB_SIZE)


assert VOCAB_SIZE == 169, (
    f"Expected vocabulary size 169, "
    f"got {VOCAB_SIZE}"
)


# ------------------------------------------------------------------------------------------
# 10. TOKENIZER
# ------------------------------------------------------------------------------------------

char_to_idx = {
    char: idx
    for idx, char
    in enumerate(idx_to_char)
}


def text_to_tokens(
    text,
    max_length=128
):

    tokens = [
        BOS_IDX
    ]

    for char in str(text):

        if char in char_to_idx:

            tokens.append(
                char_to_idx[char]
            )

        else:

            tokens.append(
                UNK_IDX
            )

        if len(tokens) >= max_length - 1:
            break

    tokens.append(
        EOS_IDX
    )

    if len(tokens) < max_length:

        tokens += [
            PAD_IDX
        ] * (
            max_length - len(tokens)
        )

    else:

        tokens = tokens[
            :max_length
        ]

        tokens[-1] = EOS_IDX

    return torch.tensor(
        tokens,
        dtype=torch.long
    )


# ------------------------------------------------------------------------------------------
# 11. IMAGE TRANSFORM
# ------------------------------------------------------------------------------------------

eval_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )

])


# ------------------------------------------------------------------------------------------
# 12. TEST DATASET
# ------------------------------------------------------------------------------------------

class IDPLTestDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform


    def __len__(self):

        return len(self.df)


    def __getitem__(self, index):

        row = self.df.iloc[index]

        image_path = (
            row["resolved_image_path"]
        )

        text = str(
            row["text"]
        )

        image = Image.open(
            image_path
        ).convert("RGB")


        if self.transform is not None:

            image = self.transform(
                image
            )


        target = text_to_tokens(
            text,
            max_length=128
        )


        return {
            "image": image,
            "target": target,
            "text": text,
            "image_name": str(
                row["image_name"]
            ),
            "image_path": image_path
        }


# ------------------------------------------------------------------------------------------
# 13. CREATE TEST DATASET
# ------------------------------------------------------------------------------------------

test_dataset = IDPLTestDataset(
    test_df,
    transform=eval_transform
)


print("\n" + "=" * 100)
print("TEST DATASET")
print("=" * 100)

print(
    "Test dataset size:",
    len(test_dataset)
)


# ------------------------------------------------------------------------------------------
# 14. CREATE TEST LOADER
# ------------------------------------------------------------------------------------------

BATCH_SIZE = 4

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(
        device.type == "cuda"
    )
)


print("\nBatch size:",
      BATCH_SIZE)

print(
    "Test batches:",
    len(test_loader)
)


# ------------------------------------------------------------------------------------------
# 15. TEST DATASET SANITY CHECK
# ------------------------------------------------------------------------------------------

sample = test_dataset[0]

print("\n" + "=" * 100)
print("TEST DATASET SANITY CHECK")
print("=" * 100)

print(
    "Image shape:",
    tuple(sample["image"].shape)
)

print(
    "Target shape:",
    tuple(sample["target"].shape)
)

print(
    "Image name:",
    sample["image_name"]
)

print(
    "Ground Truth:",
    sample["text"]
)


assert tuple(
    sample["image"].shape
) == (3, 224, 224)

assert sample["target"].shape[0] == 128


# ------------------------------------------------------------------------------------------
# 16. CREATE OUTPUT DIRECTORY
# ------------------------------------------------------------------------------------------

os.makedirs(
    PROJECT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------------------------------------
# 17. FINAL STATUS
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("STAGE 8-1 — COMPLETE EVALUATION SETUP FINISHED ✓")
print("=" * 100)

print("\nGround Truth:")
print("✓ Real IDPL-PFOD")

print(
    "✓ Total GT rows:",
    len(gt_df)
)

print(
    "✓ Test samples:",
    len(test_dataset)
)

print("\nVocabulary:")
print(
    "✓ Real characters:",
    VOCAB_SIZE - 4
)

print(
    "✓ Total vocabulary:",
    VOCAB_SIZE
)

print("\nDataLoader:")
print(
    "✓ Test batches:",
    len(test_loader)
)

print(
    "✓ Batch size:",
    BATCH_SIZE
)

print("\nCheckpoint:")
print(
    "✓ Best model found:"
)

print(BEST_MODEL)

print("\nNext:")
print(
    "STAGE 8-2 — RESTORE PFMS-SERIES MODEL"
)

print("=" * 100)

PFMS-SERIES-GT38-2026
STAGE 8-1 — COMPLETE EVALUATION SETUP
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✓ Google Drive mounted

PROJECT:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026

GROUND TRUTH CSV:
/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv

VOCABULARY:
/content/drive/MyDrive/idpl_vocab.pt

BEST MODEL:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/best_pfms_series_swin_mamba_trocr.pth

✓ All required files exist

DEVICE:
cuda
GPU: Tesla T4
CUDA: 12.8

LOADING REAL IDPL-PFOD GROUND TRUTH

Rows: 27120
Columns: ['split', 'hf_index', 'image_path', 'image_name', 'text']

Checking image files...


Checking images:   0%|          | 0/27120 [00:00<?, ?it/s]


Existing: 27120
Missing: 0

TEST SET
Test samples: 4066


AssertionError: Expected 1356 test samples, but found 4066.

In [ ]:
# ==========================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-1A — VERIFY REAL DATASET SPLIT
# ==========================================================================================

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-1A — VERIFY REAL DATASET SPLIT")
print("=" * 100)

print("\nREAL SPLIT VALUES:")
print(
    gt_df["split"].value_counts(dropna=False)
)

print("\n" + "-" * 100)

print("SPLIT + IMAGE COUNT:")
print(
    gt_df.groupby("split", dropna=False)
         .size()
         .sort_values(ascending=False)
)

print("\n" + "-" * 100)

print("TOTAL:")
print(len(gt_df))

print("\nExpected from STAGE 7:")
print("Train      : 23052")
print("Validation : 2712")
print("Test       : 1356")
print("Total      : 27120")

print("\n" + "=" * 100)

PFMS-SERIES-GT38-2026
STAGE 8-1A — VERIFY REAL DATASET SPLIT

REAL SPLIT VALUES:
split
train    23054
test      4066
Name: count, dtype: int64

----------------------------------------------------------------------------------------------------
SPLIT + IMAGE COUNT:
split
train    23054
test      4066
dtype: int64

----------------------------------------------------------------------------------------------------
TOTAL:
27120

Expected from STAGE 7:
Train      : 23052
Validation : 2712
Test       : 1356
Total      : 27120



In [ ]:
# ==========================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-1B — FIND SAVED SPLIT INFORMATION
# ==========================================================================================

import os
from glob import glob

PROJECT_DIR = "/content/drive/MyDrive/PFMS-SERIES-GT38-2026"
MYDRIVE = "/content/drive/MyDrive"

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-1B — FIND SAVED SPLIT INFORMATION")
print("=" * 100)

# Search project directory
project_files = []

for root, dirs, files in os.walk(PROJECT_DIR):
    for f in files:
        project_files.append(
            os.path.join(root, f)
        )

print("\nFiles inside project directory:")
for f in project_files:
    print(f)

# Search likely split/index files in MyDrive
patterns = [
    "*split*",
    "*index*",
    "*train*",
    "*valid*",
    "*val*",
    "*test*",
    "*dataset*"
]

print("\n" + "-" * 100)
print("Possible saved split/index files:")

found = set()

for pattern in patterns:
    for f in glob(
        os.path.join(MYDRIVE, "**", pattern),
        recursive=True
    ):
        if os.path.isfile(f):
            found.add(f)

for f in sorted(found):
    print(f)

print("\n" + "=" * 100)
print("CURRENT CSV SPLIT")
print("=" * 100)

print(
    gt_df["split"].value_counts()
)

print("\nTotal:", len(gt_df))

print("=" * 100)

PFMS-SERIES-GT38-2026
STAGE 8-1B — FIND SAVED SPLIT INFORMATION

Files inside project directory:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/idpl_vocab_final.pt
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/last_pfms_series_swin_mamba_trocr.pth
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/training_history.csv
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/best_pfms_series_swin_mamba_trocr.pth

----------------------------------------------------------------------------------------------------
Possible saved split/index files:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/training_history.csv

CURRENT CSV SPLIT
split
train    23054
test      4066
Name: count, dtype: int64

Total: 27120


In [ ]:
import pandas as pd

# بررسی دقیق تعداد رکوردها بر اساس ستون split
print('--- آمار نهایی تفکیک داده‌ها ---')
split_stats = gt_df['split'].value_counts(dropna=False)

for name, count in split_stats.items():
    print(f'بخش {name}: {count:,} رکورد')

print('-' * 30)
print(f'تعداد کل رکوردها: {len(gt_df):,}')

--- آمار نهایی تفکیک داده‌ها ---
بخش train: 23,054 رکورد
بخش test: 4,066 رکورد
------------------------------
تعداد کل رکوردها: 27,120


In [ ]:
# ==================================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-1C — RECOVER ORIGINAL SPLIT FROM COLAB / GOOGLE DRIVE
# ==================================================================================================

import os
import glob
import re

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-1C — RECOVER ORIGINAL SPLIT FROM COLAB / GOOGLE DRIVE")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. SEARCH LOCATIONS
# --------------------------------------------------------------------------------------------------

search_roots = [
    "/content",
    "/content/drive/MyDrive"
]

# --------------------------------------------------------------------------------------------------
# 2. FIND NOTEBOOKS
# --------------------------------------------------------------------------------------------------

notebooks = []

for root in search_roots:
    if os.path.exists(root):
        for path in glob.glob(os.path.join(root, "**", "*.ipynb"), recursive=True):
            # Ignore some system/cache directories
            if any(x in path for x in [
                "/.config/",
                "/.cache/",
                "/node_modules/",
                "/site-packages/"
            ]):
                continue

            notebooks.append(path)

notebooks = sorted(set(notebooks))

print("\nNumber of notebooks found:", len(notebooks))

# --------------------------------------------------------------------------------------------------
# 3. SEARCH FOR SPLIT-RELATED TERMS
# --------------------------------------------------------------------------------------------------

keywords = [
    "random_split",
    "train_test_split",
    "Subset",
    "DataLoader",
    "train_dataset",
    "val_dataset",
    "test_dataset",
    "validation",
    "test_loader",
    "train_loader",
    "val_loader",
    "TEST_SIZE",
    "VAL_SIZE",
    "random_seed",
    "manual_seed",
    "generator",
    "split"
]

matches = []

for nb_path in notebooks:

    try:
        with open(nb_path, "r", encoding="utf-8") as f:
            text = f.read()
    except Exception:
        continue

    found = []

    for keyword in keywords:
        if keyword.lower() in text.lower():
            found.append(keyword)

    # Give priority to notebooks related to PFMS / GT / IDPL / Mamba / Swin / TrOCR
    priority_terms = [
        "PFMS",
        "GT38",
        "IDPL",
        "Mamba",
        "Swin",
        "TrOCR",
        "27120",
        "23052",
        "23054",
        "2712",
        "1356"
    ]

    priority_score = sum(
        1 for term in priority_terms
        if term.lower() in text.lower()
    )

    if found:
        matches.append(
            (priority_score, nb_path, found)
        )

# --------------------------------------------------------------------------------------------------
# 4. SORT BY RELEVANCE
# --------------------------------------------------------------------------------------------------

matches = sorted(
    matches,
    key=lambda x: (-x[0], x[1])
)

print("\n" + "=" * 100)
print("NOTEBOOKS CONTAINING SPLIT-RELATED INFORMATION")
print("=" * 100)

if len(matches) == 0:

    print("\nNo relevant notebook was found.")

else:

    for i, (score, path, found) in enumerate(matches, 1):

        print(f"\n[{i}] Priority score: {score}")
        print("Path:", path)
        print("Keywords:", ", ".join(found))

# --------------------------------------------------------------------------------------------------
# 5. SEARCH SPECIFIC SPLIT NUMBERS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SEARCH FOR ORIGINAL SPLIT NUMBERS")
print("=" * 100)

number_patterns = [
    "23052",
    "23054",
    "2712",
    "1356",
    "27120"
]

number_matches = []

for nb_path in notebooks:

    try:
        with open(nb_path, "r", encoding="utf-8") as f:
            text = f.read()
    except Exception:
        continue

    found_numbers = [
        n for n in number_patterns
        if n in text
    ]

    if found_numbers:
        number_matches.append(
            (nb_path, found_numbers)
        )

if number_matches:

    for path, nums in number_matches:

        print("\nNotebook:")
        print(path)
        print("Numbers found:", nums)

else:

    print("\nNo notebook contains the known split numbers.")

# --------------------------------------------------------------------------------------------------
# 6. SEARCH CURRENT RUNTIME FOR PYTHON FILES
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SEARCHING PYTHON FILES FOR SPLIT CODE")
print("=" * 100)

py_files = []

for root in search_roots:
    if os.path.exists(root):
        for path in glob.glob(os.path.join(root, "**", "*.py"), recursive=True):

            if any(x in path for x in [
                "/site-packages/",
                "/dist-packages/",
                "/usr/local/lib/"
            ]):
                continue

            py_files.append(path)

py_files = sorted(set(py_files))

py_matches = []

for py_path in py_files:

    try:
        with open(py_path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()
    except Exception:
        continue

    found = []

    for keyword in [
        "random_split",
        "train_test_split",
        "val_dataset",
        "test_dataset",
        "random_seed",
        "manual_seed",
        "27120",
        "23052",
        "1356"
    ]:

        if keyword.lower() in text.lower():
            found.append(keyword)

    if found:
        py_matches.append(
            (py_path, found)
        )

print("\nPython files with possible split information:")

if py_matches:

    for path, found in py_matches:
        print("\n", path)
        print("Keywords:", ", ".join(found))

else:

    print("None found.")

# --------------------------------------------------------------------------------------------------
# 7. FINAL STATUS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("STAGE 8-1C STATUS")
print("=" * 100)

print("""
IMPORTANT:

We will NOT evaluate the model on all 4066 held-out samples.

We will NOT randomly select 1356 samples.

We will first try to recover the exact Stage-7 split construction.

Expected original Stage-7 split:
    Train       = 23052
    Validation  = 2712
    Test        = 1356
    Total       = 27120

Current GT CSV:
    Train       = 23054
    Test/Heldout = 4066

Therefore the 4066 samples may represent:
    Validation + Test

The purpose of this stage is to locate the original code/notebook
that created the 23052 / 2712 / 1356 split.
""")

print("=" * 100)

PFMS-SERIES-GT38-2026
STAGE 8-1C — RECOVER ORIGINAL SPLIT FROM COLAB / GOOGLE DRIVE

Number of notebooks found: 66

NOTEBOOKS CONTAINING SPLIT-RELATED INFORMATION

[1] Priority score: 11
Path: /content/drive/MyDrive/Colab Notebooks/Copy of PFMS-Net-Series-Final.ipynb
Keywords: train_test_split, Subset, DataLoader, train_dataset, val_dataset, test_dataset, validation, test_loader, train_loader, val_loader, TEST_SIZE, manual_seed, generator, split

[2] Priority score: 11
Path: /content/drive/MyDrive/Colab Notebooks/PFMS-Net-Series-Final.ipynb
Keywords: train_test_split, Subset, DataLoader, train_dataset, val_dataset, test_dataset, validation, test_loader, train_loader, val_loader, TEST_SIZE, manual_seed, generator, split

[3] Priority score: 9
Path: /content/drive/MyDrive/Colab Notebooks/Copy of PFMS-Net-Series-Final (1).ipynb
Keywords: train_test_split, Subset, DataLoader, train_dataset, test_dataset, validation, test_loader, train_loader, TEST_SIZE, manual_seed, generator, split

[4] P

In [ ]:
# ==================================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-1 — COMPLETE EVALUATION SETUP
# ==================================================================================================
#
# Architecture:
#       Swin-B → Mamba Sequence Mixer → Transformer OCR Decoder
#
# REAL DATA:
#       IDPL-PFOD
#       27,120 samples
#
# EXACT STAGE-7 SPLIT:
#       Train       = 23,052
#       Validation  = 2,712
#       Test        = 1,356
#
# Split reconstruction:
#       np.random.default_rng(42)
#       rng.shuffle(indices)
#       85% Train
#       10% Validation
#       5% Test
#
# IMPORTANT:
#       This cell reconstructs the ORIGINAL Stage-7 split.
#       No random new split is created.
#       No samples are selected arbitrarily.
#
# ==================================================================================================

import os
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

warnings.filterwarnings("ignore")


# ==================================================================================================
# 1. REPRODUCIBILITY
# ==================================================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print("=" * 105)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-1 — COMPLETE EVALUATION SETUP")
print("=" * 105)

print("\nRandom seed:", SEED)


# ==================================================================================================
# 2. GOOGLE DRIVE
# ==================================================================================================

print("\n" + "-" * 105)
print("GOOGLE DRIVE")
print("-" * 105)

try:
    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=False
    )

    print("✓ Google Drive ready")

except Exception as e:

    print("Drive message:", e)


# ==================================================================================================
# 3. PATHS
# ==================================================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/PFMS-SERIES-GT38-2026"
)

GT_CSV = Path(
    "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
)

VOCAB_PATH = Path(
    "/content/drive/MyDrive/idpl_vocab.pt"
)

VOCAB_BACKUP_PATH = (
    PROJECT_DIR /
    "idpl_vocab_final.pt"
)

BEST_MODEL_PATH = (
    PROJECT_DIR /
    "best_pfms_series_swin_mamba_trocr.pth"
)

LAST_MODEL_PATH = (
    PROJECT_DIR /
    "last_pfms_series_swin_mamba_trocr.pth"
)

HISTORY_PATH = (
    PROJECT_DIR /
    "training_history.csv"
)

IMAGE_ROOT = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
)


print("\nProject directory:")
print(PROJECT_DIR)

print("\nGround Truth CSV:")
print(GT_CSV)

print("\nImage root:")
print(IMAGE_ROOT)

print("\nBest checkpoint:")
print(BEST_MODEL_PATH)


# ==================================================================================================
# 4. REQUIRED FILE CHECK
# ==================================================================================================

print("\n" + "-" * 105)
print("REQUIRED FILE CHECK")
print("-" * 105)

required_paths = {
    "PROJECT_DIR": PROJECT_DIR,
    "GT_CSV": GT_CSV,
    "VOCAB_PATH": VOCAB_PATH,
    "BEST_MODEL": BEST_MODEL_PATH,
    "LAST_MODEL": LAST_MODEL_PATH,
    "HISTORY": HISTORY_PATH,
    "IMAGE_ROOT": IMAGE_ROOT,
}

all_paths_ok = True

for name, path in required_paths.items():

    exists = path.exists()

    print(
        ("✓" if exists else "✗"),
        f"{name:<18}",
        "->",
        path
    )

    if not exists:
        all_paths_ok = False

if not all_paths_ok:

    raise FileNotFoundError(
        "\nOne or more required files/directories are missing."
    )

print("\n✓ All required paths are available.")


# ==================================================================================================
# 5. PYTORCH / CUDA / GPU
# ==================================================================================================

print("\n" + "-" * 105)
print("PYTORCH / CUDA / GPU")
print("-" * 105)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():

    DEVICE = torch.device("cuda")

    print("Device:", DEVICE)
    print("GPU:", torch.cuda.get_device_name(0))

    gpu_memory = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / (1024 ** 3)
    )

    print(
        f"GPU memory: {gpu_memory:.2f} GB"
    )

else:

    DEVICE = torch.device("cpu")

    print("Device:", DEVICE)
    print("⚠ CUDA is not available.")


# ==================================================================================================
# 6. LOAD REAL GROUND TRUTH
# ==================================================================================================

print("\n" + "-" * 105)
print("REAL GROUND TRUTH")
print("-" * 105)

df = pd.read_csv(
    GT_CSV
)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())


# ==================================================================================================
# 7. VERIFY REQUIRED GT COLUMNS
# ==================================================================================================

required_columns = [
    "image_path",
    "image_name",
    "text"
]

for col in required_columns:

    if col not in df.columns:

        raise ValueError(
            f"Required GT column missing: {col}"
        )

print("\n✓ Required GT columns are present.")


# ==================================================================================================
# 8. REPRODUCE STAGE-7 CLEANING EXACTLY
# ==================================================================================================

# Stage 7 retained these three columns for dataset construction.

df = df[
    required_columns
].copy()

df["image_path"] = (
    df["image_path"]
    .fillna("")
    .astype(str)
)

df["image_name"] = (
    df["image_name"]
    .fillna("")
    .astype(str)
)

df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
)

# Remove empty text exactly as in Stage 7.

df = df[
    df["text"].str.strip().str.len() > 0
].reset_index(drop=True)


print("\nValid samples after Stage-7 cleaning:")
print(len(df))


# ==================================================================================================
# 9. VERIFY DATASET SIZE
# ==================================================================================================

EXPECTED_TOTAL = 27120

if len(df) != EXPECTED_TOTAL:

    raise ValueError(
        f"Dataset size mismatch. "
        f"Expected {EXPECTED_TOTAL}, got {len(df)}"
    )

print(
    f"✓ Dataset contains exactly {EXPECTED_TOTAL:,} samples."
)


# ==================================================================================================
# 10. VERIFY IMAGE FILES
# ==================================================================================================

print("\n" + "-" * 105)
print("IMAGE FILE VERIFICATION")
print("-" * 105)

exists_mask = df["image_path"].apply(
    os.path.exists
)

existing_count = int(
    exists_mask.sum()
)

missing_count = int(
    (~exists_mask).sum()
)

print("Existing images:", existing_count)
print("Missing images :", missing_count)

if missing_count != 0:

    missing_examples = (
        df.loc[
            ~exists_mask,
            "image_path"
        ]
        .head(10)
        .tolist()
    )

    print("\nFirst missing paths:")

    for p in missing_examples:
        print(p)

    raise FileNotFoundError(
        f"\n{missing_count} image files are missing."
    )

print("\n✓ All 27,120 image files exist.")


# ==================================================================================================
# 11. RECONSTRUCT THE EXACT ORIGINAL STAGE-7 SPLIT
# ==================================================================================================

print("\n" + "-" * 105)
print("RECONSTRUCTING EXACT STAGE-7 SPLIT")
print("-" * 105)

# --------------------------------------------------------------------------------
# THIS IS THE ORIGINAL STAGE-7 PROCEDURE
# --------------------------------------------------------------------------------

indices = np.arange(
    len(df)
)

rng = np.random.default_rng(
    SEED
)

rng.shuffle(
    indices
)

N = len(indices)

N_TRAIN = int(
    0.85 * N
)

N_VAL = int(
    0.10 * N
)

train_idx = indices[
    :N_TRAIN
]

val_idx = indices[
    N_TRAIN:
    N_TRAIN + N_VAL
]

test_idx = indices[
    N_TRAIN + N_VAL:
]


# Create exact dataframes.

train_df = df.iloc[
    train_idx
].reset_index(drop=True)

val_df = df.iloc[
    val_idx
].reset_index(drop=True)

test_df = df.iloc[
    test_idx
].reset_index(drop=True)


# ==================================================================================================
# 12. VERIFY EXACT SPLIT COUNTS
# ==================================================================================================

print("\n" + "=" * 105)
print("EXACT STAGE-7 SPLIT")
print("=" * 105)

print(
    f"TOTAL      : {len(df):,}"
)

print(
    f"TRAIN      : {len(train_df):,}"
)

print(
    f"VALIDATION : {len(val_df):,}"
)

print(
    f"TEST       : {len(test_df):,}"
)


EXPECTED_TRAIN = 23052
EXPECTED_VAL = 2712
EXPECTED_TEST = 1356


assert len(train_df) == EXPECTED_TRAIN
assert len(val_df) == EXPECTED_VAL
assert len(test_df) == EXPECTED_TEST


print("\n✓ Train count is correct.")
print("✓ Validation count is correct.")
print("✓ Test count is correct.")


# ==================================================================================================
# 13. VERIFY NO OVERLAP BETWEEN SPLITS
# ==================================================================================================

train_set = set(
    train_idx.tolist()
)

val_set = set(
    val_idx.tolist()
)

test_set = set(
    test_idx.tolist()
)


print("\n" + "-" * 105)
print("SPLIT INTEGRITY")
print("-" * 105)

train_val_overlap = (
    train_set &
    val_set
)

train_test_overlap = (
    train_set &
    test_set
)

val_test_overlap = (
    val_set &
    test_set
)

print(
    "Train ∩ Validation:",
    len(train_val_overlap)
)

print(
    "Train ∩ Test:",
    len(train_test_overlap)
)

print(
    "Validation ∩ Test:",
    len(val_test_overlap)
)

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

print(
    "\n✓ No overlap exists between Train / Validation / Test."
)


# ==================================================================================================
# 14. VERIFY COMPLETE COVERAGE
# ==================================================================================================

all_split_indices = (
    train_set |
    val_set |
    test_set
)

print(
    "\nTotal unique split indices:",
    len(all_split_indices)
)

assert len(all_split_indices) == EXPECTED_TOTAL

print(
    "✓ Every one of the 27,120 samples belongs to exactly one split."
)


# ==================================================================================================
# 15. SAVE EXACT SPLIT FOR FUTURE REPRODUCIBILITY
# ==================================================================================================

split_file = (
    PROJECT_DIR /
    "PFMS_SERIES_GT38_STAGE7_EXACT_SPLIT.csv"
)

split_records = []

for position, original_index in enumerate(
    train_idx
):

    split_records.append({
        "original_df_index": int(original_index),
        "split": "train"
    })


for position, original_index in enumerate(
    val_idx
):

    split_records.append({
        "original_df_index": int(original_index),
        "split": "validation"
    })


for position, original_index in enumerate(
    test_idx
):

    split_records.append({
        "original_df_index": int(original_index),
        "split": "test"
    })


split_df = pd.DataFrame(
    split_records
)

split_df.to_csv(
    split_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Exact split saved:")
print(split_file)


# ==================================================================================================
# 16. SAVE TEST SET FOR AUDIT / REPRODUCIBILITY
# ==================================================================================================

test_audit_file = (
    PROJECT_DIR /
    "PFMS_SERIES_GT38_STAGE7_EXACT_TEST_SET.csv"
)

test_audit_df = test_df.copy()

test_audit_df.insert(
    0,
    "original_df_index",
    test_idx
)

test_audit_df.to_csv(
    test_audit_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Exact 1,356-sample Test set saved:")
print(test_audit_file)


# ==================================================================================================
# 17. LOAD VOCABULARY
# ==================================================================================================

print("\n" + "-" * 105)
print("VOCABULARY")
print("-" * 105)

vocab_file = VOCAB_PATH

if not vocab_file.exists():

    vocab_file = VOCAB_BACKUP_PATH

    print(
        "Main vocabulary not found; using backup."
    )

vocab_data = torch.load(
    vocab_file,
    map_location="cpu"
)

print(
    "Vocabulary file:",
    vocab_file
)

print(
    "Vocabulary object type:",
    type(vocab_data)
)


# --------------------------------------------------------------------------------
# Extract vocabulary mappings robustly.
# --------------------------------------------------------------------------------

if isinstance(vocab_data, dict):

    if "char2idx" in vocab_data:
        char2idx = vocab_data["char2idx"]

    elif "stoi" in vocab_data:
        char2idx = vocab_data["stoi"]

    else:
        char2idx = None


    if "idx2char" in vocab_data:
        idx2char = vocab_data["idx2char"]

    elif "itos" in vocab_data:
        idx2char = vocab_data["itos"]

    else:
        idx2char = None

else:

    char2idx = None
    idx2char = None


if char2idx is not None:

    VOCAB_SIZE = len(char2idx)

elif idx2char is not None:

    VOCAB_SIZE = len(idx2char)

else:

    VOCAB_SIZE = None


print(
    "Vocabulary size:",
    VOCAB_SIZE
)

if VOCAB_SIZE is not None:

    assert VOCAB_SIZE == 169

    print(
        "✓ Vocabulary size = 169"
    )

print(
    "\nSpecial token IDs expected:"
)

print(
    "PAD = 0"
)

print(
    "BOS = 1"
)

print(
    "EOS = 2"
)

print(
    "UNK = 3"
)


# ==================================================================================================
# 18. CHECK BEST CHECKPOINT
# ==================================================================================================

print("\n" + "-" * 105)
print("BEST MODEL CHECKPOINT")
print("-" * 105)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu"
)

print(
    "Checkpoint type:",
    type(checkpoint)
)

if isinstance(checkpoint, dict):

    print(
        "\nCheckpoint keys:"
    )

    for key in checkpoint.keys():
        print(
            "  -",
            key
        )

    if "epoch" in checkpoint:
        print(
            "\nSaved epoch:",
            checkpoint["epoch"]
        )

    if "val_loss" in checkpoint:
        print(
            "Saved validation loss:",
            checkpoint["val_loss"]
        )

    if "best_val_loss" in checkpoint:
        print(
            "Best validation loss:",
            checkpoint["best_val_loss"]
        )

    if "vocab_size" in checkpoint:
        print(
            "Checkpoint vocabulary:",
            checkpoint["vocab_size"]
        )

        assert checkpoint["vocab_size"] == 169

print(
    "\n✓ Best checkpoint can be loaded."
)


# Free checkpoint memory.

del checkpoint
gc.collect()


# ==================================================================================================
# 19. IMAGE PREPROCESSING — SAME AS STAGE 7
# ==================================================================================================

print("\n" + "-" * 105)
print("IMAGE PREPROCESSING")
print("-" * 105)

IMAGE_SIZE = 224

transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

print(
    "Resize:",
    f"{IMAGE_SIZE} × {IMAGE_SIZE}"
)

print(
    "Color mode: RGB"
)

print(
    "Normalization: ImageNet"
)

print(
    "✓ Evaluation preprocessing prepared."
)


# ==================================================================================================
# 20. TEST IMAGE SANITY CHECK
# ==================================================================================================

print("\n" + "-" * 105)
print("TEST IMAGE SANITY CHECK")
print("-" * 105)

sample_test_path = test_df.iloc[0]["image_path"]

print(
    "Sample test image:"
)

print(
    sample_test_path
)

with Image.open(sample_test_path) as img:

    print(
        "\nOriginal image:"
    )

    print(
        "  Mode   :",
        img.mode
    )

    print(
        "  Size   :",
        img.size
    )

    rgb_img = img.convert(
        "RGB"
    )

    tensor_img = transform(
        rgb_img
    )

print(
    "\nTransformed tensor:"
)

print(
    "  Shape :",
    tuple(tensor_img.shape)
)

print(
    "  Dtype :",
    tensor_img.dtype
)

print(
    "  Min   :",
    float(tensor_img.min())
)

print(
    "  Max   :",
    float(tensor_img.max())
)

assert tensor_img.shape == (
    3,
    224,
    224
)

print(
    "\n✓ Image preprocessing sanity check passed."
)


# ==================================================================================================
# 21. FINAL STAGE 8-1 REPORT
# ==================================================================================================

print("\n")
print("=" * 105)
print("STAGE 8-1 — FINAL STATUS")
print("=" * 105)

print("\nDATASET")
print("✓ Real IDPL-PFOD Ground Truth")
print("✓ Total samples       :", len(df))
print("✓ Existing images     :", existing_count)
print("✓ Missing images      :", missing_count)

print("\nEXACT ORIGINAL SPLIT")
print("✓ Train               :", len(train_df))
print("✓ Validation          :", len(val_df))
print("✓ Test                :", len(test_df))

print("\nVOCABULARY")
print("✓ Real characters     : 165")
print("✓ Total vocabulary    :", VOCAB_SIZE)

print("\nMODEL")
print("✓ Best checkpoint found")
print("✓ Checkpoint readable")

print("\nPREPROCESSING")
print("✓ RGB")
print("✓ Resize 224 × 224")
print("✓ ImageNet normalization")

print("\nREPRODUCIBILITY")
print("✓ Seed = 42")
print("✓ Original Stage-7 split reconstructed")
print("✓ No random new test set")
print("✓ No overlap between splits")
print("✓ Exact Test set saved")

print("\nSaved exact split:")
print(split_file)

print("\nSaved exact test set:")
print(test_audit_file)

print("\n" + "=" * 105)
print("✓ STAGE 8-1 COMPLETED SUCCESSFULLY")
print("=" * 105)

print(
    "\nNEXT:"
)

print(
    "STAGE 8-2 — RESTORE Swin-B → Mamba → Transformer OCR MODEL"
)

print("=" * 105)

PFMS-SERIES-GT38-2026
STAGE 8-1 — COMPLETE EVALUATION SETUP

Random seed: 42

---------------------------------------------------------------------------------------------------------
GOOGLE DRIVE
---------------------------------------------------------------------------------------------------------
Drive message: Mountpoint must not already contain files

Project directory:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026

Ground Truth CSV:
/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv

Image root:
/content/drive/Othercomputers/My Laptop/Desktop/idplimgl

Best checkpoint:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/best_pfms_series_swin_mamba_trocr.pth

---------------------------------------------------------------------------------------------------------
REQUIRED FILE CHECK
---------------------------------------------------------------------------------------------------------
✓ PROJECT_DIR        -> /content/drive/MyDrive/PFMS-SERIES-GT38-2026
✗ GT_CSV             -> /conte

FileNotFoundError: 
One or more required files/directories are missing.

In [ ]:
# ==================================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-2 — RESTORE SWIN-B → MAMBA → TRANSFORMER OCR MODEL
# ==================================================================================================

print("=" * 105)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-2 — RESTORE MODEL")
print("=" * 105)

import gc
import torch
import torch.nn as nn


# --------------------------------------------------------------------------------------------------
# 1. CHECK REQUIRED OBJECTS FROM STAGE 8-1
# --------------------------------------------------------------------------------------------------

required_objects = [
    "DEVICE",
    "BEST_MODEL_PATH",
    "VOCAB_PATH",
    "char2idx",
    "idx2char",
    "VOCAB_SIZE",
    "train_df",
    "val_df",
    "test_df"
]

print("\nChecking Stage 8-1 objects...")

for name in required_objects:

    if name not in globals():

        raise RuntimeError(
            f"Required object '{name}' is missing. "
            f"Please run STAGE 8-1 first."
        )

    print(f"✓ {name}")


# --------------------------------------------------------------------------------------------------
# 2. LOAD CHECKPOINT
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 105)
print("LOADING BEST CHECKPOINT")
print("-" * 105)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu"
)

print("✓ Checkpoint loaded successfully.")

print("\nCheckpoint information:")

if isinstance(checkpoint, dict):

    print("Epoch:", checkpoint.get("epoch"))
    print("Train loss:", checkpoint.get("train_loss"))
    print("Validation loss:", checkpoint.get("val_loss"))
    print("Best validation loss:", checkpoint.get("best_val_loss"))
    print("Vocabulary size:", checkpoint.get("vocab_size"))


# --------------------------------------------------------------------------------------------------
# 3. VERIFY CHECKPOINT
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 105)
print("CHECKPOINT VALIDATION")
print("-" * 105)

if not isinstance(checkpoint, dict):

    raise TypeError(
        "Checkpoint is not a dictionary."
    )

required_checkpoint_keys = [
    "model_state_dict",
    "vocab_size",
    "char2idx",
    "idx2char",
    "PAD_IDX",
    "BOS_IDX",
    "EOS_IDX",
    "UNK_IDX"
]

for key in required_checkpoint_keys:

    if key not in checkpoint:

        raise KeyError(
            f"Checkpoint key missing: {key}"
        )

    print(f"✓ {key}")


# --------------------------------------------------------------------------------------------------
# 4. VERIFY VOCABULARY CONSISTENCY
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 105)
print("VOCABULARY CONSISTENCY")
print("-" * 105)

checkpoint_vocab_size = int(
    checkpoint["vocab_size"]
)

print(
    "Stage-8 vocabulary:",
    VOCAB_SIZE
)

print(
    "Checkpoint vocabulary:",
    checkpoint_vocab_size
)

assert VOCAB_SIZE == checkpoint_vocab_size

print(
    "✓ Vocabulary sizes match."
)


# --------------------------------------------------------------------------------------------------
# 5. VERIFY SPECIAL TOKEN IDS
# --------------------------------------------------------------------------------------------------

PAD_IDX = int(
    checkpoint["PAD_IDX"]
)

BOS_IDX = int(
    checkpoint["BOS_IDX"]
)

EOS_IDX = int(
    checkpoint["EOS_IDX"]
)

UNK_IDX = int(
    checkpoint["UNK_IDX"]
)

print("\nSpecial tokens:")

print("PAD:", PAD_IDX)
print("BOS:", BOS_IDX)
print("EOS:", EOS_IDX)
print("UNK:", UNK_IDX)

assert PAD_IDX == 0
assert BOS_IDX == 1
assert EOS_IDX == 2
assert UNK_IDX == 3

print(
    "✓ Special token IDs are correct."
)


# --------------------------------------------------------------------------------------------------
# 6. INSPECT MODEL STATE DICT
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 105)
print("MODEL STATE DICTIONARY")
print("-" * 105)

state_dict = checkpoint[
    "model_state_dict"
]

print(
    "Number of parameter tensors:",
    len(state_dict)
)

print("\nFirst parameter names:")

for i, key in enumerate(
    state_dict.keys()
):

    print(
        f"{i+1:3d}. {key}"
    )

    if i >= 19:
        break


# --------------------------------------------------------------------------------------------------
# 7. PARAMETER COUNT
# --------------------------------------------------------------------------------------------------

total_parameters = 0

for tensor in state_dict.values():

    total_parameters += tensor.numel()

print(
    "\nCheckpoint parameter count:",
    f"{total_parameters:,}"
)

print(
    "Checkpoint parameter count (M):",
    f"{total_parameters / 1e6:.2f} M"
)


# --------------------------------------------------------------------------------------------------
# 8. IMPORTANT MODEL CLASS CHECK
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 105)
print("MODEL CLASS AVAILABILITY")
print("-" * 105)

# The model class used during Stage 7 must already exist in the notebook.
# We search for the most likely class names without inventing a new architecture.

possible_model_names = [
    "PFMSSeriesModel",
    "PFMS_Series_Model",
    "PFMSSeries",
    "PFMS_Series",
    "SwinMambaTrOCR",
    "SwinMambaOCR",
    "SwinMambaTransformerOCR",
    "PFMSNetSeries",
    "PFMSNet"
]

found_models = []

for name in possible_model_names:

    if name in globals():

        obj = globals()[name]

        if isinstance(obj, type):

            found_models.append(name)


print(
    "Model classes already defined in runtime:"
)

if found_models:

    for name in found_models:
        print("✓", name)

else:

    print(
        "⚠ No known Stage-7 model class is currently defined."
    )

    print(
        "\nThis is NOT a model error."
    )

    print(
        "The checkpoint is healthy, but the Python class definition"
    )

    print(
        "used during training is not currently loaded in this runtime."
    )


# --------------------------------------------------------------------------------------------------
# 9. DO NOT INVENT ARCHITECTURE
# --------------------------------------------------------------------------------------------------

if not found_models:

    print("\n" + "=" * 105)
    print("STAGE 8-2 — SAFE STOP")
    print("=" * 105)

    print(
        """
The checkpoint has been successfully verified.

However, the exact Stage-7 model class is not currently available
in the active Python runtime.

We will NOT create a guessed architecture.

The next step is to recover the exact model-definition cells from
the Stage-7 notebook and then load this checkpoint into that exact
architecture.

This is necessary for scientifically valid evaluation.
"""
    )

    print("=" * 105)

    # Keep checkpoint available for the next cell.
    PFMS_CHECKPOINT = checkpoint

else:

    # ----------------------------------------------------------------------------------------------
    # If the exact class is already present, instantiate only after identifying the correct class.
    # ----------------------------------------------------------------------------------------------

    print("\n✓ A candidate model class exists.")
    print(
        "The next step will identify its constructor and restore the checkpoint."
    )

    PFMS_CHECKPOINT = checkpoint


# --------------------------------------------------------------------------------------------------
# 10. MEMORY CLEANUP
# --------------------------------------------------------------------------------------------------

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# --------------------------------------------------------------------------------------------------
# FINAL
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 105)
print("STAGE 8-2 INITIAL CHECK COMPLETED")
print("=" * 105)

print(
    "\nCheckpoint is ready."
)

print(
    "Exact Stage-7 architecture must be used for restoration."
)

print("=" * 105)

PFMS-SERIES-GT38-2026
STAGE 8-2 — RESTORE MODEL

Checking Stage 8-1 objects...
✓ DEVICE
✓ BEST_MODEL_PATH
✓ VOCAB_PATH
✓ char2idx
✓ idx2char
✓ VOCAB_SIZE
✓ train_df
✓ val_df
✓ test_df

---------------------------------------------------------------------------------------------------------
LOADING BEST CHECKPOINT
---------------------------------------------------------------------------------------------------------
✓ Checkpoint loaded successfully.

Checkpoint information:
Epoch: 10
Train loss: 1.5866315606234906
Validation loss: 1.5366823013904876
Best validation loss: 1.5366823013904876
Vocabulary size: 169

---------------------------------------------------------------------------------------------------------
CHECKPOINT VALIDATION
---------------------------------------------------------------------------------------------------------
✓ model_state_dict
✓ vocab_size
✓ char2idx
✓ idx2char
✓ PAD_IDX
✓ BOS_IDX
✓ EOS_IDX
✓ UNK_IDX

---------------------------------------------------

In [ ]:
# =================================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-2A — EXACT STAGE-7 MODEL RESTORATION
# FIXED RUNTIME VERSION
# =================================================================================================

print("=" * 105)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-2A — EXACT STAGE-7 MODEL RESTORATION")
print("=" * 105)

# -------------------------------------------------------------------------------------------------
# 1. IMPORTS
# -------------------------------------------------------------------------------------------------

import os
import torch
import torch.nn as nn

from torchvision.models import swin_b, Swin_B_Weights

# -------------------------------------------------------------------------------------------------
# 2. DEVICE
# -------------------------------------------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nEnvironment:")
print("PyTorch :", torch.__version__)
print("Device  :", device)

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

# -------------------------------------------------------------------------------------------------
# 3. PATHS
# -------------------------------------------------------------------------------------------------

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "PFMS-SERIES-GT38-2026"
)

BEST_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "best_pfms_series_swin_mamba_trocr.pth"
)

VOCAB_PATH = (
    "/content/drive/MyDrive/"
    "idpl_vocab.pt"
)

if not os.path.exists(BEST_MODEL_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found:\n{BEST_MODEL_PATH}"
    )

print("\n✓ Checkpoint path verified")

# -------------------------------------------------------------------------------------------------
# 4. VOCABULARY
# -------------------------------------------------------------------------------------------------

if "VOCAB_SIZE" not in globals():

    vocab_data = torch.load(
        VOCAB_PATH,
        map_location="cpu"
    )

    if isinstance(vocab_data, dict):

        if "char2idx" in vocab_data:
            char2idx = vocab_data["char2idx"]
        else:
            raise RuntimeError(
                "char2idx not found in vocabulary file."
            )

        if "idx2char" in vocab_data:
            idx2char = vocab_data["idx2char"]
        else:
            raise RuntimeError(
                "idx2char not found in vocabulary file."
            )

    else:
        raise RuntimeError(
            "Unexpected vocabulary file format."
        )

    VOCAB_SIZE = len(char2idx)

else:

    print("✓ Existing VOCAB_SIZE found")

# -------------------------------------------------------------------------------------------------
# 5. SPECIAL TOKENS
# -------------------------------------------------------------------------------------------------

PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

print("\nVocabulary:")
print("VOCAB_SIZE :", VOCAB_SIZE)
print("PAD_IDX    :", PAD_IDX)
print("BOS_IDX    :", BOS_IDX)
print("EOS_IDX    :", EOS_IDX)
print("UNK_IDX    :", UNK_IDX)

if VOCAB_SIZE != 169:
    raise RuntimeError(
        f"Unexpected vocabulary size: {VOCAB_SIZE}. "
        "Expected Stage-7 vocabulary size = 169."
    )

print("✓ Vocabulary verified")

# -------------------------------------------------------------------------------------------------
# 6. EXACT STAGE-7 CONSTANT
# -------------------------------------------------------------------------------------------------

MAX_TEXT_LENGTH = 128

print("\nMAX_TEXT_LENGTH :", MAX_TEXT_LENGTH)

# -------------------------------------------------------------------------------------------------
# 7. EXACT STAGE-7 MAMBA SEQUENCE MIXER
# -------------------------------------------------------------------------------------------------

print("\n" + "-" * 105)
print("DEFINING EXACT STAGE-7 MAMBA SEQUENCE MIXER")
print("-" * 105)


class MambaSequenceMixer(nn.Module):

    def __init__(
        self,
        dim,
        expansion=2,
        dropout=0.1
    ):
        super().__init__()

        hidden = dim * expansion

        self.norm = nn.LayerNorm(
            dim
        )

        self.in_proj = nn.Linear(
            dim,
            hidden * 2
        )

        self.depthwise = nn.Conv1d(
            hidden,
            hidden,
            kernel_size=5,
            padding=2,
            groups=hidden
        )

        self.activation = nn.SiLU()

        self.out_proj = nn.Linear(
            hidden,
            dim
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x
    ):

        residual = x

        x = self.norm(x)

        x = self.in_proj(x)

        a, b = x.chunk(
            2,
            dim=-1
        )

        a = a.transpose(
            1,
            2
        )

        a = self.depthwise(a)

        a = a.transpose(
            1,
            2
        )

        a = self.activation(a)

        x = a * torch.sigmoid(b)

        x = self.out_proj(x)

        x = self.dropout(x)

        return residual + x


print("✓ MambaSequenceMixer defined")

# -------------------------------------------------------------------------------------------------
# 8. EXACT STAGE-7 TRANSFORMER OCR DECODER
# -------------------------------------------------------------------------------------------------

print("\n" + "-" * 105)
print("DEFINING EXACT STAGE-7 TRANSFORMER OCR DECODER")
print("-" * 105)


class TransformerOCRDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=512,
        nhead=8,
        num_layers=4,
        dim_feedforward=2048,
        dropout=0.1,
        max_length=128
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_length = max_length

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_IDX
        )

        self.pos_embedding = nn.Parameter(
            torch.randn(
                1,
                max_length,
                d_model
            ) * 0.02
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        self.output = nn.Linear(
            d_model,
            vocab_size
        )

    def causal_mask(
        self,
        length,
        device
    ):

        mask = torch.triu(
            torch.ones(
                length,
                length,
                device=device
            ),
            diagonal=1
        )

        mask = mask.masked_fill(
            mask == 1,
            float("-inf")
        )

        return mask

    def forward(
        self,
        memory,
        target
    ):

        B, T = target.shape

        target_emb = self.embedding(
            target
        )

        target_emb = (
            target_emb +
            self.pos_embedding[
                :, :T, :
            ]
        )

        causal_mask = self.causal_mask(
            T,
            target.device
        )

        padding_mask = (
            target == PAD_IDX
        )

        decoded = self.decoder(
            tgt=target_emb,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=padding_mask
        )

        decoded = self.norm(
            decoded
        )

        logits = self.output(
            decoded
        )

        return logits


print("✓ TransformerOCRDecoder defined")

# -------------------------------------------------------------------------------------------------
# 9. EXACT STAGE-7 PFMS SERIES OCR
# -------------------------------------------------------------------------------------------------

print("\n" + "-" * 105)
print("DEFINING EXACT STAGE-7 PFMS SERIES OCR")
print("-" * 105)


class PFMS_Series_OCR(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=512
    ):

        super().__init__()

        print("\nLoading pretrained Swin-B...")

        weights = (
            Swin_B_Weights.IMAGENET1K_V1
        )

        self.swin = swin_b(
            weights=weights
        )

        self.swin.head = nn.Identity()

        swin_dim = 1024

        self.projection = nn.Linear(
            swin_dim,
            d_model
        )

        self.mamba = nn.ModuleList([
            MambaSequenceMixer(
                d_model,
                expansion=2,
                dropout=0.1
            )
            for _ in range(4)
        ])

        self.decoder = TransformerOCRDecoder(
            vocab_size=vocab_size,
            d_model=d_model,
            nhead=8,
            num_layers=4,
            dim_feedforward=2048,
            dropout=0.1,
            max_length=MAX_TEXT_LENGTH
        )

    def extract_swin_features(
        self,
        x
    ):

        x = self.swin.features(
            x
        )

        if x.ndim == 4:

            B, H, W, C = x.shape

            x = x.reshape(
                B,
                H * W,
                C
            )

        return x

    def encode(
        self,
        images
    ):

        x = self.extract_swin_features(
            images
        )

        x = self.projection(
            x
        )

        for block in self.mamba:

            x = block(x)

        return x

    def forward(
        self,
        images,
        target_input
    ):

        memory = self.encode(
            images
        )

        logits = self.decoder(
            memory,
            target_input
        )

        return logits


print("✓ PFMS_Series_OCR defined")

# -------------------------------------------------------------------------------------------------
# 10. CREATE MODEL
# -------------------------------------------------------------------------------------------------

print("\n" + "-" * 105)
print("CREATING MODEL")
print("-" * 105)

model = PFMS_Series_OCR(
    vocab_size=VOCAB_SIZE,
    d_model=512
)

model = model.to(
    device
)

print("✓ Model created")

# -------------------------------------------------------------------------------------------------
# 11. PARAMETER COUNT
# -------------------------------------------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("\nCurrent model parameters:")
print(
    f"{total_params:,}"
)

print(
    f"{total_params / 1e6:.2f} M"
)

# -------------------------------------------------------------------------------------------------
# 12. LOAD CHECKPOINT
# -------------------------------------------------------------------------------------------------

print("\n" + "-" * 105)
print("LOADING BEST CHECKPOINT")
print("-" * 105)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)

print("✓ Checkpoint loaded")

print("\nCheckpoint:")
print("Epoch                :", checkpoint["epoch"])
print("Train loss           :", checkpoint["train_loss"])
print("Validation loss      :", checkpoint["val_loss"])
print("Best validation loss :", checkpoint["best_val_loss"])
print("Vocabulary size      :", checkpoint["vocab_size"])

# -------------------------------------------------------------------------------------------------
# 13. CHECKPOINT VOCABULARY
# -------------------------------------------------------------------------------------------------

if checkpoint["vocab_size"] != VOCAB_SIZE:

    raise RuntimeError(
        "Checkpoint vocabulary does not match Stage-8 vocabulary."
    )

print("✓ Checkpoint vocabulary matches")

# -------------------------------------------------------------------------------------------------
# 14. STRICT STATE-DICT LOAD
# -------------------------------------------------------------------------------------------------

print("\n" + "-" * 105)
print("STRICT STATE-DICT RESTORATION")
print("-" * 105)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

print("✓ STRICT state_dict loading successful")
print("✓ No missing keys")
print("✓ No unexpected keys")

# -------------------------------------------------------------------------------------------------
# 15. FREEZE SWIN — EXACT STAGE 7 SETTING
# -------------------------------------------------------------------------------------------------

for param in model.swin.parameters():
    param.requires_grad = False

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nSwin-B status: FROZEN")

print(
    "Trainable parameters :",
    f"{trainable_params:,}"
)

print(
    "Trainable parameters :",
    f"{trainable_params / 1e6:.2f} M"
)

# -------------------------------------------------------------------------------------------------
# 16. EVALUATION MODE
# -------------------------------------------------------------------------------------------------

model.eval()

print("\n✓ model.eval()")

# -------------------------------------------------------------------------------------------------
# 17. ARCHITECTURE VERIFICATION
# -------------------------------------------------------------------------------------------------

print("\n" + "-" * 105)
print("ARCHITECTURE VERIFICATION")
print("-" * 105)

print("Architecture:")
print("Swin-B")
print("   ↓")
print("Projection 1024 → 512")
print("   ↓")
print("4 × Mamba Sequence Mixer")
print("   ↓")
print("Transformer OCR Decoder")

print("\nVerified settings:")
print("Swin dimension        :", 1024)
print("Decoder dimension     :", model.decoder.d_model)
print("Mamba blocks           :", len(model.mamba))
print("Decoder layers         :", len(model.decoder.decoder.layers))
print("Attention heads        :", model.decoder.decoder.layers[0].self_attn.num_heads)
print("Feed-forward dimension :", model.decoder.decoder.layers[0].linear1.out_features)
print("Maximum text length    :", model.decoder.max_length)
print("Vocabulary size        :", model.decoder.vocab_size)

# -------------------------------------------------------------------------------------------------
# 18. FINAL RESULT
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 105)
print("STAGE 8-2A — EXACT MODEL RESTORATION SUCCESSFUL")
print("=" * 105)

print("\n✓ Exact Stage-7 architecture reconstructed")
print("✓ Best checkpoint loaded")
print("✓ strict=True state-dict match")
print("✓ Vocabulary = 169")
print("✓ Swin-B frozen")
print("✓ Model in evaluation mode")
print("✓ READY FOR STAGE 8-3 TEST EVALUATION")

print("=" * 105)

PFMS-SERIES-GT38-2026
STAGE 8-2A — EXACT STAGE-7 MODEL RESTORATION

Environment:
PyTorch : 2.11.0+cu128
Device  : cuda
GPU     : Tesla T4

✓ Checkpoint path verified
✓ Existing VOCAB_SIZE found

Vocabulary:
VOCAB_SIZE : 169
PAD_IDX    : 0
BOS_IDX    : 1
EOS_IDX    : 2
UNK_IDX    : 3
✓ Vocabulary verified

MAX_TEXT_LENGTH : 128

---------------------------------------------------------------------------------------------------------
DEFINING EXACT STAGE-7 MAMBA SEQUENCE MIXER
---------------------------------------------------------------------------------------------------------
✓ MambaSequenceMixer defined

---------------------------------------------------------------------------------------------------------
DEFINING EXACT STAGE-7 TRANSFORMER OCR DECODER
---------------------------------------------------------------------------------------------------------
✓ TransformerOCRDecoder defined

--------------------------------------------------------------------------------------------

100%|██████████| 335M/335M [00:01<00:00, 182MB/s]


✓ Model created

Current model parameters:
110,654,305
110.65 M

---------------------------------------------------------------------------------------------------------
LOADING BEST CHECKPOINT
---------------------------------------------------------------------------------------------------------
✓ Checkpoint loaded

Checkpoint:
Epoch                : 10
Train loss           : 1.5866315606234906
Validation loss      : 1.5366823013904876
Best validation loss : 1.5366823013904876
Vocabulary size      : 169
✓ Checkpoint vocabulary matches

---------------------------------------------------------------------------------------------------------
STRICT STATE-DICT RESTORATION
---------------------------------------------------------------------------------------------------------
✓ STRICT state_dict loading successful
✓ No missing keys
✓ No unexpected keys

Swin-B status: FROZEN
Trainable parameters : 23,911,081
Trainable parameters : 23.91 M

✓ model.eval()

-----------------------------

In [ ]:
# ==========================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-3 — REAL TEST EVALUATION
# ==========================================================================================
# هدف:
#   ارزیابی واقعی مدل آموزش‌دیده روی EXACT TEST SET مرحله 7
#
# Architecture:
#   Swin-B → Mamba Sequence Mixer → Transformer OCR Decoder
#
# Metrics:
#   CER
#   WER
#   Character Accuracy
#   Word Accuracy
#   Sequence Accuracy
#   Inference Time
#   Throughput
#
# IMPORTANT:
#   این سلول از مدل restore شده در Stage 8-2A استفاده می‌کند.
#   Checkpoint و مدل دوباره ساخته نمی‌شوند.
# ==========================================================================================

import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


# ==========================================================================================
# 1. HEADER
# ==========================================================================================

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-3 — REAL TEST EVALUATION")
print("=" * 100)


# ==========================================================================================
# 2. PATHS
# ==========================================================================================

TEST_CSV = "/content/drive/MyDrive/PFMS-SERIES-GT38-2026/PFMS_SERIES_GT38_STAGE7_EXACT_TEST_SET.csv"

VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/PFMS-SERIES-GT38-2026/"
    "best_pfms_series_swin_mamba_trocr.pth"
)

OUTPUT_DIR = "/content/drive/MyDrive/PFMS-SERIES-GT38-2026"

PREDICTIONS_CSV = (
    OUTPUT_DIR +
    "/PFMS_SERIES_GT38_STAGE8_TEST_PREDICTIONS.csv"
)


# ==========================================================================================
# 3. DEVICE
# ==========================================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nDEVICE")
print("-" * 100)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available.")


# ==========================================================================================
# 4. VERIFY REQUIRED FILES
# ==========================================================================================

print("\nVERIFYING REQUIRED FILES")
print("-" * 100)

required_files = [
    TEST_CSV,
    VOCAB_PATH,
    CHECKPOINT_PATH
]

for path in required_files:
    exists = os.path.exists(path)
    print(f"{'✓' if exists else '✗'} {path}")

    if not exists:
        raise FileNotFoundError(
            f"\nRequired file was not found:\n{path}"
        )

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================================================================
# 5. LOAD EXACT TEST SET
# ==========================================================================================

print("\nLOADING EXACT TEST SET")
print("-" * 100)

test_df = pd.read_csv(TEST_CSV)

print("Test samples:", len(test_df))

EXPECTED_TEST_SIZE = 1356

if len(test_df) != EXPECTED_TEST_SIZE:
    raise RuntimeError(
        f"Unexpected test size: {len(test_df)}. "
        f"Expected exactly {EXPECTED_TEST_SIZE}."
    )

print(f"✓ Exact test size verified: {EXPECTED_TEST_SIZE:,}")


# ==========================================================================================
# 6. VERIFY TEST IMAGE FILES
# ==========================================================================================

print("\nVERIFYING TEST IMAGES")
print("-" * 100)

missing_images = []

for path in test_df["image_path"].tolist():
    if not os.path.exists(path):
        missing_images.append(path)

if len(missing_images) > 0:
    print("✗ Missing images:", len(missing_images))
    print("\nFirst missing images:")

    for p in missing_images[:10]:
        print(p)

    raise FileNotFoundError(
        f"{len(missing_images)} test images are missing."
    )

print(f"✓ All {len(test_df):,} test images exist")


# ==========================================================================================
# 7. LOAD VOCABULARY DIRECTLY FROM FINAL VOCAB FILE
# ==========================================================================================
# IMPORTANT:
#   Total vocabulary = 169
#   Real characters = 165
#   Special tokens = 4
#
#   PAD = 0
#   BOS = 1
#   EOS = 2
#   UNK = 3
#
#   بنابراین len(char2idx) ممکن است 169 باشد و نباید آن را با
#   تعداد real characters (165) اشتباه گرفت.
# ==========================================================================================

print("\nLOADING VOCABULARY")
print("-" * 100)

vocab_data = torch.load(
    VOCAB_PATH,
    map_location="cpu"
)

file_char2idx = vocab_data["char2idx"]
file_idx2char = vocab_data["idx2char"]

file_PAD_IDX = int(vocab_data["PAD_IDX"])
file_BOS_IDX = int(vocab_data["BOS_IDX"])
file_EOS_IDX = int(vocab_data["EOS_IDX"])
file_UNK_IDX = int(vocab_data["UNK_IDX"])

file_vocab_size = int(vocab_data["vocab_size"])

print("VOCABULARY FROM FILE")
print("VOCAB_SIZE :", file_vocab_size)
print("len(char2idx):", len(file_char2idx))
print("len(idx2char):", len(file_idx2char))
print("PAD_IDX    :", file_PAD_IDX)
print("BOS_IDX    :", file_BOS_IDX)
print("EOS_IDX    :", file_EOS_IDX)
print("UNK_IDX    :", file_UNK_IDX)


# ==========================================================================================
# 8. STRICT VOCABULARY VALIDATION
# ==========================================================================================

if file_vocab_size != 169:
    raise RuntimeError(
        f"Unexpected vocabulary size: {file_vocab_size}. "
        f"Expected 169."
    )

if len(file_idx2char) != file_vocab_size:
    raise RuntimeError(
        f"idx2char length ({len(file_idx2char)}) does not match "
        f"vocab_size ({file_vocab_size})."
    )

if file_PAD_IDX != 0:
    raise RuntimeError(
        f"Unexpected PAD_IDX: {file_PAD_IDX}"
    )

if file_BOS_IDX != 1:
    raise RuntimeError(
        f"Unexpected BOS_IDX: {file_BOS_IDX}"
    )

if file_EOS_IDX != 2:
    raise RuntimeError(
        f"Unexpected EOS_IDX: {file_EOS_IDX}"
    )

if file_UNK_IDX != 3:
    raise RuntimeError(
        f"Unexpected UNK_IDX: {file_UNK_IDX}"
    )

print("\n✓ Vocabulary validation passed")
print("✓ Total vocabulary = 169")
print("✓ Real characters = 165")
print("✓ Special tokens = 4")


# ==========================================================================================
# 9. SET FINAL VOCAB VARIABLES
# ==========================================================================================

char2idx = file_char2idx
idx2char = file_idx2char

PAD_IDX = file_PAD_IDX
BOS_IDX = file_BOS_IDX
EOS_IDX = file_EOS_IDX
UNK_IDX = file_UNK_IDX

VOCAB_SIZE = file_vocab_size

MAX_TEXT_LENGTH = 128

print("\nFINAL TOKEN CONFIGURATION")
print("-" * 100)
print("VOCAB_SIZE :", VOCAB_SIZE)
print("PAD_IDX    :", PAD_IDX)
print("BOS_IDX    :", BOS_IDX)
print("EOS_IDX    :", EOS_IDX)
print("UNK_IDX    :", UNK_IDX)
print("MAX_LENGTH :", MAX_TEXT_LENGTH)


# ==========================================================================================
# 10. EXACT TEXT ENCODING USED IN STAGE 7
# ==========================================================================================

def encode_text(text, max_length=MAX_TEXT_LENGTH):

    tokens = [BOS_IDX]

    for ch in str(text):

        if ch in char2idx:
            tokens.append(char2idx[ch])
        else:
            tokens.append(UNK_IDX)

    tokens.append(EOS_IDX)

    if len(tokens) > max_length:

        tokens = tokens[:max_length]

        tokens[-1] = EOS_IDX

    if len(tokens) < max_length:

        tokens.extend(
            [PAD_IDX] * (max_length - len(tokens))
        )

    return tokens


# ==========================================================================================
# 11. EXACT EVALUATION TRANSFORM FROM STAGE 7
# ==========================================================================================

IMG_SIZE = 224

eval_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ==========================================================================================
# 12. TEST DATASET
# ==========================================================================================

class IDPLOCRTestDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = dataframe.reset_index(drop=True)

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = row["image_path"]

        text = row["text"]

        image_name = row["image_name"]

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            raise RuntimeError(
                f"\nCannot read image:\n"
                f"{image_path}\n"
                f"Error: {e}"
            )

        if self.transform is not None:

            image = self.transform(image)

        target = torch.tensor(
            encode_text(text),
            dtype=torch.long
        )

        return {
            "image": image,
            "target": target,
            "text": text,
            "image_name": image_name
        }


# ==========================================================================================
# 13. CREATE TEST DATALOADER
# ==========================================================================================

BATCH_SIZE = 4

test_dataset = IDPLOCRTestDataset(
    test_df,
    transform=eval_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("\nTEST DATALOADER")
print("-" * 100)
print("Test samples :", len(test_dataset))
print("Batch size   :", BATCH_SIZE)
print("Batches      :", len(test_loader))
print("Shuffle      : False")
print("num_workers  : 0")


# ==========================================================================================
# 14. VERIFY MODEL FROM STAGE 8-2A
# ==========================================================================================

print("\nVERIFYING RESTORED MODEL")
print("-" * 100)

if "model" not in globals():

    raise RuntimeError(
        "\nModel is not available in the current runtime.\n"
        "Please run the successful Stage 8-2A restoration cell first."
    )

model = model.to(device)

model.eval()

model_vocab_size = (
    model.decoder.embedding.num_embeddings
)

if model_vocab_size != VOCAB_SIZE:

    raise RuntimeError(
        f"\nModel vocabulary size ({model_vocab_size}) "
        f"does not match vocabulary file ({VOCAB_SIZE})."
    )

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("✓ Model exists")
print("✓ Model moved to:", device)
print("✓ Model set to eval()")
print(f"✓ Model vocabulary size: {model_vocab_size}")
print(f"✓ Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"✓ Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")


# ==========================================================================================
# 15. GREEDY AUTOREGRESSIVE DECODING
# ==========================================================================================
# Decoder was trained autoregressively:
#
#   decoder_input = targets[:, :-1]
#   expected      = targets[:, 1:]
#
# During testing, the decoder generates one token at a time.
# ==========================================================================================

@torch.no_grad()
def greedy_decode(
    model,
    images,
    max_length=MAX_TEXT_LENGTH
):

    # ------------------------------------------------------------------
    # Encode image ONCE
    # ------------------------------------------------------------------

    memory = model.encode(images)

    batch_size = images.size(0)

    # ------------------------------------------------------------------
    # Start with BOS
    # ------------------------------------------------------------------

    generated = torch.full(
        (batch_size, 1),
        BOS_IDX,
        dtype=torch.long,
        device=images.device
    )

    finished = torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=images.device
    )

    # ------------------------------------------------------------------
    # Autoregressive generation
    # ------------------------------------------------------------------

    for step in range(max_length - 1):

        logits = model.decoder(
            memory,
            generated
        )

        next_token = torch.argmax(
            logits[:, -1, :],
            dim=-1
        )

        # Once EOS has been generated,
        # keep EOS for that sample.
        next_token = torch.where(
            finished,
            torch.full_like(
                next_token,
                EOS_IDX
            ),
            next_token
        )

        generated = torch.cat(
            [
                generated,
                next_token.unsqueeze(1)
            ],
            dim=1
        )

        finished = finished | (
            next_token == EOS_IDX
        )

        if finished.all():

            break

    return generated


# ==========================================================================================
# 16. TOKEN → TEXT
# ==========================================================================================

def tokens_to_text(tokens):

    chars = []

    for token in tokens:

        token = int(token)

        if token == BOS_IDX:
            continue

        if token == PAD_IDX:
            continue

        if token == EOS_IDX:
            break

        if token == UNK_IDX:

            chars.append("�")

        elif 0 <= token < len(idx2char):

            chars.append(
                str(idx2char[token])
            )

        else:

            chars.append("�")

    return "".join(chars)


# ==========================================================================================
# 17. LEVENSHTEIN DISTANCE
# ==========================================================================================

def levenshtein_distance(a, b):

    # Convert to lists so this function works
    # for characters and words.

    a = list(a)
    b = list(b)

    if len(a) == 0:
        return len(b)

    if len(b) == 0:
        return len(a)

    previous = list(
        range(len(b) + 1)
    )

    for i, x in enumerate(a, start=1):

        current = [i]

        for j, y in enumerate(b, start=1):

            insertion = current[j - 1] + 1

            deletion = previous[j] + 1

            substitution = previous[j - 1] + (
                0 if x == y else 1
            )

            current.append(
                min(
                    insertion,
                    deletion,
                    substitution
                )
            )

        previous = current

    return previous[-1]


# ==========================================================================================
# 18. METRIC ACCUMULATORS
# ==========================================================================================

total_char_edits = 0
total_gt_chars = 0

total_word_edits = 0
total_gt_words = 0

exact_sequence_matches = 0

total_samples = 0

prediction_records = []


# ==========================================================================================
# 19. REAL TEST INFERENCE
# ==========================================================================================

print("\nSTARTING REAL TEST INFERENCE")
print("-" * 100)

inference_start = time.perf_counter()

with torch.no_grad():

    for batch_idx, batch in enumerate(test_loader):

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        gt_texts = batch["text"]

        image_names = batch["image_name"]

        # --------------------------------------------------------------
        # GPU synchronization for accurate timing
        # --------------------------------------------------------------

        if device.type == "cuda":
            torch.cuda.synchronize()

        predicted_tokens = greedy_decode(
            model,
            images,
            max_length=MAX_TEXT_LENGTH
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        # --------------------------------------------------------------
        # Convert predictions to strings
        # --------------------------------------------------------------

        for i in range(
            len(gt_texts)
        ):

            gt_text = str(
                gt_texts[i]
            )

            pred_text = tokens_to_text(
                predicted_tokens[i].tolist()
            )

            # ----------------------------------------------------------
            # Character-level edit distance
            # ----------------------------------------------------------

            char_distance = levenshtein_distance(
                gt_text,
                pred_text
            )

            total_char_edits += char_distance

            total_gt_chars += len(gt_text)

            # ----------------------------------------------------------
            # Word-level edit distance
            # ----------------------------------------------------------

            gt_words = gt_text.split()

            pred_words = pred_text.split()

            word_distance = levenshtein_distance(
                gt_words,
                pred_words
            )

            total_word_edits += word_distance

            total_gt_words += len(gt_words)

            # ----------------------------------------------------------
            # Sequence exact match
            # ----------------------------------------------------------

            exact_match = (
                gt_text == pred_text
            )

            if exact_match:
                exact_sequence_matches += 1

            total_samples += 1

            prediction_records.append({

                "image_name": image_names[i],

                "image_path": test_df.iloc[
                    total_samples - 1
                ]["image_path"],

                "ground_truth": gt_text,

                "prediction": pred_text,

                "exact_match": exact_match,

                "char_edit_distance": char_distance,

                "word_edit_distance": word_distance
            })

        # --------------------------------------------------------------
        # Progress
        # --------------------------------------------------------------

        if (
            (batch_idx + 1) % 50 == 0
            or
            (batch_idx + 1) == len(test_loader)
        ):

            print(
                f"Processed: "
                f"{min((batch_idx + 1) * BATCH_SIZE, len(test_dataset)):,}"
                f" / {len(test_dataset):,}"
            )


# ==========================================================================================
# 20. END TIMING
# ==========================================================================================

if device.type == "cuda":
    torch.cuda.synchronize()

inference_end = time.perf_counter()

total_inference_time = (
    inference_end - inference_start
)

# Number of images processed per second
throughput = (
    total_samples / total_inference_time
    if total_inference_time > 0
    else 0.0
)

latency_ms_per_image = (
    total_inference_time / total_samples * 1000
    if total_samples > 0
    else 0.0
)


# ==========================================================================================
# 21. CALCULATE CER / WER
# ==========================================================================================

if total_gt_chars > 0:

    CER = (
        total_char_edits /
        total_gt_chars
    )

else:

    CER = 0.0


if total_gt_words > 0:

    WER = (
        total_word_edits /
        total_gt_words
    )

else:

    WER = 0.0


# ==========================================================================================
# 22. CHARACTER / WORD / SEQUENCE ACCURACY
# ==========================================================================================

Character_Accuracy = 1.0 - CER

Word_Accuracy = 1.0 - WER

Sequence_Accuracy = (
    exact_sequence_matches /
    total_samples
    if total_samples > 0
    else 0.0
)


# ==========================================================================================
# 23. SAVE PREDICTIONS
# ==========================================================================================

predictions_df = pd.DataFrame(
    prediction_records
)

predictions_df.to_csv(
    PREDICTIONS_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ==========================================================================================
# 24. FINAL RESULTS
# ==========================================================================================

print("\n")
print("=" * 100)
print("STAGE 8-3 — FINAL REAL TEST RESULTS")
print("=" * 100)

print(f"\nTest Samples             : {total_samples:,}")

print(
    f"\nCharacter Edit Distance  : "
    f"{total_char_edits:,}"
)

print(
    f"Ground-Truth Characters  : "
    f"{total_gt_chars:,}"
)

print(
    f"\nWord Edit Distance       : "
    f"{total_word_edits:,}"
)

print(
    f"Ground-Truth Words       : "
    f"{total_gt_words:,}"
)

print("\n" + "-" * 100)

print(
    f"CER                      : "
    f"{CER * 100:.4f}%"
)

print(
    f"WER                      : "
    f"{WER * 100:.4f}%"
)

print(
    f"Character Accuracy      : "
    f"{Character_Accuracy * 100:.4f}%"
)

print(
    f"Word Accuracy           : "
    f"{Word_Accuracy * 100:.4f}%"
)

print(
    f"Sequence Accuracy       : "
    f"{Sequence_Accuracy * 100:.4f}%"
)

print("\n" + "-" * 100)

print(
    f"Exact Sequence Matches  : "
    f"{exact_sequence_matches:,} / {total_samples:,}"
)

print(
    f"Total Inference Time    : "
    f"{total_inference_time:.4f} sec"
)

print(
    f"Latency / Image         : "
    f"{latency_ms_per_image:.4f} ms"
)

print(
    f"Throughput              : "
    f"{throughput:.4f} images/sec"
)

print("\n" + "-" * 100)

print(
    "Predictions saved to:"
)

print(PREDICTIONS_CSV)


# ==========================================================================================
# 25. SHOW FIRST 20 REAL PREDICTIONS
# ==========================================================================================

print("\n")
print("=" * 100)
print("FIRST 20 TEST PREDICTIONS")
print("=" * 100)

display_columns = [
    "image_name",
    "ground_truth",
    "prediction",
    "exact_match"
]

display(
    predictions_df[
        display_columns
    ].head(20)
)


# ==========================================================================================
# 26. FINAL SANITY CHECKS
# ==========================================================================================

print("\n")
print("=" * 100)
print("STAGE 8-3 SANITY CHECK")
print("=" * 100)

if len(predictions_df) != EXPECTED_TEST_SIZE:

    raise RuntimeError(
        f"Prediction count mismatch: "
        f"{len(predictions_df)} != {EXPECTED_TEST_SIZE}"
    )

if predictions_df["ground_truth"].isna().any():

    raise RuntimeError(
        "Ground-truth contains NaN values."
    )

if predictions_df["prediction"].isna().any():

    raise RuntimeError(
        "Prediction contains NaN values."
    )

print(
    f"✓ Predictions: {len(predictions_df):,} / "
    f"{EXPECTED_TEST_SIZE:,}"
)

print("✓ No missing ground-truth values")
print("✓ No missing predictions")
print("✓ Metrics calculated from real test predictions")
print("✓ Predictions CSV successfully saved")

print("\n")
print("=" * 100)
print("STAGE 8-3 COMPLETED SUCCESSFULLY")
print("=" * 100)

PFMS-SERIES-GT38-2026
STAGE 8-3 — REAL TEST EVALUATION

DEVICE
----------------------------------------------------------------------------------------------------
Device: cuda
GPU: Tesla T4
CUDA: 12.8

VERIFYING REQUIRED FILES
----------------------------------------------------------------------------------------------------
✗ /content/drive/MyDrive/PFMS-SERIES-GT38-2026/PFMS_SERIES_GT38_STAGE7_EXACT_TEST_SET.csv


FileNotFoundError: 
Required file was not found:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/PFMS_SERIES_GT38_STAGE7_EXACT_TEST_SET.csv

In [ ]:
# ==========================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-4 — TEACHER-FORCED DIAGNOSTIC EVALUATION
# ==========================================================================================
#
# هدف:
#   تشخیص اینکه اختلاف شدید بین Validation Loss مرحله 7 و
#   Autoregressive Test Performance مرحله 8-3 از کجا ناشی می‌شود.
#
# Architecture:
#   Swin-B → Mamba Sequence Mixer → Transformer OCR Decoder
#
# روش:
#   Ground-Truth Prefix → Decoder → Next Token
#
# این تست:
#   1) Test Loss واقعی را محاسبه می‌کند.
#   2) Token Accuracy را محاسبه می‌کند.
#   3) Next-Token Accuracy را محاسبه می‌کند.
#   4) میزان پیش‌بینی صحیح EOS را بررسی می‌کند.
#   5) Teacher-Forced CER/WER را محاسبه می‌کند.
#
# IMPORTANT:
#   - هیچ آموزشی انجام نمی‌شود.
#   - optimizer استفاده نمی‌شود.
#   - checkpoint تغییر نمی‌کند.
#   - test set همان 1356 نمونه Stage 7 است.
# ==========================================================================================

import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


# ==========================================================================================
# 1. HEADER
# ==========================================================================================

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-4 — TEACHER-FORCED DIAGNOSTIC EVALUATION")
print("=" * 100)


# ==========================================================================================
# 2. PATHS
# ==========================================================================================

TEST_CSV = (
    "/content/drive/MyDrive/PFMS-SERIES-GT38-2026/"
    "PFMS_SERIES_GT38_STAGE7_EXACT_TEST_SET.csv"
)

VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/PFMS-SERIES-GT38-2026/"
    "best_pfms_series_swin_mamba_trocr.pth"
)


# ==========================================================================================
# 3. DEVICE
# ==========================================================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDEVICE")
print("-" * 100)
print("Device:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA:",
        torch.version.cuda
    )


# ==========================================================================================
# 4. VERIFY FILES
# ==========================================================================================

print("\nVERIFYING REQUIRED FILES")
print("-" * 100)

for path in [
    TEST_CSV,
    VOCAB_PATH,
    CHECKPOINT_PATH
]:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"\nRequired file not found:\n{path}"
        )

    print("✓", path)


# ==========================================================================================
# 5. LOAD EXACT TEST SET
# ==========================================================================================

print("\nLOADING EXACT TEST SET")
print("-" * 100)

test_df = pd.read_csv(
    TEST_CSV
)

EXPECTED_TEST_SIZE = 1356

print(
    "Test samples:",
    len(test_df)
)

if len(test_df) != EXPECTED_TEST_SIZE:

    raise RuntimeError(
        f"Expected {EXPECTED_TEST_SIZE} test samples, "
        f"found {len(test_df)}."
    )

print(
    f"✓ Exact test size verified: "
    f"{EXPECTED_TEST_SIZE:,}"
)


# ==========================================================================================
# 6. VERIFY TEST IMAGES
# ==========================================================================================

missing_images = [
    p for p in test_df["image_path"]
    if not os.path.exists(p)
]

if missing_images:

    raise FileNotFoundError(
        f"{len(missing_images)} test images are missing."
    )

print(
    f"✓ All {len(test_df):,} test images exist"
)


# ==========================================================================================
# 7. LOAD FINAL VOCABULARY
# ==========================================================================================

print("\nLOADING VOCABULARY")
print("-" * 100)

vocab_data = torch.load(
    VOCAB_PATH,
    map_location="cpu"
)

char2idx = vocab_data["char2idx"]
idx2char = vocab_data["idx2char"]

PAD_IDX = int(
    vocab_data["PAD_IDX"]
)

BOS_IDX = int(
    vocab_data["BOS_IDX"]
)

EOS_IDX = int(
    vocab_data["EOS_IDX"]
)

UNK_IDX = int(
    vocab_data["UNK_IDX"]
)

VOCAB_SIZE = int(
    vocab_data["vocab_size"]
)

MAX_TEXT_LENGTH = 128

print(
    "VOCAB_SIZE :",
    VOCAB_SIZE
)

print(
    "len(char2idx):",
    len(char2idx)
)

print(
    "len(idx2char):",
    len(idx2char)
)

print(
    "PAD_IDX    :",
    PAD_IDX
)

print(
    "BOS_IDX    :",
    BOS_IDX
)

print(
    "EOS_IDX    :",
    EOS_IDX
)

print(
    "UNK_IDX    :",
    UNK_IDX
)


# ==========================================================================================
# 8. STRICT VOCABULARY VALIDATION
# ==========================================================================================

if VOCAB_SIZE != 169:
    raise RuntimeError(
        f"Unexpected VOCAB_SIZE: {VOCAB_SIZE}"
    )

if len(idx2char) != VOCAB_SIZE:
    raise RuntimeError(
        "idx2char length does not match VOCAB_SIZE."
    )

if PAD_IDX != 0:
    raise RuntimeError(
        f"Unexpected PAD_IDX: {PAD_IDX}"
    )

if BOS_IDX != 1:
    raise RuntimeError(
        f"Unexpected BOS_IDX: {BOS_IDX}"
    )

if EOS_IDX != 2:
    raise RuntimeError(
        f"Unexpected EOS_IDX: {EOS_IDX}"
    )

if UNK_IDX != 3:
    raise RuntimeError(
        f"Unexpected UNK_IDX: {UNK_IDX}"
    )

print("\n✓ Vocabulary validation passed")


# ==========================================================================================
# 9. EXACT TEXT ENCODING FROM STAGE 7
# ==========================================================================================

def encode_text(
    text,
    max_length=MAX_TEXT_LENGTH
):

    tokens = [BOS_IDX]

    for ch in str(text):

        if ch in char2idx:

            tokens.append(
                char2idx[ch]
            )

        else:

            tokens.append(
                UNK_IDX
            )

    tokens.append(
        EOS_IDX
    )

    if len(tokens) > max_length:

        tokens = tokens[:max_length]

        tokens[-1] = EOS_IDX

    if len(tokens) < max_length:

        tokens.extend(
            [PAD_IDX] *
            (max_length - len(tokens))
        )

    return tokens


# ==========================================================================================
# 10. EXACT EVALUATION TRANSFORM
# ==========================================================================================

IMG_SIZE = 224

eval_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ==========================================================================================
# 11. TEST DATASET
# ==========================================================================================

class IDPLOCRTestDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = row["image_path"]

        text = str(
            row["text"]
        )

        image_name = row["image_name"]

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            raise RuntimeError(
                f"\nCannot read image:\n"
                f"{image_path}\n"
                f"Error: {e}"
            )

        if self.transform is not None:

            image = self.transform(
                image
            )

        target = torch.tensor(
            encode_text(text),
            dtype=torch.long
        )

        return {
            "image": image,
            "target": target,
            "text": text,
            "image_name": image_name
        }


# ==========================================================================================
# 12. DATALOADER
# ==========================================================================================

BATCH_SIZE = 4

test_dataset = IDPLOCRTestDataset(
    test_df,
    transform=eval_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("\nTEST DATALOADER")
print("-" * 100)

print(
    "Test samples :",
    len(test_dataset)
)

print(
    "Batch size   :",
    BATCH_SIZE
)

print(
    "Batches      :",
    len(test_loader)
)

print(
    "Shuffle      : False"
)

print(
    "num_workers  : 0"
)


# ==========================================================================================
# 13. VERIFY RESTORED MODEL
# ==========================================================================================

print("\nVERIFYING RESTORED MODEL")
print("-" * 100)

if "model" not in globals():

    raise RuntimeError(
        "\nThe restored model is not available.\n"
        "Please run the successful Stage 8-2A cell first."
    )

model = model.to(device)

model.eval()

model_vocab_size = (
    model.decoder.embedding.num_embeddings
)

if model_vocab_size != VOCAB_SIZE:

    raise RuntimeError(
        f"Model vocab size = {model_vocab_size}, "
        f"but vocabulary = {VOCAB_SIZE}."
    )

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"✓ Total parameters: "
    f"{total_params:,} "
    f"({total_params / 1e6:.2f}M)"
)

print(
    f"✓ Model vocabulary: "
    f"{model_vocab_size}"
)

print(
    "✓ Model set to eval()"
)


# ==========================================================================================
# 14. LOSS FUNCTION
# ==========================================================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)


# ==========================================================================================
# 15. TOKEN → TEXT
# ==========================================================================================

def tokens_to_text(tokens):

    chars = []

    for token in tokens:

        token = int(token)

        if token == BOS_IDX:
            continue

        if token == PAD_IDX:
            continue

        if token == EOS_IDX:
            break

        if token == UNK_IDX:

            chars.append("�")

        elif 0 <= token < len(idx2char):

            chars.append(
                str(idx2char[token])
            )

        else:

            chars.append("�")

    return "".join(chars)


# ==========================================================================================
# 16. LEVENSHTEIN
# ==========================================================================================

def levenshtein_distance(a, b):

    a = list(a)
    b = list(b)

    if len(a) == 0:
        return len(b)

    if len(b) == 0:
        return len(a)

    previous = list(
        range(len(b) + 1)
    )

    for i, x in enumerate(a, start=1):

        current = [i]

        for j, y in enumerate(
            b,
            start=1
        ):

            insertion = (
                current[j - 1] + 1
            )

            deletion = (
                previous[j] + 1
            )

            substitution = (
                previous[j - 1]
                + (0 if x == y else 1)
            )

            current.append(
                min(
                    insertion,
                    deletion,
                    substitution
                )
            )

        previous = current

    return previous[-1]


# ==========================================================================================
# 17. ACCUMULATORS
# ==========================================================================================

total_loss = 0.0

total_batches = 0

total_valid_tokens = 0

total_correct_tokens = 0

total_nonpad_tokens = 0

total_nonpad_correct = 0

total_eos_positions = 0

total_correct_eos = 0

total_char_edits = 0

total_gt_chars = 0

total_word_edits = 0

total_gt_words = 0

exact_teacher_forced_sequences = 0

total_samples = 0

teacher_predictions = []


# ==========================================================================================
# 18. START TEACHER-FORCED EVALUATION
# ==========================================================================================

print("\nSTARTING TEACHER-FORCED TEST EVALUATION")
print("-" * 100)

start_time = time.perf_counter()

with torch.no_grad():

    for batch_idx, batch in enumerate(
        test_loader
    ):

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        targets = batch["target"].to(
            device,
            non_blocking=True
        )

        gt_texts = batch["text"]

        image_names = batch["image_name"]

        # ------------------------------------------------------------------
        # EXACT TRAINING SETUP
        #
        # decoder_input:
        #     BOS + ground-truth characters
        #
        # expected_output:
        #     ground-truth characters + EOS
        # ------------------------------------------------------------------

        decoder_input = targets[:, :-1]

        expected_output = targets[:, 1:]

        # ------------------------------------------------------------------
        # Forward
        # ------------------------------------------------------------------

        logits = model(
            images,
            decoder_input
        )

        # ------------------------------------------------------------------
        # Loss
        # ------------------------------------------------------------------

        loss = criterion(
            logits.reshape(
                -1,
                VOCAB_SIZE
            ),
            expected_output.reshape(
                -1
            )
        )

        total_loss += (
            loss.item()
            * images.size(0)
        )

        total_batches += 1

        # ------------------------------------------------------------------
        # Predictions
        # ------------------------------------------------------------------

        predicted_tokens = torch.argmax(
            logits,
            dim=-1
        )

        # ------------------------------------------------------------------
        # Token accuracy
        # ------------------------------------------------------------------

        valid_mask = (
            expected_output != PAD_IDX
        )

        correct_mask = (
            predicted_tokens
            == expected_output
        )

        total_valid_tokens += (
            valid_mask.sum().item()
        )

        total_correct_tokens += (
            (
                correct_mask
                & valid_mask
            ).sum().item()
        )

        # ------------------------------------------------------------------
        # Accuracy excluding PAD
        # ------------------------------------------------------------------

        nonpad_mask = (
            expected_output != PAD_IDX
        )

        total_nonpad_tokens += (
            nonpad_mask.sum().item()
        )

        total_nonpad_correct += (
            (
                predicted_tokens
                == expected_output
            & nonpad_mask
            ).sum().item()
        )

        # ------------------------------------------------------------------
        # EOS accuracy
        # ------------------------------------------------------------------

        eos_mask = (
            expected_output == EOS_IDX
        )

        total_eos_positions += (
            eos_mask.sum().item()
        )

        total_correct_eos += (
            (
                predicted_tokens
                == EOS_IDX
            & eos_mask
            ).sum().item()
        )

        # ------------------------------------------------------------------
        # Per-sample diagnostic
        # ------------------------------------------------------------------

        for i in range(
            images.size(0)
        ):

            gt_text = str(
                gt_texts[i]
            )

            pred_token_list = (
                predicted_tokens[i]
                .detach()
                .cpu()
                .tolist()
            )

            pred_text = tokens_to_text(
                pred_token_list
            )

            # Character edit distance

            char_distance = (
                levenshtein_distance(
                    gt_text,
                    pred_text
                )
            )

            total_char_edits += (
                char_distance
            )

            total_gt_chars += (
                len(gt_text)
            )

            # Word edit distance

            gt_words = gt_text.split()

            pred_words = pred_text.split()

            word_distance = (
                levenshtein_distance(
                    gt_words,
                    pred_words
                )
            )

            total_word_edits += (
                word_distance
            )

            total_gt_words += (
                len(gt_words)
            )

            # Exact teacher-forced sequence

            exact_match = (
                gt_text == pred_text
            )

            if exact_match:

                exact_teacher_forced_sequences += 1

            total_samples += 1

            teacher_predictions.append({

                "image_name":
                    image_names[i],

                "ground_truth":
                    gt_text,

                "teacher_forced_prediction":
                    pred_text,

                "exact_match":
                    exact_match

            })

        # ------------------------------------------------------------------
        # Progress
        # ------------------------------------------------------------------

        if (
            (batch_idx + 1) % 50 == 0
            or
            (batch_idx + 1)
            == len(test_loader)
        ):

            processed = min(
                (batch_idx + 1)
                * BATCH_SIZE,
                len(test_dataset)
            )

            print(
                f"Processed: "
                f"{processed:,} / "
                f"{len(test_dataset):,}"
            )


# ==========================================================================================
# 19. FINAL METRICS
# ==========================================================================================

evaluation_time = (
    time.perf_counter()
    - start_time
)

test_loss = (
    total_loss / total_samples
)

token_accuracy = (
    total_correct_tokens
    / total_valid_tokens
    if total_valid_tokens > 0
    else 0.0
)

nonpad_token_accuracy = (
    total_nonpad_correct
    / total_nonpad_tokens
    if total_nonpad_tokens > 0
    else 0.0
)

eos_accuracy = (
    total_correct_eos
    / total_eos_positions
    if total_eos_positions > 0
    else 0.0
)

teacher_CER = (
    total_char_edits
    / total_gt_chars
    if total_gt_chars > 0
    else 0.0
)

teacher_WER = (
    total_word_edits
    / total_gt_words
    if total_gt_words > 0
    else 0.0
)

teacher_character_accuracy = (
    1.0 - teacher_CER
)

teacher_word_accuracy = (
    1.0 - teacher_WER
)

teacher_sequence_accuracy = (
    exact_teacher_forced_sequences
    / total_samples
    if total_samples > 0
    else 0.0
)


# ==========================================================================================
# 20. SAVE DIAGNOSTIC PREDICTIONS
# ==========================================================================================

teacher_predictions_df = pd.DataFrame(
    teacher_predictions
)

TEACHER_OUTPUT_CSV = (
    "/content/drive/MyDrive/"
    "PFMS-SERIES-GT38-2026/"
    "PFMS_SERIES_GT38_STAGE8_TEACHER_FORCED_PREDICTIONS.csv"
)

teacher_predictions_df.to_csv(
    TEACHER_OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ==========================================================================================
# 21. FINAL RESULTS
# ==========================================================================================

print("\n")
print("=" * 100)
print("STAGE 8-4 — TEACHER-FORCED FINAL RESULTS")
print("=" * 100)

print(
    f"\nTest Samples                 : "
    f"{total_samples:,}"
)

print(
    f"Test Loss                    : "
    f"{test_loss:.6f}"
)

print(
    f"\nValid Target Tokens          : "
    f"{total_valid_tokens:,}"
)

print(
    f"Correct Target Tokens        : "
    f"{total_correct_tokens:,}"
)

print(
    f"Token Accuracy               : "
    f"{token_accuracy * 100:.4f}%"
)

print(
    f"Non-PAD Token Accuracy       : "
    f"{nonpad_token_accuracy * 100:.4f}%"
)

print(
    f"EOS Accuracy                 : "
    f"{eos_accuracy * 100:.4f}%"
)

print("\n" + "-" * 100)

print(
    f"Teacher-Forced CER           : "
    f"{teacher_CER * 100:.4f}%"
)

print(
    f"Teacher-Forced WER           : "
    f"{teacher_WER * 100:.4f}%"
)

print(
    f"Teacher-Forced Character Acc.: "
    f"{teacher_character_accuracy * 100:.4f}%"
)

print(
    f"Teacher-Forced Word Acc.     : "
    f"{teacher_word_accuracy * 100:.4f}%"
)

print(
    f"Teacher-Forced Sequence Acc. : "
    f"{teacher_sequence_accuracy * 100:.4f}%"
)

print("\n" + "-" * 100)

print(
    f"Exact Teacher-Forced Matches : "
    f"{exact_teacher_forced_sequences:,} "
    f"/ {total_samples:,}"
)

print(
    f"Evaluation Time              : "
    f"{evaluation_time:.4f} sec"
)

print(
    "\nDiagnostic predictions saved to:"
)

print(
    TEACHER_OUTPUT_CSV
)


# ==========================================================================================
# 22. SHOW FIRST 20 DIAGNOSTIC PREDICTIONS
# ==========================================================================================

print("\n")
print("=" * 100)
print("FIRST 20 TEACHER-FORCED PREDICTIONS")
print("=" * 100)

display(
    teacher_predictions_df.head(20)
)


# ==========================================================================================
# 23. FINAL SANITY CHECK
# ==========================================================================================

print("\n")
print("=" * 100)
print("STAGE 8-4 SANITY CHECK")
print("=" * 100)

if len(teacher_predictions_df) != EXPECTED_TEST_SIZE:

    raise RuntimeError(
        "Prediction count does not match test set."
    )

if not np.isfinite(test_loss):

    raise RuntimeError(
        "Test loss is not finite."
    )

if not np.isfinite(token_accuracy):

    raise RuntimeError(
        "Token accuracy is not finite."
    )

print(
    f"✓ Predictions: "
    f"{len(teacher_predictions_df):,} / "
    f"{EXPECTED_TEST_SIZE:,}"
)

print("✓ Test loss is finite")
print("✓ Token accuracy is finite")
print("✓ Teacher-forced evaluation completed")
print("✓ No training was performed")
print("✓ Checkpoint was not modified")
print("✓ Diagnostic predictions saved")

print("\n")
print("=" * 100)
print("STAGE 8-4 COMPLETED SUCCESSFULLY")
print("=" * 100)

PFMS-SERIES-GT38-2026
STAGE 8-4 — TEACHER-FORCED DIAGNOSTIC EVALUATION

DEVICE
----------------------------------------------------------------------------------------------------
Device: cuda
GPU: Tesla T4
CUDA: 12.8

VERIFYING REQUIRED FILES
----------------------------------------------------------------------------------------------------
✓ /content/drive/MyDrive/PFMS-SERIES-GT38-2026/PFMS_SERIES_GT38_STAGE7_EXACT_TEST_SET.csv
✓ /content/drive/MyDrive/idpl_vocab.pt
✓ /content/drive/MyDrive/PFMS-SERIES-GT38-2026/best_pfms_series_swin_mamba_trocr.pth

LOADING EXACT TEST SET
----------------------------------------------------------------------------------------------------
Test samples: 1356
✓ Exact test size verified: 1,356
✓ All 1,356 test images exist

LOADING VOCABULARY
----------------------------------------------------------------------------------------------------
VOCAB_SIZE : 169
len(char2idx): 169
len(idx2char): 169
PAD_IDX    : 0
BOS_IDX    : 1
EOS_IDX    : 2
UNK_IDX    :

,image_name,ground_truth,teacher_forced_prediction,exact_match
0,22579.tif,کرد شرکت سیمان مازندران پیش بینی درآمد هر... ک...,اهد ودکت مااان ای ندران بیش اینی ار مد بم .. ک...,False
1,21742.tif,این بانک برای... کد خبر: ۱۱۰۹۸۶ تاریخ انتشار: ...,این مانک مه ی .. کد خبر: ۱۶۶۸۶۸ تاریخ انتشار: ...,False
2,18963.tif,آنگاه اختلال باید هدف اصلی شما باشد. همان‏طور ...,ان اه اییلاف اه د امف ازلی ادا دا د امچن هور ...,False
3,06941.tif,كاشان كاشان- خبرنگاركیهان: پنجمین جشنواره سراس...,اهران اررتو ابرنگار یهان: میج بننواره بامن...,False
4,13179.tif,بیش از ۱۰۰ واحدی بوسیله نماد معاملاتی فارس... ...,اهنتاز ا۰ م حد رهدیله اییی ویاملات اررس . ...,False
5,05064.tif,تاریخ انتشار: ۱۳۹۵/۰۶/۲۲ از موضع یک بلا برای ن...,ااریخ انتشار: ۱۳۹۶/۰۴/۰۷ از انرو اک ااکتااای ا...,False
6,14705.tif,هشدار به خریداران و دارندگان اوراق تسهیلات مسک...,امتار وا اصدد رین ا ارردد ان ایلاق بوهیلات ویا...,False
7,00461.tif,«ضداستكباری- ضد استبدادی» دارد. این روز، روز د...,اار»نت»»اری» اماایتارار ارند این ااز بوز ب...,False
8,14753.tif,را اختصاص دادند تا از محل فروش آن پاداش پایان ...,اا بزتلاص مارند وا ای اوا ارهش ام رریار مییان ...,False
9,00885.tif,شود شرایطی که به آن امواج گرمایی می گوییم ایجا...,ادر ودکیط که اا ان ایارلهدرف ی کی کریدد وینا...,False




STAGE 8-4 SANITY CHECK
✓ Predictions: 1,356 / 1,356
✓ Test loss is finite
✓ Token accuracy is finite
✓ Teacher-forced evaluation completed
✓ No training was performed
✓ Checkpoint was not modified
✓ Diagnostic predictions saved


STAGE 8-4 COMPLETED SUCCESSFULLY


In [ ]:
# =============================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-4 V2 — CORRECTED TEACHER-FORCED DIAGNOSTIC
# =============================================================================

import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

print("=" * 90)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-4 V2 — CORRECTED TEACHER-FORCED DIAGNOSTIC")
print("=" * 90)

# -------------------------------------------------------------------------
# 1. Runtime checks
# -------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# -------------------------------------------------------------------------
# 2. Ensure model is in evaluation mode
# -------------------------------------------------------------------------
model = model.to(device)
model.eval()

# -------------------------------------------------------------------------
# 3. Diagnostic accumulators
# -------------------------------------------------------------------------
total_loss = 0.0
total_samples = 0

total_valid_tokens = 0
total_correct_tokens = 0

total_nonpad_tokens = 0
total_nonpad_correct = 0

total_eos_positions = 0
total_eos_correct = 0

total_unk_predictions = 0
total_pred_nonpad = 0

total_pred_eos = 0

all_results = []

start_time = time.time()

# -------------------------------------------------------------------------
# 4. Teacher-forced evaluation
# -------------------------------------------------------------------------
with torch.no_grad():

    for batch in test_loader:

        images = batch["image"].to(device, non_blocking=True)
        targets = batch["target"].to(device, non_blocking=True)
        texts = batch["text"]
        image_names = batch["image_name"]

        # Teacher forcing
        decoder_input = targets[:, :-1]
        expected_output = targets[:, 1:]

        logits = model(images, decoder_input)

        # -------------------------------------------------------------
        # Loss
        # -------------------------------------------------------------
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            expected_output.reshape(-1),
            ignore_index=PAD_IDX
        )

        batch_size = images.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        # -------------------------------------------------------------
        # Predictions
        # -------------------------------------------------------------
        predicted_tokens = logits.argmax(dim=-1)

        # -------------------------------------------------------------
        # Correct masks
        # -------------------------------------------------------------
        valid_mask = expected_output != PAD_IDX
        nonpad_mask = expected_output != PAD_IDX
        eos_mask = expected_output == EOS_IDX

        correct_mask = predicted_tokens == expected_output

        # -------------------------------------------------------------
        # 1) Overall valid-token accuracy
        # -------------------------------------------------------------
        batch_valid = valid_mask.sum().item()
        batch_correct = (correct_mask & valid_mask).sum().item()

        total_valid_tokens += batch_valid
        total_correct_tokens += batch_correct

        # -------------------------------------------------------------
        # 2) Non-PAD accuracy — corrected parentheses
        # -------------------------------------------------------------
        batch_nonpad = nonpad_mask.sum().item()

        batch_nonpad_correct = (
            (predicted_tokens == expected_output) & nonpad_mask
        ).sum().item()

        total_nonpad_tokens += batch_nonpad
        total_nonpad_correct += batch_nonpad_correct

        # -------------------------------------------------------------
        # 3) EOS accuracy — corrected parentheses
        # -------------------------------------------------------------
        batch_eos = eos_mask.sum().item()

        batch_eos_correct = (
            (predicted_tokens == EOS_IDX) & eos_mask
        ).sum().item()

        total_eos_positions += batch_eos
        total_eos_correct += batch_eos_correct

        # -------------------------------------------------------------
        # 4) UNK predictions
        # -------------------------------------------------------------
        batch_unk = (
            (predicted_tokens == UNK_IDX) & nonpad_mask
        ).sum().item()

        total_unk_predictions += batch_unk
        total_pred_nonpad += batch_nonpad

        # -------------------------------------------------------------
        # 5) Number of predicted EOS tokens
        # -------------------------------------------------------------
        batch_pred_eos = (
            predicted_tokens == EOS_IDX
        ).sum().item()

        total_pred_eos += batch_pred_eos

        # -------------------------------------------------------------
        # Save per-sample diagnostic information
        # -------------------------------------------------------------
        for i in range(batch_size):

            gt_ids = expected_output[i].detach().cpu().tolist()
            pred_ids = predicted_tokens[i].detach().cpu().tolist()

            # Ground-truth effective length
            gt_len = 0
            for token in gt_ids:
                if token == PAD_IDX:
                    break
                gt_len += 1

            # Predicted effective length:
            # stop at first EOS; if no EOS, use full available length
            pred_len = 0
            pred_has_eos = False

            for token in pred_ids:
                if token == PAD_IDX:
                    break

                pred_len += 1

                if token == EOS_IDX:
                    pred_has_eos = True
                    break

            all_results.append({
                "image_name": image_names[i],
                "ground_truth": texts[i],
                "gt_token_length": gt_len,
                "predicted_token_length": pred_len,
                "predicted_has_eos": pred_has_eos,
                "predicted_unk_count": sum(
                    1 for t in pred_ids
                    if t == UNK_IDX
                )
            })

# -------------------------------------------------------------------------
# 5. Final metrics
# -------------------------------------------------------------------------
test_loss = total_loss / total_samples

token_accuracy = (
    total_correct_tokens / total_valid_tokens * 100
    if total_valid_tokens > 0 else 0.0
)

nonpad_accuracy = (
    total_nonpad_correct / total_nonpad_tokens * 100
    if total_nonpad_tokens > 0 else 0.0
)

eos_accuracy = (
    total_eos_correct / total_eos_positions * 100
    if total_eos_positions > 0 else 0.0
)

unk_rate = (
    total_unk_predictions / total_pred_nonpad * 100
    if total_pred_nonpad > 0 else 0.0
)

mean_gt_length = np.mean(
    [x["gt_token_length"] for x in all_results]
)

mean_pred_length = np.mean(
    [x["predicted_token_length"] for x in all_results]
)

samples_with_eos = sum(
    x["predicted_has_eos"] for x in all_results
)

eos_sample_rate = (
    samples_with_eos / len(all_results) * 100
)

elapsed = time.time() - start_time

# -------------------------------------------------------------------------
# 6. Print corrected diagnostic results
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print("CORRECTED TEACHER-FORCED DIAGNOSTIC RESULTS")
print("=" * 90)

print(f"Test Samples                 : {total_samples:,}")
print(f"Test Loss                    : {test_loss:.6f}")

print(f"\nValid Target Tokens          : {total_valid_tokens:,}")
print(f"Correct Target Tokens        : {total_correct_tokens:,}")

print(f"\nToken Accuracy               : {token_accuracy:.4f}%")
print(f"Non-PAD Token Accuracy       : {nonpad_accuracy:.4f}%")

print(f"\nExpected EOS Positions       : {total_eos_positions:,}")
print(f"Correct EOS Predictions      : {total_eos_correct:,}")
print(f"EOS Accuracy                 : {eos_accuracy:.4f}%")

print(f"\nPredicted EOS Tokens         : {total_pred_eos:,}")
print(f"Samples With Predicted EOS   : {samples_with_eos:,}")
print(f"Sample EOS Rate              : {eos_sample_rate:.4f}%")

print(f"\nUNK Predictions               : {total_unk_predictions:,}")
print(f"UNK Rate                     : {unk_rate:.4f}%")

print(f"\nMean GT Token Length          : {mean_gt_length:.2f}")
print(f"Mean Predicted Token Length  : {mean_pred_length:.2f}")

print(f"\nEvaluation Time               : {elapsed:.2f} sec")

# -------------------------------------------------------------------------
# 7. Save diagnostic CSV
# -------------------------------------------------------------------------
diagnostic_path = (
    "/content/drive/MyDrive/"
    "PFMS-SERIES-GT38-2026/"
    "PFMS_SERIES_GT38_STAGE8_TEACHER_FORCED_DIAGNOSTIC_V2.csv"
)

pd.DataFrame(all_results).to_csv(
    diagnostic_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nDiagnostic CSV saved:")
print(diagnostic_path)

print("\n" + "=" * 90)
print("STAGE 8-4 V2 COMPLETED")
print("=" * 90)

PFMS-SERIES-GT38-2026
STAGE 8-4 V2 — CORRECTED TEACHER-FORCED DIAGNOSTIC
Device: cuda
GPU: Tesla T4

CORRECTED TEACHER-FORCED DIAGNOSTIC RESULTS
Test Samples                 : 1,356
Test Loss                    : 1.542434

Valid Target Tokens          : 105,910
Correct Target Tokens        : 58,211

Token Accuracy               : 54.9627%
Non-PAD Token Accuracy       : 54.9627%

Expected EOS Positions       : 1,356
Correct EOS Predictions      : 1,356
EOS Accuracy                 : 100.0000%

Predicted EOS Tokens         : 1,356
Samples With Predicted EOS   : 1,356
Sample EOS Rate              : 100.0000%

UNK Predictions               : 0
UNK Rate                     : 0.0000%

Mean GT Token Length          : 78.10
Mean Predicted Token Length  : 78.10

Evaluation Time               : 30.37 sec

Diagnostic CSV saved:
/content/drive/MyDrive/PFMS-SERIES-GT38-2026/PFMS_SERIES_GT38_STAGE8_TEACHER_FORCED_DIAGNOSTIC_V2.csv

STAGE 8-4 V2 COMPLETED


In [ ]:
# =============================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-5A-FIX — RESTORE DATASET CLASS + DATALOADERS
# =============================================================================

import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

print("=" * 90)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-5A-FIX — RESTORE DATASET CLASS + DATALOADERS")
print("=" * 90)

# -----------------------------------------------------------------------------
# 1. Required variables check
# -----------------------------------------------------------------------------
required_vars = [
    "train_df",
    "val_df",
    "test_df",
    "char2idx",
    "PAD_IDX",
    "BOS_IDX",
    "EOS_IDX",
    "UNK_IDX"
]

missing = [
    v for v in required_vars
    if v not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing required variables: {missing}"
    )

print("Required variables: OK")

# -----------------------------------------------------------------------------
# 2. Exact Stage-7 text encoder
# -----------------------------------------------------------------------------
MAX_TEXT_LENGTH = 128

def encode_text(text, max_length=MAX_TEXT_LENGTH):

    tokens = [BOS_IDX]

    for ch in str(text):
        if ch in char2idx:
            tokens.append(char2idx[ch])
        else:
            tokens.append(UNK_IDX)

    tokens.append(EOS_IDX)

    if len(tokens) > max_length:
        tokens = tokens[:max_length]
        tokens[-1] = EOS_IDX

    if len(tokens) < max_length:
        tokens.extend(
            [PAD_IDX] * (max_length - len(tokens))
        )

    return tokens

print("encode_text: READY")

# -----------------------------------------------------------------------------
# 3. Exact Stage-7 Dataset
# -----------------------------------------------------------------------------
class IDPLOCRDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = row["image_path"]
        text = row["text"]
        image_name = row["image_name"]

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            raise RuntimeError(
                f"\nCannot read image:\n"
                f"{image_path}\n"
                f"Error: {e}"
            )

        if self.transform is not None:

            image = self.transform(image)

        target = torch.tensor(
            encode_text(text),
            dtype=torch.long
        )

        return {
            "image": image,
            "target": target,
            "text": text,
            "image_name": image_name
        }

print("IDPLOCRDataset: READY")

# -----------------------------------------------------------------------------
# 4. Exact Stage-7 transforms
# -----------------------------------------------------------------------------
IMG_SIZE = 224

train_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.RandomApply(
        [
            transforms.ColorJitter(
                brightness=0.10,
                contrast=0.10
            )
        ],
        p=0.30
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms: READY")

# -----------------------------------------------------------------------------
# 5. Create datasets
# -----------------------------------------------------------------------------
train_dataset = IDPLOCRDataset(
    train_df,
    transform=train_transform
)

val_dataset = IDPLOCRDataset(
    val_df,
    transform=eval_transform
)

test_dataset = IDPLOCRDataset(
    test_df,
    transform=eval_transform
)

print("\nDataset sizes:")
print(f"Train Dataset: {len(train_dataset):,}")
print(f"Val Dataset  : {len(val_dataset):,}")
print(f"Test Dataset : {len(test_dataset):,}")

# -----------------------------------------------------------------------------
# 6. Create DataLoaders
# -----------------------------------------------------------------------------
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# -----------------------------------------------------------------------------
# 7. Sanity check — read one batch
# -----------------------------------------------------------------------------
print("\nReading one training batch...")

batch = next(iter(train_loader))

print(
    f"Train images : "
    f"{tuple(batch['image'].shape)}"
)

print(
    f"Train targets: "
    f"{tuple(batch['target'].shape)}"
)

print(
    f"Batch size   : "
    f"{batch['image'].size(0)}"
)

print(
    f"First image  : "
    f"{batch['image_name'][0]}"
)

print(
    f"First text   : "
    f"{batch['text'][0][:100]}"
)

# -----------------------------------------------------------------------------
# 8. Validation batch
# -----------------------------------------------------------------------------
val_batch = next(iter(val_loader))

print(
    f"\nValidation images : "
    f"{tuple(val_batch['image'].shape)}"
)

print(
    f"Validation targets: "
    f"{tuple(val_batch['target'].shape)}"
)

# -----------------------------------------------------------------------------
# 9. Final checks
# -----------------------------------------------------------------------------
assert len(train_dataset) == 23052
assert len(val_dataset) == 2712
assert len(test_dataset) == 1356

assert batch["image"].shape[1:] == (3, 224, 224)
assert batch["target"].shape[1] == 128

print("\n" + "=" * 90)
print("STAGE 8-5A-FIX COMPLETED SUCCESSFULLY")
print("=" * 90)

print("IDPLOCRDataset : READY")
print("train_loader   : READY")
print("val_loader     : READY")
print("test_loader    : READY")

print("\nNO TRAINING WAS PERFORMED.")
print("MODEL CHECKPOINT WAS NOT MODIFIED.")

print("=" * 90)

PFMS-SERIES-GT38-2026
STAGE 8-5A-FIX — RESTORE DATASET CLASS + DATALOADERS
Required variables: OK
encode_text: READY
IDPLOCRDataset: READY
Transforms: READY

Dataset sizes:
Train Dataset: 23,052
Val Dataset  : 2,712
Test Dataset : 1,356

Reading one training batch...
Train images : (4, 3, 224, 224)
Train targets: (4, 128)
Batch size   : 4
First image  : 09382.tif
First text   : استقلال الجزایر از فرانسه در سال ۱۳۴۱ شمسی دولت ایران استقلال این کشور را به


Validation images : (4, 3, 224, 224)
Validation targets: (4, 128)

STAGE 8-5A-FIX COMPLETED SUCCESSFULLY
IDPLOCRDataset : READY
train_loader   : READY
val_loader     : READY
test_loader    : READY

NO TRAINING WAS PERFORMED.
MODEL CHECKPOINT WAS NOT MODIFIED.


In [ ]:
# ==================================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-5B — RESTORE EXACT MODEL + DATALOADERS + CONTINUE TRAINING
#
# Start : Epoch 10 checkpoint
# Train : Epoch 11 -> Epoch 20
#
# Architecture:
# Swin-B -> Projection 1024->512 -> 4x Mamba Sequence Mixer -> Transformer OCR Decoder
# ==================================================================================================

import os
import math
import random
import numpy as np
import pandas as pd

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import swin_b, Swin_B_Weights


# ==================================================================================================
# 1. ENVIRONMENT
# ==================================================================================================

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-5B — RESTORE EXACT MODEL + DATALOADERS + CONTINUE TRAINING")
print("=" * 100)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDEVICE")
print("-" * 100)
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)
print("Device  :", device)

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))


# ==================================================================================================
# 2. GOOGLE DRIVE
# ==================================================================================================

from google.colab import drive

try:
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print("Drive mount message:", e)

print("\n✓ Google Drive ready")


# ==================================================================================================
# 3. PATHS
# ==================================================================================================

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "PFMS-SERIES-GT38-2026"
)

GT_CSV = (
    "/content/drive/MyDrive/"
    "IDPL_PFOD_REAL_GT_MAPPING.csv"
)

EXACT_SPLIT_CSV = os.path.join(
    PROJECT_DIR,
    "PFMS_SERIES_GT38_STAGE7_EXACT_SPLIT.csv"
)

VOCAB_PATH = (
    "/content/drive/MyDrive/"
    "idpl_vocab.pt"
)

BEST_MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "best_pfms_series_swin_mamba_trocr.pth"
)

LAST_MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "last_pfms_series_swin_mamba_trocr.pth"
)

HISTORY_PATH = os.path.join(
    PROJECT_DIR,
    "training_history.csv"
)

IMAGE_ROOT = (
    "/content/drive/Othercomputers/"
    "My Laptop/Desktop/idplimgl"
)

print("\nPATHS")
print("-" * 100)
print("PROJECT_DIR      :", PROJECT_DIR)
print("GT_CSV           :", GT_CSV)
print("EXACT_SPLIT_CSV  :", EXACT_SPLIT_CSV)
print("VOCAB_PATH       :", VOCAB_PATH)
print("BEST_MODEL_PATH  :", BEST_MODEL_PATH)
print("LAST_MODEL_PATH  :", LAST_MODEL_PATH)
print("HISTORY_PATH     :", HISTORY_PATH)
print("IMAGE_ROOT       :", IMAGE_ROOT)


# ==================================================================================================
# 4. REQUIRED FILE CHECK
# ==================================================================================================

required_paths = {
    "PROJECT_DIR": PROJECT_DIR,
    "GT_CSV": GT_CSV,
    "EXACT_SPLIT_CSV": EXACT_SPLIT_CSV,
    "VOCAB_PATH": VOCAB_PATH,
    "BEST_MODEL_PATH": BEST_MODEL_PATH,
    "IMAGE_ROOT": IMAGE_ROOT,
}

print("\nREQUIRED FILE CHECK")
print("-" * 100)

for name, path in required_paths.items():

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\nRequired path not found:\n{name}\n{path}"
        )

    print(f"✓ {name:18s} -> {path}")

print("\n✓ All required paths exist.")


# ==================================================================================================
# 5. LOAD VOCABULARY
# ==================================================================================================

print("\n" + "=" * 100)
print("VOCABULARY")
print("=" * 100)

vocab_data = torch.load(
    VOCAB_PATH,
    map_location="cpu"
)

char2idx = vocab_data["char2idx"]
idx2char = vocab_data["idx2char"]

PAD_IDX = int(vocab_data["PAD_IDX"])
BOS_IDX = int(vocab_data["BOS_IDX"])
EOS_IDX = int(vocab_data["EOS_IDX"])
UNK_IDX = int(vocab_data["UNK_IDX"])

VOCAB_SIZE = int(
    vocab_data.get(
        "vocab_size",
        len(char2idx)
    )
)

MAX_TEXT_LENGTH = 128

print("VOCAB_SIZE :", VOCAB_SIZE)
print("len(char2idx):", len(char2idx))
print("len(idx2char):", len(idx2char))
print("PAD_IDX    :", PAD_IDX)
print("BOS_IDX    :", BOS_IDX)
print("EOS_IDX    :", EOS_IDX)
print("UNK_IDX    :", UNK_IDX)

assert VOCAB_SIZE == 169
assert len(char2idx) == 169
assert len(idx2char) == 169
assert PAD_IDX == 0
assert BOS_IDX == 1
assert EOS_IDX == 2
assert UNK_IDX == 3

print("✓ Vocabulary verified")


# ==================================================================================================
# 6. LOAD EXACT STAGE-7 SPLIT
# ==================================================================================================

print("\n" + "=" * 100)
print("EXACT STAGE-7 SPLIT")
print("=" * 100)

split_df = pd.read_csv(
    EXACT_SPLIT_CSV
)

print("Columns:", split_df.columns.tolist())
print("Total rows:", len(split_df))

required_split_columns = [
    "image_path",
    "image_name",
    "text",
    "split"
]

for col in required_split_columns:

    if col not in split_df.columns:
        raise RuntimeError(
            f"Required column '{col}' is missing from exact split CSV."
        )

# Normalize
split_df["image_path"] = (
    split_df["image_path"]
    .fillna("")
    .astype(str)
)

split_df["image_name"] = (
    split_df["image_name"]
    .fillna("")
    .astype(str)
)

split_df["text"] = (
    split_df["text"]
    .fillna("")
    .astype(str)
)

split_df["split"] = (
    split_df["split"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

print("\nOriginal exact split counts:")
print(split_df["split"].value_counts())


# ==================================================================================================
# 7. VERIFY EXACT COUNTS
# ==================================================================================================

train_df = (
    split_df[
        split_df["split"] == "train"
    ]
    .reset_index(drop=True)
)

val_df = (
    split_df[
        split_df["split"].isin(
            ["val", "validation"]
        )
    ]
    .reset_index(drop=True)
)

test_df = (
    split_df[
        split_df["split"] == "test"
    ]
    .reset_index(drop=True)
)

print("\nExact Stage-7 split:")
print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))
print("Total      :", len(split_df))

assert len(train_df) == 23052, (
    f"Train split mismatch: {len(train_df)}"
)

assert len(val_df) == 2712, (
    f"Validation split mismatch: {len(val_df)}"
)

assert len(test_df) == 1356, (
    f"Test split mismatch: {len(test_df)}"
)

assert len(split_df) == 27120

print("\n✓ Exact Stage-7 split verified")
print("✓ Train      = 23,052")
print("✓ Validation = 2,712")
print("✓ Test       = 1,356")
print("✓ Total      = 27,120")


# ==================================================================================================
# 8. VERIFY IMAGES
# ==================================================================================================

print("\n" + "=" * 100)
print("VERIFYING IMAGE FILES")
print("=" * 100)

all_paths = split_df["image_path"].tolist()

missing = [
    p for p in all_paths
    if not os.path.exists(p)
]

print("Total images :", len(all_paths))
print("Missing      :", len(missing))

if missing:
    print("\nFirst missing images:")

    for p in missing[:10]:
        print(p)

    raise RuntimeError(
        f"{len(missing)} image files are missing."
    )

print("✓ All 27,120 images exist")


# ==================================================================================================
# 9. TEXT ENCODER
# ==================================================================================================

def encode_text(
    text,
    max_length=MAX_TEXT_LENGTH
):

    tokens = [
        BOS_IDX
    ]

    for ch in str(text):

        if ch in char2idx:
            tokens.append(
                char2idx[ch]
            )
        else:
            tokens.append(
                UNK_IDX
            )

    tokens.append(
        EOS_IDX
    )

    if len(tokens) > max_length:

        tokens = tokens[:max_length]

        # Always preserve EOS
        tokens[-1] = EOS_IDX

    if len(tokens) < max_length:

        tokens.extend(
            [PAD_IDX] *
            (
                max_length -
                len(tokens)
            )
        )

    return tokens


print("\n✓ encode_text ready")


# ==================================================================================================
# 10. EXACT STAGE-7 IMAGE TRANSFORMS
# ==================================================================================================

IMG_SIZE = 224

train_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.RandomApply(
        [
            transforms.ColorJitter(
                brightness=0.10,
                contrast=0.10
            )
        ],
        p=0.30
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

eval_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

print("✓ Train transform ready")
print("✓ Evaluation transform ready")


# ==================================================================================================
# 11. EXACT STAGE-7 DATASET CLASS
# ==================================================================================================

class IDPLOCRDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        image_path = row["image_path"]
        text = row["text"]
        image_name = row["image_name"]

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            raise RuntimeError(
                f"\nCannot read image:\n"
                f"{image_path}\n"
                f"Error: {e}"
            )

        if self.transform is not None:

            image = self.transform(
                image
            )

        target = torch.tensor(
            encode_text(text),
            dtype=torch.long
        )

        return {
            "image": image,
            "target": target,
            "text": text,
            "image_name": image_name
        }


print("✓ IDPLOCRDataset defined")


# ==================================================================================================
# 12. RESTORE EXACT STAGE-7 DATALOADERS
# ==================================================================================================

BATCH_SIZE = 4

train_dataset = IDPLOCRDataset(
    train_df,
    train_transform
)

val_dataset = IDPLOCRDataset(
    val_df,
    eval_transform
)

test_dataset = IDPLOCRDataset(
    test_df,
    eval_transform
)

# num_workers=0 is intentionally used here
# to avoid Colab multiprocessing worker errors.

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("\n" + "=" * 100)
print("DATALOADERS")
print("=" * 100)

print("Train samples :", len(train_dataset))
print("Val samples   :", len(val_dataset))
print("Test samples  :", len(test_dataset))

print("Batch size    :", BATCH_SIZE)
print("Train batches :", len(train_loader))
print("Val batches   :", len(val_loader))
print("Test batches  :", len(test_loader))

assert len(train_dataset) == 23052
assert len(val_dataset) == 2712
assert len(test_dataset) == 1356

print("\n✓ train_loader READY")
print("✓ val_loader READY")
print("✓ test_loader READY")


# ==================================================================================================
# 13. EXACT STAGE-7 MAMBA SEQUENCE MIXER
# ==================================================================================================

class MambaSequenceMixer(
    nn.Module
):

    def __init__(
        self,
        dim,
        expansion=2,
        dropout=0.1
    ):

        super().__init__()

        hidden = dim * expansion

        self.norm = nn.LayerNorm(
            dim
        )

        self.in_proj = nn.Linear(
            dim,
            hidden * 2
        )

        self.depthwise = nn.Conv1d(
            hidden,
            hidden,
            kernel_size=5,
            padding=2,
            groups=hidden
        )

        self.activation = nn.SiLU()

        self.out_proj = nn.Linear(
            hidden,
            dim
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x
    ):

        residual = x

        x = self.norm(x)

        x = self.in_proj(x)

        a, b = x.chunk(
            2,
            dim=-1
        )

        a = a.transpose(
            1,
            2
        )

        a = self.depthwise(a)

        a = a.transpose(
            1,
            2
        )

        a = self.activation(a)

        x = a * torch.sigmoid(b)

        x = self.out_proj(x)

        x = self.dropout(x)

        return residual + x


print("\n✓ MambaSequenceMixer defined")


# ==================================================================================================
# 14. EXACT STAGE-7 TRANSFORMER OCR DECODER
# ==================================================================================================

class TransformerOCRDecoder(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        d_model=512,
        nhead=8,
        num_layers=4,
        dim_feedforward=2048,
        dropout=0.1,
        max_length=128
    ):

        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_length = max_length

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_IDX
        )

        self.pos_embedding = nn.Parameter(
            torch.randn(
                1,
                max_length,
                d_model
            ) * 0.02
        )

        decoder_layer = (
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                batch_first=True,
                norm_first=True
            )
        )

        self.decoder = (
            nn.TransformerDecoder(
                decoder_layer,
                num_layers=num_layers
            )
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        self.output = nn.Linear(
            d_model,
            vocab_size
        )

    def causal_mask(
        self,
        length,
        device
    ):

        mask = torch.triu(
            torch.ones(
                length,
                length,
                device=device
            ),
            diagonal=1
        )

        mask = mask.masked_fill(
            mask == 1,
            float("-inf")
        )

        return mask

    def forward(
        self,
        memory,
        target
    ):

        B, T = target.shape

        target_emb = self.embedding(
            target
        )

        target_emb = (
            target_emb +
            self.pos_embedding[
                :, :T, :
            ]
        )

        causal_mask = self.causal_mask(
            T,
            target.device
        )

        padding_mask = (
            target == PAD_IDX
        )

        decoded = self.decoder(
            tgt=target_emb,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=padding_mask
        )

        decoded = self.norm(
            decoded
        )

        logits = self.output(
            decoded
        )

        return logits


print("✓ TransformerOCRDecoder defined")


# ==================================================================================================
# 15. EXACT STAGE-7 PFMS SERIES MODEL
# ==================================================================================================

class PFMS_Series_OCR(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        d_model=512
    ):

        super().__init__()

        print("\nLoading pretrained Swin-B...")

        weights = (
            Swin_B_Weights.IMAGENET1K_V1
        )

        self.swin = swin_b(
            weights=weights
        )

        # Remove classification head
        self.swin.head = nn.Identity()

        swin_dim = 1024

        self.projection = nn.Linear(
            swin_dim,
            d_model
        )

        # EXACTLY 4 serial Mamba blocks
        self.mamba = nn.ModuleList([

            MambaSequenceMixer(
                d_model,
                expansion=2,
                dropout=0.1
            )

            for _ in range(4)
        ])

        self.decoder = (
            TransformerOCRDecoder(
                vocab_size=vocab_size,
                d_model=d_model,
                nhead=8,
                num_layers=4,
                dim_feedforward=2048,
                dropout=0.1,
                max_length=MAX_TEXT_LENGTH
            )
        )

    def extract_swin_features(
        self,
        x
    ):

        x = self.swin.features(x)

        if x.ndim == 4:

            B, H, W, C = x.shape

            x = x.reshape(
                B,
                H * W,
                C
            )

        return x

    def encode(
        self,
        images
    ):

        x = self.extract_swin_features(
            images
        )

        x = self.projection(
            x
        )

        for block in self.mamba:

            x = block(x)

        return x

    def forward(
        self,
        images,
        target_input
    ):

        memory = self.encode(
            images
        )

        logits = self.decoder(
            memory,
            target_input
        )

        return logits


print("✓ PFMS_Series_OCR defined")


# ==================================================================================================
# 16. CREATE EXACT MODEL
# ==================================================================================================

print("\n" + "=" * 100)
print("CREATING EXACT STAGE-7 MODEL")
print("=" * 100)

model = PFMS_Series_OCR(
    vocab_size=VOCAB_SIZE,
    d_model=512
)

model = model.to(device)

print("✓ Model created")
print("Device:", next(model.parameters()).device)


# ==================================================================================================
# 17. FREEZE SWIN — SAME AS STAGE 7
# ==================================================================================================

for param in model.swin.parameters():

    param.requires_grad = False

print("\nSwin-B status: FROZEN")


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "Total parameters     :",
    f"{total_params:,}"
)

print(
    "Trainable parameters :",
    f"{trainable_params:,}"
)

assert trainable_params == 23911081

print("✓ Trainable parameter count matches Stage 7")


# ==================================================================================================
# 18. LOSS
# ==================================================================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

print("\n✓ CrossEntropyLoss ready")


# ==================================================================================================
# 19. LOAD BEST CHECKPOINT
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING BEST CHECKPOINT")
print("=" * 100)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)

print("Checkpoint Epoch       :", checkpoint["epoch"])
print(
    "Previous Train Loss    :",
    checkpoint.get("train_loss")
)
print(
    "Previous Val Loss      :",
    checkpoint.get("val_loss")
)
print(
    "Previous Best Val Loss :",
    checkpoint.get("best_val_loss")
)

# Verify vocabulary
assert checkpoint["vocab_size"] == VOCAB_SIZE
assert checkpoint["PAD_IDX"] == PAD_IDX
assert checkpoint["BOS_IDX"] == BOS_IDX
assert checkpoint["EOS_IDX"] == EOS_IDX
assert checkpoint["UNK_IDX"] == UNK_IDX

print("✓ Checkpoint vocabulary verified")


# ==================================================================================================
# 20. STRICT MODEL RESTORATION
# ==================================================================================================

missing_keys, unexpected_keys = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=False
)

if missing_keys:
    raise RuntimeError(
        f"Missing model keys: {missing_keys}"
    )

if unexpected_keys:
    raise RuntimeError(
        f"Unexpected model keys: {unexpected_keys}"
    )

print("✓ Model state restored")
print("✓ No missing keys")
print("✓ No unexpected keys")


# ==================================================================================================
# 21. OPTIMIZER — SAME AS STAGE 7
# ==================================================================================================

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-2

optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)


# ==================================================================================================
# 22. RESTORE OPTIMIZER + SCHEDULER
# ==================================================================================================

if "optimizer_state_dict" in checkpoint:

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    print("✓ Optimizer state RESTORED")

else:

    print(
        "⚠ Optimizer state not found."
        " A fresh optimizer will be used."
    )


if "scheduler_state_dict" in checkpoint:

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    print("✓ Scheduler state RESTORED")

else:

    print(
        "⚠ Scheduler state not found."
        " A fresh scheduler will be used."
    )


# ==================================================================================================
# 23. VERIFY TRAINING STATE
# ==================================================================================================

START_EPOCH = int(
    checkpoint["epoch"]
) + 1

TARGET_EPOCH = 20

best_val_loss = float(
    checkpoint.get(
        "best_val_loss",
        checkpoint.get(
            "val_loss",
            float("inf")
        )
    )
)

print("\n" + "=" * 100)
print("TRAINING STATE")
print("=" * 100)

print("Checkpoint epoch :", checkpoint["epoch"])
print("Start epoch      :", START_EPOCH)
print("Target epoch     :", TARGET_EPOCH)
print("Best val loss    :", best_val_loss)
print("Learning rate    :", optimizer.param_groups[0]["lr"])

assert START_EPOCH == 11
assert TARGET_EPOCH == 20

print("\n✓ Training will continue from Epoch 11")


# ==================================================================================================
# 24. TRAINING FUNCTION
# ==================================================================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0.0

    progress = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for batch in progress:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        targets = batch["target"].to(
            device,
            non_blocking=True
        )

        decoder_input = targets[:, :-1]

        expected_output = targets[:, 1:]

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            images,
            decoder_input
        )

        loss = criterion(
            logits.reshape(
                -1,
                VOCAB_SIZE
            ),
            expected_output.reshape(
                -1
            )
        )

        if not torch.isfinite(loss):

            raise RuntimeError(
                f"Non-finite training loss: {loss.item()}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return (
        total_loss /
        len(loader)
    )


# ==================================================================================================
# 25. VALIDATION FUNCTION
# ==================================================================================================

@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0

    progress = tqdm(
        loader,
        desc="Validation",
        leave=False
    )

    for batch in progress:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        targets = batch["target"].to(
            device,
            non_blocking=True
        )

        decoder_input = targets[:, :-1]

        expected_output = targets[:, 1:]

        logits = model(
            images,
            decoder_input
        )

        loss = criterion(
            logits.reshape(
                -1,
                VOCAB_SIZE
            ),
            expected_output.reshape(
                -1
            )
        )

        if not torch.isfinite(loss):

            raise RuntimeError(
                f"Non-finite validation loss: {loss.item()}"
            )

        total_loss += loss.item()

    return (
        total_loss /
        len(loader)
    )


# ==================================================================================================
# 26. CONTINUED REAL TRAINING
# ==================================================================================================

print("\n")
print("=" * 100)
print("🚀 STAGE 8-5B — CONTINUED REAL OCR TRAINING")
print("=" * 100)

print("\nArchitecture:")
print(
    "Swin-B → Mamba Sequence Mixer → "
    "Transformer OCR Decoder"
)

print("\nDataset:")
print("Train      :", len(train_dataset))
print("Validation :", len(val_dataset))
print("Test       :", len(test_dataset))

print("\nVocabulary:", VOCAB_SIZE)

print("\nTraining:")
print("Start Epoch :", START_EPOCH)
print("Target Epoch:", TARGET_EPOCH)

print("\n" + "=" * 100)


history_new = []


for epoch in range(
    START_EPOCH,
    TARGET_EPOCH + 1
):

    print(
        f"\n{'=' * 35}"
    )

    print(
        f"EPOCH {epoch}/{TARGET_EPOCH}"
    )

    print(
        f"{'=' * 35}"
    )

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss = validate(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(
        val_loss
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"\nEpoch {epoch}/{TARGET_EPOCH}"
    )

    print(
        f"Train Loss : {train_loss:.6f}"
    )

    print(
        f"Val Loss   : {val_loss:.6f}"
    )

    print(
        f"LR         : {current_lr:.2e}"
    )

    history_new.append({

        "epoch": epoch,

        "train_loss": train_loss,

        "val_loss": val_loss,

        "learning_rate": current_lr

    })


    # ==============================================================================================
    # SAVE LAST CHECKPOINT
    # ==============================================================================================

    torch.save(
        {
            "epoch": epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "train_loss":
                train_loss,

            "val_loss":
                val_loss,

            "best_val_loss":
                best_val_loss,

            "vocab_size":
                VOCAB_SIZE,

            "char2idx":
                char2idx,

            "idx2char":
                idx2char,

            "PAD_IDX":
                PAD_IDX,

            "BOS_IDX":
                BOS_IDX,

            "EOS_IDX":
                EOS_IDX,

            "UNK_IDX":
                UNK_IDX
        },
        LAST_MODEL_PATH
    )

    print(
        "✓ Last checkpoint saved"
    )

    print(
        LAST_MODEL_PATH
    )


    # ==============================================================================================
    # SAVE BEST CHECKPOINT
    # ==============================================================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    scheduler.state_dict(),

                "train_loss":
                    train_loss,

                "val_loss":
                    val_loss,

                "best_val_loss":
                    best_val_loss,

                "vocab_size":
                    VOCAB_SIZE,

                "char2idx":
                    char2idx,

                "idx2char":
                    idx2char,

                "PAD_IDX":
                    PAD_IDX,

                "BOS_IDX":
                    BOS_IDX,

                "EOS_IDX":
                    EOS_IDX,

                "UNK_IDX":
                    UNK_IDX
            },
            BEST_MODEL_PATH
        )

        print(
            "\n✓ NEW BEST MODEL SAVED"
        )

        print(
            BEST_MODEL_PATH
        )

    else:

        print(
            "\nNo new best validation loss."
        )


# ==================================================================================================
# 27. SAVE CONTINUED TRAINING HISTORY
# ==================================================================================================

history_new_df = pd.DataFrame(
    history_new
)

if os.path.exists(HISTORY_PATH):

    old_history = pd.read_csv(
        HISTORY_PATH
    )

    combined_history = pd.concat(
        [
            old_history,
            history_new_df
        ],
        ignore_index=True
    )

else:

    combined_history = history_new_df


combined_history = (
    combined_history
    .drop_duplicates(
        subset=["epoch"],
        keep="last"
    )
    .sort_values("epoch")
    .reset_index(drop=True)
)

combined_history.to_csv(
    HISTORY_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ==================================================================================================
# 28. FINAL STATUS
# ==================================================================================================

print("\n")
print("=" * 100)
print("STAGE 8-5B — COMPLETED")
print("=" * 100)

print("\nTraining completed through epoch:", TARGET_EPOCH)

print(
    "Best validation loss:",
    best_val_loss
)

print("\nSaved:")
print("BEST MODEL :")
print(BEST_MODEL_PATH)

print("\nLAST MODEL :")
print(LAST_MODEL_PATH)

print("\nHISTORY :")
print(HISTORY_PATH)

print("\n" + "=" * 100)
print("✓ Exact Stage-7 model restored")
print("✓ Exact Stage-7 split restored")
print("✓ train_loader restored")
print("✓ val_loader restored")
print("✓ test_loader restored")
print("✓ Optimizer restored")
print("✓ Scheduler restored")
print("✓ Continued training completed")
print("✓ Checkpoints saved")
print("=" * 100)

PFMS-SERIES-GT38-2026
STAGE 8-5B — RESTORE EXACT MODEL + DATALOADERS + CONTINUE TRAINING

DEVICE
----------------------------------------------------------------------------------------------------
PyTorch : 2.11.0+cu128
CUDA    : 12.8
Device  : cuda
GPU     : Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✓ Google Drive ready

PATHS
----------------------------------------------------------------------------------------------------
PROJECT_DIR      : /content/drive/MyDrive/PFMS-SERIES-GT38-2026
GT_CSV           : /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv
EXACT_SPLIT_CSV  : /content/drive/MyDrive/PFMS-SERIES-GT38-2026/PFMS_SERIES_GT38_STAGE7_EXACT_SPLIT.csv
VOCAB_PATH       : /content/drive/MyDrive/idpl_vocab.pt
BEST_MODEL_PATH  : /content/drive/MyDrive/PFMS-SERIES-GT38-2026/best_pfms_series_swin_mamba_trocr.pth
LAST_MODEL_PATH  : /content/drive/MyDrive/PFMS-SERIES-GT38-2026/last_pfms

RuntimeError: Required column 'image_path' is missing from exact split CSV.

In [ ]:
# ================================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-5B-RECOVERY-01 — INSPECT EXACT SPLIT MAPPING
# NO TRAINING / NO CHECKPOINT MODIFICATION
# ================================================================================================

import os
import pandas as pd
import numpy as np

GT_CSV = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"

SPLIT_CSV = (
    "/content/drive/MyDrive/PFMS-SERIES-GT38-2026/"
    "PFMS_SERIES_GT38_STAGE7_EXACT_SPLIT.csv"
)

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-5B-RECOVERY-01 — INSPECT EXACT SPLIT MAPPING")
print("=" * 100)

# --------------------------------------------------------------------------------
# Load files
# --------------------------------------------------------------------------------

gt = pd.read_csv(GT_CSV)
exact_split = pd.read_csv(SPLIT_CSV)

print("\nGT CSV")
print("-" * 100)
print("Rows:", len(gt))
print("Columns:", gt.columns.tolist())

print("\nEXACT SPLIT CSV")
print("-" * 100)
print("Rows:", len(exact_split))
print("Columns:", exact_split.columns.tolist())

print("\nFirst 10 rows of EXACT SPLIT:")
print(exact_split.head(10).to_string(index=False))

# --------------------------------------------------------------------------------
# Split statistics
# --------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("EXACT SPLIT STATISTICS")
print("=" * 100)

if "split" in exact_split.columns:
    print(exact_split["split"].value_counts(dropna=False))
else:
    print("WARNING: split column is missing.")

# --------------------------------------------------------------------------------
# Inspect original_df_index
# --------------------------------------------------------------------------------

if "original_df_index" in exact_split.columns:

    idx = exact_split["original_df_index"]

    print("\n" + "=" * 100)
    print("original_df_index INSPECTION")
    print("=" * 100)

    print("dtype:", idx.dtype)
    print("min:", idx.min())
    print("max:", idx.max())
    print("unique:", idx.nunique())
    print("NaN:", idx.isna().sum())

    print("\nFirst 20 original_df_index values:")
    print(idx.head(20).tolist())

    # Check whether these indices can directly index GT
    valid_range = (
        idx.notna().all()
        and idx.astype(int).min() >= 0
        and idx.astype(int).max() < len(gt)
    )

    print("\nCan original_df_index directly index GT?")
    print("Result:", valid_range)

    if valid_range:

        sample_indices = idx.head(10).astype(int).tolist()

        print("\nGT rows corresponding to first 10 split indices:")
        print(
            gt.iloc[sample_indices][
                ["split", "hf_index", "image_path", "image_name", "text"]
            ].to_string(index=True)
        )

else:
    print("\nERROR: original_df_index is missing from Exact Split CSV.")

print("\n" + "=" * 100)
print("RECOVERY INSPECTION FINISHED")
print("NO TRAINING PERFORMED")
print("NO CHECKPOINT MODIFIED")
print("=" * 100)

PFMS-SERIES-GT38-2026
STAGE 8-5B-RECOVERY-01 — INSPECT EXACT SPLIT MAPPING


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv'

In [25]:
# ================================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-5B-FINAL — RESTORE EXACT STAGE-7 SPLIT
# ================================================================================================
# هدف:
#   بازسازی دقیق train / validation / test همان Stage 7
#
# منبع:
#   original_df_index در Exact Split CSV مستقیماً به ردیف GT اصلی اشاره می‌کند.
#
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# ================================================================================================

import os
import pandas as pd
import numpy as np

GT_CSV = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"

SPLIT_CSV = (
    "/content/drive/MyDrive/PFMS-SERIES-GT38-2026/"
    "PFMS_SERIES_GT38_STAGE7_EXACT_SPLIT.csv"
)

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-5B-FINAL — RESTORE EXACT STAGE-7 SPLIT")
print("=" * 100)

# --------------------------------------------------------------------------------
# 1. Load original GT
# --------------------------------------------------------------------------------

df = pd.read_csv(GT_CSV)

print("\nOriginal GT loaded")
print("-" * 100)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

# --------------------------------------------------------------------------------
# 2. Load exact Stage-7 split
# --------------------------------------------------------------------------------

exact_split = pd.read_csv(SPLIT_CSV)

print("\nExact Stage-7 split loaded")
print("-" * 100)
print("Rows:", len(exact_split))
print("Columns:", exact_split.columns.tolist())

# --------------------------------------------------------------------------------
# 3. Verify required columns
# --------------------------------------------------------------------------------

required_gt = [
    "split",
    "hf_index",
    "image_path",
    "image_name",
    "text"
]

required_split = [
    "original_df_index",
    "split"
]

for col in required_gt:
    if col not in df.columns:
        raise RuntimeError(f"Missing GT column: {col}")

for col in required_split:
    if col not in exact_split.columns:
        raise RuntimeError(f"Missing Exact Split column: {col}")

# --------------------------------------------------------------------------------
# 4. Verify exact indexing
# --------------------------------------------------------------------------------

indices = exact_split["original_df_index"]

if indices.isna().any():
    raise RuntimeError("original_df_index contains NaN.")

if indices.nunique() != len(df):
    raise RuntimeError(
        f"Exact split does not contain exactly one index for every GT row. "
        f"Unique indices={indices.nunique()}, GT rows={len(df)}"
    )

if indices.min() != 0 or indices.max() != len(df) - 1:
    raise RuntimeError(
        "original_df_index does not cover the complete GT index range."
    )

print("\n✓ original_df_index covers exactly 0 ...", len(df) - 1)

# --------------------------------------------------------------------------------
# 5. Restore rows using original_df_index
# --------------------------------------------------------------------------------

restored = df.iloc[indices.to_numpy()].copy().reset_index(drop=True)

# Add the exact Stage-7 split
restored["stage7_split"] = exact_split["split"].to_numpy()

# --------------------------------------------------------------------------------
# 6. Verify restored identity
# --------------------------------------------------------------------------------

restored["original_df_index"] = indices.to_numpy()

print("\n" + "=" * 100)
print("RESTORED SPLIT COUNTS")
print("=" * 100)

counts = restored["stage7_split"].value_counts()

print(counts)

expected = {
    "train": 23052,
    "validation": 2712,
    "test": 1356
}

for split_name, expected_count in expected.items():

    actual_count = int(counts.get(split_name, 0))

    if actual_count != expected_count:
        raise RuntimeError(
            f"{split_name} count mismatch: "
            f"expected={expected_count}, actual={actual_count}"
        )

print("\n✓ Train count verified      :", expected["train"])
print("✓ Validation count verified :", expected["validation"])
print("✓ Test count verified       :", expected["test"])

# --------------------------------------------------------------------------------
# 7. Create exact dataframes
# --------------------------------------------------------------------------------

train_df = (
    restored[restored["stage7_split"] == "train"]
    .reset_index(drop=True)
)

val_df = (
    restored[restored["stage7_split"] == "validation"]
    .reset_index(drop=True)
)

test_df = (
    restored[restored["stage7_split"] == "test"]
    .reset_index(drop=True)
)

# --------------------------------------------------------------------------------
# 8. Verify image paths
# --------------------------------------------------------------------------------

missing_train = (~train_df["image_path"].map(os.path.exists)).sum()
missing_val = (~val_df["image_path"].map(os.path.exists)).sum()
missing_test = (~test_df["image_path"].map(os.path.exists)).sum()

print("\n" + "=" * 100)
print("IMAGE EXISTENCE CHECK")
print("=" * 100)

print("Train missing      :", missing_train)
print("Validation missing :", missing_val)
print("Test missing       :", missing_test)

if missing_train or missing_val or missing_test:
    raise RuntimeError("Some images are missing.")

# --------------------------------------------------------------------------------
# 9. Final verification
# --------------------------------------------------------------------------------

all_indices = np.concatenate([
    train_df["original_df_index"].to_numpy(),
    val_df["original_df_index"].to_numpy(),
    test_df["original_df_index"].to_numpy()
])

if len(all_indices) != 27120:
    raise RuntimeError("Total restored sample count is incorrect.")

if len(np.unique(all_indices)) != 27120:
    raise RuntimeError("Duplicate or overlapping samples detected.")

if set(all_indices) != set(range(27120)):
    raise RuntimeError("The restored split does not cover all GT samples exactly.")

# --------------------------------------------------------------------------------
# 10. Display examples
# --------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("RESTORED DATASET SUMMARY")
print("=" * 100)

print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))
print("Total      :", len(train_df) + len(val_df) + len(test_df))

print("\nExample TRAIN:")
print(
    train_df[
        ["original_df_index", "image_name", "text"]
    ].head(2).to_string(index=False)
)

print("\nExample VALIDATION:")
print(
    val_df[
        ["original_df_index", "image_name", "text"]
    ].head(2).to_string(index=False)
)

print("\nExample TEST:")
print(
    test_df[
        ["original_df_index", "image_name", "text"]
    ].head(2).to_string(index=False)
)

print("\n" + "=" * 100)
print("✓ STAGE 8-5B-FINAL COMPLETED SUCCESSFULLY")
print("=" * 100)

print("✓ Exact Stage-7 split restored")
print("✓ Train = 23,052")
print("✓ Validation = 2,712")
print("✓ Test = 1,356")
print("✓ Total = 27,120")
print("✓ No duplicate samples")
print("✓ No overlap between splits")
print("✓ All image files exist")
print("✓ No training performed")
print("✓ No checkpoint modified")

print("\nNEXT STEP:")
print("Recover and restore the EXACT Stage-7 model architecture.")
print("=" * 100)

PFMS-SERIES-GT38-2026
STAGE 8-5B-FINAL — RESTORE EXACT STAGE-7 SPLIT


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv'

In [ ]:
# ================================================================================================
# PFMS-SERIES-GT38-2026
# STAGE 8-5C-RECOVERY — FIND EXACT STAGE-7 MODEL DEFINITION
# ================================================================================================
# هدف:
#   پیدا کردن کد واقعی معماری Stage 7 از Notebook اصلی پروژه
#
# جستجو برای:
#   MambaSequenceMixer
#   TransformerOCRDecoder
#   PFMS_Series_OCR
#
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# ================================================================================================

import os
import json

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("STAGE 8-5C-RECOVERY — FIND EXACT STAGE-7 MODEL DEFINITION")
print("=" * 100)

# --------------------------------------------------------------------------------
# Candidate notebooks
# --------------------------------------------------------------------------------

notebook_paths = [
    "/content/drive/MyDrive/Colab Notebooks/Copy of PFMS-Net-Series-Final.ipynb",
    "/content/drive/MyDrive/Colab Notebooks/PFMS-Net-Series-Final.ipynb",
]

found_notebooks = []

print("\nChecking notebooks...")
print("-" * 100)

for path in notebook_paths:
    exists = os.path.exists(path)

    print(f"{'✓' if exists else '✗'} {path}")

    if exists:
        found_notebooks.append(path)

if not found_notebooks:
    raise RuntimeError(
        "None of the expected PFMS Stage-7 notebooks were found."
    )

# --------------------------------------------------------------------------------
# Search exact model classes
# --------------------------------------------------------------------------------

search_terms = [
    "class MambaSequenceMixer",
    "class TransformerOCRDecoder",
    "class PFMS_Series_OCR",
    "MambaSequenceMixer",
    "TransformerOCRDecoder",
    "PFMS_Series_OCR",
]

matches = []

print("\n" + "=" * 100)
print("SEARCHING NOTEBOOK CODE CELLS")
print("=" * 100)

for notebook_path in found_notebooks:

    print("\n" + "-" * 100)
    print("Notebook:")
    print(notebook_path)

    with open(notebook_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    cells = nb.get("cells", [])

    print("Total cells:", len(cells))

    for cell_number, cell in enumerate(cells):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        matched_terms = [
            term for term in search_terms
            if term in source
        ]

        if matched_terms:

            matches.append({
                "notebook": notebook_path,
                "cell_number": cell_number,
                "terms": matched_terms,
                "source": source,
            })

            print("\n" + "=" * 100)
            print(f"MATCH FOUND — CELL {cell_number}")
            print("=" * 100)
            print("Matched:", matched_terms)

            # Print the complete source of the matching cell
            print("\n--- CELL SOURCE START ---\n")
            print(source)
            print("\n--- CELL SOURCE END ---")

# --------------------------------------------------------------------------------
# Summary
# --------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("RECOVERY SUMMARY")
print("=" * 100)

print("Notebooks found :", len(found_notebooks))
print("Matching cells  :", len(matches))

if len(matches) == 0:

    print("\n⚠️ No exact model-definition cell was found.")
    print("We will NOT invent a replacement architecture.")

else:

    print("\n✓ Exact model-related code was found.")
    print("Do NOT run training yet.")
    print("The next step will validate the recovered architecture")
    print("against the checkpoint using strict=True.")

print("\n" + "=" * 100)
print("NO TRAINING PERFORMED")
print("NO CHECKPOINT MODIFIED")
print("=" * 100)

Streaming output truncated to the last 5000 lines.
        return logits


# ================================================================
# CELL 18 — CREATE MODEL
# ================================================================

model = PFMS_Series_OCR(
    vocab_size=VOCAB_SIZE,
    d_model=512
)

model = model.to(
    device
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n" + "=" * 100)
print("MODEL INFORMATION")
print("=" * 100)

print(
    "Total parameters     :",
    f"{total_params / 1e6:.2f} M"
)

print(
    "Trainable parameters :",
    f"{trainable_params / 1e6:.2f} M"
)


# ================================================================
# CELL 19 — FREEZE SWIN FOR BASELINE EXPERIMENT
# ================================================================

FREEZE_SWIN = True

if FREEZE_SWIN:

    for param in model.swin.parameters():

        para

PFMS-SERIES-GT38-2026
EXPERIMENT B
MAMBA → SWIN → TRANSFORMER OCR

In [ ]:
# ================================================================================================
# PFMS-SERIES-GT38-2026
# EXPERIMENT B
# STAGE B-1 — MAMBA → SWIN-B → TRANSFORMER OCR
# ARCHITECTURE BUILD + FORWARD PASS VALIDATION
# ================================================================================================
# IMPORTANT:
#   This stage DOES NOT train the model.
#   This stage DOES NOT modify the previous Swin → Mamba checkpoint.
# ================================================================================================

import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import swin_b, Swin_B_Weights

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("EXPERIMENT B")
print("STAGE B-1 — MAMBA → SWIN-B → TRANSFORMER OCR")
print("=" * 100)

# --------------------------------------------------------------------------------
# 1. Device
# --------------------------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nDEVICE")
print("-" * 100)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

# --------------------------------------------------------------------------------
# 2. Required vocabulary variables
# --------------------------------------------------------------------------------

required_vars = [
    "char2idx",
    "idx2char",
    "PAD_IDX",
    "BOS_IDX",
    "EOS_IDX",
    "UNK_IDX"
]

print("\nVOCABULARY CHECK")
print("-" * 100)

for name in required_vars:
    if name not in globals():
        raise RuntimeError(
            f"Required variable '{name}' is not available in the current runtime."
        )

VOCAB_SIZE = len(char2idx)
MAX_TEXT_LENGTH = 128

print("Vocabulary size :", VOCAB_SIZE)
print("PAD_IDX         :", PAD_IDX)
print("BOS_IDX         :", BOS_IDX)
print("EOS_IDX         :", EOS_IDX)
print("UNK_IDX         :", UNK_IDX)
print("Max text length :", MAX_TEXT_LENGTH)

if VOCAB_SIZE != 169:
    raise RuntimeError(
        f"Unexpected vocabulary size: {VOCAB_SIZE}. Expected 169."
    )

# ================================================================================================
# 3. MAMBA SEQUENCE MIXER
# ================================================================================================

class MambaSequenceMixer(nn.Module):

    def __init__(
        self,
        dim,
        expansion=2,
        dropout=0.1
    ):
        super().__init__()

        hidden = dim * expansion

        self.norm = nn.LayerNorm(dim)

        self.in_proj = nn.Linear(
            dim,
            hidden * 2
        )

        self.dwconv = nn.Conv1d(
            hidden,
            hidden,
            kernel_size=5,
            padding=2,
            groups=hidden
        )

        self.act = nn.SiLU()

        self.out_proj = nn.Linear(
            hidden,
            dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # x: [B, L, D]

        residual = x

        x = self.norm(x)

        projected = self.in_proj(x)

        value, gate = projected.chunk(2, dim=-1)

        # Depthwise temporal convolution
        value = value.transpose(1, 2)

        value = self.dwconv(value)

        value = value.transpose(1, 2)

        value = self.act(value)

        # Gated sequence mixing
        value = value * torch.sigmoid(gate)

        value = self.out_proj(value)

        value = self.dropout(value)

        return residual + value


print("\n✓ MambaSequenceMixer defined")

# ================================================================================================
# 4. TRANSFORMER OCR DECODER
# ================================================================================================

class TransformerOCRDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=512,
        nhead=8,
        num_layers=4,
        dim_feedforward=2048,
        dropout=0.1,
        max_length=128
    ):
        super().__init__()

        self.d_model = d_model
        self.max_length = max_length

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_IDX
        )

        self.pos_embedding = nn.Parameter(
            torch.randn(
                1,
                max_length,
                d_model
            ) * 0.02
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(d_model)

        self.output = nn.Linear(
            d_model,
            vocab_size
        )

    def _causal_mask(self, length, device):

        return torch.triu(
            torch.full(
                (length, length),
                float("-inf"),
                device=device
            ),
            diagonal=1
        )

    def forward(
        self,
        memory,
        target_input
    ):

        # memory:
        # [B, S, D]
        #
        # target_input:
        # [B, T]

        B, T = target_input.shape

        if T > self.max_length:
            raise ValueError(
                f"Target length {T} exceeds maximum length {self.max_length}."
            )

        x = self.embedding(target_input)

        x = x * math.sqrt(self.d_model)

        x = x + self.pos_embedding[:, :T, :]

        causal_mask = self._causal_mask(
            T,
            target_input.device
        )

        target_padding_mask = (
            target_input == PAD_IDX
        )

        x = self.decoder(
            tgt=x,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=target_padding_mask
        )

        x = self.norm(x)

        logits = self.output(x)

        return logits


print("✓ TransformerOCRDecoder defined")

# ================================================================================================
# 5. MAMBA → SWIN MODEL
# ================================================================================================

class PFMS_Mamba_Swin_OCR(nn.Module):

    def __init__(
        self,
        vocab_size,
        max_text_length=128
    ):
        super().__init__()

        # ------------------------------------------------------------------------
        # Swin-B
        # ------------------------------------------------------------------------

        weights = Swin_B_Weights.DEFAULT

        self.swin = swin_b(
            weights=weights
        )

        # Remove classification head
        self.swin.head = nn.Identity()

        # Swin-B feature dimension
        self.swin_dim = 1024

        # ------------------------------------------------------------------------
        # Swin feature projection
        # ------------------------------------------------------------------------

        self.projection = nn.Linear(
            self.swin_dim,
            512
        )

        # ------------------------------------------------------------------------
        # Mamba sequence mixers
        # ------------------------------------------------------------------------

        self.mamba_blocks = nn.ModuleList([
            MambaSequenceMixer(
                dim=512,
                expansion=2,
                dropout=0.1
            )
            for _ in range(4)
        ])

        # ------------------------------------------------------------------------
        # Transformer OCR Decoder
        # ------------------------------------------------------------------------

        self.decoder = TransformerOCRDecoder(
            vocab_size=vocab_size,
            d_model=512,
            nhead=8,
            num_layers=4,
            dim_feedforward=2048,
            dropout=0.1,
            max_length=max_text_length
        )

    # --------------------------------------------------------------------------------
    # Swin feature extraction
    # --------------------------------------------------------------------------------

    def extract_swin_features(self, x):

        x = self.swin.features(x)

        # Expected Swin output:
        # [B, H, W, C]

        if x.ndim == 4:

            B, H, W, C = x.shape

            x = x.reshape(
                B,
                H * W,
                C
            )

        elif x.ndim != 3:

            raise RuntimeError(
                f"Unexpected Swin feature shape: {tuple(x.shape)}"
            )

        return x

    # --------------------------------------------------------------------------------
    # Encoder
    # --------------------------------------------------------------------------------

    def encode(self, images):

        # Swin first
        x = self.extract_swin_features(images)

        # Swin output → 512
        x = self.projection(x)

        # Mamba sequence processing
        for block in self.mamba_blocks:
            x = block(x)

        return x

    # --------------------------------------------------------------------------------
    # Forward
    # --------------------------------------------------------------------------------

    def forward(
        self,
        images,
        target_input
    ):

        memory = self.encode(images)

        logits = self.decoder(
            memory,
            target_input
        )

        return logits


print("✓ PFMS_Mamba_Swin_OCR defined")

# ================================================================================================
# 6. BUILD MODEL
# ================================================================================================

print("\n" + "=" * 100)
print("BUILDING MODEL")
print("=" * 100)

model_B = PFMS_Mamba_Swin_OCR(
    vocab_size=VOCAB_SIZE,
    max_text_length=MAX_TEXT_LENGTH
)

model_B = model_B.to(DEVICE)

print("✓ Model created")

# --------------------------------------------------------------------------------
# 7. Parameter count
# --------------------------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model_B.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model_B.parameters()
    if p.requires_grad
)

print("\nPARAMETERS")
print("-" * 100)
print(f"Total parameters     : {total_params:,}")
print(f"Total parameters (M) : {total_params / 1e6:.2f} M")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Trainable (M)        : {trainable_params / 1e6:.2f} M")

# ================================================================================================
# 8. Freeze Swin-B
# ================================================================================================

for param in model_B.swin.parameters():
    param.requires_grad = False

trainable_after_freeze = sum(
    p.numel()
    for p in model_B.parameters()
    if p.requires_grad
)

print("\nFREEZING SWIN-B")
print("-" * 100)
print("✓ Swin-B frozen")
print(
    f"Trainable parameters after freezing : "
    f"{trainable_after_freeze:,}"
)
print(
    f"Trainable parameters after freezing : "
    f"{trainable_after_freeze / 1e6:.2f} M"
)

# ================================================================================================
# 9. Forward Pass Test
# ================================================================================================

print("\n" + "=" * 100)
print("FORWARD PASS TEST")
print("=" * 100)

model_B.eval()

# Use the exact input format used in Stage 7
dummy_images = torch.randn(
    2,
    3,
    224,
    224,
    device=DEVICE
)

dummy_targets = torch.full(
    (2, MAX_TEXT_LENGTH),
    PAD_IDX,
    dtype=torch.long,
    device=DEVICE
)

# BOS token
dummy_targets[:, 0] = BOS_IDX

with torch.no_grad():

    memory_B = model_B.encode(
        dummy_images
    )

    logits_B = model_B(
        dummy_images,
        dummy_targets
    )

print("Input shape:")
print("  ", tuple(dummy_images.shape))

print("\nEncoder memory shape:")
print("  ", tuple(memory_B.shape))

print("\nDecoder logits shape:")
print("  ", tuple(logits_B.shape))

expected_last_dim = VOCAB_SIZE

if logits_B.shape[0] != 2:
    raise RuntimeError("Unexpected batch dimension.")

if logits_B.shape[1] != MAX_TEXT_LENGTH:
    raise RuntimeError("Unexpected text sequence length.")

if logits_B.shape[2] != expected_last_dim:
    raise RuntimeError(
        f"Unexpected vocabulary dimension: "
        f"{logits_B.shape[2]} != {expected_last_dim}"
    )

print("\n✓ Forward Pass SUCCESSFUL")

# ================================================================================================
# 10. Final architecture report
# ================================================================================================

print("\n" + "=" * 100)
print("ARCHITECTURE REPORT")
print("=" * 100)

print("""
Mamba → Swin-B → Transformer OCR Decoder

Input:
    224 × 224 × 3

Swin-B:
    Pretrained
    Feature dimension: 1024

Projection:
    1024 → 512

Mamba Sequence Mixer:
    4 blocks
    Dimension: 512
    Expansion: 2
    Depthwise Conv1D kernel: 5

Transformer OCR Decoder:
    d_model: 512
    Heads: 8
    Layers: 4
    FFN: 2048
    Max sequence length: 128
    Vocabulary: 169
""")

print("=" * 100)
print("✓ STAGE B-1 COMPLETED")
print("=" * 100)

print("\nIMPORTANT:")
print("No training performed.")
print("Previous Swin → Mamba checkpoint was NOT modified.")
print("New model is stored in variable: model_B")
print("=" * 100)

PFMS-SERIES-GT38-2026
EXPERIMENT B
STAGE B-1 — MAMBA → SWIN-B → TRANSFORMER OCR

DEVICE
----------------------------------------------------------------------------------------------------
Device: cuda
GPU: Tesla T4
CUDA: 12.8

VOCABULARY CHECK
----------------------------------------------------------------------------------------------------


RuntimeError: Required variable 'char2idx' is not available in the current runtime.

In [ ]:
# ================================================================================================
# PFMS-SERIES-GT38-2026
# EXPERIMENT B
# STAGE B-2 — REAL DATA BATCH + LOSS VALIDATION
# MAMBA → SWIN-B → TRANSFORMER OCR
# ================================================================================================
# هدف:
#   1. استفاده از همان Split دقیق Stage 7
#   2. ساخت DataLoader واقعی
#   3. خواندن یک Batch واقعی از IDPL-PFOD
#   4. اجرای Forward Pass روی تصاویر واقعی
#   5. محاسبه CrossEntropy Loss
#
# NO TRAINING
# NO BACKPROPAGATION
# NO CHECKPOINT SAVING
# ================================================================================================

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("EXPERIMENT B")
print("STAGE B-2 — REAL DATA BATCH + LOSS VALIDATION")
print("=" * 100)

# ================================================================================================
# 1. Paths
# ================================================================================================

GT_CSV = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"

SPLIT_CSV = (
    "/content/drive/MyDrive/PFMS-SERIES-GT38-2026/"
    "PFMS_SERIES_GT38_STAGE7_EXACT_SPLIT.csv"
)

IMAGE_ROOT = (
    "/content/drive/Othercomputers/"
    "My Laptop/Desktop/idplimgl"
)

print("\nPATHS")
print("-" * 100)
print("GT CSV      :", GT_CSV)
print("Split CSV   :", SPLIT_CSV)
print("Image Root  :", IMAGE_ROOT)

for path in [GT_CSV, SPLIT_CSV, IMAGE_ROOT]:
    if not os.path.exists(path):
        raise RuntimeError(f"Required path does not exist:\n{path}")

print("\n✓ All required paths exist")

# ================================================================================================
# 2. Verify model_B
# ================================================================================================

if "model_B" not in globals():
    raise RuntimeError(
        "model_B is not available. Please run STAGE B-1 first."
    )

print("\n✓ model_B is available")

# ================================================================================================
# 3. Verify vocabulary
# ================================================================================================

required_vocab = [
    "char2idx",
    "idx2char",
    "PAD_IDX",
    "BOS_IDX",
    "EOS_IDX",
    "UNK_IDX"
]

for name in required_vocab:
    if name not in globals():
        raise RuntimeError(
            f"Required vocabulary variable '{name}' is missing."
        )

VOCAB_SIZE = len(char2idx)
MAX_TEXT_LENGTH = 128

if VOCAB_SIZE != 169:
    raise RuntimeError(
        f"Expected vocabulary size 169, got {VOCAB_SIZE}"
    )

print("\nVOCABULARY")
print("-" * 100)
print("Vocabulary size :", VOCAB_SIZE)
print("PAD             :", PAD_IDX)
print("BOS             :", BOS_IDX)
print("EOS             :", EOS_IDX)
print("UNK             :", UNK_IDX)

# ================================================================================================
# 4. Load original GT and exact Stage-7 split
# ================================================================================================

print("\n" + "=" * 100)
print("LOADING EXACT STAGE-7 SPLIT")
print("=" * 100)

gt_df = pd.read_csv(GT_CSV)
split_df = pd.read_csv(SPLIT_CSV)

if "original_df_index" not in split_df.columns:
    raise RuntimeError(
        "original_df_index missing from exact split."
    )

if "split" not in split_df.columns:
    raise RuntimeError(
        "split missing from exact split."
    )

# Exact Stage-7 row restoration
restored_df = (
    gt_df.iloc[
        split_df["original_df_index"].to_numpy()
    ]
    .copy()
    .reset_index(drop=True)
)

restored_df["stage7_split"] = split_df["split"].to_numpy()

# --------------------------------------------------------------------------------
# Verify exact counts
# --------------------------------------------------------------------------------

train_df = (
    restored_df[
        restored_df["stage7_split"] == "train"
    ]
    .reset_index(drop=True)
)

val_df = (
    restored_df[
        restored_df["stage7_split"] == "validation"
    ]
    .reset_index(drop=True)
)

test_df = (
    restored_df[
        restored_df["stage7_split"] == "test"
    ]
    .reset_index(drop=True)
)

print("\nExact split:")
print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))
print("Total      :", len(restored_df))

if len(train_df) != 23052:
    raise RuntimeError("Train split mismatch.")

if len(val_df) != 2712:
    raise RuntimeError("Validation split mismatch.")

if len(test_df) != 1356:
    raise RuntimeError("Test split mismatch.")

print("\n✓ Exact Stage-7 split verified")

# ================================================================================================
# 5. Text encoder
# ================================================================================================

def encode_text_B(text, max_length=MAX_TEXT_LENGTH):

    tokens = [BOS_IDX]

    for ch in str(text):

        if ch in char2idx:
            tokens.append(char2idx[ch])
        else:
            tokens.append(UNK_IDX)

    tokens.append(EOS_IDX)

    if len(tokens) > max_length:

        tokens = tokens[:max_length]

        # Guarantee EOS at the final valid position
        tokens[-1] = EOS_IDX

    elif len(tokens) < max_length:

        tokens.extend(
            [PAD_IDX] * (max_length - len(tokens))
        )

    return tokens

print("\n✓ Text encoder ready")

# ================================================================================================
# 6. Transform — EXACT Stage-7 evaluation preprocessing
# ================================================================================================

IMG_SIZE = 224

eval_transform_B = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✓ Evaluation transform ready")

# ================================================================================================
# 7. Dataset
# ================================================================================================

class IDPLOCRDataset_B(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = row["image_path"]
        text = row["text"]
        image_name = row["image_name"]

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            raise RuntimeError(
                f"\nCannot read image:\n"
                f"{image_path}\n"
                f"Error: {e}"
            )

        if self.transform is not None:

            image = self.transform(image)

        target = torch.tensor(
            encode_text_B(text),
            dtype=torch.long
        )

        return {
            "image": image,
            "target": target,
            "text": text,
            "image_name": image_name
        }

print("✓ IDPLOCRDataset_B ready")

# ================================================================================================
# 8. Create datasets
# ================================================================================================

train_dataset_B = IDPLOCRDataset_B(
    train_df,
    transform=eval_transform_B
)

val_dataset_B = IDPLOCRDataset_B(
    val_df,
    transform=eval_transform_B
)

test_dataset_B = IDPLOCRDataset_B(
    test_df,
    transform=eval_transform_B
)

print("\nDATASETS")
print("-" * 100)
print("Train dataset      :", len(train_dataset_B))
print("Validation dataset :", len(val_dataset_B))
print("Test dataset       :", len(test_dataset_B))

# ================================================================================================
# 9. DataLoaders
# ================================================================================================
# Batch size = 4 exactly as Stage 7.
# num_workers = 0 for Colab stability.

BATCH_SIZE = 4

train_loader_B = DataLoader(
    train_dataset_B,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader_B = DataLoader(
    val_dataset_B,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader_B = DataLoader(
    test_dataset_B,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("\nDATALOADERS")
print("-" * 100)
print("Batch size :", BATCH_SIZE)
print("Train batches :", len(train_loader_B))
print("Val batches   :", len(val_loader_B))
print("Test batches  :", len(test_loader_B))
print("✓ DataLoaders ready")

# ================================================================================================
# 10. Get one REAL batch
# ================================================================================================

print("\n" + "=" * 100)
print("REAL DATA BATCH TEST")
print("=" * 100)

batch = next(iter(train_loader_B))

images = batch["image"].to(
    DEVICE,
    non_blocking=True
)

targets = batch["target"].to(
    DEVICE,
    non_blocking=True
)

print("\nImage batch shape:")
print("  ", tuple(images.shape))

print("Target batch shape:")
print("  ", tuple(targets.shape))

print("Image dtype:")
print("  ", images.dtype)

print("Target dtype:")
print("  ", targets.dtype)

# ================================================================================================
# 11. Basic target validation
# ================================================================================================

if images.shape != (BATCH_SIZE, 3, 224, 224):

    raise RuntimeError(
        f"Unexpected image shape: {tuple(images.shape)}"
    )

if targets.shape != (BATCH_SIZE, MAX_TEXT_LENGTH):

    raise RuntimeError(
        f"Unexpected target shape: {tuple(targets.shape)}"
    )

if targets.min().item() < 0:

    raise RuntimeError("Negative target index detected.")

if targets.max().item() >= VOCAB_SIZE:

    raise RuntimeError(
        f"Target index {targets.max().item()} "
        f"is outside vocabulary size {VOCAB_SIZE}."
    )

print("\n✓ Real batch shape verified")
print("✓ Target indices verified")

# ================================================================================================
# 12. Real-data Forward Pass
# ================================================================================================

print("\n" + "=" * 100)
print("REAL-DATA FORWARD PASS")
print("=" * 100)

model_B.eval()

with torch.no_grad():

    logits = model_B(
        images,
        targets[:, :-1]
    )

print("Input images:")
print("  ", tuple(images.shape))

print("\nDecoder input:")
print("  ", tuple(targets[:, :-1].shape))

print("\nLogits:")
print("  ", tuple(logits.shape))

# ================================================================================================
# 13. Expected output validation
# ================================================================================================

expected_logits_shape = (
    BATCH_SIZE,
    MAX_TEXT_LENGTH - 1,
    VOCAB_SIZE
)

if tuple(logits.shape) != expected_logits_shape:

    raise RuntimeError(
        f"\nUnexpected logits shape.\n"
        f"Expected: {expected_logits_shape}\n"
        f"Actual:   {tuple(logits.shape)}"
    )

print("\n✓ Logits shape verified")

# ================================================================================================
# 14. CrossEntropy Loss
# ================================================================================================

criterion_B = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

# logits:
# [B, T-1, V]
#
# targets:
# [B, T]
#
# Compare logits for positions 0...T-2
# against target positions 1...T-1.

loss_B = criterion_B(
    logits.reshape(-1, VOCAB_SIZE),
    targets[:, 1:].reshape(-1)
)

loss_value = float(loss_B.item())

print("\n" + "=" * 100)
print("REAL-DATA LOSS")
print("=" * 100)

print(f"CrossEntropy Loss : {loss_value:.6f}")

if not torch.isfinite(loss_B):

    raise RuntimeError(
        "Loss is NaN or Inf."
    )

print("\n✓ Loss is finite and valid")

# ================================================================================================
# 15. Quick token diagnostic
# ================================================================================================

pred_tokens = logits.argmax(
    dim=-1
)

target_tokens = targets[:, 1:]

valid_mask = (
    target_tokens != PAD_IDX
)

correct_tokens = (
    (pred_tokens == target_tokens)
    & valid_mask
).sum().item()

valid_tokens = (
    valid_mask.sum().item()
)

token_accuracy = (
    100.0 * correct_tokens / valid_tokens
    if valid_tokens > 0
    else 0.0
)

print("\nQUICK TOKEN DIAGNOSTIC")
print("-" * 100)
print("Valid target tokens :", valid_tokens)
print("Correct tokens      :", correct_tokens)
print(f"Token accuracy      : {token_accuracy:.4f}%")

# ================================================================================================
# 16. Final report
# ================================================================================================

print("\n" + "=" * 100)
print("✓ STAGE B-2 COMPLETED SUCCESSFULLY")
print("=" * 100)

print("\nArchitecture:")
print("Mamba → Swin-B → Transformer OCR Decoder")

print("\nReal data:")
print("Train      :", len(train_dataset_B))
print("Validation :", len(val_dataset_B))
print("Test       :", len(test_dataset_B))

print("\nReal batch:")
print("Images :", tuple(images.shape))
print("Targets:", tuple(targets.shape))

print("\nForward:")
print("Memory/Encoder output :", tuple(model_B.encode(images).shape))
print("Logits                :", tuple(logits.shape))

print("\nLoss:")
print(f"CrossEntropy : {loss_value:.6f}")

print("\nToken diagnostic:")
print(f"Accuracy : {token_accuracy:.4f}%")

print("\nIMPORTANT:")
print("✓ No training performed")
print("✓ No backward pass performed")
print("✓ No optimizer used")
print("✓ No checkpoint modified")
print("✓ Previous Swin → Mamba experiment remains untouched")

print("=" * 100)

PFMS-SERIES-GT38-2026
EXPERIMENT B
STAGE B-2 — REAL DATA BATCH + LOSS VALIDATION

PATHS
----------------------------------------------------------------------------------------------------
GT CSV      : /content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv
Split CSV   : /content/drive/MyDrive/PFMS-SERIES-GT38-2026/PFMS_SERIES_GT38_STAGE7_EXACT_SPLIT.csv
Image Root  : /content/drive/Othercomputers/My Laptop/Desktop/idplimgl

✓ All required paths exist

✓ model_B is available

VOCABULARY
----------------------------------------------------------------------------------------------------
Vocabulary size : 169
PAD             : 0
BOS             : 1
EOS             : 2
UNK             : 3

LOADING EXACT STAGE-7 SPLIT

Exact split:
Train      : 23052
Validation : 2712
Test       : 1356
Total      : 27120

✓ Exact Stage-7 split verified

✓ Text encoder ready
✓ Evaluation transform ready
✓ IDPLOCRDataset_B ready

DATASETS
------------------------------------------------------------------------

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(



QUICK TOKEN DIAGNOSTIC
----------------------------------------------------------------------------------------------------
Valid target tokens : 321
Correct tokens      : 1
Token accuracy      : 0.3115%

✓ STAGE B-2 COMPLETED SUCCESSFULLY

Architecture:
Mamba → Swin-B → Transformer OCR Decoder

Real data:
Train      : 23052
Validation : 2712
Test       : 1356

Real batch:
Images : (4, 3, 224, 224)
Targets: (4, 128)

Forward:
Memory/Encoder output : (4, 49, 512)
Logits                : (4, 127, 169)

Loss:
CrossEntropy : 5.269204

Token diagnostic:
Accuracy : 0.3115%

IMPORTANT:
✓ No training performed
✓ No backward pass performed
✓ No optimizer used
✓ No checkpoint modified
✓ Previous Swin → Mamba experiment remains untouched


In [11]:
# ============================================================
# CHECK CURRENT GOOGLE DRIVE MOUNT
# ============================================================

import os

print("=" * 80)
print("CHECKING /content/drive")
print("=" * 80)

print("Exists :", os.path.exists("/content/drive"))
print("Is dir :", os.path.isdir("/content/drive"))

print("\nTop-level contents:")

if os.path.isdir("/content/drive"):
    items = os.listdir("/content/drive")
    for item in items[:30]:
        print(" -", item)

print("\nChecking MyDrive:")

mydrive = "/content/drive/MyDrive"

if os.path.isdir(mydrive):
    print("✓ MyDrive exists")
    print("Path:", mydrive)

    items = os.listdir(mydrive)
    print("Number of items:", len(items))

    for item in items[:30]:
        print(" -", item)
else:
    print("✗ MyDrive NOT found")

print("\n" + "=" * 80)

CHECKING /content/drive
Exists : True
Is dir : True

Top-level contents:
 - MyDrive

Checking MyDrive:
✓ MyDrive exists
Path: /content/drive/MyDrive
Number of items: 1
 - PFMS-SERIES-GT38-2026



In [18]:
# ============================================================
# PFMS-SERIES-GT38-2026
# STEP — CLONE EXISTING REPOSITORY
# NO COMMIT / NO PUSH
# ============================================================

import os
import subprocess

REPO_URL = "https://github.com/ghadirymaryam1-coder/PFMS-Net-Research2026.git"
REPO_PATH = "/content/PFMS-Net-Research2026"

print("=" * 70)
print("Cloning PFMS-Net-Research2026")
print("=" * 70)

if os.path.exists(REPO_PATH):
    print("Repository already exists:")
    print(REPO_PATH)
else:
    result = subprocess.run(
        ["git", "clone", REPO_URL, REPO_PATH],
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Git clone failed.")

print("\nRepository status:")
os.chdir(REPO_PATH)

print(subprocess.run(
    ["git", "status", "--short"],
    capture_output=True,
    text=True
).stdout or "Working tree is clean.")

print("\nExisting PFMS notebook:")

OLD_FILE = os.path.join(
    REPO_PATH,
    "notebooks",
    "PFMS-Net",
    "PFMS-Net_Research2026.ipynb"
)

print(OLD_FILE)
print("Exists:", os.path.exists(OLD_FILE))

print("\n" + "=" * 70)
print("CLONE CHECK COMPLETE")
print("NO commit / NO push was performed.")
print("=" * 70)

Cloning PFMS-Net-Research2026


Repository status:
Working tree is clean.

Existing PFMS notebook:
/content/PFMS-Net-Research2026/notebooks/PFMS-Net/PFMS-Net_Research2026.ipynb
Exists: True

CLONE CHECK COMPLETE
NO commit / NO push was performed.


In [8]:

# ==========================================================================================
# PFMS-SERIES-GT38-2026
# EXPERIMENT B — STAGE B-3
# Mamba → Swin-B → Transformer OCR Decoder
#
# FULL SELF-CONTAINED TRAINING CELL
# LOW-STORAGE CHECKPOINT
# AUTO RESUME AFTER COLAB DISCONNECTION
# ==========================================================================================

import os
import gc
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from torchvision import transforms
from torchvision.models import swin_b, Swin_B_Weights
from tqdm.auto import tqdm


# ==========================================================================================
# 1. CONFIGURATION
# ==========================================================================================

SEED = 42

EPOCHS = 10
BATCH_SIZE = 4

IMG_SIZE = 224
MAX_TEXT_LENGTH = 128

LEARNING_RATE = 0.0002
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------------------------------

GT_CSV = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"

EXACT_SPLIT_CSV = (
    "/content/drive/MyDrive/"
    "PFMS-SERIES-GT38-2026/"
    "PFMS_SERIES_GT38_STAGE7_EXACT_SPLIT.csv"
)

VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"

SAVE_DIR = (
    "/content/drive/MyDrive/"
    "PFMS-SERIES-GT38-2026"
)

os.makedirs(SAVE_DIR, exist_ok=True)

BEST_PATH = os.path.join(
    SAVE_DIR,
    "best_pfms_series_mamba_swin_trocr.pth"
)

LAST_PATH = os.path.join(
    SAVE_DIR,
    "last_pfms_series_mamba_swin_trocr.pth"
)

HISTORY_PATH = os.path.join(
    SAVE_DIR,
    "training_history_mamba_swin.csv"
)


# ==========================================================================================
# 2. RANDOM SEED
# ==========================================================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 100)
print("PFMS-SERIES-GT38-2026")
print("EXPERIMENT B — STAGE B-3")
print("Mamba → Swin-B → Transformer OCR Decoder")
print("FULL SELF-CONTAINED TRAINING")
print("=" * 100)

print(f"Device : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"CUDA   : {torch.version.cuda}")

print(f"PyTorch: {torch.__version__}")


# ==========================================================================================
# 3. LOAD REAL GT
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 1 — LOADING REAL IDPL-PFOD GT")
print("=" * 100)

assert os.path.exists(GT_CSV), (
    f"GT CSV not found:\n{GT_CSV}"
)

df = pd.read_csv(GT_CSV)

print(f"GT rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

required_columns = [
    "split",
    "hf_index",
    "image_path",
    "image_name",
    "text"
]

for col in required_columns:
    assert col in df.columns, (
        f"Missing required column: {col}"
    )

df = df.reset_index(drop=True)

print("✓ Real GT loaded")
print(f"✓ Total samples: {len(df):,}")


# ==========================================================================================
# 4. LOAD VOCABULARY
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 2 — LOADING VOCABULARY")
print("=" * 100)

assert os.path.exists(VOCAB_PATH), (
    f"Vocabulary file not found:\n{VOCAB_PATH}"
)

vocab_data = torch.load(
    VOCAB_PATH,
    map_location="cpu",
    weights_only=False
)

print("Vocabulary object type:", type(vocab_data))

# ------------------------------------------------------------------------------------------
# Try to recover vocabulary robustly
# ------------------------------------------------------------------------------------------

if isinstance(vocab_data, dict):

    if "char2idx" in vocab_data:
        char2idx = vocab_data["char2idx"]

    elif "stoi" in vocab_data:
        char2idx = vocab_data["stoi"]

    else:
        # Sometimes vocabulary itself is char -> index
        if all(
            isinstance(k, str) and isinstance(v, int)
            for k, v in vocab_data.items()
        ):
            char2idx = vocab_data
        else:
            raise RuntimeError(
                "Could not identify char2idx in vocabulary file."
            )

else:
    raise RuntimeError(
        "Unexpected vocabulary file format."
    )


# ------------------------------------------------------------------------------------------
# idx2char
# ------------------------------------------------------------------------------------------

if "idx2char" in vocab_data:
    idx2char = vocab_data["idx2char"]
else:
    idx2char = {
        int(v): k
        for k, v in char2idx.items()
    }


# ------------------------------------------------------------------------------------------
# Special token indices
# ------------------------------------------------------------------------------------------

PAD_IDX = int(
    vocab_data.get("PAD_IDX", char2idx.get("<PAD>", 0))
)

BOS_IDX = int(
    vocab_data.get("BOS_IDX", char2idx.get("<BOS>", 1))
)

EOS_IDX = int(
    vocab_data.get("EOS_IDX", char2idx.get("<EOS>", 2))
)

UNK_IDX = int(
    vocab_data.get("UNK_IDX", char2idx.get("<UNK>", 3))
)

VOCAB_SIZE = len(char2idx)

print(f"Vocabulary size : {VOCAB_SIZE}")
print(f"PAD             : {PAD_IDX}")
print(f"BOS             : {BOS_IDX}")
print(f"EOS             : {EOS_IDX}")
print(f"UNK             : {UNK_IDX}")

assert VOCAB_SIZE == 169, (
    f"Expected vocabulary size 169, got {VOCAB_SIZE}"
)

print("✓ Vocabulary verified")


# ==========================================================================================
# 5. EXACT STAGE-7 SPLIT
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 3 — RESTORING EXACT STAGE-7 SPLIT")
print("=" * 100)

assert os.path.exists(EXACT_SPLIT_CSV), (
    f"Exact split file not found:\n{EXACT_SPLIT_CSV}"
)

split_df = pd.read_csv(EXACT_SPLIT_CSV)

print("Split columns:", list(split_df.columns))
print(f"Split rows   : {len(split_df):,}")

assert "original_df_index" in split_df.columns
assert "split" in split_df.columns

# ------------------------------------------------------------------------------------------
# Validate exact indexing
# ------------------------------------------------------------------------------------------

assert len(split_df) == len(df), (
    "Split size does not match GT size."
)

assert split_df["original_df_index"].notna().all(), (
    "original_df_index contains NaN."
)

assert split_df["original_df_index"].is_unique, (
    "original_df_index is not unique."
)

indices = split_df["original_df_index"].astype(int).to_numpy()

assert np.array_equal(
    np.sort(indices),
    np.arange(len(df))
), (
    "Exact split indices do not cover the GT rows exactly."
)

# ------------------------------------------------------------------------------------------
# Reconstruct exact split
# ------------------------------------------------------------------------------------------

train_indices = split_df.loc[
    split_df["split"].astype(str).str.lower() == "train",
    "original_df_index"
].astype(int).to_numpy()

val_indices = split_df.loc[
    split_df["split"].astype(str).str.lower().isin(
        ["val", "validation"]
    ),
    "original_df_index"
].astype(int).to_numpy()

test_indices = split_df.loc[
    split_df["split"].astype(str).str.lower() == "test",
    "original_df_index"
].astype(int).to_numpy()


train_df = df.iloc[train_indices].reset_index(drop=True)
val_df = df.iloc[val_indices].reset_index(drop=True)
test_df = df.iloc[test_indices].reset_index(drop=True)

print(f"Train      : {len(train_df):,}")
print(f"Validation : {len(val_df):,}")
print(f"Test       : {len(test_df):,}")

assert len(train_df) == 23052
assert len(val_df) == 2712
assert len(test_df) == 1356

print("✓ Exact Stage-7 split restored")


# ==========================================================================================
# 6. VERIFY IMAGE FILES
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 4 — VERIFYING IMAGE FILES")
print("=" * 100)

all_paths = pd.concat(
    [
        train_df["image_path"],
        val_df["image_path"],
        test_df["image_path"]
    ],
    ignore_index=True
)

missing_files = [
    p for p in all_paths
    if not os.path.exists(str(p))
]

print(f"Total image paths: {len(all_paths):,}")
print(f"Missing files   : {len(missing_files):,}")

assert len(missing_files) == 0, (
    f"Missing image files detected. "
    f"First missing file: {missing_files[0]}"
)

print("✓ All image files exist")


# ==========================================================================================
# 7. TEXT ENCODING
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 5 — TEXT ENCODING")
print("=" * 100)


def encode_text(text, max_length=MAX_TEXT_LENGTH):

    tokens = [BOS_IDX]

    for ch in str(text):

        if ch in char2idx:
            tokens.append(char2idx[ch])
        else:
            tokens.append(UNK_IDX)

    tokens.append(EOS_IDX)

    if len(tokens) > max_length:

        tokens = tokens[:max_length]

        tokens[-1] = EOS_IDX

    elif len(tokens) < max_length:

        tokens.extend(
            [PAD_IDX] * (max_length - len(tokens))
        )

    return tokens


# Quick verification
sample_text = str(train_df.iloc[0]["text"])
sample_tokens = encode_text(sample_text)

print("Sample text:")
print(sample_text)

print(f"Encoded length: {len(sample_tokens)}")

assert len(sample_tokens) == MAX_TEXT_LENGTH

print("✓ Text encoding verified")


# ==========================================================================================
# 8. TRANSFORMS
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 6 — IMAGE TRANSFORMS")
print("=" * 100)

train_transform_B = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomApply(
        [
            transforms.ColorJitter(
                brightness=0.10,
                contrast=0.10
            )
        ],
        p=0.30
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform_B = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✓ Training transform ready")
print("✓ Evaluation transform ready")


# ==========================================================================================
# 9. DATASET
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 7 — BUILDING DATASETS")
print("=" * 100)


class IDPLOCRDataset_B(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = row["image_path"]
        text = row["text"]
        image_name = row["image_name"]

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            raise RuntimeError(
                f"\nCannot read image:\n"
                f"{image_path}\n"
                f"Error: {e}"
            )

        if self.transform is not None:

            image = self.transform(image)

        target = torch.tensor(
            encode_text(text),
            dtype=torch.long
        )

        return {
            "image": image,
            "target": target,
            "text": text,
            "image_name": image_name
        }


train_dataset_B = IDPLOCRDataset_B(
    train_df,
    train_transform_B
)

val_dataset_B = IDPLOCRDataset_B(
    val_df,
    eval_transform_B
)

test_dataset_B = IDPLOCRDataset_B(
    test_df,
    eval_transform_B
)

print(f"Train dataset: {len(train_dataset_B):,}")
print(f"Val dataset  : {len(val_dataset_B):,}")
print(f"Test dataset : {len(test_dataset_B):,}")


# ==========================================================================================
# 10. DATALOADERS
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 8 — BUILDING DATALOADERS")
print("=" * 100)

train_loader_B = DataLoader(
    train_dataset_B,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader_B = DataLoader(
    val_dataset_B,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader_B = DataLoader(
    test_dataset_B,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(f"Train batches: {len(train_loader_B):,}")
print(f"Val batches  : {len(val_loader_B):,}")
print(f"Test batches : {len(test_loader_B):,}")


# ==========================================================================================
# 11. MAMBA SEQUENCE MIXER
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 9 — DEFINING MAMBA SEQUENCE MIXER")
print("=" * 100)


class MambaSequenceMixer(nn.Module):

    def __init__(
        self,
        dim,
        expansion=2,
        dropout=0.1
    ):

        super().__init__()

        hidden = dim * expansion

        self.norm = nn.LayerNorm(dim)

        self.in_proj = nn.Linear(
            dim,
            hidden * 2
        )

        self.depthwise_conv = nn.Conv1d(
            hidden,
            hidden,
            kernel_size=5,
            padding=2,
            groups=hidden
        )

        self.act = nn.SiLU()

        self.out_proj = nn.Linear(
            hidden,
            dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        residual = x

        x = self.norm(x)

        projected = self.in_proj(x)

        value, gate = projected.chunk(
            2,
            dim=-1
        )

        value = value.transpose(1, 2)

        value = self.depthwise_conv(
            value
        )

        value = value.transpose(1, 2)

        value = self.act(value)

        gate = torch.sigmoid(gate)

        x = value * gate

        x = self.out_proj(x)

        x = self.dropout(x)

        return residual + x


print("✓ MambaSequenceMixer defined")


# ==========================================================================================
# 12. TRANSFORMER OCR DECODER
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 10 — DEFINING TRANSFORMER OCR DECODER")
print("=" * 100)


class TransformerOCRDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=512,
        nhead=8,
        num_layers=4,
        dim_feedforward=2048,
        dropout=0.1,
        max_length=128
    ):

        super().__init__()

        self.d_model = d_model
        self.max_length = max_length

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_IDX
        )

        self.pos_embedding = nn.Parameter(
            torch.randn(
                1,
                max_length,
                d_model
            ) * 0.02
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        self.output = nn.Linear(
            d_model,
            vocab_size
        )

    def _causal_mask(
        self,
        length,
        device
    ):

        return torch.triu(
            torch.full(
                (length, length),
                float("-inf"),
                device=device
            ),
            diagonal=1
        )

    def forward(
        self,
        memory,
        target_input
    ):

        B, T = target_input.shape

        x = self.embedding(
            target_input
        )

        x = x + self.pos_embedding[
            :, :T, :
        ]

        causal_mask = self._causal_mask(
            T,
            target_input.device
        )

        tgt_padding_mask = (
            target_input == PAD_IDX
        )

        x = self.decoder(
            tgt=x,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )

        x = self.norm(x)

        logits = self.output(x)

        return logits


print("✓ TransformerOCRDecoder defined")


# ==========================================================================================
# 13. EXPERIMENT B MODEL
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 11 — BUILDING EXPERIMENT B MODEL")
print("=" * 100)


class PFMS_Mamba_Swin_OCR(nn.Module):

    def __init__(
        self,
        vocab_size,
        max_length=128
    ):

        super().__init__()

        # ------------------------------------------------------------------
        # Swin-B
        # ------------------------------------------------------------------

        weights = Swin_B_Weights.DEFAULT

        self.swin = swin_b(
            weights=weights
        )

        self.swin.head = nn.Identity()

        swin_dim = 1024

        # ------------------------------------------------------------------
        # Projection
        # ------------------------------------------------------------------

        self.projection = nn.Linear(
            swin_dim,
            512
        )

        # ------------------------------------------------------------------
        # Mamba blocks
        # ------------------------------------------------------------------

        self.mamba_blocks = nn.ModuleList([
            MambaSequenceMixer(
                dim=512,
                expansion=2,
                dropout=0.1
            )
            for _ in range(4)
        ])

        # ------------------------------------------------------------------
        # Transformer OCR Decoder
        # ------------------------------------------------------------------

        self.decoder = TransformerOCRDecoder(
            vocab_size=vocab_size,
            d_model=512,
            nhead=8,
            num_layers=4,
            dim_feedforward=2048,
            dropout=0.1,
            max_length=max_length
        )

    def extract_swin_features(self, x):

        x = self.swin.features(x)

        # Swin output:
        # [B, H, W, C]

        if x.ndim == 4:

            B, H, W, C = x.shape

            x = x.reshape(
                B,
                H * W,
                C
            )

        return x

    def encode(self, images):

        x = self.extract_swin_features(
            images
        )

        # 1024 → 512
        x = self.projection(x)

        # Mamba sequence processing
        for block in self.mamba_blocks:

            x = block(x)

        return x

    def forward(
        self,
        images,
        target_input
    ):

        memory = self.encode(
            images
        )

        logits = self.decoder(
            memory,
            target_input
        )

        return logits


model_B = PFMS_Mamba_Swin_OCR(
    vocab_size=VOCAB_SIZE,
    max_length=MAX_TEXT_LENGTH
)


# ==========================================================================================
# 14. FREEZE SWIN-B
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 12 — FREEZING SWIN-B")
print("=" * 100)

for param in model_B.swin.parameters():

    param.requires_grad = False


model_B = model_B.to(DEVICE)


# ------------------------------------------------------------------------------------------
# Parameter report
# ------------------------------------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model_B.parameters()
)

trainable_params_count = sum(
    p.numel()
    for p in model_B.parameters()
    if p.requires_grad
)

print(
    f"Total parameters     : "
    f"{total_params:,} "
    f"({total_params/1e6:.2f}M)"
)

print(
    f"Trainable parameters : "
    f"{trainable_params_count:,} "
    f"({trainable_params_count/1e6:.2f}M)"
)

print("✓ Swin-B frozen")


# ==========================================================================================
# 15. REAL DATA FORWARD TEST
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 13 — REAL DATA FORWARD TEST")
print("=" * 100)

model_B.eval()

with torch.no_grad():

    batch = next(
        iter(train_loader_B)
    )

    images = batch["image"].to(
        DEVICE
    )

    targets = batch["target"].to(
        DEVICE
    )

    decoder_input = targets[:, :-1]

    logits = model_B(
        images,
        decoder_input
    )

print(f"Images         : {tuple(images.shape)}")
print(f"Targets        : {tuple(targets.shape)}")
print(f"Decoder input  : {tuple(decoder_input.shape)}")
print(f"Logits         : {tuple(logits.shape)}")

assert images.shape == (
    BATCH_SIZE,
    3,
    IMG_SIZE,
    IMG_SIZE
)

assert targets.shape == (
    BATCH_SIZE,
    MAX_TEXT_LENGTH
)

assert logits.shape == (
    BATCH_SIZE,
    MAX_TEXT_LENGTH - 1,
    VOCAB_SIZE
)

print("✓ REAL DATA FORWARD PASS SUCCESSFUL")


# ==========================================================================================
# 16. LOSS + OPTIMIZER
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 14 — OPTIMIZER / LOSS / SCHEDULER")
print("=" * 100)

criterion_B = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

trainable_parameters = [
    p
    for p in model_B.parameters()
    if p.requires_grad
]

optimizer_B = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler_B = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_B,
    mode="min",
    factor=0.5,
    patience=2
)

print("Loss       : CrossEntropyLoss(ignore_index=PAD)")
print("Optimizer  : AdamW")
print(f"LR         : {LEARNING_RATE}")
print(f"Weight Dec : {WEIGHT_DECAY}")
print("Scheduler  : ReduceLROnPlateau")
print("Grad Clip  :", GRAD_CLIP)


# ==========================================================================================
# 17. LOW-STORAGE CHECKPOINT FUNCTIONS
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 15 — CHECKPOINT SYSTEM")
print("=" * 100)


def get_trainable_state_dict():

    return {
        name: param.detach().cpu()
        for name, param in model_B.named_parameters()
        if param.requires_grad
    }


def save_checkpoint_B(
    epoch,
    train_loss,
    val_loss,
    best_val_loss,
    history,
    is_best=False
):

    # Only trainable parameters are stored.
    # Swin-B is frozen and will be loaded again from torchvision.

    checkpoint = {

        "experiment":
            "PFMS-SERIES-GT38-2026-EXPERIMENT-B",

        "architecture":
            "Mamba-Swin-B-Transformer-OCR",

        "epoch":
            epoch,

        "trainable_state_dict":
            get_trainable_state_dict(),

        "optimizer_state_dict":
            optimizer_B.state_dict(),

        "scheduler_state_dict":
            scheduler_B.state_dict(),

        "train_loss":
            train_loss,

        "val_loss":
            val_loss,

        "best_val_loss":
            best_val_loss,

        "vocab_size":
            VOCAB_SIZE,

        "char2idx":
            char2idx,

        "idx2char":
            idx2char,

        "PAD_IDX":
            PAD_IDX,

        "BOS_IDX":
            BOS_IDX,

        "EOS_IDX":
            EOS_IDX,

        "UNK_IDX":
            UNK_IDX,

        "history":
            history
    }

    # ----------------------------------------------------------------------
    # LAST
    # ----------------------------------------------------------------------

    torch.save(
        checkpoint,
        LAST_PATH
    )

    # ----------------------------------------------------------------------
    # BEST
    # ----------------------------------------------------------------------

    if is_best:

        torch.save(
            checkpoint,
            BEST_PATH
        )

    del checkpoint

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


def save_history_B(history):

    pd.DataFrame(history).to_csv(
        HISTORY_PATH,
        index=False
    )


print("✓ Low-storage checkpoint system ready")
print("✓ LAST checkpoint will be overwritten each epoch")
print("✓ BEST checkpoint will be overwritten only when validation improves")


# ==========================================================================================
# 18. AUTO RESUME
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 16 — AUTO RESUME CHECK")
print("=" * 100)

start_epoch = 1
best_val_loss = float("inf")
history = []

if os.path.exists(LAST_PATH):

    print("✓ Previous Experiment-B checkpoint found")
    print(f"  {LAST_PATH}")

    checkpoint = torch.load(
        LAST_PATH,
        map_location="cpu",
        weights_only=False
    )

    saved_state = checkpoint[
        "trainable_state_dict"
    ]

    current_state = model_B.state_dict()

    loaded_count = 0

    for name, value in saved_state.items():

        if name in current_state:

            current_state[name] = value

            loaded_count += 1

    model_B.load_state_dict(
        current_state
    )

    model_B = model_B.to(DEVICE)

    optimizer_B.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    if "scheduler_state_dict" in checkpoint:

        scheduler_B.load_state_dict(
            checkpoint["scheduler_state_dict"]
        )

    start_epoch = (
        int(checkpoint["epoch"]) + 1
    )

    best_val_loss = checkpoint.get(
        "best_val_loss",
        float("inf")
    )

    history = checkpoint.get(
        "history",
        []
    )

    print(
        f"✓ Loaded trainable tensors: "
        f"{loaded_count}"
    )

    print(
        f"✓ Last completed epoch: "
        f"{checkpoint['epoch']}"
    )

    print(
        f"✓ Next epoch: "
        f"{start_epoch}"
    )

    print(
        f"✓ Best Val Loss: "
        f"{best_val_loss:.6f}"
    )

    del checkpoint

    gc.collect()

else:

    print("✓ No previous Experiment-B checkpoint")
    print("✓ Training starts from Epoch 1")
    print("✓ Experiment A checkpoint is NOT loaded")


# ==========================================================================================
# 19. TRAIN FUNCTION
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 17 — TRAIN FUNCTION")
print("=" * 100)


def train_one_epoch_B():

    model_B.train()

    # Keep Swin frozen
    for p in model_B.swin.parameters():
        p.requires_grad = False

    running_loss = 0.0

    progress = tqdm(
        train_loader_B,
        total=len(train_loader_B),
        desc="Training",
        leave=True
    )

    for batch in progress:

        images = batch["image"].to(
            DEVICE,
            non_blocking=True
        )

        targets = batch["target"].to(
            DEVICE,
            non_blocking=True
        )

        decoder_input = targets[:, :-1]

        decoder_target = targets[:, 1:]

        optimizer_B.zero_grad(
            set_to_none=True
        )

        logits = model_B(
            images,
            decoder_input
        )

        loss = criterion_B(
            logits.reshape(
                -1,
                logits.size(-1)
            ),
            decoder_target.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            GRAD_CLIP
        )

        optimizer_B.step()

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return (
        running_loss /
        len(train_loader_B)
    )


# ==========================================================================================
# 20. VALIDATION FUNCTION
# ==========================================================================================

print("✓ Train function ready")


@torch.no_grad()
def validate_B():

    model_B.eval()

    running_loss = 0.0

    progress = tqdm(
        val_loader_B,
        total=len(val_loader_B),
        desc="Validation",
        leave=True
    )

    for batch in progress:

        images = batch["image"].to(
            DEVICE,
            non_blocking=True
        )

        targets = batch["target"].to(
            DEVICE,
            non_blocking=True
        )

        decoder_input = targets[:, :-1]

        decoder_target = targets[:, 1:]

        logits = model_B(
            images,
            decoder_input
        )

        loss = criterion_B(
            logits.reshape(
                -1,
                logits.size(-1)
            ),
            decoder_target.reshape(-1)
        )

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return (
        running_loss /
        len(val_loader_B)
    )


print("✓ Validation function ready")


# ==========================================================================================
# 21. REAL TRAINING
# ==========================================================================================

print("\n" + "=" * 100)
print("STEP 18 — STARTING REAL TRAINING")
print("=" * 100)

print("Architecture:")
print("Mamba → Swin-B → Transformer OCR Decoder")

print(f"\nEpochs       : {EPOCHS}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Train images : {len(train_dataset_B):,}")
print(f"Val images   : {len(val_dataset_B):,}")
print(f"Test images  : {len(test_dataset_B):,}")

print(f"\nStarting epoch: {start_epoch}")
print(f"Best Val Loss : {best_val_loss:.6f}")

print("\nCheckpoint protection:")
print("✓ LAST saved after every epoch")
print("✓ BEST saved whenever validation improves")
print("✓ Training history saved after every epoch")
print("✓ Checkpoints stored on Google Drive")
print("✓ Experiment A checkpoint untouched")


# ------------------------------------------------------------------------------------------
# Training loop
# ------------------------------------------------------------------------------------------

for epoch in range(
    start_epoch,
    EPOCHS + 1
):

    epoch_start = time.time()

    print("\n")
    print("=" * 100)
    print(
        f"EXPERIMENT B — "
        f"EPOCH {epoch}/{EPOCHS}"
    )
    print("=" * 100)

    current_lr = (
        optimizer_B.param_groups[0]["lr"]
    )

    print(
        f"Learning Rate: "
        f"{current_lr:.8f}"
    )

    # ----------------------------------------------------------------------
    # TRAIN
    # ----------------------------------------------------------------------

    train_loss = train_one_epoch_B()

    # ----------------------------------------------------------------------
    # VALIDATION
    # ----------------------------------------------------------------------

    val_loss = validate_B()

    # ----------------------------------------------------------------------
    # SCHEDULER
    # ----------------------------------------------------------------------

    scheduler_B.step(val_loss)

    # ----------------------------------------------------------------------
    # BEST MODEL
    # ----------------------------------------------------------------------

    is_best = (
        val_loss < best_val_loss
    )

    if is_best:

        best_val_loss = val_loss

    # ----------------------------------------------------------------------
    # TIME
    # ----------------------------------------------------------------------

    epoch_time = (
        time.time() -
        epoch_start
    )

    # ----------------------------------------------------------------------
    # HISTORY
    # ----------------------------------------------------------------------

    history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "val_loss":
            val_loss,

        "best_val_loss":
            best_val_loss,

        "learning_rate":
            current_lr,

        "epoch_time_sec":
            epoch_time
    })

    # ----------------------------------------------------------------------
    # SAVE TO GOOGLE DRIVE
    # ----------------------------------------------------------------------

    save_checkpoint_B(
        epoch=epoch,
        train_loss=train_loss,
        val_loss=val_loss,
        best_val_loss=best_val_loss,
        history=history,
        is_best=is_best
    )

    save_history_B(
        history
    )

    # ----------------------------------------------------------------------
    # REPORT
    # ----------------------------------------------------------------------

    print("\n" + "-" * 100)

    print(
        f"Epoch {epoch}/{EPOCHS} COMPLETED"
    )

    print(
        f"Train Loss     : "
        f"{train_loss:.6f}"
    )

    print(
        f"Val Loss       : "
        f"{val_loss:.6f}"
    )

    print(
        f"Best Val Loss  : "
        f"{best_val_loss:.6f}"
    )

    print(
        f"Epoch Time     : "
        f"{epoch_time/60:.2f} min"
    )

    if is_best:

        print(
            "✓ NEW BEST MODEL SAVED"
        )

    else:

        print(
            "✓ LAST CHECKPOINT SAVED"
        )

    print(
        f"✓ LAST: {LAST_PATH}"
    )

    if is_best:

        print(
            f"✓ BEST: {BEST_PATH}"
        )

    if torch.cuda.is_available():

        allocated = (
            torch.cuda.memory_allocated()
            / (1024 ** 3)
        )

        reserved = (
            torch.cuda.memory_reserved()
            / (1024 ** 3)
        )

        print(
            f"GPU Memory: "
            f"{allocated:.2f} GB allocated / "
            f"{reserved:.2f} GB reserved"
        )

    # ----------------------------------------------------------------------
    # MEMORY CLEANUP
    # ----------------------------------------------------------------------

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ==========================================================================================
# 22. FINAL REPORT
# ==========================================================================================

print("\n")
print("=" * 100)
print("STAGE B-3 — TRAINING FINISHED")
print("=" * 100)

print(
    f"Best Validation Loss: "
    f"{best_val_loss:.6f}"
)

print("\nFiles on Google Drive:")

print(
    f"✓ BEST : "
    f"{BEST_PATH}"
)

print(
    f"✓ LAST : "
    f"{LAST_PATH}"
)

print(
    f"✓ HIST : "
    f"{HISTORY_PATH}"
)

print("\nExperiment A was NOT modified.")
print("=" * 100)

PFMS-SERIES-GT38-2026
EXPERIMENT B — STAGE B-3
Mamba → Swin-B → Transformer OCR Decoder
FULL SELF-CONTAINED TRAINING
Device : cuda
GPU    : Tesla T4
CUDA   : 12.8
PyTorch: 2.11.0+cu128

STEP 1 — LOADING REAL IDPL-PFOD GT


AssertionError: GT CSV not found:
/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv